# RSL – Letter II: código de análise

**Does early-season burn configuration constrain subsequent fires in Cerrado Indigenous Lands?**
Dias, Alcântara & Rodrigues — *Remote Sensing Letters*

O código roda no Google Colab, com Google Earth Engine e Google Drive. Cada bloco é uma célula do Colab. Rode os blocos **na ordem numérica e na mesma sessão**. Todos gravam em `MyDrive/RSL_LETTER_II/` (as exportações do Earth Engine passam por `MyDrive/RSL_LETTER_II_exports/`); se a sessão cair, rode de novo o bloco 00 e continue de onde parou, porque o que já existe não é refeito.

O arquivo `RSL_LetterII_pipeline.ipynb` traz os 18 blocos na mesma ordem, um por célula.

## Sequência

| Bloco | Arquivo | O que faz | Saídas principais |
|---|---|---|---|
| 00 | `00_configuracao.py` | Drive, Earth Engine, as 14 TIs com BRIF (código FUNAI, recorte no Cerrado, EPSG:5880), período pós-BRIF e funções comuns | `01_boundaries/tis_brif_cerrado.gpkg` |
| 01 | `01_cicatrizes_mapbiomas.py` | MapBiomas Fogo Col. 4 mensal → raster do mês da queima por TI e polígonos de cicatriz | `02_cache/cicatrizes_*.gpkg` |
| 02 | `02_elegibilidade_amostra.py` | Triagem das 14 TIs: imagens Landsat 8/9 e encontros analisáveis → amostra final com o motivo de cada exclusão | `03_tables/amostra_final_tis.csv` |
| 03 | `03_rasters_30m.py` | Pilhas de 30 m por TI-ano: mês da queima, cobertura, NBR mensal mínimo do Landsat, relevo e barreiras | `07_exports/rasters/*.tif` |
| 04 | `04_era5_diario.py` | ERA5-Land diário: VPD, umidade, vento e chuva | `02_cache/era5_diario_*.parquet` |
| 05 | `05_focos_modis.py` | Focos MODIS C6.1 (MOD14A1 + MYD14A1, 1 km, FireMask ≥ 8) por pixel e dia | `02_cache/modis_focos_*.parquet` |
| 06 | `06_estacao_seca.py` | Recorte do artigo 1 (mai–jul × ago–out) e seca local por TI (CHIRPS, sensibilidade) | `03_tables/estacoes_por_ti.json` |
| 07 | `07_tipologia_cicatrizes.py` | Tipologia descritiva das cicatrizes iniciais, tempo no dia do foco e focos noturnos | `04_stats/B06_*, D1_*, D2_*` |
| 08 | `08_configuracao_celulas.py` | Quantidade × configuração (m_eff, LPI, borda, nº de manchas) em células de 2,5, 5 e 10 km | `02_cache/celulas_*.parquet` |
| 09 | `09_configuracao_modelo_nulo.py` | **Modelo nulo:** cicatrizes reposicionadas ao acaso (mesma forma e área) → SES, configuração independente da quantidade | `02_cache/celulas_*_nulo.parquet` |
| 10 | `10_encontros_bordas.py` | Sequência das queimadas e encontros por segmento de borda (chegada, interrupção, continuidade), pseudo-cicatrizes de referência e regra simétrica × antiga | `02_cache/segmentos_*_md.parquet` |
| 11 | `11_detectabilidade.py` | Detectabilidade: curva do limiar de NBR, sensibilidade do Landsat e teste MODIS independente do NBR (núcleos de cicatriz × núcleos não queimados) | `04_stats/B08b_*` |
| 12 | `12_modelos.py` | Modelos referência × referência + configuração (célula e segmento), validação fora da amostra | `04_stats/B09_*` |
| 13 | `13_inferencia_poucas_TIs.py` | Inferência para 8 TIs: wild cluster bootstrap (Webb) e jackknife por TI; SES e controle de detectabilidade | `04_stats/B09b_*` |
| 14 | `14_robustez.py` | Robustez: recorte sazonal, escala, critérios dos encontros e combustível | `04_stats/B10_*` |
| 15 | `15_sintese_texto.py` | Métodos e Resultados em PT e EN, com os números lidos das tabelas | `08_textos/*.docx` |
| 16 | `16_figuras_letter.py` | Figuras 1–4, S2, S3 e Tabela 1 | `05_figures/letter/` |
| 17 | `17_figuras_sensoriamento.py` | Figuras espaciais R1–R4: área de estudo, Landsat falsa-cor + MODIS, teste MODIS e SES no mapa | `05_figures/letter/` |

Os blocos 14 e 17 usam funções definidas em blocos anteriores (08, 10, 11, 12 e 16). Por isso, rode-os na mesma sessão desses blocos.

## Pastas no Drive (`MyDrive/RSL_LETTER_II/`)

| Subpasta | Conteúdo |
|---|---|
| `00_raw` | `artigo1_tis.csv` (classe e FRRI do artigo 1), preenchido pelo usuário |
| `01_boundaries` | TIs recortadas no Cerrado |
| `02_cache` | Checkpoints (cicatrizes, ERA5, focos MODIS, células, segmentos, modelo nulo) |
| `03_tables` | Amostra final e estações por TI |
| `04_stats` | Tabelas de resultados (CSV) |
| `05_figures/letter` | Figuras da revista (600 dpi, TIFF/PDF/PNG) |
| `06_logs` | Registros das tarefas e da amostra |
| `07_exports` | Rasters de 30 m, mês da queima, ERA5, MODIS, sequências |
| `08_textos` | Métodos/Resultados (PT e EN) e figuras com legendas (.docx) |

Para usar outra pasta, troque `PROJECT_NAME` no bloco 00.

### Rodada do zero

Esta versão processa tudo do zero, sem reaproveitar nada de rodadas anteriores:
- pasta nova no Drive (`RSL_LETTER_II`), com checkpoints próprios;
- nomes de tarefas e arquivos novos (`RUN_TAG = "r2_..."`), que nunca coincidem com os antigos (`v4_...`);
- busca de arquivos exportados restrita à pasta `RSL_LETTER_II_exports`;
- ativo novo das TIs no Earth Engine (`RSL2_tis_poligonais_cerrado`).

Os checkpoints que aparecem durante a rodada são os da própria rodada nova: se a sessão cair, o bloco continua de onde parou.

## Figuras

Todas as figuras saem a **600 dpi** em três formatos: TIFF (compressão LZW), PDF vetorial e PNG. A largura é a da mancha de texto da revista (14 cm), e os mapas têm escala, norte e grade de coordenadas.

## Dados de entrada

- **Ativos do Earth Engine do autor:** `tis_poligonais` (FUNAI) e `biome_border` (limite do Cerrado).
- **Coleções públicas do Earth Engine:**
  - MapBiomas Fogo Col. 4 e Cobertura Col. 10;
  - Landsat 8/9 C2 L2;
  - MODIS MOD14A1 e MYD14A1 C6.1;
  - ERA5-Land diário;
  - CHIRPS;
  - SRTM;
  - HydroSHEDS.
- **Tabela opcional `00_raw/artigo1_tis.csv`** (colunas `terrai_cod`, `classe`, `frri`): classe e FRRI de cada TI, vindos do primeiro artigo.

## Princípios da análise

- Período pós-BRIF (2014–2024); 2013 é o ano de transição e nunca é analisado.
- Ausência de fogo junto a uma cicatriz não conta como contenção: só entram segmentos com chegada do fogo.
- Nenhuma queimada é atribuída às BRIFs nem chamada de prescrita.
- A configuração só é tratada como relevante além da quantidade quando há, ao mesmo tempo:
  - associação no sentido esperado;
  - melhora do ajuste;
  - melhora da previsão fora da amostra.


## 00 configuracao

In [ ]:
# ##########################################################################
# RSL LETTER II - Does early-season burn configuration constrain subsequent
# fires in Cerrado Indigenous Lands?  (Remote Sensing Letters)
# Dias, Alcantara & Rodrigues - codigo de analise (Google Colab + Earth Engine)
#
# BLOCO 00 - Configuracao: Google Drive + Earth Engine + 14 TIs com BRIF + funcoes comuns
# Roda em TODA sessao, antes de qualquer outro bloco.
#
# SEQUENCIA DOS BLOCOS (rode nesta ordem; cada um grava checkpoints e pode ser
# rodado de novo sem refazer o que ja existe):
#   00 configuracao                      09 configuracao pelo modelo nulo (SES)
#   01 cicatrizes MapBiomas Fogo         10 encontros nas bordas (segmentos)
#   02 elegibilidade -> amostra final    11 detectabilidade (NBR x MODIS)
#   03 rasters de 30 m por TI            12 modelos referencia x configuracao
#   04 ERA5-Land diario                  13 inferencia para poucas TIs
#   05 focos MODIS (MOD14A1 + MYD14A1)   14 robustez
#   06 estacao seca (artigo 1 + local)   15 sintese em texto (PT e EN)
#   07 tipologia das cicatrizes iniciais 16 figuras e tabela da letter
#   08 quantidade x configuracao (celula)17 figuras de sensoriamento remoto
#
# Periodo: so pos-BRIF (2014-2024); 2013 = transicao (nunca analisado);
# 2012-2013 so como historico de fogo (defasagens t-1 e t-2).
# Resultado = evidencia observacional: nada e atribuido as BRIFs e nenhuma
# queimada e chamada de prescrita sem documentacao independente.
# ##########################################################################

!pip -q install geemap geopandas pyogrio shapely tqdm rich pyarrow rasterio statsmodels scipy matplotlib pyfixest scikit-learn joblib wildboottest python-docx

import io
import os
import json
import time
import glob
import warnings
from pathlib import Path
from datetime import datetime

import ee
import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from google.colab import drive, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

warnings.filterwarnings("ignore")
import logging
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
console = Console()
BLOCKS_VERSION = "2026-10-17-final"   # todos os blocos conferem esta versao

# ============================================================
# 1. USER CONFIGURATION
# ============================================================

PROJECT_NAME = "RSL_LETTER_II"          # pasta no Drive: MyDrive/RSL_LETTER_II (exportacoes do EE: MyDrive/RSL_LETTER_II_exports)
PROJECT_ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")

FORCE_REPROCESS = False

# --- Periodo -------------------------------------------------------------
START_YEAR      = 2002
PRE_PERIOD      = (2002, 2012)   # pre-BRIF (so referencia ao artigo 1)
TRANSITION_YEAR = 2013           # ACT Funai-Ibama assinado em 2013; brigadas comecam em 2014
POST_PERIOD     = (2014, 2025)   # pos-BRIF (definicao da dissertacao)
END_YEAR        = 2024           # a letter analisa 2014-2024
YEARS = list(range(POST_PERIOD[0], END_YEAR + 1))
N_LAGS = 2                       # cicatrizes de 2012-2013 so como defasagem de 2014-2015

# --- Estacao seca: recorte do ARTIGO 1 (principal) ------------------------
#   inicio da seca  = mai-jul -> cicatrizes anteriores
#   restante da seca = ago-out -> queimadas posteriores
# Sensibilidade (bloco 06): seca local por TI (CHIRPS), mesma divisao.
DRY_MONTHS = (5, 6, 7, 8, 9, 10)
EARLY      = (5, 6, 7)
LATE       = (8, 9, 10)
SEASON_MAIN = "artigo1"
ELNINO     = [2015, 2023]

# --- Escalas espaciais ----------------------------------------------------
PIX_M      = 30
CELL_SIZES = [2490, 4980, 9990]    # ~2,5 / 5 / 10 km (sensibilidade); principal = 4980

# --- Earth Engine ---------------------------------------------------------
EE_PROJECT       = "smart-firmament-481617-n0"
EE_ASSET_PROJECT = "smart-firmament-481617-n0"
EE_ASSET_TIS     = f"projects/{EE_ASSET_PROJECT}/assets/tis_poligonais"
EE_ASSET_BIOME   = f"projects/{EE_ASSET_PROJECT}/assets/biome_border"
CLIPPED_ASSET_ID = f"projects/{EE_ASSET_PROJECT}/assets/RSL2_tis_poligonais_cerrado"   # ativo novo (rodada do zero)

# --- As 14 TIs com BRIF documentada no artigo submetido -----------------
# Tabela 1 de Dias et al. (2026). Selecao SEMPRE pelo codigo FUNAI.
COHORT_14 = {
    4602:  ("Ava-Canoeiro",        31382.34),
    2401:  ("Apinaye",             142342.10),
    5201:  ("Bakairi",             62123.55),
    7702:  ("Cachoeirinha",        24428.54),   # 7701 = poligono duplicado
    13601: ("Governador",          34003.31),
    20001: ("Kadiweu",             172119.93),
    20701: ("Kanela",              125575.79),
    23301: ("Kraolandia",          306030.23),
    25701: ("Limao Verde",         5209.87),
    33401: ("Paresi",              562192.89),
    33501: ("Parque do Araguaia",  1359314.28),
    36601: ("Porquinhos",          79406.80),
    50301: ("Xerente",             171257.31),
    49901: ("Xacriaba",            23223.94),
}
AREA_TOL_PCT    = 5.0
CERRADO_MIN_PCT = 10.0
EXPECTED_N_TIS  = 14
POSSIBLE_NAME_FIELDS = ["terrai_nom", "terrai_nome", "nome", "Nome", "NOME", "nome_ti", "NOME_TI",
                        "name", "Name", "NAME"]
POSSIBLE_CODE_FIELDS = ["terrai_cod", "cod_funai", "codigo_funai", "COD_FUNAI", "terrai_codi", "codigo"]

EE_EXPORT_FOLDER = f"{PROJECT_NAME}_exports"
MONITOR_INTERVAL = 60
MONITOR_TIMEOUT  = 18000

AREA_CRS = "EPSG:5880"     # SIRGAS 2000 / Brazil Polyconic
CELL_M   = 4980

FIG_DPI = 600              # todas as figuras (Remote Sensing Letters)
CAT_ORDER = ["Inicial pequena e fraca", "Inicial pequena e intensa",
             "Inicial grande", "Restante da seca"]
CAT_COLORS = {"Inicial pequena e fraca": "#1b7837", "Inicial pequena e intensa": "#7fbf7b",
              "Inicial grande": "#c2a5cf", "Restante da seca": "#b2182b"}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "figure.dpi": 150,
    "savefig.dpi": FIG_DPI,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ============================================================
# 2. SAFE GOOGLE DRIVE MOUNT
# ============================================================

if Path("/content/drive/MyDrive").exists():
    console.print("[green]OK Google Drive already mounted.[/green]")
else:
    drive.mount("/content/drive")
    console.print("[green]OK Google Drive mounted.[/green]")

# ============================================================
# 3. FOLDER STRUCTURE
# ============================================================

DIRS = {
    "raw":        PROJECT_ROOT / "00_raw",
    "boundaries": PROJECT_ROOT / "01_boundaries",
    "cache":      PROJECT_ROOT / "02_cache",
    "tables":     PROJECT_ROOT / "03_tables",
    "stats":      PROJECT_ROOT / "04_stats",
    "figures":    PROJECT_ROOT / "05_figures",
    "logs":       PROJECT_ROOT / "06_logs",
    "exports":    PROJECT_ROOT / "07_exports",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
console.print(f"[green]OK Project folders ready:[/green] {PROJECT_ROOT}")

# ============================================================
# 4. EARTH ENGINE INITIALIZATION
# ============================================================

EE_FORCE_AUTH = False

def _ee_init() -> str:
    try:
        ee.Initialize(project=EE_PROJECT)
        return "ok"
    except Exception as e:
        return str(e)

if EE_FORCE_AUTH:
    ee.Authenticate(force=True)
msg = _ee_init()
if msg != "ok":
    if "required permission" in msg or "USER_PROJECT_DENIED" in msg:
        console.print(f"[yellow]A conta logada nao tem permissao no projeto {EE_PROJECT}. "
                      "Refazendo o login: escolha a conta DONA do projeto.[/yellow]")
    ee.Authenticate(force=True)
    msg = _ee_init()
if msg != "ok":
    raise RuntimeError(
        f"Earth Engine nao inicializou no projeto {EE_PROJECT}:\n{msg}\n\n"
        "Saidas possiveis:\n"
        "  1) Rode de novo e, no login, escolha a conta dona do projeto.\n"
        "  2) Ou, logado como dono, de a sua outra conta o papel 'Service Usage Consumer' em\n"
        f"     https://console.cloud.google.com/iam-admin/iam?project={EE_PROJECT}\n"
        "  3) Ou troque EE_PROJECT/EE_ASSET_PROJECT por um projeto em que esta conta tenha acesso.")
console.print(f"[green]OK Earth Engine initialized: {EE_PROJECT}[/green]")

PROJ = ee.Projection(AREA_CRS)

# ============================================================
# 4b. ANOS DISPONIVEIS NOS DADOS
# ============================================================

FIRE_ASSET = ("projects/mapbiomas-public/assets/brazil/fire/collection4/"
              "mapbiomas_fire_collection4_monthly_burned_v1")
FIRE_BAND  = lambda y: f"burned_monthly_{y}"
LULC_ASSET         = "projects/mapbiomas-public/assets/brazil/lulc/v1"
LULC_COLLECTION_ID = 10.0
LULC_VERSION       = "v1"

fire_last = max(int(b.rsplit("_", 1)[1]) for b in ee.Image(FIRE_ASSET).bandNames().getInfo()
                if b.startswith("burned_monthly_"))
lulc_last = int(ee.ImageCollection(LULC_ASSET)
                .filter(ee.Filter.eq("collection_id", LULC_COLLECTION_ID))
                .filter(ee.Filter.eq("version", LULC_VERSION))
                .aggregate_max("year").getInfo())
last_ok = min(END_YEAR, fire_last, lulc_last)
if last_ok < END_YEAR:
    console.print(f"[yellow]WARNING MapBiomas Fogo vai ate {fire_last} e a Cobertura ate {lulc_last}. "
                  f"Analise rodando {POST_PERIOD[0]}-{last_ok}.[/yellow]")
YEARS = [y for y in YEARS if y <= last_ok]
SCAR_YEARS = list(range(YEARS[0] - N_LAGS, YEARS[-1] + 1))
assert TRANSITION_YEAR not in YEARS and all(POST_PERIOD[0] <= y <= POST_PERIOD[1] for y in YEARS)
console.print(f"[cyan]Anos analisados (pos-BRIF):[/cyan] {YEARS[0]}-{YEARS[-1]} | "
              f"[cyan]so defasagem do fogo:[/cyan] {SCAR_YEARS[0]}-{YEARS[0] - 1} | "
              f"[cyan]transicao (nunca analisada):[/cyan] {TRANSITION_YEAR}")

# ============================================================
# 5. CACHE / TABLE UTILITIES
# ============================================================

def cache_path(name, ext="parquet"):
    return DIRS["cache"] / f"{name}.{ext}"

def table_path(name, ext="csv"):
    return DIRS["tables"] / f"{name}.{ext}"

def stats_path(name, ext="csv"):
    return DIRS["stats"] / f"{name}.{ext}"

def fig_path(name, ext="png"):
    return DIRS["figures"] / f"{name}.{ext}"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def read_table(path):
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix in (".gpkg", ".geojson"):
        return gpd.read_file(path)
    raise ValueError(f"Unsupported format: {path}")

def write_table(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".parquet":
        df.to_parquet(path, index=False)
    elif path.suffix == ".csv":
        df.to_csv(path, index=False)
    elif path.suffix == ".gpkg":
        df.to_file(path, driver="GPKG")
    else:
        raise ValueError(f"Unsupported format: {path}")

def checkpoint_load_or_run(name, func, ext="parquet", force=FORCE_REPROCESS):
    """Carrega o checkpoint se existir; senao roda func() e salva."""
    path = cache_path(name, ext)
    if path.exists() and not force:
        console.print(f"[cyan]OK Loading checkpoint:[/cyan] {path.name}")
        return read_table(path)
    console.print(f"[yellow]-> Processing:[/yellow] {name}")
    df = func()
    write_table(df, path)
    console.print(f"[green]OK Checkpoint saved:[/green] {path}")
    return df

# ============================================================
# 6. EARTH ENGINE EXPORT -> MONITOR -> DRIVE DOWNLOAD
# ============================================================

def get_active_tasks() -> dict:
    try:
        return {t["description"]: t["id"] for t in ee.data.getTaskList()
                if t.get("state") in ("RUNNING", "READY")}
    except Exception:
        return {}

def ee_monitor(ids_by_name: dict) -> dict:
    """Acompanha as tarefas PELO ID. Devolve {nome: estado}."""
    pending = dict(ids_by_name)
    done, t0 = {}, time.time()
    if not pending:
        return done
    console.print(f"[cyan]Monitoring {len(pending)} tasks...[/cyan]")
    while pending:
        try:
            for st in ee.data.getTaskStatus(list(pending.values())):
                name = next((n for n, i in pending.items() if i == st.get("id")), None)
                if name is None:
                    continue
                state = st.get("state", "")
                if state == "COMPLETED":
                    done[name] = state; pending.pop(name)
                    console.print(f"  [green]OK {name}: COMPLETED[/green]")
                elif state in ("FAILED", "CANCELLED"):
                    done[name] = state; pending.pop(name)
                    console.print(f"  [red]X {name}: {state} - {st.get('error_message', '')}[/red]")
        except Exception as e:
            console.print(f"[yellow]WARNING checking status: {e}[/yellow]")
        if not pending:
            break
        if time.time() - t0 > MONITOR_TIMEOUT:
            for n in pending:
                done[n] = "TIMEOUT"
            console.print(f"[red]WARNING timeout. Pending: {sorted(pending)}[/red]")
            break
        console.print(f"  [dim]{len(done)}/{len(done) + len(pending)} | {len(pending)} pending | "
                      f"{MONITOR_INTERVAL}s...[/dim]")
        time.sleep(MONITOR_INTERVAL)
    bad = [n for n, s in done.items() if s != "COMPLETED"]
    if bad:
        save_json({"failed": bad, "at": datetime.now()}, DIRS["logs"] / f"failed_tasks_{int(time.time())}.json")
    return done

def launch_or_attach(names: list, make_task) -> dict:
    active = get_active_tasks()
    ids = {}
    for n in names:
        if n in active:
            ids[n] = active[n]
            console.print(f"[cyan]OK Task already active: {n}[/cyan]")
            continue
        task = make_task(n)
        task.start()
        ids[n] = task.id
        console.print(f"[yellow]-> Task launched: {n}[/yellow]")
        time.sleep(0.5)
    return ids

def _drive_call_with_retry(request_fn, max_retries=6):
    from googleapiclient.errors import HttpError
    for attempt in range(max_retries):
        try:
            return request_fn()
        except HttpError as e:
            if e.resp.status == 403 and "rateLimitExceeded" in str(e) and attempt < max_retries - 1:
                wait = min(2 ** attempt * 2, 60)
                console.print(f"  [yellow]Rate limit hit, waiting {wait}s...[/yellow]")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Drive API retries exhausted.")

def _export_folder_id(svc):
    """id da pasta EE_EXPORT_FOLDER no Drive (None se ainda nao existe). A busca de arquivos
    exportados fica restrita a ela: arquivos de rodadas antigas em outras pastas nunca sao usados."""
    res = _drive_call_with_retry(lambda: svc.files().list(
        q=f"name='{EE_EXPORT_FOLDER}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        spaces="drive", fields="files(id, name)").execute())
    f = res.get("files", [])
    return f[0]["id"] if f else None

def drive_download(filenames: list, dest_dir: Path, quiet: bool = False) -> list:
    auth.authenticate_user()
    svc = build("drive", "v3", cache_discovery=False)
    folder = _export_folder_id(svc)
    got = []
    for fn in filenames:
        dest = Path(dest_dir) / fn
        if dest.exists():
            got.append(dest)
            continue
        if folder is None:                      # pasta de exportacao ainda nao existe: nada a baixar
            continue
        res = _drive_call_with_retry(lambda: svc.files().list(
            q=f"name='{fn}' and '{folder}' in parents and trashed=false", spaces="drive",
            orderBy="createdTime desc", fields="files(id, name)").execute())
        time.sleep(1.5)
        files = res.get("files", [])
        if not files:
            if not quiet:
                console.print(f"  [yellow]WARNING not found on Drive: {fn}[/yellow]")
            continue
        buf = io.BytesIO()
        downloader = MediaIoBaseDownload(buf, svc.files().get_media(fileId=files[0]["id"]))
        finished = False
        while not finished:
            _, finished = _drive_call_with_retry(lambda: downloader.next_chunk())
        dest.write_bytes(buf.getvalue())
        got.append(dest)
        console.print(f"  [green]OK {fn}[/green]")
        time.sleep(1.5)
    return got

def run_ee_exports(specs: dict, ext: str, dest_dir: Path) -> list:
    """specs = {descricao: funcao_sem_argumentos -> ee.batch.Task (nao iniciada)}."""
    dest_dir = Path(dest_dir); dest_dir.mkdir(parents=True, exist_ok=True)
    need = [d for d in specs if not (dest_dir / f"{d}.{ext}").exists()]
    console.print(f"  Ja baixados: {len(specs) - len(need)} | A processar: {len(need)}")
    if need:
        drive_download([f"{d}.{ext}" for d in need], dest_dir, quiet=True)
        need = [d for d in need if not (dest_dir / f"{d}.{ext}").exists()]
    if need:
        ids = launch_or_attach(need, lambda d: specs[d]())
        status = ee_monitor(ids)
        drive_download([f"{d}.{ext}" for d in need if status.get(d) == "COMPLETED"], dest_dir)
    paths = [dest_dir / f"{d}.{ext}" for d in specs if (dest_dir / f"{d}.{ext}").exists()]
    if len(paths) < len(specs):
        console.print(f"[red]WARNING {len(specs) - len(paths)} arquivo(s) faltando - "
                      "rode o bloco de novo mais tarde.[/red]")
    return paths

# ============================================================
# 7. AS 14 TIs COM BRIF - pelo CODIGO FUNAI
#    tis_poligonais -> 14 codigos -> recorte no Cerrado (biome_border) -> >= 10%
#    Poligonos sem simplificar; areas em EPSG:5880, margem de erro de 1 m.
# ============================================================

EXACT = ee.ErrorMargin(1)

def _as_number(v):
    return ee.Number(ee.Algorithms.If(ee.Algorithms.ObjectType(v).equals("String"),
                                      ee.Number.parse(v), v))

def _check_fields():
    props = ee.Feature(ee.FeatureCollection(EE_ASSET_TIS).first()).propertyNames().getInfo()
    code_field = next((c for c in POSSIBLE_CODE_FIELDS if c in props), None)
    name_field = next((c for c in POSSIBLE_NAME_FIELDS if c in props), None)
    if code_field is None:
        raise RuntimeError(f"Nao achei o campo de codigo FUNAI em {EE_ASSET_TIS}. Campos: {props}.")
    console.print(f"[dim]Campos usados: codigo='{code_field}', nome='{name_field}'[/dim]")
    return code_field, name_field

def build_tis_server(code_field, name_field, codes):
    raw_fc = ee.FeatureCollection(EE_ASSET_TIS).filter(ee.Filter.Or(
        ee.Filter.inList(code_field, codes),
        ee.Filter.inList(code_field, [str(c) for c in codes])))
    biome_geom = ee.FeatureCollection(EE_ASSET_BIOME).geometry(EXACT)

    def _clip(f):
        f = ee.Feature(f)
        geom = f.geometry()
        inter = geom.intersection(biome_geom, EXACT, PROJ)
        return ee.Feature(inter, {
            "terrai_cod":      _as_number(f.get(code_field)),
            "nome_ativo":      f.get(name_field) if name_field else "",
            "area_total_ha":   geom.area(EXACT, PROJ).divide(1e4),
            "area_cerrado_ha": inter.area(EXACT, PROJ).divide(1e4),
        })
    return raw_fc.map(_clip)

def prepare_tis_asset() -> ee.FeatureCollection:
    code_field, name_field = _check_fields()
    codes = list(COHORT_14)
    clipped = build_tis_server(code_field, name_field, codes)

    info = clipped.map(lambda f: ee.Feature(None, ee.Feature(f).toDictionary())).getInfo()
    df = pd.DataFrame([x["properties"] for x in info["features"]])
    if df.empty:
        raise RuntimeError(f"Nenhum dos 14 codigos encontrado em {EE_ASSET_TIS}.")
    df["terrai_cod"] = df["terrai_cod"].astype(int)
    df = df.groupby("terrai_cod", as_index=False).agg(
        nome_ativo=("nome_ativo", "first"), n_poligonos=("nome_ativo", "size"),
        area_total_ha=("area_total_ha", "sum"), area_cerrado_ha=("area_cerrado_ha", "sum"))

    faltam = [f"{c} ({COHORT_14[c][0]})" for c in codes if c not in set(df["terrai_cod"])]
    if faltam:
        raise RuntimeError(f"Codigo(s) FUNAI nao encontrado(s) em {EE_ASSET_TIS}: {', '.join(faltam)}.")
    df["ti_name"] = df["terrai_cod"].map(lambda c: COHORT_14[c][0])
    df["area_artigo_ha"] = df["terrai_cod"].map(lambda c: COHORT_14[c][1])
    df["pct_cerrado"] = 100 * df["area_cerrado_ha"] / df["area_total_ha"]
    df["dif_area_vs_artigo_%"] = 100 * (df["area_cerrado_ha"] / df["area_artigo_ha"] - 1)
    df.to_csv(DIRS["stats"] / "tis_conferencia_codigo_nome_area.csv", index=False)
    console.print("[bold]Conferencia das 14 TIs (codigo FUNAI -> nome no ativo, area no Cerrado x Tabela 1)[/bold]")
    display(df[["terrai_cod", "ti_name", "nome_ativo", "n_poligonos", "area_cerrado_ha",
                "area_artigo_ha", "dif_area_vs_artigo_%", "pct_cerrado"]].round(2))
    off = df[df["dif_area_vs_artigo_%"].abs() > AREA_TOL_PCT]
    if len(off):
        console.print(f"[yellow]WARNING area difere > {AREA_TOL_PCT}% da Tabela 1 em: "
                      f"{', '.join(off['ti_name'])}.[/yellow]")

    kept = df[df["pct_cerrado"] >= CERRADO_MIN_PCT].sort_values("terrai_cod").reset_index(drop=True)
    kept["ti_id"] = np.arange(1, len(kept) + 1)
    if len(kept) != EXPECTED_N_TIS:
        console.print(f"[red]WARNING ficaram {len(kept)} TIs (esperado {EXPECTED_N_TIS}).[/red]")
    kept.to_csv(DIRS["stats"] / "tis_cerrado_overlap_pct.csv", index=False)

    try:
        cur = ee.FeatureCollection(CLIPPED_ASSET_ID).aggregate_array("terrai_cod").getInfo()
        if sorted(int(c) for c in cur) == sorted(kept["terrai_cod"].tolist()):
            console.print(f"[green]OK {CLIPPED_ASSET_ID} ja existe com as {len(kept)} TIs - reaproveitando.[/green]")
            return ee.FeatureCollection(CLIPPED_ASSET_ID)
    except Exception:
        pass

    id_map = ee.Dictionary({str(c): int(i) for c, i in zip(kept["terrai_cod"], kept["ti_id"])})
    nm_map = ee.Dictionary({str(c): n for c, n in zip(kept["terrai_cod"], kept["ti_name"])})
    keep_codes = kept["terrai_cod"].tolist()

    def _props(f):
        f = ee.Feature(f)
        key = ee.Number(f.get("terrai_cod")).format("%d")
        return f.set({"ti_id": id_map.get(key), "ti_name": nm_map.get(key),
                      "area_ha": f.get("area_cerrado_ha"),
                      "area_km2": ee.Number(f.get("area_cerrado_ha")).divide(100)})
    out = clipped.filter(ee.Filter.inList("terrai_cod", keep_codes)).map(_props)

    try:
        ee.data.deleteAsset(CLIPPED_ASSET_ID)
    except Exception:
        pass
    task = ee.batch.Export.table.toAsset(collection=out, description="RSL2_tis_poligonais_cerrado",
                                         assetId=CLIPPED_ASSET_ID)
    task.start()
    console.print(f"[yellow]-> Criando o ativo {CLIPPED_ASSET_ID}...[/yellow]")
    while True:
        st = task.status()
        if st["state"] in ("COMPLETED", "FAILED", "CANCELLED"):
            break
        time.sleep(20)
    if st["state"] != "COMPLETED":
        raise RuntimeError(f"Falha ao criar {CLIPPED_ASSET_ID}: {st}")
    return ee.FeatureCollection(CLIPPED_ASSET_ID)

tis_fc = prepare_tis_asset()
tis = geemap.ee_to_gdf(tis_fc)
tis = (tis.set_crs("EPSG:4326") if tis.crs is None else tis).to_crs(AREA_CRS)
tis["ti_id"] = tis["ti_id"].astype(int)
tis["terrai_cod"] = pd.to_numeric(tis["terrai_cod"]).astype(int)
tis = (tis.dissolve(by="ti_id", as_index=False, aggfunc={"terrai_cod": "first", "ti_name": "first",
                                                         "area_ha": "sum", "area_km2": "sum"})
          .sort_values("ti_id").reset_index(drop=True))
tis.to_file(DIRS["boundaries"] / "tis_brif_cerrado.gpkg", driver="GPKG")

ti_img = ee.Image(0).int().paint(tis_fc, "ti_id")

# Etiquetas dos checkpoints (nunca reaproveitar arquivos de outro conjunto de TIs/anos)
# "r2" = rodada do zero na pasta RSL_LETTER_II: nomes de tarefas e arquivos NOVOS, que nunca
# coincidem com os de rodadas antigas (v4_...) -> nada antigo e reaproveitado
RUN_TAG    = f"r2_{len(tis)}tp"
TAG_YEARS  = f"{YEARS[0]}_{YEARS[-1]}_{RUN_TAG}"
SCARS_NAME = f"cicatrizes_{SCAR_YEARS[0]}_{YEARS[-1]}_{RUN_TAG}"
FOCOS_NAME = f"modis_focos_v1_{TAG_YEARS}"     # focos MODIS (bloco 05)
TAG_SEG    = f"{TAG_YEARS}_md"                  # tudo o que usa focos MODIS (blocos 07, 10-17)

# ============================================================
# 8. TABELA DO PRIMEIRO ARTIGO: classe e FRRI de cada TI
#    Preencha 00_raw/artigo1_tis.csv (terrai_cod, classe, frri).
# ============================================================

def load_artigo1() -> pd.DataFrame:
    manual = DIRS["raw"] / "artigo1_tis.csv"
    base = tis[["ti_id", "terrai_cod", "ti_name"]].rename(columns={"ti_name": "nome"})
    if not manual.exists():
        modelo = DIRS["raw"] / "artigo1_tis_MODELO.csv"
        base.assign(classe="", frri="").to_csv(modelo, index=False)
        console.print(f"[yellow]WARNING sem {manual.name}: comparacoes com o FRRI ficam em branco. "
                      f"Preencha o modelo {modelo} e salve como {manual.name}.[/yellow]")
        return base.assign(classe=np.nan, frri=np.nan)
    a = pd.read_csv(manual)
    key = "terrai_cod" if "terrai_cod" in a.columns else "ti_id"
    a = base.merge(a[[key, "classe", "frri"]], on=key, how="left")
    a["frri"] = pd.to_numeric(a["frri"], errors="coerce")
    return a[["ti_id", "terrai_cod", "nome", "classe", "frri"]]

artigo1 = load_artigo1()
artigo1.to_csv(table_path("artigo1_tis"), index=False)

# ============================================================
# 8b. FUNCOES COMUNS DOS BLOCOS 02-17
# ============================================================

def load_sample_ids() -> list:
    """ti_id das TIs elegiveis (bloco 02). Sem a triagem, PARA (nao roda as 14 por engano)."""
    p = table_path("amostra_final_tis")
    if not p.exists():
        raise RuntimeError(f"Amostra final nao encontrada ({p.name}) - rode o bloco 02.")
    a = pd.read_csv(p)
    ids = a.loc[a["elegivel"].astype(bool), "ti_id"].astype(int).tolist()
    console.print(f"[dim]Amostra final: {len(ids)} TIs[/dim]")
    return ids

def load_seasons() -> dict:
    """{esquema: {ti_id: (meses_iniciais, meses_restantes)}}. 'artigo1' sempre; 'local' do bloco 06."""
    s = {"artigo1": {int(t): (tuple(EARLY), tuple(LATE)) for t in tis["ti_id"]}}
    p = table_path("estacoes_por_ti", "json")
    if p.exists():
        with open(p, encoding="utf-8") as f:
            raw = json.load(f)
        for sch, d in raw.items():
            s[sch] = {int(k): (tuple(v["inicio"]), tuple(v["restante"])) for k, v in d.items()}
    return s

# --- Rasters de 30 m por TI (bloco 03) ------------------------------------
RASTER_DIR   = DIRS["exports"] / "rasters"
LS_MONTHS    = list(range(4, 12))     # abr-nov
NODATA       = -32768
YEAR_BANDS   = (["month", "cover"] + [f"nbr{m:02d}" for m in LS_MONTHS]
                + [f"cnt{m:02d}" for m in LS_MONTHS])
STATIC_BANDS = ["ti", "elev", "slope10", "barrier", "river"]

# --- Uso do solo -> combustivel (MapBiomas Col. 10)
#   1 natural | 2 pastagem e mosaico | 3 agricultura e silvicultura | 4 NAO combustivel | 0 sem dado
COVER_CLASSES = {
    1: [3, 4, 5, 6, 11, 12, 13, 29, 32, 49, 50],
    2: [15, 21],
    3: [9, 18, 19, 20, 35, 36, 39, 40, 41, 46, 47, 48, 62],
    4: [22, 23, 24, 25, 30, 31, 33],
}
FUEL_MODES = {"todos_vegetados": (1, 2, 3), "nativa+pastagem": (1, 2), "nativa": (1,)}
FUEL_MAIN  = "todos_vegetados"
FUEL_RUNS  = ["todos_vegetados", "nativa+pastagem"]
FUEL_TAG   = {"todos_vegetados": "tv", "nativa+pastagem": "np", "nativa": "n"}

def fuel_from_cover(cover, mode=FUEL_MAIN):
    return np.isin(cover, FUEL_MODES[mode])

def raster_name(ti_id, year=None):
    return (f"l9_{RUN_TAG}_v3_static_ti{int(ti_id)}" if year is None
            else f"l9_{RUN_TAG}_v3_yr_ti{int(ti_id)}_{int(year)}")

RASTER_BUFFER_M = 2000

def read_raster(ti_id, year=None) -> dict:
    """Le um raster do bloco 03 -> {banda: array, '_transform', '_shape', '_crs'}, na grade da TI."""
    import rasterio
    from rasterio.windows import Window
    from rasterio.transform import Affine
    p = RASTER_DIR / f"{raster_name(ti_id, year)}.tif"
    if not p.exists():
        raise FileNotFoundError(f"{p.name} nao existe - rode o bloco 03.")
    names = STATIC_BANDS if year is None else YEAR_BANDS
    minx, miny, maxx, maxy = tis.set_index("ti_id").loc[ti_id].geometry.bounds
    x0 = float(np.floor((minx - RASTER_BUFFER_M) / PIX_M) * PIX_M)
    y0 = float(np.floor((miny - RASTER_BUFFER_M) / PIX_M) * PIX_M)
    x1 = float(np.ceil((maxx + RASTER_BUFFER_M) / PIX_M) * PIX_M)
    y1 = float(np.ceil((maxy + RASTER_BUFFER_M) / PIX_M) * PIX_M)
    W, H = int(round((x1 - x0) / PIX_M)), int(round((y1 - y0) / PIX_M))
    with rasterio.open(p) as src:
        col = int(round((x0 - src.transform.c) / PIX_M))
        row = int(round((src.transform.f - y1) / PIX_M))
        win, crs = Window(col, row, W, H), src.crs
        out = {}
        for i, n in enumerate(names):
            fill = -128 if n.startswith("nbr") else 0
            out[n] = src.read(i + 1, window=win, boundless=True, fill_value=fill)
    out["_transform"], out["_shape"], out["_crs"] = Affine(PIX_M, 0, x0, 0, -PIX_M, y1), (H, W), crs
    return out

def ti_frame(ti_id):
    """Recorte de cada TI: retangulo da TI + 2 km, grade de 30 m alinhada em EPSG:5880."""
    minx, miny, maxx, maxy = tis.set_index("ti_id").loc[ti_id].geometry.bounds
    x0 = float(np.floor((minx - RASTER_BUFFER_M) / PIX_M) * PIX_M)
    y0 = float(np.floor((miny - RASTER_BUFFER_M) / PIX_M) * PIX_M)
    x1 = float(np.ceil((maxx + RASTER_BUFFER_M) / PIX_M) * PIX_M)
    y1 = float(np.ceil((maxy + RASTER_BUFFER_M) / PIX_M) * PIX_M)
    region = ee.Geometry.Rectangle([x0, y0, x1, y1], PROJ, False)
    transform = [float(PIX_M), 0.0, x0, 0.0, -float(PIX_M), y1]
    n_px = int((x1 - x0) / PIX_M * (y1 - y0) / PIX_M)
    return region, transform, n_px

def export_image_task(img, name, region, transform):
    return ee.batch.Export.image.toDrive(
        image=img, description=name, folder=EE_EXPORT_FOLDER, fileNamePrefix=name,
        region=region, crs=AREA_CRS, crsTransform=transform, maxPixels=1e13, fileFormat="GeoTIFF")

def drive_download_tif(name, dest_dir, quiet=False):
    """Baixa um GeoTIFF exportado pelo EE; se veio em pedacos, junta num arquivo so."""
    import rasterio
    from rasterio.merge import merge as rio_merge
    dest_dir = Path(dest_dir); dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{name}.tif"
    if dest.exists():
        return dest
    auth.authenticate_user()
    svc = build("drive", "v3", cache_discovery=False)
    folder = _export_folder_id(svc)
    if folder is None:
        return None
    res = _drive_call_with_retry(lambda: svc.files().list(
        q=f"name contains '{name}' and '{folder}' in parents and trashed=false", spaces="drive",
        orderBy="createdTime desc", fields="files(id, name)").execute())
    files = [f for f in res.get("files", [])
             if f["name"] == f"{name}.tif" or (f["name"].startswith(f"{name}-") and f["name"].endswith(".tif"))]
    # o mesmo arquivo exportado mais de uma vez fica com nomes repetidos no Drive: usa so o
    # mais recente de cada nome (a lista ja vem do mais novo para o mais antigo)
    _vistos = set()
    files = [f for f in files if not (f["name"] in _vistos or _vistos.add(f["name"]))]
    inteiro = [f for f in files if f["name"] == f"{name}.tif"]
    if inteiro:                                   # arquivo inteiro tem prioridade sobre pedacos
        files = inteiro
    if not files:
        if not quiet:
            console.print(f"  [yellow]WARNING nao achei {name} no Drive[/yellow]")
        return None
    parts = []
    for f in files:
        p = dest_dir / f["name"]
        buf = io.BytesIO()
        dl = MediaIoBaseDownload(buf, svc.files().get_media(fileId=f["id"]))
        done = False
        while not done:
            _, done = _drive_call_with_retry(lambda: dl.next_chunk())
        p.write_bytes(buf.getvalue())
        parts.append(p)
        time.sleep(1.0)
    if len(parts) > 1:
        srcs = [rasterio.open(p) for p in parts]
        mosaic, tr = rio_merge(srcs)
        meta = srcs[0].meta.copy()
        meta.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=tr, compress="deflate")
        for s in srcs:
            s.close()
        with rasterio.open(dest, "w", **meta) as dst:
            dst.write(mosaic)
        for p in parts:
            p.unlink(missing_ok=True)
    else:
        parts[0].rename(dest)
    console.print(f"  [green]OK {dest.name}[/green]")
    return dest

def run_image_exports(specs: dict, dest_dir) -> list:
    """specs = {nome: (funcao -> ee.Image, region, transform)}. Devolve os nomes que ainda faltam."""
    dest_dir = Path(dest_dir)
    need = [n for n in specs if not (dest_dir / f"{n}.tif").exists()]
    console.print(f"  Ja baixados: {len(specs) - len(need)} | A processar: {len(need)}")
    for n in need:
        drive_download_tif(n, dest_dir, quiet=True)
    need = [n for n in need if not (dest_dir / f"{n}.tif").exists()]
    if need:
        def _make(n):
            make, region, transform = specs[n]
            return export_image_task(make(), n, region, transform)
        ids = launch_or_attach(need, _make)
        status = ee_monitor(ids)
        for n in need:
            if status.get(n) == "COMPLETED":
                drive_download_tif(n, dest_dir)
    return [n for n in specs if not (dest_dir / f"{n}.tif").exists()]

_STATIC = {}

def static_of(ti_id):
    if ti_id not in _STATIC:
        _STATIC[ti_id] = read_raster(ti_id)
    return _STATIC[ti_id]

RES_METODO = "curva_q_v2"

def residualize_conf(df, conf_vars, q="q_inicio", nbins=50):
    """conf_*_res = configuracao MENOS o que a quantidade explica (curva por faixas de q).
    Usado so como sensibilidade; a medida principal e o SES do modelo nulo (bloco 09)."""
    df = df.copy()
    qv = df[q].to_numpy(float)
    pos = qv > 0
    edges = np.unique(np.quantile(qv[pos], np.linspace(0, 1, nbins + 1))) if pos.sum() > nbins else None
    for v in conf_vars:
        if v not in df.columns:
            continue
        y = df[v].to_numpy(float)
        pred = np.full(len(df), np.nan)
        z = ~pos & ~np.isnan(y)
        if z.any():
            pred[~pos] = np.nanmean(y[z])
        okp = pos & ~np.isnan(y)
        if edges is not None and okp.sum() > nbins:
            b = np.clip(np.searchsorted(edges, qv[okp], side="right") - 1, 0, len(edges) - 2)
            qm = pd.Series(qv[okp]).groupby(b).mean()
            ym = pd.Series(y[okp]).groupby(b).mean()
            pred[pos] = np.interp(qv[pos], qm.to_numpy(), ym.to_numpy())
        df[f"{v}_res"] = y - pred
    df["res_metodo"] = RES_METODO
    return df

def _meff(mask):
    """Manchas (vizinhanca de 8) -> (soma das areas^2, maior area, n de manchas), em pixels.
    m_eff (Jaeger 2000) = soma(a_i^2) / area de referencia."""
    from scipy import ndimage
    lab, n = ndimage.label(mask, structure=np.ones((3, 3), int))
    if n == 0:
        return 0.0, 0.0, 0
    a = np.bincount(lab.ravel())[1:].astype(float)
    return float((a ** 2).sum()), float(a.max()), int(n)

def season_masks(month, ti_id, scheme, seasons):
    """antes = queimado antes do inicio da seca | inicio = cicatrizes iniciais | resto = restante."""
    ini, res = seasons[scheme][int(ti_id)]
    return {"antes": (month > 0) & (month < min(ini)),
            "inicio": np.isin(month, ini),
            "resto": np.isin(month, res)}

def check_version():
    if globals().get("BLOCKS_VERSION") != "2026-10-17-final":
        raise RuntimeError("Rode o bloco 00 (configuracao) desta versao antes deste bloco.")

# ============================================================
# 9. SUMMARY
# ============================================================

summary = Table(title="RSL - TIs com BRIF (Cerrado)")
summary.add_column("ti_id"); summary.add_column("TI"); summary.add_column("area (ha)")
summary.add_column("classe"); summary.add_column("FRRI")
for _, r in tis.merge(artigo1.drop(columns=["terrai_cod"]), on="ti_id", how="left").iterrows():
    summary.add_row(str(r["ti_id"]), str(r["ti_name"]), f"{r.get('area_ha', np.nan):,.0f}",
                    str(r.get("classe", "")), f"{r.get('frri', np.nan):.3f}")
console.print(summary)

save_json({"project": PROJECT_NAME, "created_at": datetime.now(), "ee_project": EE_PROJECT,
           "tis_asset": CLIPPED_ASSET_ID, "transition_year": TRANSITION_YEAR,
           "post_period": POST_PERIOD, "years": YEARS, "scar_years": SCAR_YEARS,
           "early": EARLY, "late": LATE, "cell_m": CELL_M, "focos": "MODIS MOD14A1+MYD14A1 C6.1",
           "ti_names": tis["ti_name"].tolist()}, DIRS["logs"] / "project_metadata.json")
console.print(f"[bold green]OK BLOCO 00 completo (versao {BLOCKS_VERSION}).[/bold green]")


## 01 cicatrizes_mapbiomas

In [ ]:
# ======================================================================
# BLOCO 01 - Cicatrizes do MapBiomas Fogo (Colecao 4, mensal, 30 m)
# Depende do bloco 00.
# ======================================================================
#
# Cicatriz = pixels queimados no MESMO mes e ano, contiguos (vizinhanca de 8),
# dentro de cada TI. O valor do pixel no MapBiomas Fogo Col. 4 mensal e o
# proprio MES da queima (1-12), banda "burned_monthly_{ano}".
#
#   1) Earth Engine: exporta o raster do mes da queima - uma imagem por TI,
#      uma banda por ano (2012-2024) + mascara da TI. EPSG:5880, 30 m, mesma
#      grade dos rasters do bloco 03.
#   2) Colab: transforma em poligonos (mesmo mes, vizinhanca de 8), com bordas
#      exatamente nos pixels de 30 m (sem simplificar).
#
# Output:
#   07_exports/fogo_mes/l9_<tag>_firemonth_ti<id>.tif   (um por TI)
#   02_cache/cicatrizes_<anos>_<tag>.gpkg  (campos: ti_id, year, month, area_ha)
# ======================================================================

check_version()

import rasterio
from rasterio.features import shapes
from shapely.geometry import shape

FIRE_DIR    = DIRS["exports"] / "fogo_mes"
TASK_PREFIX = f"l9_{RUN_TAG}_firemonth"

def fire_name(ti_id):
    return f"{TASK_PREFIX}_ti{int(ti_id)}"

# ============================================================
# 1. EXPORTACAO DO RASTER (uma imagem por TI, uma banda por ano)
# ============================================================

def fire_image(ti_id):
    fire = ee.Image(FIRE_ASSET)
    bands = [ti_img.eq(int(ti_id)).rename("ti")]
    bands += [fire.select(FIRE_BAND(y)).unmask(0).rename(f"m{y}") for y in SCAR_YEARS]
    return ee.Image.cat(bands).toUint8()

def run_fire_rasters():
    specs = {}
    for t in tis["ti_id"].astype(int):
        region, transform, _ = ti_frame(t)
        specs[fire_name(t)] = (lambda t=t: fire_image(t), region, transform)
    return run_image_exports(specs, FIRE_DIR)

# ============================================================
# 2. POLIGONOS (mesmo mes, vizinhanca de 8, sem simplificar)
# ============================================================

def vectorize_ti(ti_id):
    with rasterio.open(FIRE_DIR / f"{fire_name(ti_id)}.tif") as src:
        arr = src.read()
        tr = src.transform
    inside = arr[0] == 1
    rows = []
    for k, y in enumerate(SCAR_YEARS, start=1):
        month = arr[k].astype(np.int16)
        mask = inside & (month > 0)
        if not mask.any():
            continue
        for geom, val in shapes(month, mask=mask, connectivity=8, transform=tr):
            rows.append({"ti_id": int(ti_id), "year": int(y), "month": int(val), "geometry": shape(geom)})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=AREA_CRS)

def run_scars_pipeline() -> gpd.GeoDataFrame:
    missing = run_fire_rasters()
    if missing:
        raise RuntimeError(f"Faltam rasters: {missing} - rode o bloco 01 de novo quando as tarefas terminarem.")
    parts = []
    for t in tqdm(tis["ti_id"].astype(int), desc="Vetorizando TIs"):
        g = vectorize_ti(t)
        console.print(f"  [dim]TI {t}: {len(g):,} cicatrizes[/dim]")
        parts.append(g)
    g = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs=AREA_CRS)
    g["area_ha"] = g.geometry.area / 1e4
    return g[["ti_id", "year", "month", "area_ha", "geometry"]]

scars_raw = checkpoint_load_or_run(SCARS_NAME, run_scars_pipeline, ext="gpkg")

# ============================================================
# 3. CONFERENCIA
# ============================================================

chk = (scars_raw.assign(periodo=np.select([scars_raw["month"].isin(EARLY), scars_raw["month"].isin(LATE)],
                                          ["mai-jul", "ago-out"], "outro"))
                .groupby(["year", "periodo"])["area_ha"].sum().unstack(fill_value=0).round(0))
display(chk)
chk.to_csv(stats_path("cicatrizes_area_por_ano_periodo"))

console.print(f"[bold green]OK BLOCO 01 completo: {len(scars_raw):,} cicatrizes.[/bold green]")


## 02 elegibilidade_amostra

In [ ]:
# ======================================================================
# BLOCO 02 - Elegibilidade das 14 TIs com BRIF -> amostra final da letter
# Depende do bloco 00 e dos rasters do bloco 01 (07_exports/fogo_mes).
# ======================================================================
#
# Ficam na amostra final as TIs que tiverem, ao mesmo tempo:
#   (a) IMAGENS: em cada ano, maior intervalo entre datas Landsat 8/9 com
#       cena <= MAX_CLOUD_L % de nuvem, em jun-out, <= MAX_GAP_DIAS; e
#       >= MIN_ANOS_IMG anos assim em 2014-2024. Sentinel-2 nao entra (sem
#       cobertura regular em 2014-2016).
#   (b) ENCONTROS: queimadas do restante da seca (cada mes de ago-out, pixels
#       conectados) encostando, a ate CONTACT_TOL_M, em cicatriz do inicio da
#       seca do mesmo ano, com >= MIN_CONTATO_M de borda em contato, em anos
#       com imagem adequada; >= MIN_ENCONTROS por TI.
#         ordem "clara"      : >= 2 meses de diferenca (ex.: jun x ago)
#         ordem "a verificar": meses vizinhos (jul x ago)
#
# Encontros calculados nos rasters de 30 m do bloco 01:
#   borda da queimada = pixels dela com vizinho (8 direcoes) fora dela
#   contato           = pixels de borda a ate 2 pixels de uma cicatriz inicial
#   contato (m)       = n de pixels de contato x 30 m
#
# Limitacoes (declarar no texto): nuvem = da CENA (CLOUD_COVER), nao da TI;
# encontros do MapBiomas mensal, antes da conferencia nas imagens (bloco 10).
#
# Output:
#   03_tables/amostra_final_tis.csv  (as 14, com elegivel e motivo de exclusao)
#   04_stats/elegibilidade_encontros_ti_ano.csv
#   02_cache/encontros_candidatos_<...>.gpkg (1 ponto por encontro)
#   06_logs/amostra_final.json
#   05_figures/02_elegibilidade.png
# ======================================================================

check_version()

import rasterio
from scipy import ndimage

# ============================================================
# 1. CRITERIOS
# ============================================================

IMG_MONTHS     = (6, 10)
MAX_CLOUD_L    = 30
MAX_GAP_DIAS   = 32          # = 2 revisitas do Landsat
MIN_ANOS_IMG   = 6           # de 11 anos
CONTACT_TOL_M  = 45          # 1,5 pixel
MIN_CONTATO_M  = 300
MIN_ENCONTROS  = 20

ANALYSIS_YEARS = list(YEARS)
FIRE_DIR = DIRS["exports"] / "fogo_mes"
CONN8 = np.ones((3, 3), int)
TOL_PX = int(np.ceil(CONTACT_TOL_M / PIX_M))

# ============================================================
# 2. (a) IMAGENS: datas limpas por TI e ano (so metadados)
# ============================================================

def clean_dates(ti_geom):
    d0, d1 = f"{ANALYSIS_YEARS[0]}-01-01", f"{ANALYSIS_YEARS[-1] + 1}-01-01"
    mm = ee.Filter.calendarRange(IMG_MONTHS[0], IMG_MONTHS[1], "month")
    out = {}
    for cid in ["LANDSAT/LC08/C02/T1_L2", "LANDSAT/LC09/C02/T1_L2"]:
        col = (ee.ImageCollection(cid).filterBounds(ti_geom).filterDate(d0, d1)
               .filter(mm).filter(ee.Filter.lt("CLOUD_COVER", MAX_CLOUD_L)))
        out[cid.split("/")[1]] = col.aggregate_array("system:time_start").getInfo()
    return out

def max_gap(days, year):
    start = pd.Timestamp(year, IMG_MONTHS[0], 1)
    end = pd.Timestamp(year, IMG_MONTHS[1], 1) + pd.offsets.MonthEnd(0)
    pts = sorted({start, end, *[d for d in days if start <= d <= end]})
    return int(max((b - a).days for a, b in zip(pts[:-1], pts[1:])))

def run_imagery():
    rows = []
    for r in tqdm(list(tis.itertuples()), desc="Imagens por TI"):
        geom = tis_fc.filter(ee.Filter.eq("ti_id", int(r.ti_id))).geometry()
        by_sensor = {k: pd.DatetimeIndex(pd.to_datetime(v, unit="ms").normalize().unique())
                     for k, v in clean_dates(geom).items()}
        all_d = pd.DatetimeIndex(sorted(set().union(*[set(d) for d in by_sensor.values()])))
        for y in ANALYSIS_YEARS:
            dy = all_d[all_d.year == y]
            row = {"ti_id": r.ti_id, "year": y}
            for k, d in by_sensor.items():
                row[f"n_datas_{k}"] = int((d.year == y).sum())
            row.update({"n_datas_limpas": len(dy), "max_gap_dias": max_gap(list(dy), y)})
            rows.append(row)
    return pd.DataFrame(rows)

img = checkpoint_load_or_run(f"B01b_imagens_landsat_{ANALYSIS_YEARS[0]}_{ANALYSIS_YEARS[-1]}_c{MAX_CLOUD_L}_{RUN_TAG}",
                             run_imagery, ext="parquet")
img["imagem_ok"] = img["max_gap_dias"] <= MAX_GAP_DIAS

# ============================================================
# 3. (b) ENCONTROS NO RASTER DE 30 m
# ============================================================

def encounters_ti(ti_id):
    with rasterio.open(FIRE_DIR / f"l9_{RUN_TAG}_firemonth_ti{int(ti_id)}.tif") as src:
        arr, tr = src.read(), src.transform
    inside = arr[0] == 1
    enc_rows, area_rows = [], []
    for y in ANALYSIS_YEARS:
        m = arr[1 + SCAR_YEARS.index(y)].astype(np.int16)
        E = inside & np.isin(m, EARLY)
        L_all = inside & np.isin(m, LATE)
        area_rows.append({"ti_id": ti_id, "year": y, "precoce": float(E.sum() * 0.09),
                          "tardia": float(L_all.sum() * 0.09)})
        if not E.any() or not L_all.any():
            continue
        e_near = ndimage.maximum_filter(np.where(E, m, 0), size=2 * TOL_PX + 1)
        for mo in LATE:
            Lm = inside & (m == mo)
            if not Lm.any():
                continue
            lab, n = ndimage.label(Lm, structure=CONN8)
            edge = Lm & ~ndimage.binary_erosion(Lm, structure=CONN8, border_value=0)
            contact = edge & (e_near > 0)
            if not contact.any():
                continue
            ids_c = lab[contact]
            n_contact = np.bincount(ids_c, minlength=n + 1)
            n_edge = np.bincount(lab[edge], minlength=n + 1)
            n_px = np.bincount(lab.ravel(), minlength=n + 1)
            emax = np.zeros(n + 1, np.int16)
            np.maximum.at(emax, ids_c, e_near[contact])
            hits = np.nonzero(n_contact * PIX_M >= MIN_CONTATO_M)[0]
            hits = hits[hits > 0]
            if len(hits) == 0:
                continue
            cents = ndimage.center_of_mass(Lm, lab, hits)
            for k, (r, c) in zip(hits, cents):
                gap = int(mo - emax[k])
                enc_rows.append({
                    "scar_id": f"{ti_id}_{y}_{mo}_{k}", "ti_id": ti_id, "year": y,
                    "month_late": int(mo), "month_early_max": int(emax[k]),
                    "contato_m": float(n_contact[k] * PIX_M), "perimetro_m": float(n_edge[k] * PIX_M),
                    "frac_borda_em_contato": float(n_contact[k] / max(n_edge[k], 1)),
                    "area_queimada_ha": float(n_px[k] * 0.09),
                    "ordem": "clara" if gap >= 2 else "a verificar",
                    "x": tr.c + (c + 0.5) * PIX_M, "y": tr.f - (r + 0.5) * PIX_M})
    del arr
    return enc_rows, area_rows

names_all = tis.set_index("ti_id")["ti_name"].to_dict()

def run_encounters():
    enc_rows, area_rows = [], []
    for t in tqdm(tis["ti_id"].astype(int).tolist(), desc="Encontros por TI"):
        e, a = encounters_ti(t)
        enc_rows += e
        area_rows += a
        console.print(f"  [dim]{names_all[t]}: {len(e):,} encontros[/dim]")
    enc = pd.DataFrame(enc_rows)
    g = gpd.GeoDataFrame(enc, geometry=gpd.points_from_xy(enc["x"], enc["y"]), crs=AREA_CRS) \
        if len(enc) else gpd.GeoDataFrame(columns=["ti_id", "year", "ordem", "contato_m", "geometry"],
                                          geometry="geometry", crs=AREA_CRS)
    write_table(pd.DataFrame(area_rows), cache_path(f"elegibilidade_areas_{RUN_TAG}", "parquet"))
    return g

enc = checkpoint_load_or_run(f"encontros_candidatos_{ANALYSIS_YEARS[0]}_{ANALYSIS_YEARS[-1]}"
                             f"_t{CONTACT_TOL_M}_m{MIN_CONTATO_M}_{RUN_TAG}", run_encounters, ext="gpkg")
areas = read_table(cache_path(f"elegibilidade_areas_{RUN_TAG}", "parquet"))

enc_ty = (pd.DataFrame(enc.drop(columns="geometry"))
          .groupby(["ti_id", "year"])
          .agg(n_encontros=("scar_id", "size"),
               n_ordem_clara=("ordem", lambda o: int((o == "clara").sum())),
               contato_km=("contato_m", lambda c: c.sum() / 1000))
          .reset_index())
ty = (img.merge(enc_ty, on=["ti_id", "year"], how="left")
         .merge(areas, on=["ti_id", "year"], how="left"))
for c in ["n_encontros", "n_ordem_clara", "contato_km", "precoce", "tardia"]:
    ty[c] = ty[c].fillna(0)
ty.to_csv(stats_path("elegibilidade_encontros_ti_ano"), index=False)

# ============================================================
# 4. ELEGIBILIDADE POR TI E MOTIVO DAS EXCLUSOES
# ============================================================

ok_years = ty[ty["imagem_ok"]]
per_ti = (tis[["ti_id", "terrai_cod", "ti_name", "area_ha"]]
          .merge(ty.groupby("ti_id").agg(anos_imagem_ok=("imagem_ok", "sum"),
                                          gap_mediano_dias=("max_gap_dias", "median")).reset_index(),
                 on="ti_id", how="left")
          .merge(ok_years.groupby("ti_id").agg(encontros=("n_encontros", "sum"),
                                                encontros_ordem_clara=("n_ordem_clara", "sum"),
                                                contato_km=("contato_km", "sum")).reset_index(),
                 on="ti_id", how="left")
          .fillna({"anos_imagem_ok": 0, "encontros": 0, "encontros_ordem_clara": 0, "contato_km": 0}))

def motivo(r):
    m = []
    if r["anos_imagem_ok"] < MIN_ANOS_IMG:
        m.append(f"imagens: {int(r['anos_imagem_ok'])} ano(s) com intervalo <= {MAX_GAP_DIAS} d "
                 f"(minimo {MIN_ANOS_IMG})")
    if r["encontros"] < MIN_ENCONTROS:
        m.append(f"encontros: {int(r['encontros'])} em anos com imagem adequada (minimo {MIN_ENCONTROS})")
    return "; ".join(m)

per_ti["motivo_exclusao"] = per_ti.apply(motivo, axis=1)
per_ti["elegivel"] = per_ti["motivo_exclusao"] == ""
per_ti = per_ti.sort_values(["elegivel", "encontros"], ascending=[False, False])
per_ti.to_csv(table_path("amostra_final_tis"), index=False)
save_json({"criterios": {"anos": [ANALYSIS_YEARS[0], ANALYSIS_YEARS[-1]], "IMG_MONTHS": IMG_MONTHS,
                         "sensores": "Landsat 8/9 C2 L2", "MAX_CLOUD_L": MAX_CLOUD_L,
                         "MAX_GAP_DIAS": MAX_GAP_DIAS, "MIN_ANOS_IMG": MIN_ANOS_IMG,
                         "CONTACT_TOL_M": CONTACT_TOL_M, "MIN_CONTATO_M": MIN_CONTATO_M,
                         "MIN_ENCONTROS": MIN_ENCONTROS},
           "elegiveis": per_ti.loc[per_ti["elegivel"], "ti_name"].tolist(),
           "excluidas": dict(zip(per_ti.loc[~per_ti["elegivel"], "ti_name"],
                                 per_ti.loc[~per_ti["elegivel"], "motivo_exclusao"]))},
          DIRS["logs"] / "amostra_final.json")

AMOSTRA_TI_IDS = per_ti.loc[per_ti["elegivel"], "ti_id"].astype(int).tolist()

t = Table(title=f"Amostra final: {len(AMOSTRA_TI_IDS)} de {len(per_ti)} TIs com BRIF")
for c in ["TI", "anos c/ imagem", "encontros", "ordem clara", "contato (km)", "situacao"]:
    t.add_column(c)
for _, r in per_ti.iterrows():
    t.add_row(str(r["ti_name"]), str(int(r["anos_imagem_ok"])), str(int(r["encontros"])),
              str(int(r["encontros_ordem_clara"])), f"{r['contato_km']:.1f}",
              "[green]entra[/green]" if r["elegivel"] else f"[red]sai[/red] - {r['motivo_exclusao']}")
console.print(t)

# ============================================================
# 5. FIGURA DE CONFERENCIA (a versao da revista e a FigR1 do bloco 17)
# ============================================================

names = per_ti.set_index("ti_id")["ti_name"]
order = per_ti["ti_id"].tolist()
mat = ty.pivot_table(index="ti_id", columns="year", values="n_encontros", fill_value=0).reindex(order)
okm = ty.pivot_table(index="ti_id", columns="year", values="imagem_ok", aggfunc="first").reindex(order)
fig, ax = plt.subplots(figsize=(1.2 + 0.55 * mat.shape[1], 0.9 + 0.38 * mat.shape[0]))
im = ax.imshow(mat.values, cmap="Blues", aspect="auto")
vmax = np.nanmax(mat.values) if mat.size else 0
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = int(mat.values[i, j]); ok_ = bool(okm.values[i, j])
        ax.text(j, i, "x" if not ok_ else (str(v) if v else ""), ha="center", va="center", fontsize=7,
                color="#b2182b" if not ok_ else ("white" if v > vmax / 2 else "black"))
ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, fontsize=8)
ax.set_yticks(range(mat.shape[0]))
ax.set_yticklabels([f"{names[i]}{'' if i in AMOSTRA_TI_IDS else '  (sai)'}" for i in mat.index], fontsize=8)
ax.grid(False)
ax.set_title("Encontros candidatos (ago-out encostando em mai-jul)\nx = ano sem imagem adequada", fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.7, label="encontros")
fig.savefig(fig_path("02_elegibilidade"))
plt.show()

console.print(f"[bold green]OK BLOCO 02 completo: amostra final = {len(AMOSTRA_TI_IDS)} TIs "
              f"({', '.join(names[i] for i in AMOSTRA_TI_IDS)}).[/bold green]")


## 03 rasters_30m

In [ ]:
# ======================================================================
# BLOCO 03 - Pilhas de 30 m por TI (fogo, cobertura, Landsat NBR, relevo, barreiras)
# Depende dos blocos 00 e 02 (amostra final).
# ======================================================================
#
# Tudo o que vem depois (sequencia, configuracao, encontros, modelos) e
# calculado sobre estas pilhas, para qualquer recorte sazonal e escala.
#
# 1) Por TI e por ano (2012-2024; 2012-2013 so como historico):
#      month : mes da queima no MapBiomas Fogo Col. 4 (0 = nao queimou)
#      cover : MapBiomas Cobertura Col. 10 em 4 grupos (1 natural, 2 pastagem/
#              mosaico, 3 agricultura/silvicultura, 4 nao combustivel, 0 sem dado)
#      nbrMM : NBR MINIMO do mes MM (abr-nov), Landsat 8/9 C2 L2 (x100), so
#              observacoes limpas (QA_PIXEL sem nuvem, sombra, cirro); calculado
#              ate 500 m de onde houve fogo no ano (fora: -128 = sem dado)
#      cntMM : n de observacoes limpas do Landsat no mes MM
# 2) Camadas fixas por TI:
#      ti, elev (SRTM), slope10 (graus x 10),
#      barrier : agua / nao vegetado / urbano / mineracao em >= 80% dos anos
#                + rios HydroSHEDS (ordem <= 7) [+ estradas, se ROADS_ASSET]
#      river   : so os rios
#
# Download direto do Earth Engine (ee.data.computePixels, endereco de alto
# volume), em blocos de ate 1024 x 1024 pixels em paralelo, sem tarefas no
# painel e sem passar pelo Drive de exportacao. Bloco pesado demais e dividido
# em 4; recusa por excesso de pedidos -> espera e tenta de novo; arquivo ja
# existente -> pula (se o Colab cair, rode de novo).
#
# Projecao EPSG:5880, grade de 30 m alinhada; recorte = retangulo da TI + 2 km.
# Output: 07_exports/rasters/l9_<tag>_v3_yr_ti<id>_<ano>.tif e ..._static_ti<id>.tif
# ======================================================================

check_version()

import concurrent.futures as cf
import shutil
import rasterio
from rasterio.transform import Affine

# ============================================================
# 1. PARAMETERS
# ============================================================

BARRIER_CLASSES   = [22, 23, 24, 25, 30, 31, 33]
BARRIER_MIN_SHARE = 0.8
RIVER_MAX_ORD     = 7
RIVER_BUFFER_M    = 30
ROADS_ASSET       = ""          # opcional: ativo de linhas de estradas
ROADS_BUFFER_M    = 15
NODATA8           = -128
NEAR_BURN_M       = 500
N_WORKERS         = 10
TILE_PX           = 1024
MIN_TILE          = 128
MAX_TRIES         = 6

RASTER_DIR.mkdir(parents=True, exist_ok=True)
ti_ids = load_sample_ids()
console.print(f"[cyan]TIs com raster:[/cyan] " + ", ".join(tis.set_index("ti_id").loc[ti_ids, "ti_name"])
              + f" | anos: {SCAR_YEARS[0]}-{SCAR_YEARS[-1]}")

# ============================================================
# 2. IMAGENS
# ============================================================

lulc_ic = (ee.ImageCollection(LULC_ASSET)
           .filter(ee.Filter.eq("collection_id", LULC_COLLECTION_ID))
           .filter(ee.Filter.eq("version", LULC_VERSION)))

def lulc(y):
    return lulc_ic.filter(ee.Filter.eq("year", y)).first().select("classification")

def landsat_nbr(img):
    img = ee.Image(img)
    clear = img.select("QA_PIXEL").bitwiseAnd(0b11111).eq(0)   # fill, nuvem dilatada, cirro, nuvem, sombra
    sr = img.select(["SR_B5", "SR_B7"]).multiply(0.0000275).add(-0.2)
    return (sr.normalizedDifference(["SR_B5", "SR_B7"]).toFloat().rename("nbr")
              .updateMask(clear).copyProperties(img, ["system:time_start"]))

L89 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").select(["SR_B5", "SR_B7", "QA_PIXEL"])
       .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").select(["SR_B5", "SR_B7", "QA_PIXEL"])))
_EMPTY = ee.Image.constant(0).toFloat().rename("nbr").updateMask(ee.Image.constant(0))
_MIN_COUNT = ee.Reducer.min().combine(ee.Reducer.count(), sharedInputs=True)

def year_image(y, region):
    month = ee.Image(FIRE_ASSET).select(FIRE_BAND(y)).unmask(0).toInt8().rename("month")
    frm = [c for g in COVER_CLASSES for c in COVER_CLASSES[g]]
    to = [g for g in COVER_CLASSES for _ in COVER_CLASSES[g]]
    cover = lulc(y).remap(frm, to, 0).unmask(0).toInt8().rename("cover")
    nbrs, cnts = [], []
    if y < YEARS[0]:          # 2012-2013: so historico de fogo, sem Landsat
        for m in LS_MONTHS:
            nbrs.append(ee.Image.constant(NODATA8).toInt8().rename(f"nbr{m:02d}"))
            cnts.append(ee.Image.constant(0).toInt8().rename(f"cnt{m:02d}"))
        return ee.Image.cat([month, cover] + nbrs + cnts)
    near = month.gt(0).focalMax(radius=NEAR_BURN_M, kernelType="circle", units="meters")
    col = (L89.filterBounds(region).filterDate(f"{y}-01-01", f"{y + 1}-01-01")
              .map(landsat_nbr).map(lambda i: ee.Image(i).updateMask(near)))
    for m in LS_MONTHS:
        cm = col.filter(ee.Filter.calendarRange(m, m, "month")).merge(ee.ImageCollection([_EMPTY]))
        red = cm.reduce(_MIN_COUNT)
        nbrs.append(red.select("nbr_min").multiply(100).round().unmask(NODATA8).toInt8().rename(f"nbr{m:02d}"))
        cnts.append(red.select("nbr_count").unmask(0).toInt8().rename(f"cnt{m:02d}"))
    return ee.Image.cat([month, cover] + nbrs + cnts)

def static_image():
    elev = ee.Image("USGS/SRTMGL1_003").select("elevation")
    slope10 = ee.Terrain.slope(elev).multiply(10).round()
    share = ee.Image.cat([lulc(y).remap(BARRIER_CLASSES, [1] * len(BARRIER_CLASSES), 0).unmask(0)
                          for y in YEARS]).reduce(ee.Reducer.mean())
    perm = share.gte(BARRIER_MIN_SHARE)
    rivers = (ee.FeatureCollection("WWF/HydroSHEDS/v1/FreeFlowingRivers")
              .filter(ee.Filter.lte("RIV_ORD", RIVER_MAX_ORD))
              .map(lambda f: ee.Feature(f).buffer(RIVER_BUFFER_M)))
    river = ee.Image(0).paint(rivers, 1)
    barrier = perm.Or(river)
    if ROADS_ASSET:
        roads = ee.FeatureCollection(ROADS_ASSET).map(lambda f: ee.Feature(f).buffer(ROADS_BUFFER_M))
        barrier = barrier.Or(ee.Image(0).paint(roads, 1))
    return ee.Image.cat([ti_img.rename("ti"), elev.rename("elev"), slope10.rename("slope10"),
                         barrier.rename("barrier"), river.rename("river")]).unmask(0).toInt16()

# ============================================================
# 3. DOWNLOAD DIRETO EM BLOCOS
# ============================================================

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
ee.Initialize(project=EE_PROJECT, opt_url="https://earthengine-highvolume.googleapis.com")

def _fetch(img, bands, x0, y1, w, h):
    req = {"expression": img, "fileFormat": "NUMPY_NDARRAY",
           "grid": {"dimensions": {"width": int(w), "height": int(h)},
                    "affineTransform": {"scaleX": float(PIX_M), "shearX": 0.0, "translateX": float(x0),
                                        "shearY": 0.0, "scaleY": -float(PIX_M), "translateY": float(y1)},
                    "crsCode": AREA_CRS}}
    arr = ee.data.computePixels(req)
    return np.stack([arr[b] for b in bands])

def fetch_tile(img, bands, x0, y1, w, h):
    for k in range(MAX_TRIES):
        try:
            return _fetch(img, bands, x0, y1, w, h)
        except Exception as e:
            msg = str(e).lower()
            heavy = any(s in msg for s in ["timed out", "memory", "too large", "payload", "deadline"])
            if heavy and max(w, h) > MIN_TILE:
                hw, hh = (w + 1) // 2, (h + 1) // 2
                out = None
                for dx, dy, pw, ph in [(0, 0, hw, hh), (hw, 0, w - hw, hh), (0, hh, hw, h - hh), (hw, hh, w - hw, h - hh)]:
                    if pw <= 0 or ph <= 0:
                        continue
                    a = fetch_tile(img, bands, x0 + dx * PIX_M, y1 - dy * PIX_M, pw, ph)
                    if out is None:
                        out = np.zeros((a.shape[0], h, w), a.dtype)
                    out[:, dy:dy + ph, dx:dx + pw] = a
                return out
            time.sleep(min(2 ** k * 3, 60))
    raise RuntimeError(f"Bloco ({x0:.0f}, {y1:.0f}, {w}x{h}) falhou {MAX_TRIES} vezes.")

def write_stack(name, img, bands, dtype, ti_id):
    dest = RASTER_DIR / f"{name}.tif"
    if dest.exists():
        return "ja existe"
    _, transform, _ = ti_frame(ti_id)
    x0, y1 = transform[2], transform[5]
    reg = tis.set_index("ti_id").loc[ti_id].geometry.bounds
    W = int(round((float(np.ceil((reg[2] + RASTER_BUFFER_M) / PIX_M) * PIX_M) - x0) / PIX_M))
    H = int(round((y1 - float(np.floor((reg[1] - RASTER_BUFFER_M) / PIX_M) * PIX_M)) / PIX_M))
    out = np.zeros((len(bands), H, W), dtype)
    jobs = [(c, r, min(TILE_PX, W - c), min(TILE_PX, H - r))
            for r in range(0, H, TILE_PX) for c in range(0, W, TILE_PX)]
    with cf.ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(fetch_tile, img, bands, x0 + c * PIX_M, y1 - r * PIX_M, w, h): (c, r, w, h)
                for c, r, w, h in jobs}
        for f in cf.as_completed(futs):
            c, r, w, h = futs[f]
            out[:, r:r + h, c:c + w] = f.result().astype(dtype)
    tmp = Path("/content") / f"{name}.tif"
    prof = dict(driver="GTiff", height=H, width=W, count=len(bands), dtype=np.dtype(dtype).name,
                crs=AREA_CRS, transform=Affine(PIX_M, 0, x0, 0, -PIX_M, y1),
                compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(tmp, "w", **prof) as dst:
        dst.write(out)
    shutil.copy(tmp, dest)
    tmp.unlink()
    return f"{W}x{H} px, {len(jobs)} bloco(s)"

def run_rasters():
    todo = []
    for t in ti_ids:
        todo.append((raster_name(t), lambda: static_image(), STATIC_BANDS, np.int16, t))
        for y in SCAR_YEARS:
            todo.append((raster_name(t, y), (lambda y=y, t=t: year_image(y, ti_frame(t)[0])),
                         YEAR_BANDS, np.int8, t))
    falta = [j for j in todo if not (RASTER_DIR / f"{j[0]}.tif").exists()]
    console.print(f"  Ja prontos: {len(todo) - len(falta)} | A baixar: {len(falta)}")
    erros = []
    for name, make, bands, dtype, t in tqdm(falta, desc="Rasters"):
        t0 = time.time()
        try:
            info = write_stack(name, make(), bands, dtype, t)
            console.print(f"  [green]OK {name}[/green] [dim]{info}, {time.time() - t0:.0f} s[/dim]")
        except Exception as e:
            erros.append(name)
            console.print(f"  [red]X {name}: {str(e)[:200]}[/red]")
    return erros

missing_rasters = run_rasters()
ee.Initialize(project=EE_PROJECT)          # volta ao endereco normal

# ============================================================
# 4. CONFERENCIA
# ============================================================

if not missing_rasters:
    t0, y0 = ti_ids[0], YEARS[-1]
    st, yr = read_raster(t0), read_raster(t0, y0)
    inside = st["ti"] == t0
    obs = {m: float((yr[f"cnt{m:02d}"][inside & (yr["month"] > 0)] > 0).mean()) for m in LS_MONTHS}
    console.print(f"[cyan]TI {t0}, {y0}:[/cyan] queimado {float((yr['month'][inside] > 0).mean()):.1%} | "
                  f"nativa {float((yr['cover'][inside] == 1).mean()):.1%} | barreira {float(st['barrier'][inside].mean()):.1%}")
    console.print("[cyan]Areas queimadas com observacao Landsat limpa, por mes:[/cyan] "
                  + " | ".join(f"{m:02d}: {v:.0%}" for m, v in obs.items()))
    console.print("[bold green]OK BLOCO 03 completo.[/bold green]")
else:
    console.print(f"[red]{len(missing_rasters)} raster(s) com erro - rode o bloco 03 de novo: {missing_rasters[:5]}[/red]")


## 04 era5_diario

In [ ]:
# ======================================================================
# BLOCO 04 - ERA5-Land diario (abr-nov) nas TIs com BRIF
# Depende do bloco 00.
# ======================================================================
#
# Usos: tempo no dia em que o fogo chegou a cada borda (bloco 10); VPD e chuva
# da estacao por celula (bloco 08); tempo no dia das queimadas iniciais (07).
#   vpd_max   : deficit de pressao de vapor no horario mais quente (kPa), da
#               temperatura maxima e do ponto de orvalho medio
#   rh_min    : umidade relativa minima aproximada (%)
#   wind      : vento a 10 m (m/s), das componentes u/v medias
#   precip_mm : chuva do dia (mm)
# Pontos = centros dos pixels do ERA5-Land (~9 km) sobre as TIs + 10 km.
#
# Output: 07_exports/era5/l9_<tag>_era5v2_AAAA.csv | 02_cache/era5_diario_v2_<anos>.parquet
# ======================================================================

check_version()

ERA5_DAILY    = "ECMWF/ERA5_LAND/DAILY_AGGR"
ERA5_BUFFER_M = 10000
TASK_ERA5     = f"l9_{RUN_TAG}_era5v2"
ERA5_DIR      = DIRS["exports"] / "era5"

era5_ic   = ee.ImageCollection(ERA5_DAILY)
era5_proj = era5_ic.first().projection()
era5_area = tis_fc.map(lambda f: ee.Feature(f).buffer(ERA5_BUFFER_M, 500)).geometry()

def _pt_props(f):
    f = ee.Feature(f)
    lon = ee.Number(f.get("longitude"))
    lat = ee.Number(f.get("latitude"))
    return f.set({"lon": lon, "lat": lat,
                  "point_id": lon.format("%.3f").cat("_").cat(lat.format("%.3f"))})

era5_points = (ee.Image.pixelLonLat().reproject(era5_proj)
               .sample(region=era5_area, projection=era5_proj, geometries=True)
               .map(_pt_props))
console.print(f"[cyan]Pontos ERA5 dentro da area:[/cyan] {era5_points.size().getInfo()}")

def _sat_vp(t_c):
    return t_c.expression("0.6108*exp(17.27*T/(T+237.3))", {"T": t_c})

def daily_vars(img):
    img = ee.Image(img)
    t  = img.select("temperature_2m_max").subtract(273.15)
    td = img.select("dewpoint_temperature_2m").subtract(273.15)
    es, ea = _sat_vp(t), _sat_vp(td)
    u = img.select("u_component_of_wind_10m")
    v = img.select("v_component_of_wind_10m")
    return (es.subtract(ea).rename("vpd_max")
              .addBands(ea.divide(es).multiply(100).rename("rh_min"))
              .addBands(u.hypot(v).rename("wind"))
              .addBands(img.select("total_precipitation_sum").multiply(1000).rename("precip_mm"))
              .set("date", img.date().format("YYYY-MM-dd")))

def era5_task(y):
    def _make():
        col = era5_ic.filterDate(f"{y}-04-01", f"{y}-12-01").map(daily_vars)

        def _per_day(img):
            img = ee.Image(img)
            return (img.reduceRegions(collection=era5_points, reducer=ee.Reducer.first(), scale=11132)
                       .map(lambda f: ee.Feature(f).set("date", img.get("date")).setGeometry(None)))
        table = col.map(_per_day).flatten()
        return ee.batch.Export.table.toDrive(
            collection=table, description=f"{TASK_ERA5}_{y}", folder=EE_EXPORT_FOLDER,
            fileNamePrefix=f"{TASK_ERA5}_{y}", fileFormat="CSV",
            selectors=["point_id", "lon", "lat", "date", "vpd_max", "rh_min", "wind", "precip_mm"])
    return _make

def run_era5_pipeline() -> pd.DataFrame:
    paths = run_ee_exports({f"{TASK_ERA5}_{y}": era5_task(y) for y in YEARS}, "csv", ERA5_DIR)
    if len(paths) < len(YEARS):
        raise RuntimeError("Faltam anos do ERA5 - rode o bloco 04 de novo mais tarde.")
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    return df.dropna(subset=["vpd_max"])

era5_daily = checkpoint_load_or_run(f"era5_diario_v2_{TAG_YEARS}", run_era5_pipeline, ext="parquet")
era5_daily["date"] = pd.to_datetime(era5_daily["date"])

chk = era5_daily.groupby(era5_daily["date"].dt.month)[["vpd_max", "rh_min", "wind"]].mean().round(2)
chk.index.name = "mes"
display(chk)
if chk["vpd_max"].idxmax() not in (8, 9, 10):
    console.print("[red]WARNING pico de VPD fora de ago-out - confira as bandas do ERA5.[/red]")
console.print(f"[bold green]OK BLOCO 04 completo: {era5_daily['point_id'].nunique()} pontos x "
              f"{era5_daily['date'].nunique()} dias.[/bold green]")


## 05 focos_modis

In [ ]:
# ======================================================================
# BLOCO 05 - Focos de calor MODIS C6.1 (Terra MOD14A1 + Aqua MYD14A1, 1 km) por pixel e dia
# Depende do bloco 00 (e do 01 para o mapa de conferencia). Mesma colecao do artigo 1.
# ======================================================================

check_version()

# ============================================================
# BLOCK 4 - MODIS C6.1 daily active fires (MOD14A1 + MYD14A1), Earth Engine
#
# Por que MODIS: continuidade com o artigo 1 (mesma colecao, mesmo limiar
# principal FireMask >= 8) e serie homogenea em todo o pos-BRIF (2014-2024).
#
# O que sai: UM PONTO POR PIXEL MODIS COM FOGO POR DIA E SATELITE, no centro
# do pixel da grade sinusoidal (~926 m). Colunas no MESMO formato que o bloco 07
# e o bloco 10 ja usavam (latitude, longitude, acq_date, frp, daynight,
# confidence), mais satellite e firemask:
#   confidence : FireMask 7 = "l" (baixa), 8 = "n" (nominal), 9 = "h" (alta)
#                -> bloco 07/bloco 10 descartam "l" = serie principal >= 8 do artigo 1
#   frp        : MaxFRP x 0,1 (MW) - FRP maximo do pixel no dia
#   daynight   : QA bit 2 (1 = algoritmo diurno "D", 0 = noturno "N")
#   acq_time   : nao existe no produto diario (fica vazio)
#
# Consequencias da resolucao de 1 km (declarar nos Metodos):
#   - chegada  : foco a ate FOCO_ARR_M = 1000 m da borda, do lado de fora
#   - entrada  : centro do pixel a >= FOCO_IN_M = 1000 m DENTRO da cicatriz
#                (o pixel inteiro fica dentro da cicatriz) -> so cicatrizes
#                largas (> ~2 km) podem ter entrada vista pelo MODIS; nas
#                demais a continuidade vem so do Landsat
#   - data     : a data do foco e o dia da chegada usado no tempo (ERA5)
#
# Output:
#   07_exports/modis_focos/modis_focos_<ano>_<tag>.csv  (1 por ano, do EE)
#   02_cache/modis_focos_v1_<anos>_<tag>.parquet        (lido pelo bloco 07 e bloco 10)
#   04_stats/B04_modis_focos_ano_mes.csv
#   05_figures/B04_modis_focos_mapa.{tif,png,pdf}       (600 dpi)
# ============================================================

import json as _json

# ============================================================
# 1. PARAMETERS
# ============================================================

MODIS_COLS   = {"Terra": "MODIS/061/MOD14A1", "Aqua": "MODIS/061/MYD14A1"}
FOCO_MONTHS  = (3, 4, 5, 6, 7, 8, 9, 10, 11)   # mar-nov: cobre o recorte do artigo 1 e o local (bloco 06)
FOCO_MIN_FM  = 7                               # exporta >= 7; a analise usa >= 8 (descarta "l")
FOCO_BUFFER_M = 2000                           # TIs + 2 km (mesmo recorte dos rasters)
FOCO_ARR_M   = 1000                            # usado no bloco 10 (chegada)
FOCO_IN_M    = 1000                            # usado no bloco 10 (entrada) e no bloco 07 (ligacao foco-cicatriz)
FOCO_DIR     = DIRS["exports"] / "modis_focos"
FOCO_DIR.mkdir(parents=True, exist_ok=True)
FOCOS_NAME   = f"modis_focos_v1_{TAG_YEARS}"   # checkpoint lido pelo bloco 07 e bloco 10
TAG_SEG      = f"{TAG_YEARS}_md"               # sufixo dos checkpoints que usam focos (bloco 07, bloco 10...)
FIG_DPI_RSL  = 600

# ============================================================
# 2. REGIAO: as 14 TIs + 2 km (barato; o bloco 07/bloco 10 filtram a amostra)
# ============================================================

region = tis_fc.map(lambda f: ee.Feature(f).buffer(FOCO_BUFFER_M, 100)).union(100).geometry()

def modis_points_year(y: int) -> ee.FeatureCollection:
    fcs = []
    for sat, cid in MODIS_COLS.items():
        col = (ee.ImageCollection(cid)
               .filterDate(f"{y}-01-01", f"{y + 1}-01-01")
               .filter(ee.Filter.calendarRange(min(FOCO_MONTHS), max(FOCO_MONTHS), "month"))
               .filterBounds(region))

        def to_points(img, sat=sat):
            img = ee.Image(img)
            fm = img.select("FireMask")
            day = img.select("QA").bitwiseAnd(4).rightShift(2)            # bit 2: 1 = dia
            stack = (fm.rename("firemask")
                     .addBands(img.select("MaxFRP").multiply(0.1).rename("frp"))
                     .addBands(day.rename("day"))
                     .updateMask(fm.gte(FOCO_MIN_FM)))
            pts = stack.sample(region=region, projection=fm.projection(), geometries=True,
                               dropNulls=True, tileScale=4)            # centro do pixel nativo
            d = img.date().format("YYYY-MM-dd")
            return pts.map(lambda f: f.set({"acq_date": d, "satellite": sat}))
        fcs.append(col.map(to_points).flatten())
    return ee.FeatureCollection(fcs).flatten()

def foco_name(y):
    return f"modis_focos_{y}_{RUN_TAG}"

def make_task(y):
    return ee.batch.Export.table.toDrive(
        collection=modis_points_year(y), description=foco_name(y), folder=EE_EXPORT_FOLDER,
        fileNamePrefix=foco_name(y), fileFormat="CSV",
        selectors=["firemask", "frp", "day", "acq_date", "satellite", ".geo"])

# ============================================================
# 3. EXPORTAR (1 tarefa por ano) -> MONITORAR -> BAIXAR (funcoes do bloco 00)
# ============================================================

def run_modis_pipeline() -> pd.DataFrame:
    specs = {foco_name(y): (lambda y=y: make_task(y)) for y in YEARS}
    paths = run_ee_exports(specs, "csv", FOCO_DIR)
    if len(paths) < len(YEARS):
        raise RuntimeError("Faltam anos de focos MODIS - rode o bloco 05 de novo quando as tarefas terminarem.")
    parts = []
    for p in paths:
        d = pd.read_csv(p)
        if d.empty:
            continue
        xy = d[".geo"].map(lambda g: _json.loads(g)["coordinates"])
        d["longitude"] = xy.str[0]; d["latitude"] = xy.str[1]
        parts.append(d.drop(columns=".geo"))
    v = pd.concat(parts, ignore_index=True)
    v["acq_date"] = pd.to_datetime(v["acq_date"])
    v = v[v["acq_date"].dt.year.isin(YEARS) & v["acq_date"].dt.month.isin(FOCO_MONTHS)]
    v["firemask"] = v["firemask"].astype(int)
    v["confidence"] = v["firemask"].map({7: "l", 8: "n", 9: "h"})
    v["daynight"] = np.where(v["day"].astype(int) == 1, "D", "N")
    v["acq_time"] = np.nan
    v["sensor"] = "MODIS"
    v = v.drop_duplicates(["latitude", "longitude", "acq_date", "satellite"])

    tis_buf = tis.to_crs(AREA_CRS).copy()
    tis_buf["geometry"] = tis_buf.buffer(FOCO_BUFFER_M)
    g = gpd.GeoDataFrame(v, geometry=gpd.points_from_xy(v["longitude"], v["latitude"]), crs="EPSG:4326")
    g = gpd.sjoin(g, tis_buf.to_crs("EPSG:4326")[["ti_id", "geometry"]], predicate="within", how="inner")
    g = g.drop(columns=["index_right", "geometry", "day"])
    # TIs vizinhas com buffers sobrepostos: o foco fica na TI cujo limite esta mais perto
    g = g.drop_duplicates(["latitude", "longitude", "acq_date", "satellite"])
    cols = ["ti_id", "latitude", "longitude", "acq_date", "acq_time", "satellite", "sensor",
            "firemask", "confidence", "frp", "daynight"]
    return pd.DataFrame(g[cols]).reset_index(drop=True)

# ============================================================
# 4. RUN
# ============================================================

focos_raw = checkpoint_load_or_run(FOCOS_NAME, run_modis_pipeline, ext="parquet")
focos_raw["acq_date"] = pd.to_datetime(focos_raw["acq_date"])

# ============================================================
# 5. CONFERENCIAS
# ============================================================

chk = (focos_raw[focos_raw["confidence"] != "l"]
       .assign(ano=lambda d: d["acq_date"].dt.year, mes=lambda d: d["acq_date"].dt.month)
       .pivot_table(index="ano", columns="mes", values="frp", aggfunc="size", fill_value=0))
chk.to_csv(stats_path("B04_modis_focos_ano_mes"))
display(chk)
fm = focos_raw["firemask"].value_counts(normalize=True).sort_index()
console.print(f"[cyan]FireMask 7/8/9:[/cyan] " + " | ".join(f"{k}: {v:.1%}" for k, v in fm.items())
              + f" | [cyan]noturnos (>= 8):[/cyan] "
              f"{(focos_raw.loc[focos_raw['confidence'] != 'l', 'daynight'] == 'N').mean():.1%}"
              + f" | [cyan]Terra/Aqua:[/cyan] {focos_raw['satellite'].value_counts().to_dict()}")

# Mapa de conferencia (espacial): uma TI media, um ano - focos sobre as cicatrizes do MapBiomas
try:
    if "scars_raw" not in globals():
        scars_raw = read_table(cache_path(SCARS_NAME, "gpkg"))
    t0 = int(tis.sort_values("area_ha").iloc[len(tis) // 2]["ti_id"])
    yr_show = int(focos_raw.loc[focos_raw["ti_id"] == t0, "acq_date"].dt.year.mode().iat[0])
    s = scars_raw[(scars_raw["ti_id"] == t0) & (scars_raw["year"] == yr_show)]
    f = focos_raw[(focos_raw["ti_id"] == t0) & (focos_raw["acq_date"].dt.year == yr_show) &
                  (focos_raw["confidence"] != "l")]
    fg = gpd.GeoDataFrame(f, geometry=gpd.points_from_xy(f["longitude"], f["latitude"]),
                          crs="EPSG:4326").to_crs(AREA_CRS)
    fig, ax = plt.subplots(figsize=(14 / 2.54, 14 / 2.54))
    tis[tis["ti_id"] == t0].boundary.plot(ax=ax, color="black", lw=0.6)
    s[s["month"].isin(EARLY)].plot(ax=ax, color="#1f5fc8", lw=0)
    s[s["month"].isin(LATE)].plot(ax=ax, color="#f28322", lw=0)
    # quadrado de ~1 km = pegada nominal do pixel MODIS
    fg.assign(geometry=fg.buffer(463, cap_style=3)).boundary.plot(
        ax=ax, color=np.where(fg["acq_date"].dt.month.isin(EARLY), "#0b2a66", "#7a1f3d"), lw=0.4)
    ax.set_title(f"{tis.set_index('ti_id').loc[t0, 'ti_name']} {yr_show}: MapBiomas mai-jul (azul) e "
                 f"ago-out (laranja); quadrados = pixels MODIS com fogo (>= 8)", fontsize=8)
    ax.set_axis_off()
    for ext, kw in [("tif", {"pil_kwargs": {"compression": "tiff_lzw"}}), ("png", {}), ("pdf", {})]:
        fig.savefig(fig_path("B04_modis_focos_mapa", ext), dpi=FIG_DPI_RSL, facecolor="white", **kw)
    plt.show()
except Exception as e:
    console.print(f"[dim]mapa de conferencia pulado: {str(e)[:100]}[/dim]")

console.print(f"[bold green]OK BLOCO 05 completo: {len(focos_raw):,} pixels-dia com fogo "
              f"({(focos_raw['confidence'] != 'l').sum():,} com FireMask >= 8).[/bold green]")


## 06 estacao_seca

In [ ]:
# ======================================================================
# BLOCO 06 - Inicio e restante da seca: recorte do artigo 1 + recorte local
# Depende do bloco 00.
# ======================================================================
#
# PRINCIPAL ("artigo1"): igual para todas as TIs - seca = mai-out;
#   inicio = mai-jul; restante = ago-out.
# SENSIBILIDADE ("local", "local_100"): a seca de CADA TI pela chuva local
#   (CHIRPS, media mensal 2014-2024 sobre a TI): sequencia continua de meses
#   com chuva < limiar (60 / 100 mm) contendo o mes mais seco, dividida pela
#   mesma regra (metade inicial x final; n impar -> mes do meio no inicio).
#   Meses limitados a abr-nov (Landsat do bloco 03). Seca local < 2 meses ->
#   a TI fica com o recorte do artigo 1.
#
# Output: 03_tables/estacoes_por_ti.json | 04_stats/chuva_mensal_por_ti.csv,
#         estacoes_por_ti.csv | 05_figures/06_estacoes_por_ti.png
# ======================================================================

check_version()

LOCAL_SCHEMES = {"local": 60, "local_100": 100}
MONTH_LIMITS  = (4, 11)

def monthly_precip() -> pd.DataFrame:
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(f"{YEARS[0]}-01-01", f"{YEARS[-1] + 1}-01-01")
    n_years = len(YEARS)
    imgs = [chirps.filter(ee.Filter.calendarRange(m, m, "month")).sum().divide(n_years).rename(f"p{m:02d}")
            for m in range(1, 13)]
    fc = ee.Image.cat(imgs).reduceRegions(collection=tis_fc, reducer=ee.Reducer.mean(), scale=5566)
    info = fc.map(lambda f: ee.Feature(None, ee.Feature(f).toDictionary())).getInfo()
    df = pd.DataFrame([x["properties"] for x in info["features"]])
    long = df.melt(id_vars=["ti_id"], value_vars=[f"p{m:02d}" for m in range(1, 13)],
                   var_name="mes", value_name="chuva_mm")
    long["mes"] = long["mes"].str[1:].astype(int)
    long["ti_id"] = long["ti_id"].astype(int)
    return long.sort_values(["ti_id", "mes"]).reset_index(drop=True)

precip = checkpoint_load_or_run(f"chuva_mensal_{TAG_YEARS}", monthly_precip, ext="parquet")
precip.to_csv(stats_path("chuva_mensal_por_ti"), index=False)

def local_season(p_by_month: pd.Series, limiar: float):
    dry = p_by_month < limiar
    driest = int(p_by_month.idxmin())
    run = [driest]
    m = driest
    while True:
        m = 12 if m == 1 else m - 1
        if not dry[m] or m in run:
            break
        run.insert(0, m)
    m = driest
    while True:
        m = 1 if m == 12 else m + 1
        if not dry[m] or m in run:
            break
        run.append(m)
    run = [m for m in run if MONTH_LIMITS[0] <= m <= MONTH_LIMITS[1]]
    if len(run) < 2:
        return None
    k = int(np.ceil(len(run) / 2))
    return run[:k], run[k:]

seasons_out, rows = {}, []
for sch, limiar in LOCAL_SCHEMES.items():
    seasons_out[sch] = {}
    for ti, g in precip.groupby("ti_id"):
        res = local_season(g.set_index("mes")["chuva_mm"], limiar)
        fallback = res is None
        ini, rest = (list(EARLY), list(LATE)) if fallback else res
        seasons_out[sch][str(int(ti))] = {"inicio": ini, "restante": rest}
        rows.append({"esquema": sch, "ti_id": int(ti), "limiar_mm": limiar,
                     "inicio": "-".join(map(str, ini)), "restante": "-".join(map(str, rest)),
                     "usou_artigo1": fallback})
for ti in tis["ti_id"].astype(int):
    rows.append({"esquema": "artigo1", "ti_id": ti, "limiar_mm": np.nan,
                 "inicio": "-".join(map(str, EARLY)), "restante": "-".join(map(str, LATE)),
                 "usou_artigo1": True})

with open(table_path("estacoes_por_ti", "json"), "w", encoding="utf-8") as f:
    json.dump(seasons_out, f, indent=2)
est = pd.DataFrame(rows).merge(tis[["ti_id", "ti_name"]], on="ti_id")
est.to_csv(stats_path("estacoes_por_ti"), index=False)

SEASONS = load_seasons()
display(est.pivot_table(index="ti_name", columns="esquema", values="inicio", aggfunc="first").add_suffix(" (inicio)")
           .join(est.pivot_table(index="ti_name", columns="esquema", values="restante", aggfunc="first")
                    .add_suffix(" (restante)")))

ids = sorted(precip["ti_id"].unique())
nc = 4
nr = int(np.ceil(len(ids) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(3.2 * nc, 2.4 * nr), sharex=True, sharey=True)
names = tis.set_index("ti_id")["ti_name"]
for ax, ti in zip(np.ravel(axs), ids):
    g = precip[precip["ti_id"] == ti]
    ini, rest = SEASONS["local"][int(ti)]
    cols = ["#1b7837" if m in ini else "#b2182b" if m in rest else "#9e9e9e" for m in g["mes"]]
    ax.bar(g["mes"], g["chuva_mm"], color=cols)
    ax.axhline(LOCAL_SCHEMES["local"], color="black", lw=0.7, ls="--")
    for m in EARLY + LATE:
        ax.axvspan(m - 0.5, m + 0.5, color="#e0e0e0", zorder=0, lw=0)
    ax.set_title(str(names.get(ti, ti)), fontsize=8)
    ax.set_xticks(range(1, 13)); ax.tick_params(labelsize=6)
for ax in np.ravel(axs)[len(ids):]:
    ax.set_axis_off()
fig.suptitle("Chuva media mensal (CHIRPS). Verde = inicio, vermelho = restante da seca local; "
             "faixa cinza = recorte do artigo 1", fontsize=9)
fig.savefig(fig_path("06_estacoes_por_ti"))
plt.show()
console.print("[bold green]OK BLOCO 06 completo: esquemas " + ", ".join(SEASONS) + ".[/bold green]")


## 07 tipologia_cicatrizes

In [ ]:
# ======================================================================
# BLOCO 07 - Tipologia DESCRITIVA das cicatrizes do inicio da seca (focos MODIS 1 km)
# Depende dos blocos 00, 01 (cicatrizes), 02 (amostra), 04 (ERA5), 05 (MODIS) e 06.
# ======================================================================

check_version()

# ============================================================
# BLOCK 6 - Descriptive typology of early-season burns
#
# MUDANCA (2026-10-17): focos MODIS C6.1 (MOD14A1 + MYD14A1, FireMask >= 8),
#   como no artigo 1. Cada foco vai para a cicatriz mais proxima a ate
#   FOCO_LINK_M = 1000 m (pixel de ~1 km). FRP = MaxFRP diario do pixel.
#   Foco noturno = algoritmo noturno do MODIS (QA bit 2 = 0; passagens
#   ~22h30 Terra e ~01h30 Aqua).
#   Checkpoint com sufixo "_md" (o de VIIRS fica intacto).
#
# Descreve as cicatrizes ANTERIORES (inicio da seca). So descricao:
#   - nenhuma queimada e chamada de "manejo" ou "prescrita";
#   - nada e atribuido as BRIFs.
#
# Tipos (recorte principal do artigo 1):
#   "Inicial pequena e fraca"  : inicio + area < mediana do restante da mesma TI
#                                + (sem foco ou FRP max < p25 do restante)
#   "Inicial pequena e intensa": inicio + pequena, mas FRP alto
#   "Inicial grande"           : inicio, area >= mediana
#   "Restante da seca"         : cicatrizes do restante (referencia)
#   "Fragmento < 1 ha"         : abaixo de MIN_SCAR_HA (fora dos limiares e dos focos)
#
# Descricoes: D1 tempo no dia do 1o foco (percentil ERA5); D2 foco noturno
#   (controlando a area); D3 tipologia por TI x FRRI do artigo 1.
#
# Output:
#   02_cache/cicatrizes_tipologia_<tag>_md.gpkg
#   04_stats/B06_tipologia_resumo.csv, D1_tempo.csv, D2_foco_noturno*.csv, D3_tipologia_por_TI.csv
#   05_figures/D1_tempo.png
# ============================================================

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.spatial import cKDTree

# ============================================================
# 1. PARAMETERS
# ============================================================

FOCOS_NAME   = globals().get("FOCOS_NAME", f"modis_focos_v1_{TAG_YEARS}")
TAG_SEG      = globals().get("TAG_SEG", f"{TAG_YEARS}_md")
FOCO_LINK_M  = 1000      # foco -> cicatriz mais proxima ate 1 km (pixel MODIS)
MIN_SCAR_HA  = 1.0
N_BOOT_D1    = 999
SEED         = 42
SEASONS      = load_seasons()
AMOSTRA      = load_sample_ids()

# ============================================================
# 2. LOAD
# ============================================================

if "scars_raw" not in globals():
    scars_raw = read_table(cache_path(SCARS_NAME, "gpkg"))
if "focos_raw" not in globals():
    focos_raw = read_table(cache_path(FOCOS_NAME, "parquet"))
if "era5_daily" not in globals():
    era5_daily = read_table(cache_path(f"era5_diario_v2_{TAG_YEARS}", "parquet"))

scars = scars_raw[scars_raw["year"].isin(YEARS) & scars_raw["ti_id"].isin(AMOSTRA)].to_crs(AREA_CRS).copy()
scars["area_ha"] = scars.geometry.area / 1e4
scars["scar_id"] = np.arange(len(scars))

focos = focos_raw.copy()
focos["confidence"] = focos["confidence"].astype(str).str.lower().str[0]
focos = focos[focos["confidence"] != "l"].copy()                 # FireMask >= 8
focos["acq_date"] = pd.to_datetime(focos["acq_date"])
focos["det_id"] = np.arange(len(focos))
focos = gpd.GeoDataFrame(focos.drop(columns=["ti_id"], errors="ignore"),
                         geometry=gpd.points_from_xy(focos["longitude"], focos["latitude"]),
                         crs="EPSG:4326").to_crs(AREA_CRS)
era5 = era5_daily.copy()
era5["date"] = pd.to_datetime(era5["date"])
console.print(f"[cyan]{len(scars):,} cicatrizes ({len(AMOSTRA)} TIs da amostra) | {len(focos):,} focos MODIS[/cyan]")

# ============================================================
# 3. FOCOS MODIS -> CICATRIZES
# ============================================================

def attach_focos(scars, focos):
    """Cada foco vai para a cicatriz mais proxima (ate 1 km) do mesmo ano e do
    mesmo mes ou do mes seguinte (o MapBiomas data a cicatriz pela imagem)."""
    focos = focos.copy()
    focos["y"] = focos["acq_date"].dt.year
    focos["m"] = focos["acq_date"].dt.month
    matches = []
    for (yr, mo), s in scars.groupby(["year", "month"]):
        cand = focos[(focos["y"] == yr) & (focos["m"].isin([mo, mo - 1]))]
        if cand.empty:
            continue
        matches.append(gpd.sjoin_nearest(cand, s[["scar_id", "month", "geometry"]],
                                         max_distance=FOCO_LINK_M, distance_col="dist"))
    if not matches:
        m = pd.DataFrame(columns=["det_id", "scar_id", "daynight", "frp", "acq_date"])
    else:
        m = pd.concat(matches, ignore_index=True)
        m["same_month"] = (m["m"] == m["month"]).astype(int)
        m = (m.sort_values(["det_id", "same_month", "dist"], ascending=[True, False, True])
               .drop_duplicates("det_id"))
    agg = m.groupby("scar_id").agg(
        n_day=("daynight", lambda s: int((s == "D").sum())),
        n_night=("daynight", lambda s: int((s == "N").sum())),
        frp_max=("frp", "max"),
        first_date=("acq_date", "min"))
    out = scars.merge(agg, on="scar_id", how="left")
    out[["n_day", "n_night"]] = out[["n_day", "n_night"]].fillna(0).astype(int)
    out["n_det"] = out["n_day"] + out["n_night"]
    return out

FRAG = f"Fragmento < {MIN_SCAR_HA:g} ha"
_big = scars["area_ha"] >= MIN_SCAR_HA
s = gpd.GeoDataFrame(
    pd.concat([attach_focos(scars[_big], focos),
               scars[~_big].assign(n_day=0, n_night=0, n_det=0)], ignore_index=True),
    geometry="geometry", crs=AREA_CRS)
console.print(f"[cyan]Cicatrizes >= {MIN_SCAR_HA:g} ha:[/cyan] {_big.sum():,} "
              f"({scars.loc[_big, 'area_ha'].sum() / scars['area_ha'].sum():.1%} da area) | "
              f"[cyan]fragmentos:[/cyan] {(~_big).sum():,}")

# ============================================================
# 4. TIPOLOGIA
# ============================================================

def typology(s, scheme=SEASON_MAIN):
    s = s.copy()
    ini = s.apply(lambda r: r["month"] in SEASONS[scheme][int(r["ti_id"])][0], axis=1)
    res = s.apply(lambda r: r["month"] in SEASONS[scheme][int(r["ti_id"])][1], axis=1)
    s["periodo"] = np.select([ini, res], ["inicio", "restante"], "outro")
    big = s["area_ha"] >= MIN_SCAR_HA
    rest = s[(s["periodo"] == "restante") & big]
    med = rest.groupby("ti_id")["area_ha"].median()
    frp_p25 = rest.loc[rest["n_det"] > 0, "frp_max"].quantile(0.25)
    pequena = s["area_ha"] < s["ti_id"].map(med).fillna(rest["area_ha"].median())
    fraca = (s["n_det"] == 0) | (s["frp_max"] < frp_p25)
    s["tipo"] = np.select(
        [~big, ini & pequena & fraca, ini & pequena, ini, s["periodo"].eq("restante")],
        [FRAG] + CAT_ORDER, "Fora da seca (nov-abr)")
    lim = {"esquema": scheme, "sensor_focos": "MODIS MOD14A1+MYD14A1 (FireMask >= 8)",
           "area_minima_ha": MIN_SCAR_HA, "mediana_restante_por_TI_ha": med.round(1).to_dict(),
           "frp_p25_restante_MW": round(float(frp_p25), 2)}
    return s, lim

s, lim = typology(s)
save_json(lim, DIRS["logs"] / "B06_limiares_tipologia_md.json")
tipo_tab = s.groupby("tipo").agg(n=("scar_id", "size"), area_ha=("area_ha", "sum"),
                                 area_mediana_ha=("area_ha", "median"),
                                 com_foco=("n_det", lambda n: (n > 0).mean()))
tipo_tab.to_csv(stats_path("B06_tipologia_resumo"))
display(tipo_tab.round(2))

# ============================================================
# 5. D1 - TEMPO NO DIA DA QUEIMA (descricao)
# ============================================================

def weather_percentiles(s, era5):
    d = s[s["first_date"].notna() & s["tipo"].isin(CAT_ORDER)].copy()
    if d.empty:
        return d
    pts = era5.drop_duplicates("point_id")[["point_id", "lon", "lat"]].reset_index(drop=True)
    tree = cKDTree(pts[["lon", "lat"]].to_numpy())
    cent = d.geometry.centroid.to_crs("EPSG:4326")
    _, idx = tree.query(np.c_[cent.x, cent.y])
    d["point_id"] = pts.loc[idx, "point_id"].to_numpy()
    d["date"] = pd.to_datetime(d["first_date"]).dt.normalize()
    e = era5.copy()
    e["mo"] = e["date"].dt.month
    for var in ["vpd_max", "rh_min", "wind"]:
        e[f"P_{var}"] = e.groupby(["point_id", "mo"])[var].rank(method="max", pct=True)
    return d.merge(e[["point_id", "date", "P_vpd_max", "P_rh_min", "P_wind"]], on=["point_id", "date"], how="left")

d1 = weather_percentiles(s, era5)
rng = np.random.default_rng(SEED)
rows = []
for cat in CAT_ORDER:
    x = d1[d1["tipo"] == cat]
    if len(x) < 5:
        continue
    for var in ["P_vpd_max", "P_rh_min", "P_wind"]:
        v = x[var].dropna()
        tis_x = x["ti_id"].unique()
        boots = [x.set_index("ti_id").loc[rng.choice(tis_x, len(tis_x))][var].mean() for _ in range(N_BOOT_D1)]
        rows.append({"tipo": cat, "variavel": var, "n": len(v), "media_percentil": v.mean(),
                     "ic95_inf": np.nanpercentile(boots, 2.5), "ic95_sup": np.nanpercentile(boots, 97.5)})
D1 = pd.DataFrame(rows)
D1.to_csv(stats_path("D1_tempo"), index=False)
console.print("\n[bold]D1 - percentil do VPD no dia do 1o foco MODIS (0,5 = dia tipico do mes)[/bold]")
display(D1[D1["variavel"] == "P_vpd_max"].round(3))

fig, ax = plt.subplots(figsize=(4.8, 4.0))
ax.plot([0, 1], [0, 1], color="black", lw=0.8, ls="--", label="Dia tipico")
for cat in CAT_ORDER:
    v = np.sort(d1.loc[d1["tipo"] == cat, "P_vpd_max"].dropna().to_numpy())
    if len(v) >= 5:
        ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=CAT_COLORS[cat], lw=1.6,
                label=f"{cat} (n={len(v)})")
ax.set_xlabel("Percentil do VPD no dia do 1o foco MODIS\n(0 = dia mais umido do mes; 1 = mais seco)")
ax.set_ylabel("Proporcao acumulada")
ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.savefig(fig_path("D1_tempo"))
plt.show()

# ============================================================
# 6. D2 - FOCO NOTURNO (descricao, controlando a area)
# ============================================================

d2 = s[(s["n_det"] > 0) & s["tipo"].isin(CAT_ORDER)].copy()
d2["noturno"] = (d2["n_night"] > 0).astype(int)
d2["log_area"] = np.log(d2["area_ha"])
d2["cat"] = pd.Categorical(d2["tipo"], categories=[c for c in reversed(CAT_ORDER) if c in set(d2["tipo"])])
m2 = smf.glm("noturno ~ C(cat) + log_area", data=d2, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": d2["ti_id"]})
D2 = pd.DataFrame({"razao_chances": np.exp(m2.params), "ic95_inf": np.exp(m2.conf_int()[0]),
                   "ic95_sup": np.exp(m2.conf_int()[1]), "p": m2.pvalues})
D2.to_csv(stats_path("D2_foco_noturno"))
d2_taxa = d2.groupby("tipo")["noturno"].agg(["size", "mean"])
d2_taxa.to_csv(stats_path("D2_foco_noturno_taxa"))
console.print("\n[bold]D2 - chance de foco noturno MODIS (referencia = restante da seca)[/bold]")
display(d2_taxa.round(3))
display(D2.round(3))

# ============================================================
# 7. D3 - TIPOLOGIA POR TI x FRRI DO ARTIGO 1
# ============================================================

ini_area = s[s["periodo"] == "inicio"].groupby(["ti_id", "tipo"])["area_ha"].sum().unstack(fill_value=0)
D3 = ini_area.div(ini_area.sum(axis=1), axis=0).add_prefix("frac_").reset_index().merge(artigo1, on="ti_id", how="left")
D3.to_csv(stats_path("D3_tipologia_por_TI"), index=False)
display(D3.round(3))
ok = D3.dropna(subset=["frri"])
if len(ok) >= 4 and "frac_Inicial pequena e fraca" in ok:
    r, p = stats.spearmanr(ok["frac_Inicial pequena e fraca"], ok["frri"])
    console.print(f"Spearman (fracao 'pequena e fraca' x FRRI): rho = {r:.2f}, p = {p:.3f}, n = {len(ok)} "
                  "(descritivo; poucas TIs)")
    save_json({"rho": float(r), "p": float(p), "n": int(len(ok))}, DIRS["logs"] / "D3_spearman_frri.json")

# ============================================================
# 8. SAVE
# ============================================================

out = s.drop(columns=["first_date"]).assign(first_date=s["first_date"].astype(str))
write_table(out, cache_path(f"cicatrizes_tipologia_{TAG_SEG}", "gpkg"))
console.print("[bold green]OK BLOCO 07 completo.[/bold green]")


## 08 configuracao_celulas

In [ ]:
# ======================================================================
# BLOCO 08 - Quantidade x configuracao das cicatrizes iniciais, por celula
# Depende dos blocos 00, 03 (rasters), 04 (ERA5) e 06 (estacoes).
# ======================================================================
#
# Objetivo especifico 2. Para cada celula (principal 4980 m), ano e recorte,
# sobre o COMBUSTIVEL (cobertura vegetada; fora: nao vegetado, mineracao,
# aquicultura e agua):
#   QUANTIDADE   q_inicio (fracao queimada no inicio), q_inicio_viz (8 vizinhas),
#                q_ant1/q_ant2 (t-1, t-2)
#   CONFIGURACAO do combustivel restante no fim do inicio da seca:
#                conf_split  = log(m_eff depois / antes) (Jaeger 2000)
#                conf_lpi    = maior mancha / area queimavel
#                conf_borda  = borda cicatriz-combustivel (m/ha)
#                conf_npatch = manchas de combustivel por km2
#                (a versao independente da quantidade e o SES do bloco 09)
#   DESFECHO     y_resto = fracao queimada no restante; y_cont = maior mancha
#                queimada no restante / area queimavel
#   CONTROLES    nativa, pastagem, agricultura, declividade, barreira, rios,
#                VPD do restante e chuva da seca (ERA5), centroide
# Manchas cortadas na borda da celula contam so pela parte dentro (cutting-out).
#
# Output: 02_cache/celulas_<esquema>_<tamanho>_<comb>_<tag>.parquet
#         04_stats/B07_correlacao_quantidade_configuracao.csv
# ======================================================================

check_version()

from scipy import ndimage
from scipy.spatial import cKDTree
from pyproj import Transformer

MIN_QUEIMAVEL = 0.25
CONN8 = np.ones((3, 3), int)
SEASONS = load_seasons()
AMOSTRA = load_sample_ids()

if "era5_daily" not in globals():
    era5_daily = read_table(cache_path(f"era5_diario_v2_{TAG_YEARS}", "parquet"))
era5_daily["date"] = pd.to_datetime(era5_daily["date"])
era5_m = (era5_daily.assign(year=era5_daily["date"].dt.year, mes=era5_daily["date"].dt.month)
          .groupby(["point_id", "year", "mes"])
          .agg(vpd=("vpd_max", "mean"), chuva=("precip_mm", "sum")).reset_index())
era5_pts = era5_daily.drop_duplicates("point_id")[["point_id", "lon", "lat"]].reset_index(drop=True)
era5_tree = cKDTree(era5_pts[["lon", "lat"]].to_numpy())
era5_m_idx = era5_m.set_index(["point_id", "year", "mes"])
_to_ll = Transformer.from_crs(AREA_CRS, "EPSG:4326", always_xy=True)

def weather_for(xs, ys, year, ini, res):
    lon, lat = _to_ll.transform(np.asarray(xs), np.asarray(ys))
    _, idx = era5_tree.query(np.c_[lon, lat])
    pid = era5_pts.loc[idx, "point_id"].to_numpy()
    vpd, chuva = [], []
    for p in pid:
        try:
            g = era5_m_idx.loc[(p, year)]
            vpd.append(g.loc[g.index.isin(res), "vpd"].mean())
            chuva.append(g.loc[g.index.isin(list(ini) + list(res)), "chuva"].sum())
        except KeyError:
            vpd.append(np.nan); chuva.append(np.nan)
    return np.array(vpd), np.array(chuva), pid

def _edge(scar, fuel):
    nb = np.zeros_like(fuel)
    nb[1:, :] |= fuel[:-1, :]; nb[:-1, :] |= fuel[1:, :]
    nb[:, 1:] |= fuel[:, :-1]; nb[:, :-1] |= fuel[:, 1:]
    return float((scar & nb).sum()) * PIX_M

def cells_ti_year(ti, year, scheme, cell_m, fuel_mode=FUEL_MAIN):
    st = static_of(ti)
    yr, l1, l2 = read_raster(ti, year), read_raster(ti, year - 1), read_raster(ti, year - 2)
    inside = st["ti"] == ti
    queim = inside & fuel_from_cover(yr["cover"], fuel_mode) & (st["barrier"] == 0)
    sm = season_masks(yr["month"], ti, scheme, SEASONS)
    E = sm["inicio"] & queim
    L = sm["resto"] & queim
    fuel0 = queim & ~sm["antes"]
    fuel1 = fuel0 & ~E
    P1 = (l1["month"] > 0) & queim
    P2 = (l2["month"] > 0) & queim
    slope = st["slope10"] / 10.0
    cp = int(round(cell_m / PIX_M))
    H, W = queim.shape
    tr = st["_transform"]
    rows = []
    for i in range(H // cp):
        for j in range(W // cp):
            sl = (slice(i * cp, (i + 1) * cp), slice(j * cp, (j + 1) * cp))
            A = int(queim[sl].sum())
            if A < MIN_QUEIMAVEL * cp * cp:
                continue
            e, l, f0, f1 = E[sl], L[sl], fuel0[sl], fuel1[sl]
            s0, _, _ = _meff(f0)
            s1, mx1, n1 = _meff(f1)
            _, mxL, _ = _meff(l)
            _, _, ne = _meff(e)
            ins = inside[sl]
            rows.append({
                "ti_id": ti, "year": year, "row": i, "col": j,
                "x": tr.c + (j + 0.5) * cp * PIX_M, "y": tr.f - (i + 0.5) * cp * PIX_M,
                "area_queimavel_ha": A * 0.09,
                "q_inicio": e.sum() / A, "q_ant1": P1[sl].sum() / A, "q_ant2": P2[sl].sum() / A,
                "conf_split": np.log(s1 / s0) if (s0 > 0 and s1 > 0) else np.nan,
                "conf_lpi": mx1 / A,
                "conf_borda": _edge(e, f1) / (A * 0.09),
                "conf_npatch": n1 / (A * 0.09 / 100),
                "n_cicatrizes": ne,
                "y_resto": l.sum() / A,
                "y_cont": mxL / A,
                "nativa": float((yr["cover"][sl][ins] == 1).mean()) if ins.any() else np.nan,
                "pastagem": float((yr["cover"][sl][ins] == 2).mean()) if ins.any() else np.nan,
                "agricultura": float((yr["cover"][sl][ins] == 3).mean()) if ins.any() else np.nan,
                "declividade": float(slope[sl][ins].mean()) if ins.any() else np.nan,
                "barreira": float(st["barrier"][sl][ins].mean()) if ins.any() else np.nan,
                "rios": float(st["river"][sl][ins].mean()) if ins.any() else np.nan,
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    ini, res = SEASONS[scheme][int(ti)]
    df["vpd_resto"], df["chuva_seca"], df["era5_point"] = weather_for(df["x"], df["y"], year, ini, res)
    key = df.set_index(["row", "col"])["q_inicio"]
    viz = []
    for r, c in zip(df["row"], df["col"]):
        vals = [key.get((r + dr, c + dc)) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
        vals = [v for v in vals if v is not None and not np.isnan(v)]
        viz.append(np.mean(vals) if vals else np.nan)
    df["q_inicio_viz"] = viz
    return df

CONF_VARS = ["conf_split", "conf_lpi", "conf_borda", "conf_npatch"]

def build_cells(scheme=SEASON_MAIN, cell_m=CELL_M, force=False, fuel_mode=FUEL_MAIN) -> pd.DataFrame:
    p = cache_path(f"celulas_{scheme}_{cell_m}_{FUEL_TAG[fuel_mode]}_{TAG_YEARS}", "parquet")
    if p.exists() and not (force or FORCE_REPROCESS):
        df = pd.read_parquet(p)
        if df.get("res_metodo", pd.Series([""])).iloc[0] != RES_METODO:
            df = residualize_conf(df, CONF_VARS)
            df.to_parquet(p, index=False)
        return df
    parts = []
    for ti in tqdm(AMOSTRA, desc=f"Celulas {scheme} {cell_m} m"):
        for y in YEARS:
            parts.append(cells_ti_year(ti, y, scheme, cell_m, fuel_mode))
    df = pd.concat([d for d in parts if len(d)], ignore_index=True)
    df["cell_id"] = df["ti_id"].astype(str) + "_" + df["row"].astype(str) + "_" + df["col"].astype(str)
    df = residualize_conf(df, CONF_VARS)
    df["esquema"], df["cell_m"], df["combustivel"] = scheme, cell_m, fuel_mode
    df.to_parquet(p, index=False)
    return df

cells_by_fuel = {}
for fm in FUEL_RUNS:
    cells_by_fuel[fm] = build_cells(fuel_mode=fm)
    c_ = cells_by_fuel[fm]
    console.print(f"[green]OK [{fm}] {c_['cell_id'].nunique():,} celulas x {c_['year'].nunique()} anos = {len(c_):,} linhas[/green]")
cells = cells_by_fuel[FUEL_MAIN]

cols_q = ["q_inicio", "q_ant1"] + CONF_VARS + [f"{v}_res" for v in CONF_VARS]
corr = pd.concat([c_[cols_q].corr(method="spearman").loc[["q_inicio"], CONF_VARS + [f"{v}_res" for v in CONF_VARS]]
                  .assign(combustivel=fm) for fm, c_ in cells_by_fuel.items()]).set_index("combustivel")
corr.to_csv(stats_path("B07_correlacao_quantidade_configuracao"))
console.print("[bold]Spearman: quantidade (q_inicio) x configuracao bruta e residual[/bold]")
display(corr.round(2))
console.print("[bold green]OK BLOCO 08 completo.[/bold green]")


## 09 configuracao_modelo_nulo

In [ ]:
# ======================================================================
# BLOCO 09 - Configuracao das cicatrizes contra um modelo nulo (mesma quantidade, posicao ao acaso)
# Depende dos blocos 00, 03 (rasters), 06 (estacoes) e 08 (celulas).
# ======================================================================

check_version()

# ============================================================
# BLOCK 7b - Null-model configuration (standardized effect size, SES)
#
# Problema do bloco 08: na celula, m_eff e densidade de borda sao quase a propria
# quantidade (Spearman -1,00 e 0,97 com q_inicio), e o residuo pela curva
# depende da forma escolhida para a curva.
#
# Solucao (padrao em ecologia da paisagem): para CADA celula-ano, as
# cicatrizes do inicio da seca sao REPOSICIONADAS AO ACASO dentro do
# combustivel da propria celula, N_NULL vezes:
#   - cada cicatriz (mancha conectada, vizinhanca de 8) mantem a FORMA e a AREA;
#   - so pode cair sobre combustivel disponivel (fuel0) e sem sobrepor outra;
#   - cicatriz que nao cabe em lugar nenhum fica onde estava (contada em n_fixas).
# Quantidade, numero e tamanho das cicatrizes ficam IGUAIS ao observado; so a
# POSICAO muda. Para cada metrica de configuracao do combustivel restante:
#   SES = (observado - media do nulo) / desvio do nulo
#   p_nulo = fracao das replicas <= observado
# Leitura (conf_split = log m_eff depois/antes):
#   SES < 0 -> as cicatrizes fragmentam o combustivel MAIS do que o acaso
#              (ficaram espalhadas / atravessadas no combustivel)
#   SES > 0 -> fragmentam MENOS (agrupadas, encostadas em barreiras ou na borda)
# Assim a configuracao e independente da quantidade POR CONSTRUCAO, nao por
# uma regressao. E isto que testa a hipotese do professor: "mesma proporcao
# queimada, organizacao espacial diferente".
#
# Custo: uma replica = reposicionar as cicatrizes e rotular o combustivel.
# Roda em paralelo por TI-ano e guarda checkpoint por TI (se cair, continua).
#
# Output:
#   02_cache/nulo/nulo_<esquema>_<cel>_<comb>_ti<id>_<tag>.parquet (por TI)
#   02_cache/celulas_<esquema>_<cel>_<comb>_<tag>_nulo.parquet  (celulas do bloco 08 + SES)
#   04_stats/B07b_ses_resumo.csv, B07b_correlacao_ses_quantidade.csv
#   05_figures/B07b_ses_mapa_exemplo.{tif,png,pdf}, B07b_ses_x_quantidade.{tif,png,pdf} (600 dpi)
# ============================================================

from scipy import ndimage
from joblib import Parallel, delayed

# ============================================================
# 1. PARAMETERS
# ============================================================

N_NULL       = 29          # replicas por celula-ano (p minimo = 1/30)
TRIES        = 40          # tentativas de posicao por cicatriz
MIN_SCAR_PX  = 2           # manchas de 1 pixel ficam fixas (ruido do MapBiomas)
N_JOBS       = -1
SEED         = 2026
NULO_DIR     = DIRS["cache"] / "nulo"
NULO_DIR.mkdir(exist_ok=True)
CONN8 = np.ones((3, 3), int)
SEASONS = load_seasons()
AMOSTRA = load_sample_ids()
FUEL_RUNS_NULO = [FUEL_MAIN]          # acrescente "nativa+pastagem" se quiser o nulo nas duas
SES_VARS = ["conf_split", "conf_lpi", "conf_npatch"]
SES_CAP  = 5.0             # |SES| limitado (nulo quase sem variancia gera razoes enormes)

# ============================================================
# 2. METRICAS E REPOSICIONAMENTO (funcoes puras, testaveis)
# ============================================================

def config_metrics(f0, e, s0, A):
    """Configuracao do combustivel restante (f0 sem as cicatrizes e) numa celula."""
    s1, mx1, n1 = _meff(f0 & ~e)
    split = np.log(s1 / s0) if (s0 > 0 and s1 > 0) else np.nan
    return split, mx1 / A, n1 / (A * 0.09 / 100)

def random_layout(e, f0, rng, tries=TRIES, min_px=MIN_SCAR_PX):
    """Mesmas cicatrizes (forma e area), posicoes ao acaso dentro de f0, sem sobrepor.
    Devolve (mascara nova, n de cicatrizes que nao couberam e ficaram no lugar)."""
    H, W = e.shape
    lab, n = ndimage.label(e, structure=CONN8)
    if n == 0:
        return e.copy(), 0
    objs = ndimage.find_objects(lab)
    sizes = np.bincount(lab.ravel())[1:]
    order = np.argsort(-sizes)                      # maiores primeiro (mais dificeis de encaixar)
    out = np.zeros_like(e)
    fixed_mask = np.zeros_like(e)
    n_fix = 0
    for k in order:
        sl = objs[k]
        shp = lab[sl] == (k + 1)
        h, w = shp.shape
        placed = False
        if sizes[k] >= min_px and h <= H and w <= W:
            r0s = rng.integers(0, H - h + 1, tries)
            c0s = rng.integers(0, W - w + 1, tries)
            for r0, c0 in zip(r0s, c0s):
                tgt = (slice(r0, r0 + h), slice(c0, c0 + w))
                if f0[tgt][shp].all() and not out[tgt][shp].any():
                    out[tgt][shp] = True
                    placed = True
                    break
        if not placed:                               # fica no lugar observado
            fixed_mask[sl] |= shp
            n_fix += 1
    # cicatrizes fixas entram por ultimo; se colidirem com uma reposicionada, a reposicionada perde
    out |= fixed_mask
    return out, n_fix

def null_cell(e, f0, n_null, rng):
    """SES de cada metrica para uma celula. Devolve dict."""
    A = int(f0.sum())
    s0, _, _ = _meff(f0)
    obs = config_metrics(f0, e, s0, A)
    sims, fixes = [], []
    for _ in range(n_null):
        en, nf = random_layout(e, f0, rng)
        sims.append(config_metrics(f0, en, s0, A))
        fixes.append(nf)
    sims = np.array(sims, float)
    out = {"n_cicatrizes_cel": int(ndimage.label(e, structure=CONN8)[1]),
           "n_fixas_media": float(np.mean(fixes))}
    for j, v in enumerate(SES_VARS):
        mu, sd = np.nanmean(sims[:, j]), np.nanstd(sims[:, j], ddof=1)
        out[f"{v}_obs"] = obs[j]
        out[f"{v}_nulo_media"] = mu
        out[f"{v}_nulo_dp"] = sd
        if not np.isfinite(obs[j]) or not np.isfinite(mu):
            ses = np.nan
        elif not (sd > 1e-9):                       # nulo sem variancia: nenhuma posicao muda a metrica
            ses = 0.0 if abs(obs[j] - mu) < 1e-9 else float(np.sign(obs[j] - mu) * SES_CAP)
        else:
            ses = float(np.clip((obs[j] - mu) / sd, -SES_CAP, SES_CAP))
        out[f"{v}_ses"] = ses
        # p de posto medio: empates contam meio (evita p = 1 quando o nulo e igual ao observado)
        out[f"{v}_p_nulo"] = (float(np.mean(sims[:, j] < obs[j]) + 0.5 * np.mean(sims[:, j] == obs[j]))
                              if np.isfinite(obs[j]) else np.nan)
    return out

# ============================================================
# 3. UMA TI-ANO (mesmas mascaras do bloco 08)
# ============================================================

def null_ti_year(ti, year, scheme, cell_m, fuel_mode, seed):
    st = static_of(ti)
    yr = read_raster(ti, year)
    inside = st["ti"] == ti
    queim = inside & fuel_from_cover(yr["cover"], fuel_mode) & (st["barrier"] == 0)
    sm = season_masks(yr["month"], ti, scheme, SEASONS)
    E = sm["inicio"] & queim
    fuel0 = queim & ~sm["antes"]
    cp = int(round(cell_m / PIX_M))
    H, W = queim.shape
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(H // cp):
        for j in range(W // cp):
            sl = (slice(i * cp, (i + 1) * cp), slice(j * cp, (j + 1) * cp))
            q = queim[sl]
            if q.sum() < 0.25 * cp * cp:              # MIN_QUEIMAVEL do bloco 08
                continue
            e, f0 = E[sl], fuel0[sl]
            if not e.any() or f0.sum() == 0:
                continue
            r = null_cell(e, f0, N_NULL, rng)
            r.update({"ti_id": ti, "year": year, "row": i, "col": j})
            rows.append(r)
    return rows

def run_null(scheme=SEASON_MAIN, cell_m=CELL_M, fuel_mode=FUEL_MAIN) -> pd.DataFrame:
    parts = []
    for ti in AMOSTRA:
        p = NULO_DIR / f"nulo_{scheme}_{cell_m}_{FUEL_TAG[fuel_mode]}_ti{ti}_{TAG_YEARS}.parquet"
        if p.exists() and not FORCE_REPROCESS:
            parts.append(pd.read_parquet(p)); continue
        static_of(ti)                                  # carrega uma vez antes de paralelizar
        res = Parallel(n_jobs=N_JOBS, prefer="threads")(
            delayed(null_ti_year)(ti, y, scheme, cell_m, fuel_mode, SEED + 1000 * ti + y) for y in YEARS)
        d = pd.DataFrame([r for rr in res for r in rr])
        d.to_parquet(p, index=False)
        console.print(f"  [dim]{tis.set_index('ti_id').loc[ti, 'ti_name']}: {len(d):,} celulas-ano com cicatriz[/dim]")
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

# ============================================================
# 4. RUN + JUNTA COM AS CELULAS DO bloco 08
# ============================================================

cells_null = {}
for fm in FUEL_RUNS_NULO:
    console.print(f"[cyan]Modelo nulo - {fm}, {N_NULL} replicas por celula-ano[/cyan]")
    nd = run_null(fuel_mode=fm)
    c = pd.read_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[fm]}_{TAG_YEARS}", "parquet"))
    c = c.merge(nd, on=["ti_id", "year", "row", "col"], how="left")
    # celulas sem cicatriz: configuracao indefinida -> SES = 0 (igual ao nulo) + indicador
    c["sem_cicatriz"] = c["conf_split_ses"].isna() & (c["q_inicio"] == 0)
    for v in SES_VARS:
        c[f"{v}_ses_0"] = c[f"{v}_ses"].fillna(0.0)
    c.to_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[fm]}_{TAG_YEARS}_nulo", "parquet"), index=False)
    cells_null[fm] = c

cn = cells_null[FUEL_MAIN]
sub = cn[cn["q_inicio"] > 0].dropna(subset=["conf_split_ses"])
corr = sub[["q_inicio", "conf_split", "conf_lpi", "conf_npatch"] + [f"{v}_ses" for v in SES_VARS]] \
          .corr(method="spearman").loc[["q_inicio"]]
corr.to_csv(stats_path("B07b_correlacao_ses_quantidade"))
console.print("[bold]Spearman com a quantidade: metricas brutas x SES do modelo nulo[/bold]")
display(corr.round(2))
resumo = pd.DataFrame({
    "celulas_ano_com_cicatriz": [len(sub)],
    "frac_mais_fragmentado_que_acaso (p_nulo<=0.05)": [(sub["conf_split_p_nulo"] <= 0.05).mean()],
    "frac_menos_fragmentado_que_acaso (p_nulo>=0.95)": [(sub["conf_split_p_nulo"] >= 0.95).mean()],
    "ses_split_mediana": [sub["conf_split_ses"].median()],
    "cicatrizes_fixas_media": [sub["n_fixas_media"].mean()]})
resumo.to_csv(stats_path("B07b_ses_resumo"), index=False)
display(resumo.round(3))

# ============================================================
# 5. FIGURAS (espaciais, 600 dpi)
# ============================================================

def _save3(fig, nome):
    for ext, kw in [("tif", {"pil_kwargs": {"compression": "tiff_lzw"}}), ("png", {}), ("pdf", {})]:
        fig.savefig(fig_path(nome, ext), dpi=600, facecolor="white", **kw)
    plt.show(); plt.close(fig)

# (a) exemplo: uma celula observada x uma replica nula, mesma quantidade
try:
    k = sub.assign(a=lambda d: (d["q_inicio"] - 0.25).abs()).sort_values(["a", "n_cicatrizes_cel"],
                                                                         ascending=[True, False]).iloc[0]
    ti, y, i, j = int(k["ti_id"]), int(k["year"]), int(k["row"]), int(k["col"])
    st, yr = static_of(ti), read_raster(ti, y)
    cp = int(round(CELL_M / PIX_M)); sl = (slice(i * cp, (i + 1) * cp), slice(j * cp, (j + 1) * cp))
    inside = st["ti"] == ti
    queim = inside & fuel_from_cover(yr["cover"], FUEL_MAIN) & (st["barrier"] == 0)
    sm = season_masks(yr["month"], ti, SEASON_MAIN, SEASONS)
    e, f0 = (sm["inicio"] & queim)[sl], (queim & ~sm["antes"])[sl]
    en, _ = random_layout(e, f0, np.random.default_rng(1))
    from matplotlib.colors import ListedColormap
    cm3 = ListedColormap(["#f4f3ef", "#a9cf8f", "#1f5fc8"])
    fig, axs = plt.subplots(1, 2, figsize=(14 / 2.54, 7.4 / 2.54), layout="constrained")
    for ax, ee_, tt in [(axs[0], e, f"Observed (SES = {k['conf_split_ses']:.2f})"),
                        (axs[1], en, "One null layout (same scars, random positions)")]:
        im = np.where(ee_, 2, np.where(f0, 1, 0))
        ax.imshow(im, cmap=cm3, vmin=0, vmax=2, interpolation="nearest")
        ax.add_patch(plt.Rectangle((cp * 0.06, cp * 0.92), 1000 / PIX_M, cp * 0.02, color="black"))
        ax.text(cp * 0.06 + 500 / PIX_M, cp * 0.89, "1 km", ha="center", va="bottom", fontsize=7)
        ax.set_title(tt, fontsize=8); ax.set_axis_off()
    fig.suptitle(f"{tis.set_index('ti_id').loc[ti, 'ti_name']} {y}, {CELL_M / 1000:.2f}-km cell: "
                 f"early scars (blue) {100 * k['q_inicio']:.0f}% of burnable fuel (green)", fontsize=8)
    _save3(fig, "B07b_ses_mapa_exemplo")
except Exception as ex:
    console.print(f"[dim]figura de exemplo pulada: {str(ex)[:100]}[/dim]")

fig, axs = plt.subplots(1, 2, figsize=(14 / 2.54, 6.2 / 2.54), layout="constrained")
axs[0].scatter(sub["q_inicio"], sub["conf_split"], s=2, alpha=0.25, color="#9a9992", rasterized=True)
axs[0].set_xlabel("Early burned fraction"); axs[0].set_ylabel("log(m_eff after / before)")
axs[0].set_title(f"raw: Spearman {corr.loc['q_inicio', 'conf_split']:.2f}", fontsize=8, loc="left")
axs[1].scatter(sub["q_inicio"], sub["conf_split_ses"], s=2, alpha=0.25, color="#1f5fc8", rasterized=True)
axs[1].axhline(0, color="black", lw=0.6)
axs[1].set_xlabel("Early burned fraction"); axs[1].set_ylabel("SES vs null model")
axs[1].set_title(f"null-model SES: Spearman {corr.loc['q_inicio', 'conf_split_ses']:.2f}", fontsize=8, loc="left")
_save3(fig, "B07b_ses_x_quantidade")

console.print("[bold green]OK BLOCO 09 completo: SES gravado nas celulas (*_nulo.parquet).[/bold green]")


## 10 encontros_bordas

In [ ]:
# ======================================================================
# BLOCO 10 - Sequencia das queimadas e encontros por segmento de borda (Landsat + MODIS)
# Depende dos blocos 00, 02, 03 (rasters), 04 (ERA5), 05 (MODIS) e 06 (estacoes).
# ======================================================================

check_version()
# ======================================================================

# ============================================================
# BLOCK 8 - Burn sequence + encounters at edge segments
#
# MUDANCA (2026-10-17): os focos ativos agora sao MODIS C6.1 (MOD14A1 +
#   MYD14A1, 1 km, FireMask >= 8), como no artigo 1. Raios ajustados a 1 km:
#     chegada : foco a ate FOCO_ARR_M (1000 m) da borda, do lado de fora
#     entrada : centro do pixel MODIS a >= FOCO_IN_M (1000 m) dentro da cicatriz
#   As colunas internas mantem o prefixo "viirs_" (viirs_chegada, viirs_dentro,
#   ordem "landsat+viirs"...) para nao quebrar bloco 10/bloco 12/bloco 16: leia "foco ativo".
#   Os checkpoints ganham o sufixo "_md" (TAG_SEG): os de VIIRS ficam intactos.
#
# Objetivo 1 - SEQUENCIA e QUALIDADE: cada cicatriz do inicio da seca
#   (MapBiomas) tem a ordem conferida antes de virar "anterior":
#     Landsat : mes da 1a queda de NBR (>= NBR_DROP) nos pixels da cicatriz
#     MODIS   : 1o foco da cicatriz antes do 1o foco que chega a borda = confirma
#   ordem_q = landsat+viirs | landsat | viirs | so_mapbiomas | contraditoria
#
# Objetivo 3 - ENCONTROS (unidade = SEGMENTO DE BORDA):
#   segmento  = trecho da borda de uma cicatriz anterior voltado para
#               vegetacao ainda nao queimada, cortado a cada SEG_LEN m
#   CHEGADA   = o fogo do restante queimou >= ARRIVAL da faixa externa
#               (MapBiomas) OU ha foco MODIS do restante a <= 1 km da borda,
#               do lado de fora. Sem chegada -> nao conta.
#   continuidade : nova queda de NBR em >= CONT_MIN da faixa interna bem
#                  observada, ou foco MODIS do restante >= 1 km dentro
#   interrupcao  : faixa interna bem observada e sem entrada
#   indeterminado: observacao insuficiente ou ordem nao verificavel
#
# NOVO - estado da faixa interna (para tratar a deteccao assimetrica do NBR):
#   nbr_pre_int   : NBR medio da faixa interna no fim do inicio da seca
#   drop_int_med  : queda mediana de NBR na faixa interna (pre - min do restante)
#   (o bloco 11 usa essas colunas; o bloco 12 pode usar nbr_pre_int como covariavel)
#
# REFERENCIA: pseudo-cicatrizes de mesma forma, deslocadas 2-10 km para
#   vegetacao nao queimada da mesma TI; mesma regra (Landsat + MODIS).
# ============================================================

from scipy import ndimage
from scipy.spatial import cKDTree
import rasterio

# ============================================================
# 1. PARAMETERS (o bloco 14 varia SEG_LEN, STRIP e ARRIVAL)
# ============================================================

FOCOS_NAME = globals().get("FOCOS_NAME", f"modis_focos_v1_{TAG_YEARS}")
TAG_SEG    = globals().get("TAG_SEG", f"{TAG_YEARS}_md")
ENC_DEFAULT = dict(
    seg_len=300,       # comprimento do segmento (m)
    strip=90,          # largura das faixas externa e interna (m)
    arrival=0.3,       # fracao da faixa externa queimada para haver chegada
    cont_min=0.2,      # fracao da faixa interna com nova queda de NBR = continuidade
    obs_min=0.5,       # fracao da faixa interna com Landsat antes e durante o restante
    nbr_drop=10,       # queda de NBR (x100) que conta como queimada
    min_seg_px=5,      # segmento com >= 5 pixels de borda (150 m)
    win=990,           # janela das covariaveis de quantidade/configuracao (m)
    viirs_arr=1000,    # (MODIS) foco a ate 1 km da borda, do lado de fora = chegada
    viirs_in=1000,     # (MODIS) centro do pixel >= 1 km dentro da cicatriz = entrada
    ref_min=2000, ref_max=10000, ref_tries=25,
)
SEASONS = load_seasons()
AMOSTRA = load_sample_ids()
CROSS = ndimage.generate_binary_structure(2, 1)
CONN8 = np.ones((3, 3), int)
SEQ_DIR = DIRS["exports"] / "sequencia_md"
SEQ_DIR.mkdir(parents=True, exist_ok=True)
rng_ref = np.random.default_rng(2024)

# --- Focos MODIS e ERA5 ---------------------------------------------------
if "focos_raw" not in globals():
    focos_raw = read_table(cache_path(FOCOS_NAME, "parquet"))
vi = focos_raw.copy()
vi = vi[vi["confidence"].astype(str).str.lower().str[0] != "l"].copy()   # FireMask >= 8 (artigo 1)
vi["acq_date"] = pd.to_datetime(vi["acq_date"])
_g = gpd.GeoSeries(gpd.points_from_xy(vi["longitude"], vi["latitude"]), crs="EPSG:4326").to_crs(AREA_CRS)
vi["x"], vi["y"] = _g.x.to_numpy(), _g.y.to_numpy()
vi["year"], vi["mes"] = vi["acq_date"].dt.year, vi["acq_date"].dt.month
console.print(f"[cyan]Focos MODIS (FireMask >= 8):[/cyan] {len(vi):,}")

if "era5_daily" not in globals():
    era5_daily = read_table(cache_path(f"era5_diario_v2_{TAG_YEARS}", "parquet"))
e5 = era5_daily.copy()
e5["date"] = pd.to_datetime(e5["date"])
e5["mo"] = e5["date"].dt.month
for v in ["vpd_max", "rh_min", "wind"]:
    e5[f"P_{v}"] = e5.groupby(["point_id", "mo"])[v].rank(method="max", pct=True)
e5_idx = e5.set_index(["point_id", "date"])[["vpd_max", "wind", "rh_min", "P_vpd_max", "P_wind", "P_rh_min"]]
e5_pts = e5.drop_duplicates("point_id")[["point_id", "lon", "lat"]].reset_index(drop=True)
e5_tree = cKDTree(e5_pts[["lon", "lat"]].to_numpy())
from pyproj import Transformer
_to_ll8 = Transformer.from_crs(AREA_CRS, "EPSG:4326", always_xy=True)

# ============================================================
# 2. LANDSAT: estado no fim do inicio, nova queda no restante, mes da queda
# ============================================================

def landsat_layers(yr, ini, res, nbr_drop):
    # sem dado = -128 (arquivos de 8 bits) ou -32768 (16 bits); NBR valido vai de -100 a 100
    nbr = {m: np.where(yr[f"nbr{m:02d}"] <= -101, np.nan, yr[f"nbr{m:02d}"].astype(float)) for m in LS_MONTHS}
    pre = np.full(next(iter(nbr.values())).shape, np.nan)
    for m in [m for m in LS_MONTHS if m <= max(ini)]:
        pre = np.where(~np.isnan(nbr[m]), nbr[m], pre)
    rm = [m for m in res if m in nbr]
    stack = np.stack([nbr[m] for m in rm]) if rm else np.full((1,) + pre.shape, np.nan)
    with np.errstate(all="ignore"):
        min_res = np.nanmin(stack, axis=0)
    obs_ok = ~np.isnan(pre) & ~np.isnan(min_res)
    drop = np.where(obs_ok, pre - min_res, np.nan)
    nova = obs_ok & (drop >= nbr_drop)
    ls_mes = np.zeros(pre.shape, np.int16)
    prev = np.full(pre.shape, np.nan)
    for m in LS_MONTHS:
        cur = nbr[m]
        hit = (ls_mes == 0) & ~np.isnan(cur) & ~np.isnan(prev) & ((prev - cur) >= nbr_drop)
        ls_mes[hit] = m
        prev = np.where(~np.isnan(cur), cur, prev)
    return obs_ok, nova, ls_mes, pre, drop

# ============================================================
# 3. SEGMENTOS DE UMA "CICATRIZ" (real ou pseudo)
# ============================================================

def _px(xs, ys, tr, shape):
    c = np.floor((np.asarray(xs) - tr.c) / PIX_M).astype(int)
    r = np.floor((tr.f - np.asarray(ys)) / PIX_M).astype(int)
    ok = (r >= 0) & (r < shape[0]) & (c >= 0) & (c < shape[1])
    return r, c, ok

def segments_for(S, labS, ctx, P, kind):
    """S: mascara das cicatrizes (reais ou pseudo); labS: rotulos; ctx: camadas do TI-ano."""
    fuel1, L, obs_ok, nova, tr = ctx["fuel1"], ctx["L"], ctx["obs_ok"], ctx["nova"], ctx["tr"]
    H, W = S.shape
    facing = fuel1 & ~S
    B = S & ndimage.binary_dilation(facing, structure=CROSS)
    if not B.any():
        return pd.DataFrame()
    tpx = max(1, int(round(P["seg_len"] / PIX_M)))
    rr, cc = np.nonzero(B)
    ntc = W // tpx + 1
    key = labS[rr, cc].astype(np.int64) * (ntc * (H // tpx + 1)) + (rr // tpx) * ntc + (cc // tpx)
    uk, seg_of_b, nb = np.unique(key, return_inverse=True, return_counts=True)
    keep = nb[seg_of_b] >= P["min_seg_px"]
    rr, cc, seg_of_b = rr[keep], cc[keep], seg_of_b[keep]
    if len(rr) == 0:
        return pd.DataFrame()
    _, seg_of_b = np.unique(seg_of_b, return_inverse=True)
    nseg = seg_of_b.max() + 1
    segmap = np.full((H, W), -1, np.int64)
    segmap[rr, cc] = seg_of_b
    Bk = segmap >= 0
    dist, inds = ndimage.distance_transform_edt(~Bk, return_indices=True)
    near = segmap[inds[0], inds[1]]
    scar_near = labS[inds[0], inds[1]]
    wpx = P["strip"] / PIX_M
    # faixa externa: combustivel fora da cicatriz; pixel de agricultura nao conta
    ext = (dist <= wpx) & ~S & fuel1 & (ctx["cover"] != 3)
    intr = (dist <= wpx) & S & (labS == scar_near)
    deep = (dist <= 10) & S & (labS == scar_near)                 # ate 300 m para dentro
    d_in = ndimage.distance_transform_edt(S)

    def bc(mask, w=None):
        ids = near[mask]
        return np.bincount(ids, weights=None if w is None else w[mask], minlength=nseg)

    io = intr & obs_ok
    seg = pd.DataFrame({
        "seg": np.arange(nseg),
        "scar": np.bincount(seg_of_b, weights=labS[rr, cc], minlength=nseg) / np.bincount(seg_of_b, minlength=nseg),
        "comprimento_m": np.bincount(seg_of_b, minlength=nseg) * PIX_M,
        "r": np.bincount(seg_of_b, weights=rr, minlength=nseg) / np.bincount(seg_of_b, minlength=nseg),
        "c": np.bincount(seg_of_b, weights=cc, minlength=nseg) / np.bincount(seg_of_b, minlength=nseg),
        "n_ext": bc(ext), "n_ext_q": bc(ext & L),
        "n_int": bc(intr), "n_int_obs": bc(io), "n_int_nova": bc(io & nova),
        "n_int_L": bc(intr & L),
        "decl_ext": bc(ext, ctx["slope"]),
        "nbr_pre_int": bc(io, np.nan_to_num(ctx["pre"])),
        "drop_int_soma": bc(io, np.nan_to_num(ctx["drop"])),
    })
    seg["scar"] = seg["scar"].round().astype(int)
    seg["decl_ext"] = seg["decl_ext"] / seg["n_ext"].replace(0, np.nan)
    seg["nbr_pre_int"] = seg["nbr_pre_int"] / seg["n_int_obs"].replace(0, np.nan) / 100.0
    seg["drop_int_med"] = seg.pop("drop_int_soma") / seg["n_int_obs"].replace(0, np.nan) / 100.0
    wmax = np.zeros(nseg)
    np.maximum.at(wmax, near[deep], d_in[deep])
    seg["largura_local_m"] = 2 * wmax * PIX_M
    m_ext = ctx["month"][ext & L]; s_ext = near[ext & L]; f_ext = ctx["fire"][ext & L]
    if len(s_ext):
        tmp = pd.DataFrame({"seg": s_ext, "mes": m_ext, "fire": f_ext})
        moda = tmp.groupby("seg").agg(mes_chegada=("mes", lambda x: x.mode().iat[0]),
                                      fire=("fire", lambda x: x.mode().iat[0]))
        seg = seg.merge(moda, left_on="seg", right_index=True, how="left")
    else:
        seg["mes_chegada"], seg["fire"] = np.nan, np.nan

    # FOCOS MODIS: chegada (fora, perto da borda) e entrada (dentro, longe da borda)
    vl, ve = ctx["viirs_res"], ctx["viirs_ini"]
    seg["viirs_chegada"], seg["data_chegada_viirs"], seg["viirs_dentro"] = 0, pd.NaT, 0
    if len(vl):
        r, c, ok = _px(vl["x"], vl["y"], tr, S.shape)
        r, c, d = r[ok], c[ok], vl["acq_date"].to_numpy()[ok]
        out = ~S[r, c] & (dist[r, c] <= P["viirs_arr"] / PIX_M)
        inn = S[r, c] & (d_in[r, c] >= P["viirs_in"] / PIX_M)
        if out.any():
            t = pd.DataFrame({"seg": near[r[out], c[out]], "d": d[out]}).groupby("seg")["d"].agg(["size", "min"])
            seg.loc[t.index, "viirs_chegada"] = t["size"].to_numpy()
            seg.loc[t.index, "data_chegada_viirs"] = t["min"].to_numpy()
        if inn.any():
            t = pd.Series(near[r[inn], c[inn]]).value_counts()
            seg.loc[t.index, "viirs_dentro"] = t.to_numpy()
    seg["data_1o_foco_cicatriz"] = pd.NaT
    if kind == "cicatriz" and len(ve):
        r, c, ok = _px(ve["x"], ve["y"], tr, S.shape)
        r, c, d = r[ok], c[ok], ve["acq_date"].to_numpy()[ok]
        hit = labS[r, c] > 0
        if hit.any():
            first = pd.DataFrame({"scar": labS[r[hit], c[hit]], "d": d[hit]}).groupby("scar")["d"].min()
            seg["data_1o_foco_cicatriz"] = seg["scar"].map(first)
    seg["tipo_seg"] = kind
    return seg

# ============================================================
# 4. PSEUDO-CICATRIZES (referencia sem queima anterior)
# ============================================================

def place_pseudo(labS, scar_ids, ctx, P):
    H, W = labS.shape
    fuel1, burned_before = ctx["fuel1"], ctx["E"] | ctx["antes"]
    barrier, inside = ctx["barrier"], ctx["inside"]
    pseudo = np.zeros((H, W), np.int32)
    objs = ndimage.find_objects(labS)
    k = 0
    for sid in scar_ids:
        sl = objs[sid - 1]
        if sl is None:
            continue
        shape_mask = labS[sl] == sid
        h, w = shape_mask.shape
        for _ in range(P["ref_tries"]):
            ang = rng_ref.uniform(0, 2 * np.pi)
            dd = rng_ref.uniform(P["ref_min"], P["ref_max"]) / PIX_M
            r0 = int(sl[0].start + dd * np.sin(ang)); c0 = int(sl[1].start + dd * np.cos(ang))
            if r0 < 0 or c0 < 0 or r0 + h > H or c0 + w > W:
                continue
            tgt = (slice(r0, r0 + h), slice(c0, c0 + w))
            n = shape_mask.sum()
            if (inside[tgt] & shape_mask).sum() < n:
                continue
            if (fuel1[tgt] & shape_mask).sum() < 0.8 * n:
                continue
            if ((burned_before[tgt] | barrier[tgt]) & shape_mask).sum() > 0.05 * n:
                continue
            if (pseudo[tgt][shape_mask] > 0).any():
                continue
            k += 1
            pseudo[tgt][shape_mask] = k
            break
    return pseudo

# ============================================================
# 5. UM TI-ANO COMPLETO
# ============================================================

def tile_conf(fuel0, fuel1, E, queim, P1, P2, P):
    """quantidade e configuracao em janelas de ~1 km (mesmas metricas do bloco 08)."""
    tp = max(1, int(round(P["win"] / PIX_M)))
    H, W = fuel1.shape
    nr, nc = H // tp + 1, W // tp + 1
    out = {k: np.full((nr, nc), np.nan) for k in ["q", "a1", "a2", "split", "lpi", "nat"]}
    for i in range(nr):
        for j in range(nc):
            sl = (slice(i * tp, (i + 1) * tp), slice(j * tp, (j + 1) * tp))
            A = queim[sl].sum()
            out["nat"][i, j] = queim[sl].mean() if queim[sl].size else np.nan
            if A < 10:
                continue
            s0, _, _ = _meff(fuel0[sl]); s1, mx1, _ = _meff(fuel1[sl])
            out["q"][i, j] = (E[sl] & queim[sl]).sum() / A
            out["a1"][i, j] = P1[sl].sum() / A
            out["a2"][i, j] = P2[sl].sum() / A
            out["split"][i, j] = np.log(s1 / s0) if (s0 > 0 and s1 > 0) else np.nan
            out["lpi"][i, j] = mx1 / A
    return out, tp

def ti_year_segments(ti, year, scheme, P, save_seq=False, fuel_mode=FUEL_MAIN):
    st = static_of(ti)
    yr, l1, l2 = read_raster(ti, year), read_raster(ti, year - 1), read_raster(ti, year - 2)
    ini, res = SEASONS[scheme][int(ti)]
    tr = st["_transform"]
    inside = st["ti"] == ti
    barrier = st["barrier"] == 1
    queim = inside & fuel_from_cover(yr["cover"], fuel_mode) & ~barrier
    sm = season_masks(yr["month"], ti, scheme, SEASONS)
    E = sm["inicio"] & inside
    L = sm["resto"] & inside
    fuel0 = queim & ~sm["antes"]
    fuel1 = fuel0 & ~E
    obs_ok, nova, ls_mes, pre, drop = landsat_layers(yr, ini, res, P["nbr_drop"])
    fire, _ = ndimage.label(L, structure=CONN8)
    v = vi[(vi["year"] == year)]
    ctx = dict(fuel1=fuel1, L=L, E=E, antes=sm["antes"], obs_ok=obs_ok, nova=nova, tr=tr, cover=yr["cover"],
               slope=st["slope10"] / 10.0, month=yr["month"], fire=fire, barrier=barrier, inside=inside,
               pre=pre, drop=drop,
               viirs_res=v[v["mes"].isin(res)], viirs_ini=v[v["mes"].isin(ini)])

    labE, nE = ndimage.label(E, structure=CONN8)
    segs = segments_for(E, labE, ctx, P, "cicatriz")
    if segs.empty:
        return pd.DataFrame(), None
    lab_px = labE[E]
    lm = ls_mes[E]
    before = np.bincount(lab_px, weights=((lm > 0) & (lm <= max(ini))), minlength=nE + 1)
    after = np.bincount(lab_px, weights=(lm > max(ini)), minlength=nE + 1)
    segs["ls_antes"], segs["ls_depois"] = before[segs["scar"]], after[segs["scar"]]
    segs["area_cicatriz_ha"] = np.bincount(lab_px, minlength=nE + 1)[segs["scar"]] * 0.09
    mes_cic = pd.Series(yr["month"][E]).groupby(lab_px).agg(lambda x: x.mode().iat[0])
    segs["mes_cicatriz"] = segs["scar"].map(mes_cic)

    arr = (segs["n_ext_q"] / segs["n_ext"].replace(0, np.nan) >= P["arrival"]) | (segs["viirs_chegada"] > 0)
    pseudo = place_pseudo(labE, sorted(segs.loc[arr, "scar"].unique()), ctx, P)
    refs = segments_for(pseudo > 0, pseudo, ctx, P, "referencia") if pseudo.any() else pd.DataFrame()
    if len(refs):
        refs["area_cicatriz_ha"] = np.bincount(pseudo[pseudo > 0], minlength=pseudo.max() + 1)[refs["scar"]] * 0.09
        refs["ls_antes"] = refs["ls_depois"] = 0
        refs["mes_cicatriz"] = np.nan
    allseg = pd.concat([segs, refs], ignore_index=True)

    P1 = (l1["month"] > 0) & queim
    P2 = (l2["month"] > 0) & queim
    tc, tp = tile_conf(fuel0, fuel1, E, queim, P1, P2, P)
    ti_r = (allseg["r"] // tp).astype(int).clip(0, tc["q"].shape[0] - 1)
    ti_c = (allseg["c"] // tp).astype(int).clip(0, tc["q"].shape[1] - 1)
    for k, col in [("q", "q_inicio_win"), ("a1", "q_ant1_win"), ("a2", "q_ant2_win"),
                   ("split", "conf_split_win"), ("lpi", "conf_lpi_win"), ("nat", "combustivel_win")]:
        allseg[col] = tc[k][ti_r, ti_c]
    win = max(3, int(round(P["win"] / PIX_M)))
    bar_f = ndimage.uniform_filter(barrier.astype(float), size=win)
    riv_f = ndimage.uniform_filter((st["river"] == 1).astype(float), size=win)
    rr, cc = allseg["r"].round().astype(int), allseg["c"].round().astype(int)
    allseg["barreira_win"], allseg["rios_win"] = bar_f[rr, cc], riv_f[rr, cc]
    for g, col in [(1, "nativa_win"), (2, "pastagem_win"), (3, "agricultura_win")]:
        allseg[col] = ndimage.uniform_filter(((yr["cover"] == g) & inside).astype(float), size=win)[rr, cc]
    allseg["x"] = tr.c + (allseg["c"] + 0.5) * PIX_M
    allseg["y"] = tr.f - (allseg["r"] + 0.5) * PIX_M
    allseg["ti_id"], allseg["year"], allseg["esquema"], allseg["combustivel"] = ti, year, scheme, fuel_mode
    allseg["fonte_focos"] = "MODIS"

    if save_seq:
        seqr = np.zeros(E.shape, np.uint8)
        seqr[E] = 1; seqr[L] = 2; seqr[E & nova] = 3; seqr[pseudo > 0] = 4
        prof = dict(driver="GTiff", height=E.shape[0], width=E.shape[1], count=1, dtype="uint8",
                    crs=st["_crs"], transform=tr, compress="deflate", nodata=0)
        with rasterio.open(SEQ_DIR / f"seq_ti{ti}_{year}.tif", "w", **prof) as dst:
            dst.write(seqr, 1)

    mb = yr["month"]
    okp = inside & (mb > 0) & (ls_mes > 0)
    conc = {"ti_id": ti, "year": year, "esquema": scheme,
            "n_pixels_comparados": int(okp.sum()),
            "mesmo_mes": float((mb[okp] == ls_mes[okp]).mean()) if okp.any() else np.nan,
            "ate_1_mes": float((np.abs(mb[okp] - ls_mes[okp]) <= 1).mean()) if okp.any() else np.nan}
    return allseg, conc

# ============================================================
# 6. CLASSIFICACAO: ORDEM, CHEGADA E RESULTADO
# ============================================================

def classify_segments(df, P):
    df = df.copy()
    df["frac_ext_queimada"] = df["n_ext_q"] / df["n_ext"].replace(0, np.nan)
    df["frac_int_obs"] = df["n_int_obs"] / df["n_int"].replace(0, np.nan)
    df["frac_int_nova"] = df["n_int_nova"] / df["n_int_obs"].replace(0, np.nan)
    df["chegada_mapbiomas"] = df["frac_ext_queimada"] >= P["arrival"]
    df["chegada"] = df["chegada_mapbiomas"] | (df["viirs_chegada"] > 0)

    df["ordem_landsat"] = np.select(
        [(df["ls_antes"] >= 5) & (df["ls_antes"] >= df["ls_depois"]),
         (df["ls_depois"] >= 5) & (df["ls_depois"] > df["ls_antes"])],
        ["confirma", "contradiz"], "sem_dado")
    has_v = df["data_1o_foco_cicatriz"].notna() & df["data_chegada_viirs"].notna()
    df["ordem_viirs"] = np.where(~has_v, "sem_dado",
                                 np.where(pd.to_datetime(df["data_1o_foco_cicatriz"]) <
                                          pd.to_datetime(df["data_chegada_viirs"]), "confirma", "contradiz"))
    ol, ov = df["ordem_landsat"], df["ordem_viirs"]
    df["ordem_q"] = np.select(
        [(ol == "contradiz") | (ov == "contradiz"),
         (ol == "confirma") & (ov == "confirma"), ol == "confirma", ov == "confirma"],
        ["contraditoria", "landsat+viirs", "landsat", "viirs"], "so_mapbiomas")
    df["gap_meses"] = df["mes_chegada"] - df["mes_cicatriz"]
    ref = df["tipo_seg"] == "referencia"
    df.loc[ref, ["ordem_landsat", "ordem_viirs", "ordem_q"]] = "nao_se_aplica"
    ordem_ok = ref | df["ordem_q"].isin(["landsat+viirs", "landsat", "viirs"]) | \
               ((df["ordem_q"] == "so_mapbiomas") & (df["gap_meses"] >= 2))

    bem_obs = df["frac_int_obs"] >= P["obs_min"]
    entrou_ls = bem_obs & (df["frac_int_nova"] >= P["cont_min"])
    entrou = entrou_ls | (df["viirs_dentro"] > 0)
    entrou_mb = ref & (df["n_int_L"] / df["n_int"].replace(0, np.nan) >= P["cont_min"])

    df["resultado"] = np.select(
        [~df["chegada"], ~ordem_ok, entrou, bem_obs],
        ["sem_chegada", "indeterminado", "continuidade", "interrupcao"], "indeterminado")
    df["resultado_mb"] = np.select(
        [~df["chegada"], ~ordem_ok, entrou | entrou_mb, bem_obs],
        ["sem_chegada", "indeterminado", "continuidade", "interrupcao"], "indeterminado")
    df["regra_continuidade"] = "simetrica_v1"
    df["motivo"] = np.select(
        [~df["chegada"], ~ordem_ok, entrou, bem_obs],
        ["fogo nao chegou a borda", "ordem nao verificada/contraditoria", "", ""],
        "observacao insuficiente da faixa interna")
    df["interrupcao"] = (df["resultado"] == "interrupcao").astype(float)
    df.loc[~df["resultado"].isin(["interrupcao", "continuidade"]), "interrupcao"] = np.nan
    df["fire_id"] = np.where(df["fire"].notna(),
                             df["ti_id"].astype(str) + "_" + df["year"].astype(str) + "_f" +
                             df["fire"].fillna(-1).astype(int).astype(str),
                             df["ti_id"].astype(str) + "_" + df["year"].astype(str) + "_v" + df["seg"].astype(str))
    return df

def add_weather(df):
    df = df.copy()
    dt = pd.to_datetime(df["data_chegada_viirs"])
    mes_dt = pd.to_datetime(pd.DataFrame({"year": df["year"].astype(int),
                                          "month": df["mes_chegada"].fillna(9).astype(int), "day": 15}))
    df["data_chegada"] = dt.fillna(mes_dt).dt.normalize()
    df["fonte_data"] = np.where(dt.notna(), "modis", "mapbiomas_mes")
    lon, lat = _to_ll8.transform(df["x"].to_numpy(), df["y"].to_numpy())
    _, idx = e5_tree.query(np.c_[lon, lat])
    df["era5_point"] = e5_pts.loc[idx, "point_id"].to_numpy()
    w = e5_idx.reindex(pd.MultiIndex.from_arrays([df["era5_point"], df["data_chegada"]])).reset_index(drop=True)
    for c in w.columns:
        df[c] = w[c].to_numpy()
    return df

# ============================================================
# 7. TODAS AS TIs DA AMOSTRA
# ============================================================

def param_tag(P):
    return f"s{P['seg_len']}_f{P['strip']}_a{int(P['arrival'] * 100)}"

def seg_cache_name(scheme=SEASON_MAIN, P=None, fuel_mode=FUEL_MAIN):
    P = {**ENC_DEFAULT, **(P or {})}
    return f"segmentos_{scheme}_{param_tag(P)}_{FUEL_TAG[fuel_mode]}_{TAG_SEG}"

def build_segments(scheme=SEASON_MAIN, P=None, force=False, save_seq=False, fuel_mode=FUEL_MAIN):
    P = {**ENC_DEFAULT, **(P or {})}
    p = cache_path(seg_cache_name(scheme, P, fuel_mode), "parquet")
    if p.exists() and not (force or FORCE_REPROCESS):
        return pd.read_parquet(p), None
    parts, concs = [], []
    for ti in tqdm(AMOSTRA, desc=f"Segmentos {scheme} {param_tag(P)}"):
        for y in YEARS:
            seg, conc = ti_year_segments(ti, y, scheme, P, save_seq=save_seq, fuel_mode=fuel_mode)
            if len(seg):
                parts.append(seg)
            if conc:
                concs.append(conc)
    if not parts:
        raise RuntimeError("Nenhum segmento encontrado - confira os rasters do bloco 03 e a amostra do bloco 02.")
    df = classify_segments(pd.concat(parts, ignore_index=True), P)
    df = add_weather(df)
    df["seg_uid"] = (df["ti_id"].astype(str) + "_" + df["year"].astype(str) + "_" +
                     df["tipo_seg"].str[0] + df["seg"].astype(str))
    for c in ["data_chegada_viirs", "data_1o_foco_cicatriz", "data_chegada"]:
        df[c] = pd.to_datetime(df[c])
    df.to_parquet(p, index=False)
    return df, pd.DataFrame(concs)

segs_by_fuel, conc = {}, None
for fm in FUEL_RUNS:
    s_, c_ = build_segments(save_seq=(fm == FUEL_MAIN), fuel_mode=fm)
    segs_by_fuel[fm] = s_
    if fm == FUEL_MAIN:
        conc = c_
    console.print(f"[green]OK [{fm}] {len(s_):,} segmentos[/green]")
segs = segs_by_fuel[FUEL_MAIN]
console.print("[bold]Resultado por definicao de combustivel[/bold]")
display(pd.concat({fm: s_.groupby(["tipo_seg", "resultado"]).size() for fm, s_ in segs_by_fuel.items()},
                  axis=1).fillna(0).astype(int))

# ============================================================
# 8. RESUMOS
# ============================================================

res_tab = (segs.groupby(["tipo_seg", "resultado"]).agg(n=("seg_uid", "size"),
                                                         km=("comprimento_m", lambda x: x.sum() / 1000))
               .reset_index())
res_tab.to_csv(stats_path("B08_resultado_segmentos"), index=False)
display(res_tab)

ordem_tab = (segs[(segs["tipo_seg"] == "cicatriz") & segs["chegada"]]
             .groupby(["ti_id", "ordem_q"]).size().unstack(fill_value=0))
ordem_tab.to_csv(stats_path("B08_ordem_qualidade"))
display(ordem_tab)

# quanto o MODIS contribui (com 1 km ele pesa menos que o VIIRS: declarar)
ch = segs[segs["chegada"]]
fontes = pd.DataFrame({
    "chegada_so_mapbiomas": (ch["chegada_mapbiomas"] & (ch["viirs_chegada"] == 0)).groupby(ch["tipo_seg"]).mean(),
    "chegada_com_modis": (ch["viirs_chegada"] > 0).groupby(ch["tipo_seg"]).mean(),
    "entrada_vista_pelo_modis": (ch["viirs_dentro"] > 0).groupby(ch["tipo_seg"]).mean(),
    "data_chegada_modis": (ch["fonte_data"] == "modis").groupby(ch["tipo_seg"]).mean()}).round(3)
fontes.to_csv(stats_path("B08_contribuicao_modis"))
console.print("[bold]Contribuicao dos focos MODIS (fracao dos segmentos com chegada)[/bold]")
display(fontes)

if conc is not None and len(conc):
    conc.to_csv(stats_path("B08_concordancia_mapbiomas_landsat"), index=False)
    display(conc.groupby("ti_id")[["mesmo_mes", "ate_1_mes"]].mean().round(2))


# ============================================================
# 9. REGRA SIMETRICA x REGRA ANTIGA (sensibilidade)
#    resultado    = regra SIMETRICA: entrada detectada do MESMO jeito nas bordas de
#                   cicatriz e de referencia (Landsat + MODIS) -> principal
#    resultado_mb = regra antiga: na referencia o MapBiomas tambem conta entrada
#                   (o MapBiomas guarda um so mes por pixel/ano e nunca mostra a
#                   cicatriz queimando de novo) -> so sensibilidade
# ============================================================

rows = []
for fm, s_ in segs_by_fuel.items():
    for regra, col in [("antiga (MapBiomas na referencia)", "resultado_mb"), ("simetrica (Landsat + MODIS)", "resultado")]:
        for tipo, g in s_.groupby("tipo_seg"):
            vc = g[col].value_counts()
            n_i, n_c = int(vc.get("interrupcao", 0)), int(vc.get("continuidade", 0))
            rows.append({"combustivel": fm, "regra": regra, "tipo_seg": tipo, "interrupcao": n_i,
                         "continuidade": n_c, "indeterminado": int(vc.get("indeterminado", 0)),
                         "sem_chegada": int(vc.get("sem_chegada", 0)),
                         "% interrupcao": round(100 * n_i / max(n_i + n_c, 1), 1)})
comp = pd.DataFrame(rows)
comp.to_csv(stats_path("B08a_regra_antiga_x_simetrica"), index=False)
console.print("[bold]Interrupcao x continuidade: regra simetrica (principal) x antiga (sensibilidade)[/bold]")
display(comp.set_index(["combustivel", "regra", "tipo_seg"]))

anal = segs[segs["resultado"].isin(["interrupcao", "continuidade"])]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
pv = segs.pivot_table(index="tipo_seg", columns="resultado", values="seg_uid", aggfunc="size", fill_value=0)
pv.plot.barh(stacked=True, ax=axs[0], color={"continuidade": "#b2182b", "interrupcao": "#1b7837",
                                              "indeterminado": "#bdbdbd", "sem_chegada": "#f0f0f0"})
axs[0].set_xlabel("Segmentos"); axs[0].set_ylabel("")
axs[0].legend(fontsize=7, frameon=False)
p_int = anal.groupby(["tipo_seg"])["interrupcao"].mean()
axs[1].bar(p_int.index, p_int.values, color=["#1b7837", "#9e9e9e"][:len(p_int)])
axs[1].set_ylim(0, 1)
axs[1].set_ylabel("Proporcao de interrupcao\n(segmentos com chegada e resultado definido)")
fig.savefig(fig_path("B08_resultados_md"))
plt.show()

console.print(f"[bold green]OK BLOCO 10 completo: {len(segs):,} segmentos "
              f"({len(anal):,} com resultado definido).[/bold green]")


## 11 detectabilidade

In [ ]:
# ======================================================================
# BLOCO 11 - Detectabilidade: a interrupcao nas bordas e real ou e o NBR que nao ve fogo sobre area queimada?
# Depende dos blocos 00, 03 (rasters), 05 (MODIS), 06 e 10 (segmentos).
# ======================================================================

check_version()

# ============================================================
# BLOCK 8b - Detectability checks
#
# O problema: no bloco 10 a CONTINUIDADE e uma nova queda de NBR na faixa interna.
#   Na REFERENCIA o interior e vegetacao (NBR ~0,4): qualquer fogo derruba o NBR.
#   Na CICATRIZ o interior ja queimou (NBR ~0) e tem pouco combustivel: uma
#   reentrada real mexe pouco no NBR. A interrupcao nas cicatrizes pode estar
#   inflada pelo DETECTOR, nao pelo fogo. Tres verificacoes:
#
# (A) CURVA DO LIMIAR: % de interrupcao (cicatriz x referencia) refeita com o
#     limiar da queda media de NBR na faixa interna variando de 0,02 a 0,20,
#     e com a queda RELATIVA ao estado anterior (dNBR / sqrt(|NBR antes|),
#     forma do RdNBR, Miller & Thode 2007; NBR antes com piso de 0,05).
#     Se o contraste so existe em limiares altos, ele e do detector.
#
# (B) SENSIBILIDADE DO LANDSAT COM ENTRADA CONFIRMADA PELO MODIS: nos segmentos
#     em que um pixel MODIS com fogo do restante esta >= 1 km DENTRO da
#     cicatriz (o fogo entrou de fato), em que fracao o Landsat tambem marcou
#     continuidade? Mesmo calculo na referencia. Diferenca = vies de deteccao.
#
# (C) TESTE INDEPENDENTE DO NBR (MODIS): densidade de pixels-dia MODIS com fogo
#     no restante da seca (FireMask >= 8) dentro dos NUCLEOS das cicatrizes do
#     inicio (pixels a > 1 km da borda: o pixel MODIS inteiro cabe dentro) x
#     nucleos de combustivel ainda nao queimado no fim do inicio da seca,
#     na mesma TI-ano. Versao "local": so o combustivel a ate 5 km de uma
#     cicatriz. Razao de taxas por Poisson com efeito fixo de TI-ano e IC por
#     bootstrap de TIs. Nao depende do NBR nem da regra de borda: se a razao
#     for << 1, as cicatrizes de fato barram o fogo posterior.
#
# Output:
#   04_stats/B08b_curva_limiar.csv, B08b_sensibilidade_landsat.csv,
#            B08b_modis_nucleos_ti_ano.csv, B08b_modis_razao_taxas.csv
#   05_figures/B08b_detectabilidade.{tif,png,pdf}   (600 dpi)
# ============================================================

from scipy import ndimage
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ============================================================
# 1. PARAMETERS
# ============================================================

TAG_SEG    = globals().get("TAG_SEG", f"{TAG_YEARS}_md")
FOCOS_NAME = globals().get("FOCOS_NAME", f"modis_focos_v1_{TAG_YEARS}")
LIMIARES   = np.round(np.arange(0.02, 0.205, 0.01), 2)
PRE_PISO   = 0.05
OBS_MIN    = 0.5
CORE_M     = [1000, 1500]          # nucleo: > 1 km da borda (principal) e > 1,5 km (sensibilidade)
LOCAL_M    = 5000                  # versao local: combustivel a ate 5 km de uma cicatriz
N_BOOT     = 2000
SEASONS    = load_seasons()
AMOSTRA    = load_sample_ids()
rng        = np.random.default_rng(8)

segs = pd.read_parquet(cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet"))
if "drop_int_med" not in segs.columns:
    raise RuntimeError("Segmentos sem 'drop_int_med' - rode o bloco 10 novo (MODIS) antes do bloco 11.")

# ============================================================
# 2. (A) CURVA DO LIMIAR
# ============================================================

d = segs[segs["resultado"].isin(["interrupcao", "continuidade"]) & (segs["frac_int_obs"] >= OBS_MIN)
         & (segs["viirs_dentro"] == 0)].copy()                   # so o Landsat decide
d["drop_rel"] = d["drop_int_med"] / np.sqrt(np.maximum(np.abs(d["nbr_pre_int"]), PRE_PISO))
rows = []
for t in LIMIARES:
    for tipo, g in d.groupby("tipo_seg"):
        rows.append({"limiar": t, "tipo_seg": tipo, "n": len(g),
                     "interrupcao_abs_%": 100 * (g["drop_int_med"] < t).mean(),
                     "interrupcao_rel_%": 100 * (g["drop_rel"] < t / np.sqrt(0.4)).mean()})
curva = pd.DataFrame(rows)
curva.to_csv(stats_path("B08b_curva_limiar"), index=False)
estado = d.groupby("tipo_seg")[["nbr_pre_int", "drop_int_med", "drop_rel"]].describe().T
estado.to_csv(stats_path("B08b_estado_faixa_interna"))
console.print("[bold](A) Estado da faixa interna antes do fogo posterior (NBR) e queda observada[/bold]")
display(estado.round(3))
console.print("[bold](A) % de interrupcao pelo limiar de queda media de NBR[/bold]")
display(curva.pivot_table(index="limiar", columns="tipo_seg", values=["interrupcao_abs_%", "interrupcao_rel_%"]).round(1))

# ============================================================
# 3. (B) SENSIBILIDADE DO LANDSAT DADA ENTRADA CONFIRMADA PELO MODIS
# ============================================================

c = segs[segs["chegada"] & (segs["viirs_dentro"] > 0) & (segs["frac_int_obs"] >= OBS_MIN)].copy()
c["landsat_viu"] = c["frac_int_nova"] >= ENC_DEFAULT["cont_min"] if "ENC_DEFAULT" in globals() else c["frac_int_nova"] >= 0.2
sens = c.groupby("tipo_seg").agg(n_segmentos=("seg_uid", "size"), landsat_viu_entrada=("landsat_viu", "mean"),
                                 queda_mediana=("drop_int_med", "median"), nbr_antes_mediano=("nbr_pre_int", "median"))
sens.to_csv(stats_path("B08b_sensibilidade_landsat"))
console.print("[bold](B) Entrada confirmada pelo MODIS: em que fracao o Landsat tambem viu?[/bold]")
display(sens.round(3))

# ============================================================
# 4. (C) MODIS NOS NUCLEOS: CICATRIZ x COMBUSTIVEL NAO QUEIMADO
# ============================================================

if "focos_raw" not in globals():
    focos_raw = read_table(cache_path(FOCOS_NAME, "parquet"))
fo = focos_raw[focos_raw["confidence"].astype(str).str[0] != "l"].copy()
fo["acq_date"] = pd.to_datetime(fo["acq_date"])
fo = fo.drop_duplicates(["latitude", "longitude", "acq_date"])   # pixel-dia (Terra e Aqua no mesmo dia = 1)
_g = gpd.GeoSeries(gpd.points_from_xy(fo["longitude"], fo["latitude"]), crs="EPSG:4326").to_crs(AREA_CRS)
fo["x"], fo["y"] = _g.x.to_numpy(), _g.y.to_numpy()
fo["year"], fo["mes"] = fo["acq_date"].dt.year, fo["acq_date"].dt.month

def core_counts(ti, y, core_m, scheme=SEASON_MAIN, fuel_mode=FUEL_MAIN, keep_masks=False):
    st, yr = static_of(ti), read_raster(ti, y)
    ini, res = SEASONS[scheme][int(ti)]
    inside = st["ti"] == ti
    queim = inside & fuel_from_cover(yr["cover"], fuel_mode) & (st["barrier"] == 0)
    sm_ = season_masks(yr["month"], ti, scheme, SEASONS)
    E = sm_["inicio"] & queim
    U = queim & ~sm_["antes"] & ~E
    k = core_m / PIX_M
    coreE = ndimage.distance_transform_edt(E) > k
    coreU = ndimage.distance_transform_edt(U) > k
    nearE = ndimage.distance_transform_edt(~E) <= LOCAL_M / PIX_M if E.any() else np.zeros_like(E)
    tr = st["_transform"]
    p = fo[(fo["ti_id"] == ti) & (fo["year"] == y) & fo["mes"].isin(res)]
    c_ = np.floor((p["x"].to_numpy() - tr.c) / PIX_M).astype(int)
    r_ = np.floor((tr.f - p["y"].to_numpy()) / PIX_M).astype(int)
    ok = (r_ >= 0) & (r_ < E.shape[0]) & (c_ >= 0) & (c_ < E.shape[1])
    r_, c_ = r_[ok], c_[ok]
    out = {"ti_id": ti, "year": y, "core_m": core_m,
           "n_cicatriz": int(coreE[r_, c_].sum()), "km2_cicatriz": coreE.sum() * 9e-4,
           "n_naoqueimado": int(coreU[r_, c_].sum()), "km2_naoqueimado": coreU.sum() * 9e-4,
           "n_naoqueimado_local": int((coreU & nearE)[r_, c_].sum()),
           "km2_naoqueimado_local": (coreU & nearE).sum() * 9e-4}
    if keep_masks:
        return out, dict(coreE=coreE, coreU=coreU, E=E, L=sm_["resto"] & inside, inside=inside, tr=tr,
                         pts=(p["x"].to_numpy()[ok], p["y"].to_numpy()[ok]))
    return out

rows = [core_counts(ti, y, cm) for cm in CORE_M for ti in tqdm(AMOSTRA, desc=f"Nucleos MODIS") for y in YEARS]
nuc = pd.DataFrame(rows)
nuc.to_csv(stats_path("B08b_modis_nucleos_ti_ano"), index=False)

def rate_ratio(df, ref="naoqueimado"):
    """Razao de taxas cicatriz / referencia: Poisson com efeito fixo de TI-ano + bootstrap de TIs."""
    dd = df[(df["km2_cicatriz"] > 0) & (df[f"km2_{ref}"] > 0)]
    long = pd.concat([pd.DataFrame({"ti_id": dd["ti_id"], "ty": dd["ti_id"].astype(str) + "_" + dd["year"].astype(str),
                                    "cic": 1, "n": dd["n_cicatriz"], "km2": dd["km2_cicatriz"]}),
                      pd.DataFrame({"ti_id": dd["ti_id"], "ty": dd["ti_id"].astype(str) + "_" + dd["year"].astype(str),
                                    "cic": 0, "n": dd[f"n_{ref}"], "km2": dd[f"km2_{ref}"]})])
    pooled = (dd["n_cicatriz"].sum() / dd["km2_cicatriz"].sum()) / (dd[f"n_{ref}"].sum() / dd[f"km2_{ref}"].sum())
    try:
        m = smf.glm("n ~ cic + C(ty)", data=long, family=sm.families.Poisson(),
                    offset=np.log(long["km2"])).fit(cov_type="cluster", cov_kwds={"groups": long["ti_id"]})
        rr_glm = float(np.exp(m.params["cic"]))
    except Exception:
        rr_glm = np.nan
    ids = dd["ti_id"].unique()
    boots = []
    for _ in range(N_BOOT):
        b = pd.concat([dd[dd["ti_id"] == t] for t in rng.choice(ids, len(ids))])
        den = b[f"n_{ref}"].sum() / b[f"km2_{ref}"].sum()
        if den > 0:
            boots.append((b["n_cicatriz"].sum() / b["km2_cicatriz"].sum()) / den)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan))
    return {"referencia": ref, "ti_anos": len(dd), "focos_cicatriz": int(dd["n_cicatriz"].sum()),
            "km2_cicatriz": dd["km2_cicatriz"].sum(), "focos_ref": int(dd[f"n_{ref}"].sum()),
            "km2_ref": dd[f"km2_{ref}"].sum(), "razao_taxas_agregada": pooled, "razao_taxas_poisson_FE": rr_glm,
            "ic95_inf_boot_TI": lo, "ic95_sup_boot_TI": hi}

rr = pd.DataFrame([dict(core_m=cm, **rate_ratio(nuc[nuc["core_m"] == cm], ref))
                   for cm in CORE_M for ref in ["naoqueimado", "naoqueimado_local"]])
per_ti = (nuc[nuc["core_m"] == CORE_M[0]].groupby("ti_id")
          [["n_cicatriz", "km2_cicatriz", "n_naoqueimado", "km2_naoqueimado"]].sum())
per_ti["razao_taxas"] = (per_ti["n_cicatriz"] / per_ti["km2_cicatriz"]) / (per_ti["n_naoqueimado"] / per_ti["km2_naoqueimado"])
per_ti["TI"] = per_ti.index.map(tis.set_index("ti_id")["ti_name"])
rr.to_csv(stats_path("B08b_modis_razao_taxas"), index=False)
per_ti.to_csv(stats_path("B08b_modis_razao_taxas_por_TI"))
console.print("[bold](C) Focos MODIS do restante da seca: nucleo de cicatriz / nucleo nao queimado[/bold]")
display(rr.round(3)); display(per_ti.round(3))

# ============================================================
# 5. FIGURA (600 dpi): mapa de um TI-ano + as tres verificacoes
# ============================================================

try:
    ex = nuc[nuc["core_m"] == CORE_M[0]].assign(t=lambda x: x["n_cicatriz"] + x["n_naoqueimado"]) \
            .sort_values(["km2_cicatriz", "t"], ascending=False).iloc[0]
    _, M = core_counts(int(ex["ti_id"]), int(ex["year"]), CORE_M[0], keep_masks=True)
    from matplotlib.colors import ListedColormap
    tr = M["tr"]; Hh, Ww = M["E"].shape
    ext = (tr.c, tr.c + Ww * PIX_M, tr.f - Hh * PIX_M, tr.f)
    cls = np.zeros(M["E"].shape, np.uint8)
    cls[M["inside"]] = 1; cls[M["L"]] = 2; cls[M["E"]] = 3; cls[M["coreE"]] = 4; cls[M["coreU"] & ~M["L"]] = 5
    cls[M["coreU"] & M["L"]] = 6
    cmap = ListedColormap(["white", "#efeee9", "#f7c79a", "#9ec2ee", "#1f5fc8", "#cfe3c3", "#e98b4a"])
    fig = plt.figure(figsize=(17 / 2.54, 12.5 / 2.54), layout="constrained")
    gs = fig.add_gridspec(2, 3, height_ratios=[1.35, 1])
    ax = fig.add_subplot(gs[0, :])
    f = int(np.ceil(max(cls.shape) / 1600))
    ax.imshow(cls[::f, ::f], cmap=cmap, vmin=0, vmax=6, extent=ext, interpolation="nearest", rasterized=True)
    ax.scatter(*M["pts"], s=3, color="#7a1f3d", lw=0, zorder=3)
    tis[tis["ti_id"] == int(ex["ti_id"])].boundary.plot(ax=ax, color="black", lw=0.5)
    ax.set_aspect("equal"); ax.set_axis_off()
    x0 = ext[0] + 0.04 * (ext[1] - ext[0]); y0 = ext[2] + 0.05 * (ext[3] - ext[2])
    ax.add_patch(plt.Rectangle((x0, y0), 10000, (ext[3] - ext[2]) * 0.012, color="black"))
    ax.text(x0 + 5000, y0 + (ext[3] - ext[2]) * 0.02, "10 km", ha="center", va="bottom", fontsize=7)
    import matplotlib.patches as mp
    ax.legend(handles=[mp.Patch(color="#9ec2ee", label="Early scar (May–Jul)"),
                       mp.Patch(color="#1f5fc8", label="Early-scar core (> 1 km from edge)"),
                       mp.Patch(color="#cfe3c3", label="Unburned fuel core"),
                       mp.Patch(color="#f7c79a", label="Burned Aug–Oct (MapBiomas)"),
                       plt.Line2D([], [], marker="o", ls="", ms=3, color="#7a1f3d", label="MODIS fire pixel, Aug–Oct")],
              loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=6.5, frameon=False)
    ax.set_title(f"(a) {tis.set_index('ti_id').loc[int(ex['ti_id']), 'ti_name']} {int(ex['year'])}: "
                 f"MODIS later-fire pixels in scar cores vs unburned cores", fontsize=8, loc="left")
    ax1 = fig.add_subplot(gs[1, 0])
    for tipo, cor in [("cicatriz", "#1f5fc8"), ("referencia", "#8f8e88")]:
        g = curva[curva["tipo_seg"] == tipo]
        ax1.plot(g["limiar"], g["interrupcao_abs_%"], color=cor, lw=1.5, label={"cicatriz": "Scar", "referencia": "Reference"}[tipo])
        ax1.plot(g["limiar"], g["interrupcao_rel_%"], color=cor, lw=1.0, ls="--")
    ax1.axvline(0.10, color="black", lw=0.5, ls=":")
    ax1.set_xlabel("NBR-drop threshold"); ax1.set_ylabel("Interrupted (%)")
    ax1.set_title("(b) threshold (dashed: relative)", fontsize=8, loc="left"); ax1.legend(fontsize=6.5, frameon=False)
    ax2 = fig.add_subplot(gs[1, 1])
    if len(sens):
        ax2.bar(range(len(sens)), 100 * sens["landsat_viu_entrada"], color=["#1f5fc8", "#8f8e88"][:len(sens)])
        ax2.set_xticks(range(len(sens))); ax2.set_xticklabels([{"cicatriz": "Scar", "referencia": "Ref."}.get(i, i) for i in sens.index])
        for k_, v in enumerate(sens["n_segmentos"]):
            ax2.text(k_, 2, f"n={v:,}", ha="center", fontsize=6, color="white")
    ax2.set_ylim(0, 100); ax2.set_ylabel("Landsat flagged entry (%)")
    ax2.set_title("(c) entry confirmed by MODIS", fontsize=8, loc="left")
    ax3 = fig.add_subplot(gs[1, 2])
    r0 = rr[rr["core_m"] == CORE_M[0]].set_index("referencia")
    for k_, ref in enumerate(["naoqueimado", "naoqueimado_local"]):
        if ref in r0.index:
            v = r0.loc[ref]
            ax3.errorbar(k_, v["razao_taxas_agregada"], yerr=[[v["razao_taxas_agregada"] - v["ic95_inf_boot_TI"]],
                         [v["ic95_sup_boot_TI"] - v["razao_taxas_agregada"]]], fmt="o", color="#1f5fc8", capsize=3)
    ax3.axhline(1, color="black", lw=0.6, ls="--")
    ax3.set_xticks([0, 1]); ax3.set_xticklabels(["all unburned", "≤ 5 km of scar"], fontsize=7)
    ax3.set_yscale("log"); ax3.set_ylabel("MODIS rate ratio, scar / unburned")
    ax3.set_title("(d) NBR-independent test", fontsize=8, loc="left")
    for ext_, kw in [("tif", {"pil_kwargs": {"compression": "tiff_lzw"}}), ("png", {}), ("pdf", {})]:
        fig.savefig(fig_path("B08b_detectabilidade", ext_), dpi=600, facecolor="white", **kw)
    plt.show(); plt.close(fig)
except Exception as e_:
    console.print(f"[yellow]figura pulada: {str(e_)[:150]}[/yellow]")

console.print("[bold green]OK BLOCO 11 completo.[/bold green]")


## 12 modelos

In [ ]:
# ======================================================================
# BLOCO 12 - Modelos: referencia (quantidade) x referencia + configuracao
# Depende do bloco 00 e dos checkpoints dos blocos 08 (celulas) e 10 (segmentos).
# ======================================================================
#
# Hipotese: com quantidade queimada antes, vegetacao, relevo, tempo e barreiras
# comparaveis, cicatrizes que fragmentam mais o combustivel estao associadas a
# MENOS continuidade do fogo posterior.
#
# ESCALA 1 - CELULA (bloco 08, 4980 m): y_resto e y_cont
#   REFERENCIA: q_inicio + q_inicio_viz + q_ant1 + q_ant2 + pastagem + agricultura
#               + vpd_resto + chuva_seca | celula + ano
#   + CONFIG  : conf_lpi + conf_npatch (bruta e residual); m_eff/borda so residuais
#   Wald do bloco, R2 within, previsao deixando uma TI de fora.
# ESCALA 2 - SEGMENTO (bloco 10; interrupcao x continuidade)
#   (a) borda de cicatriz x borda de referencia (pseudo-cicatriz)
#   (b) so cicatrizes: referencia x + configuracao (fragmentacao e maior mancha
#       na janela de ~1 km, area e largura local da cicatriz)
#   EP agrupado por incendio e TI; validacao cruzada deixando incendios / TIs de fora.
# A inferencia valida para 8 TIs (wild bootstrap, jackknife) esta no bloco 13.
#
# Output: 04_stats/B09_*.csv
# ======================================================================

check_version()

import pyfixest as pf
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

REGRA_PRINCIPAL = "simetrica (Landsat + MODIS)"
REGRA_SENS      = "antiga (MapBiomas na referencia)"

CELL_REF  = ["q_inicio", "q_inicio_viz", "q_ant1", "q_ant2", "pastagem", "agricultura", "vpd_resto", "chuva_seca"]
CELL_CONF = ["conf_lpi", "conf_npatch"]
CELL_CONF_ALT = ["conf_split_res", "conf_borda_res"]
CELL_POOL_EXTRA = ["declividade", "barreira", "rios"]
SEG_REF  = ["q_inicio_win", "q_ant1_win", "q_ant2_win", "pastagem_win", "agricultura_win", "decl_ext",
            "barreira_win", "rios_win", "P_vpd_max", "P_wind", "P_rh_min"]
SEG_CONF = ["conf_split_win", "conf_lpi_win", "log_area_cicatriz", "largura_local_100m"]
N_FOLDS  = 5

# ============================================================
# 1. CELULA
# ============================================================

def demean_twoway(M, g1, g2, tol=1e-10, maxit=2000):
    M = M.astype(float).copy()
    n1, n2 = g1.max() + 1, g2.max() + 1
    c1, c2 = np.bincount(g1, minlength=n1), np.bincount(g2, minlength=n2)
    for _ in range(maxit):
        old = M.copy()
        for j in range(M.shape[1]):
            M[:, j] -= (np.bincount(g1, M[:, j], n1) / c1)[g1]
            M[:, j] -= (np.bincount(g2, M[:, j], n2) / c2)[g2]
        if np.max(np.abs(M - old)) < tol:
            break
    return M

def r2_within(d, y, xs):
    g1 = pd.factorize(d["cell_id"])[0]; g2 = pd.factorize(d["year"])[0]
    Z = demean_twoway(d[[y] + xs].to_numpy(), g1, g2)
    yy, X = Z[:, 0], Z[:, 1:]
    b, *_ = np.linalg.lstsq(X, yy, rcond=None)
    return 1 - ((yy - X @ b) ** 2).sum() / (yy ** 2).sum()

def wald_block(names, b, V, block):
    idx = [names.index(v) for v in block if v in names]
    bb, VV = b[idx], V[np.ix_(idx, idx)]
    chi = float(bb @ np.linalg.pinv(VV) @ bb)
    return chi, len(idx), float(stats.chi2.sf(chi, len(idx)))

def zscore(d, cols):
    d = d.copy()
    for c in cols:
        s = d[c].std()
        d[c] = (d[c] - d[c].mean()) / s if s and s > 0 else 0.0
    return d

def fit_cells(cells, outcome="y_resto", conf=CELL_CONF, label=""):
    d = cells.dropna(subset=[outcome] + CELL_REF + conf).copy()
    d = zscore(d, CELL_REF + conf)
    m_conf = pf.feols(f"{outcome} ~ " + " + ".join(CELL_REF + conf) + " | cell_id + year", data=d,
                      vcov={"CRV1": "ti_id"})
    names = list(m_conf._coefnames)
    b = m_conf.coef().reindex(names).to_numpy(); V = np.asarray(m_conf._vcov)
    chi, k, p = wald_block(names, b, V, conf)
    tid = m_conf.tidy()
    coefs = tid.loc[[c for c in conf if c in tid.index]].copy()
    coefs.insert(0, "modelo", label or outcome)
    r2r, r2c = r2_within(d, outcome, CELL_REF), r2_within(d, outcome, CELL_REF + conf)
    pool = CELL_REF + CELL_POOL_EXTRA
    dp = zscore(d.dropna(subset=CELL_POOL_EXTRA).copy(), CELL_POOL_EXTRA)
    Yd = pd.get_dummies(dp["year"].astype(str), prefix="a", drop_first=True).astype(float)
    err = {"ref": [], "conf": []}
    for ti in sorted(dp["ti_id"].unique()):
        tr, te = dp["ti_id"] != ti, dp["ti_id"] == ti
        for k_, xs in [("ref", pool), ("conf", pool + conf)]:
            X = np.c_[np.ones(len(dp)), dp[xs].to_numpy(), Yd.to_numpy()]
            bb, *_ = np.linalg.lstsq(X[tr.to_numpy()], dp.loc[tr, outcome], rcond=None)
            err[k_].append(((dp.loc[te, outcome] - X[te.to_numpy()] @ bb) ** 2).to_numpy())
    rmse_r = float(np.sqrt(np.concatenate(err["ref"]).mean()))
    rmse_c = float(np.sqrt(np.concatenate(err["conf"]).mean()))
    summ = {"modelo": label or outcome, "n": len(d), "n_celulas": d["cell_id"].nunique(),
            "n_TIs": d["ti_id"].nunique(), "r2_within_ref": r2r, "r2_within_conf": r2c,
            "ganho_r2_within": r2c - r2r, "wald_chi2_conf": chi, "wald_gl": k, "wald_p": p,
            "rmse_fora_TI_ref": rmse_r, "rmse_fora_TI_conf": rmse_c,
            "ganho_previsao_fora_TI_%": 100 * (rmse_r - rmse_c) / rmse_r}
    return summ, coefs

# ============================================================
# 2. SEGMENTO
# ============================================================

def prep_segments(segs):
    d = segs[segs["resultado"].isin(["interrupcao", "continuidade"])].copy()
    d["interrupcao"] = (d["resultado"] == "interrupcao").astype(int)
    d["cicatriz"] = (d["tipo_seg"] == "cicatriz").astype(int)
    d["log_area_cicatriz"] = np.log(d["area_cicatriz_ha"].clip(lower=0.09))
    d["largura_local_100m"] = d["largura_local_m"] / 100
    d["ano"] = pd.Categorical(d["year"], categories=sorted(d["year"].unique()))
    sem_id = d["fire_id"].astype(str).str.contains("_v")
    d["fire_id"] = np.where(sem_id, d["ti_id"].astype(str) + "_" + d["year"].astype(str) + "_semid", d["fire_id"])
    return d

def glm_cluster(formula, d):
    m = smf.glm(formula, data=d, family=sm.families.Binomial())
    g = np.column_stack([pd.factorize(d["fire_id"])[0], pd.factorize(d["ti_id"])[0]])
    try:
        return m.fit(cov_type="cluster", cov_kwds={"groups": g}), "incendio + TI"
    except Exception:
        return m.fit(cov_type="cluster", cov_kwds={"groups": g[:, 1]}), "TI"

def cv_scores(d, xs_ref, xs_conf, splitter, groups):
    out = {"ref": np.full(len(d), np.nan), "conf": np.full(len(d), np.nan)}
    for tr, te in splitter.split(d, d["interrupcao"], groups):
        dtr, dte = d.iloc[tr], d.iloc[te]
        if dtr["interrupcao"].nunique() < 2:
            continue
        for k, xs in [("ref", xs_ref), ("conf", xs_conf)]:
            try:
                m = smf.glm("interrupcao ~ " + " + ".join(xs) + " + C(ano)", data=dtr,
                            family=sm.families.Binomial()).fit()
                out[k][te] = m.predict(dte).to_numpy()
            except Exception:
                pass
    ok = ~np.isnan(out["ref"]) & ~np.isnan(out["conf"])
    y = d["interrupcao"].to_numpy()[ok]
    res = {}
    for k in ["ref", "conf"]:
        p = np.clip(out[k][ok], 1e-6, 1 - 1e-6)
        res[k] = {"log_loss": log_loss(y, p), "brier": brier_score_loss(y, p),
                  "auc": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan}
    return res, int(ok.sum())

def fit_segments(segs, conf=SEG_CONF, label=""):
    d = prep_segments(segs)
    sc = zscore(d[d["cicatriz"] == 1].dropna(subset=SEG_REF + conf).copy(), SEG_REF + conf)
    m_ref, cl = glm_cluster("interrupcao ~ " + " + ".join(SEG_REF) + " + C(ano)", sc)
    m_conf, _ = glm_cluster("interrupcao ~ " + " + ".join(SEG_REF + conf) + " + C(ano)", sc)
    w = m_conf.wald_test(", ".join(f"{v} = 0" for v in conf), scalar=True)
    coefs = pd.DataFrame({"modelo": label or "segmento", "coef": m_conf.params[conf],
                          "razao_chances_1dp": np.exp(m_conf.params[conf]),
                          "ic95_inf": np.exp(m_conf.conf_int().loc[conf, 0]),
                          "ic95_sup": np.exp(m_conf.conf_int().loc[conf, 1]),
                          "p": m_conf.pvalues[conf]})
    cv_f, _ = cv_scores(sc.reset_index(drop=True), SEG_REF, SEG_REF + conf,
                        GroupKFold(n_splits=N_FOLDS), sc["fire_id"].to_numpy())
    cv_t, _ = cv_scores(sc.reset_index(drop=True), SEG_REF, SEG_REF + conf,
                        LeaveOneGroupOut(), sc["ti_id"].to_numpy())
    summ = {"modelo": label or "segmento", "n_segmentos": len(sc), "n_incendios": sc["fire_id"].nunique(),
            "n_TIs": sc["ti_id"].nunique(), "prop_interrupcao": sc["interrupcao"].mean(),
            "agrupamento_EP": cl, "aic_ref": m_ref.aic, "aic_conf": m_conf.aic,
            "wald_chi2_conf": float(w.statistic), "wald_p": float(w.pvalue),
            "cv_incendio_logloss_ref": cv_f["ref"]["log_loss"], "cv_incendio_logloss_conf": cv_f["conf"]["log_loss"],
            "cv_incendio_auc_ref": cv_f["ref"]["auc"], "cv_incendio_auc_conf": cv_f["conf"]["auc"],
            "cv_TI_logloss_ref": cv_t["ref"]["log_loss"], "cv_TI_logloss_conf": cv_t["conf"]["log_loss"],
            "cv_TI_auc_ref": cv_t["ref"]["auc"], "cv_TI_auc_conf": cv_t["conf"]["auc"],
            "cv_TI_brier_ref": cv_t["ref"]["brier"], "cv_TI_brier_conf": cv_t["conf"]["brier"]}
    return summ, coefs

def fit_scar_vs_reference(segs):
    d = zscore(prep_segments(segs).dropna(subset=SEG_REF), SEG_REF)
    m, cl = glm_cluster("interrupcao ~ cicatriz + " + " + ".join(SEG_REF) + " + C(ano)", d)
    return {"n": len(d), "n_cicatriz": int(d["cicatriz"].sum()), "n_referencia": int((1 - d["cicatriz"]).sum()),
            "prop_interrupcao_cicatriz": d.loc[d["cicatriz"] == 1, "interrupcao"].mean(),
            "prop_interrupcao_referencia": d.loc[d["cicatriz"] == 0, "interrupcao"].mean(),
            "razao_chances_cicatriz": float(np.exp(m.params["cicatriz"])),
            "ic95_inf": float(np.exp(m.conf_int().loc["cicatriz", 0])),
            "ic95_sup": float(np.exp(m.conf_int().loc["cicatriz", 1])),
            "p": float(m.pvalues["cicatriz"]), "agrupamento_EP": cl}

# ============================================================
# 3. RUN - as duas definicoes de combustivel
# ============================================================

cell_rows, cell_coefs, seg_rows, seg_coefs_all, ref_rows = [], [], [], [], []
seg_sens_rows, seg_sens_coefs, seg_corr = [], [], []
for fm in FUEL_RUNS:
    cells = residualize_conf(pd.read_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[fm]}_{TAG_YEARS}", "parquet")),
                             ["conf_split", "conf_lpi", "conf_borda", "conf_npatch"])
    segs = pd.read_parquet(cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[fm]}_{TAG_SEG}", "parquet"))

    for outcome in ["y_resto", "y_cont"]:
        for conf, lab in [(CELL_CONF, "bruta"), ([f"{c}_res" for c in CELL_CONF], "sem quantidade"),
                          (CELL_CONF_ALT, "m_eff e borda sem quantidade")]:
            s_, c_ = fit_cells(cells, outcome, conf, f"{outcome} | config {lab}")
            s_["combustivel"] = fm
            cell_rows.append(s_); cell_coefs.append(c_.assign(combustivel=fm))

    r_ = fit_scar_vs_reference(segs); r_.update(combustivel=fm, regra=REGRA_PRINCIPAL); ref_rows.append(r_)
    r_ = fit_scar_vs_reference(segs.assign(resultado=segs["resultado_mb"]))
    r_.update(combustivel=fm, regra=REGRA_SENS); ref_rows.append(r_)

    s_, c_ = fit_segments(segs, SEG_CONF, "segmento | config")
    s_["combustivel"] = fm
    seg_rows.append(s_); seg_coefs_all.append(c_.assign(combustivel=fm))
    for conf, lab in [(["conf_split_win", "conf_lpi_win"], "segmento | so paisagem (sem area/largura)"),
                      (["log_area_cicatriz", "largura_local_100m"], "segmento | so cicatriz (area/largura)")]:
        s2, c2 = fit_segments(segs, conf, lab)
        s2["combustivel"] = fm
        seg_sens_rows.append(s2); seg_sens_coefs.append(c2.assign(combustivel=fm))
    dc = prep_segments(segs); dc = dc[dc["cicatriz"] == 1]
    seg_corr.append(dc[["q_inicio_win"] + SEG_CONF].corr(method="spearman").round(2).assign(combustivel=fm))

cell_summary = pd.DataFrame(cell_rows); cell_coefs = pd.concat(cell_coefs)
ref_vs = pd.DataFrame(ref_rows)
seg_summary = pd.DataFrame(seg_rows); seg_coefs = pd.concat(seg_coefs_all)
pd.DataFrame(seg_sens_rows).to_csv(stats_path("B09_segmento_sensibilidade_resumo"), index=False)
pd.concat(seg_sens_coefs).to_csv(stats_path("B09_segmento_sensibilidade_coeficientes"))
pd.concat(seg_corr).to_csv(stats_path("B09_segmento_correlacoes"))
cell_summary.to_csv(stats_path("B09_celula_resumo"), index=False)
cell_coefs.to_csv(stats_path("B09_celula_coeficientes"))
ref_vs.to_csv(stats_path("B09_cicatriz_vs_referencia"), index=False)
seg_summary.to_csv(stats_path("B09_segmento_resumo"), index=False)
seg_coefs.to_csv(stats_path("B09_segmento_coeficientes"))

console.print("[bold]CELULA[/bold]"); display(cell_summary.set_index(["combustivel", "modelo"]).round(4))
console.print("[bold]SEGMENTO (a): cicatriz x referencia[/bold]"); display(ref_vs.set_index(["combustivel", "regra"]).round(3))
console.print("[bold]SEGMENTO (b): referencia x configuracao[/bold]"); display(seg_summary.set_index("combustivel").T.round(4))
display(seg_coefs.round(3))

# ============================================================
# 4. CRITERIO DE EVIDENCIA (proposta, item 7)
# ============================================================

vrows = []
for fm in FUEL_RUNS:
    c_main = cell_summary[(cell_summary["combustivel"] == fm) & (cell_summary["modelo"] == "y_resto | config bruta")].iloc[0]
    s_main = seg_summary[seg_summary["combustivel"] == fm].iloc[0]
    vrows += [
        {"combustivel": fm, "escala": "celula",
         "configuracao associada (Wald p < 0,05)": c_main["wald_p"] < 0.05,
         "melhora o ajuste": c_main["ganho_r2_within"] > 0,
         "melhora a previsao fora da amostra": c_main["ganho_previsao_fora_TI_%"] > 0},
        {"combustivel": fm, "escala": "segmento",
         "configuracao associada (Wald p < 0,05)": s_main["wald_p"] < 0.05,
         "melhora o ajuste": s_main["aic_conf"] < s_main["aic_ref"],
         "melhora a previsao fora da amostra": bool((s_main["cv_incendio_logloss_conf"] < s_main["cv_incendio_logloss_ref"])
                                                    and (s_main["cv_TI_logloss_conf"] < s_main["cv_TI_logloss_ref"]))}]
verdict = pd.DataFrame(vrows)
verdict.to_csv(stats_path("B09_criterio_evidencia"), index=False)
display(verdict)
console.print("[bold green]OK BLOCO 12 completo.[/bold green]")


## 13 inferencia_poucas_TIs

In [ ]:
# ======================================================================
# BLOCO 13 - Configuracao alem da quantidade, com inferencia valida para poucas TIs
# Depende dos blocos 00, 09 (celulas com SES) e 10 (segmentos).
# ======================================================================

check_version()

# ============================================================
# BLOCK 9b - Few-cluster inference and null-model configuration
#
# Por que: com 8 TIs, o erro-padrao agrupado por TI (CRV1) subestima a
# incerteza e rejeita demais (Cameron, Gelbach & Miller 2008; MacKinnon &
# Webb 2017). Aqui:
#   CELULA (linear, efeitos fixos de celula e ano):
#     configuracao = SES do modelo nulo (bloco 09), independente da quantidade
#     por construcao; p-valor por WILD CLUSTER BOOTSTRAP com pesos de Webb
#     (6 pontos, recomendado para G < 12), 9999 replicas, cluster = TI;
#     + jackknife por TI (CR3) e previsao deixando uma TI de fora.
#   SEGMENTO (logistico, bordas de cicatriz com resultado definido):
#     configuracao = area e largura da cicatriz + SES da celula onde o
#     segmento esta; EP agrupado (incendio + TI) e EP JACKKNIFE POR TI (CR3);
#     OR deixando cada TI de fora (amplitude). Cicatriz x referencia idem.
#   DETECTABILIDADE: o modelo de segmento e refeito com o estado da faixa
#     interna (NBR antes, nbr_pre_int) e o tempo desde a cicatriz (gap_meses)
#     como covariaveis: se a configuracao so "funciona" pelo NBR, ela some.
#
# Criterio de evidencia (proposta, item 7), agora com inferencia de poucos
# clusters: associacao (p do wild bootstrap ou IC jackknife < 0,05), melhora do
# ajuste e da previsao fora da amostra, no sentido esperado.
#
# Output: 04_stats/B09b_*.csv | 05_figures/B09b_efeitos.{tif,png,pdf} (600 dpi)
# ============================================================

import subprocess, sys
try:
    import wildboottest  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wildboottest"], check=True)
import pyfixest as pf
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

# ============================================================
# 1. PARAMETERS
# ============================================================

TAG_SEG   = globals().get("TAG_SEG", f"{TAG_YEARS}_md")
B_WILD    = 9999
SEED      = 11
CELL_REF  = ["q_inicio", "q_inicio_viz", "q_ant1", "q_ant2", "pastagem", "agricultura", "vpd_resto", "chuva_seca"]
CELL_SES  = ["conf_split_ses_0", "conf_lpi_ses_0", "conf_npatch_ses_0"]
SEG_REF   = ["q_inicio_win", "q_ant1_win", "q_ant2_win", "pastagem_win", "agricultura_win", "decl_ext",
             "barreira_win", "rios_win", "P_vpd_max", "P_wind", "P_rh_min"]
SEG_CONF  = ["log_area_cicatriz", "largura_local_100m", "ses_cel_split"]
SEG_DETEC = ["nbr_pre_int", "gap_meses"]
EXPECT    = {"conf_split_ses_0": +1, "conf_lpi_ses_0": +1, "conf_npatch_ses_0": -1,   # celula: y_resto
             "log_area_cicatriz": +1, "largura_local_100m": +1, "ses_cel_split": -1}  # segmento: OR interrupcao

def zscore(d, cols):
    d = d.copy()
    for c in cols:
        s = d[c].std()
        d[c] = (d[c] - d[c].mean()) / s if s and s > 0 else 0.0
    return d

def jackknife_ti(fit_fn, d, names):
    """CR3: re-estima deixando cada TI de fora. Devolve EP jackknife e amplitude dos coeficientes."""
    ids = sorted(d["ti_id"].unique()); G = len(ids)
    B = []
    for t in ids:
        try:
            b = fit_fn(d[d["ti_id"] != t])
            B.append([b.get(n, np.nan) for n in names])
        except Exception:
            B.append([np.nan] * len(names))
    B = np.array(B, float)
    mu = np.nanmean(B, axis=0)
    se = np.sqrt((G - 1) / G * np.nansum((B - mu) ** 2, axis=0))
    return se, np.nanmin(B, axis=0), np.nanmax(B, axis=0)

# ============================================================
# 2. CELULA - SES DO MODELO NULO + WILD CLUSTER BOOTSTRAP
# ============================================================

cells = pd.read_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[FUEL_MAIN]}_{TAG_YEARS}_nulo", "parquet"))
cells["sem_cicatriz"] = cells["sem_cicatriz"].astype(float)
dc = cells.dropna(subset=["y_resto"] + CELL_REF + CELL_SES).copy()
dc = zscore(dc, CELL_REF + CELL_SES)
f_ref  = "y_resto ~ " + " + ".join(CELL_REF + ["sem_cicatriz"]) + " | cell_id + year"
f_conf = "y_resto ~ " + " + ".join(CELL_REF + ["sem_cicatriz"] + CELL_SES) + " | cell_id + year"
m_ref  = pf.feols(f_ref, data=dc, vcov={"CRV1": "ti_id"})
m_conf = pf.feols(f_conf, data=dc, vcov={"CRV1": "ti_id"})

cell_rows = []
for v in CELL_SES:
    try:
        wb = m_conf.wildboottest(param=v, reps=B_WILD, cluster="ti_id", weights_type="webb", seed=SEED)
        p_wild = float(wb["Pr(>|t|)"]) if "Pr(>|t|)" in wb else float(wb.iloc[-1])
    except Exception as e:
        console.print(f"[yellow]wild bootstrap falhou para {v}: {str(e)[:100]}[/yellow]"); p_wild = np.nan
    cell_rows.append({"variavel": v, "coef_pp_por_dp": 100 * float(m_conf.coef()[v]),
                      "ep_crv1_pp": 100 * float(m_conf.se()[v]), "p_crv1": float(m_conf.pvalue()[v]),
                      "p_wild_webb": p_wild})
cell_tab = pd.DataFrame(cell_rows).set_index("variavel")

def _fit_cell_coefs(d_):
    return pf.feols(f_conf, data=d_, vcov="iid").coef().to_dict()
se_jk, lo_jk, hi_jk = jackknife_ti(_fit_cell_coefs, dc, CELL_SES)
cell_tab["ep_jackknife_TI_pp"] = 100 * se_jk
cell_tab["min_deixando_1_TI_pp"], cell_tab["max_deixando_1_TI_pp"] = 100 * lo_jk, 100 * hi_jk
cell_tab["sinal_esperado"] = [EXPECT[v] for v in cell_tab.index]

# previsao fora da TI (sem efeito fixo de celula; com relevo/barreira e ano)
pool = CELL_REF + ["sem_cicatriz", "declividade", "barreira", "rios"]
dp = zscore(dc.dropna(subset=["declividade", "barreira", "rios"]), ["declividade", "barreira", "rios"])
Yd = pd.get_dummies(dp["year"].astype(str), drop_first=True).astype(float).to_numpy()
err = {"ref": [], "conf": []}
for t in sorted(dp["ti_id"].unique()):
    tr_, te_ = (dp["ti_id"] != t).to_numpy(), (dp["ti_id"] == t).to_numpy()
    for k, xs in [("ref", pool), ("conf", pool + CELL_SES)]:
        X = np.c_[np.ones(len(dp)), dp[xs].to_numpy(), Yd]
        b, *_ = np.linalg.lstsq(X[tr_], dp["y_resto"].to_numpy()[tr_], rcond=None)
        err[k].append((dp["y_resto"].to_numpy()[te_] - X[te_] @ b) ** 2)
rmse = {k: float(np.sqrt(np.concatenate(v).mean())) for k, v in err.items()}
def _r2w(m):
    v = getattr(m, "_r2_within", None)
    return float(v) if v is not None else np.nan
cell_summ = {"n": len(dc), "n_TIs": dc["ti_id"].nunique(), "r2_within_ref": _r2w(m_ref),
             "r2_within_conf": _r2w(m_conf), "rmse_fora_TI_ref": rmse["ref"],
             "rmse_fora_TI_conf": rmse["conf"], "ganho_previsao_%": 100 * (rmse["ref"] - rmse["conf"]) / rmse["ref"]}
cell_tab.to_csv(stats_path("B09b_celula_ses_coeficientes"))
pd.DataFrame([cell_summ]).to_csv(stats_path("B09b_celula_ses_resumo"), index=False)
console.print("[bold]CELULA - configuracao pelo modelo nulo (p.p. de area queimada depois por 1 DP de SES)[/bold]")
display(cell_tab.round(4)); display(pd.DataFrame([cell_summ]).round(4))

# ============================================================
# 3. SEGMENTO - SES DA CELULA NO SEGMENTO + JACKKNIFE POR TI
# ============================================================

segs = pd.read_parquet(cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet"))
size = CELL_M
orig = (cells.assign(c0=lambda d: d["x"] - (d["col"] + 0.5) * size, f0=lambda d: d["y"] + (d["row"] + 0.5) * size)
             .groupby("ti_id")[["c0", "f0"]].median())
segs = segs.merge(orig, left_on="ti_id", right_index=True, how="left")
segs["row"] = np.floor((segs["f0"] - segs["y"]) / size).astype("Int64")
segs["col"] = np.floor((segs["x"] - segs["c0"]) / size).astype("Int64")
segs = segs.merge(cells[["ti_id", "year", "row", "col", "conf_split_ses_0"]].rename(
    columns={"conf_split_ses_0": "ses_cel_split"}).astype({"row": "Int64", "col": "Int64"}),
    on=["ti_id", "year", "row", "col"], how="left")

def prep(segs):
    d = segs[segs["resultado"].isin(["interrupcao", "continuidade"])].copy()
    d["interrupcao"] = (d["resultado"] == "interrupcao").astype(int)
    d["cicatriz"] = (d["tipo_seg"] == "cicatriz").astype(int)
    d["log_area_cicatriz"] = np.log(d["area_cicatriz_ha"].clip(lower=0.09))
    d["largura_local_100m"] = d["largura_local_m"] / 100
    sem_id = d["fire_id"].astype(str).str.contains("_v")
    d["fire_id"] = np.where(sem_id, d["ti_id"].astype(str) + "_" + d["year"].astype(str) + "_semid", d["fire_id"])
    return d

def glm_fit(formula, d, cluster=True):
    m = smf.glm(formula, data=d, family=sm.families.Binomial())
    if not cluster:
        return m.fit()
    g = np.column_stack([pd.factorize(d["fire_id"])[0], pd.factorize(d["ti_id"])[0]])
    try:
        return m.fit(cov_type="cluster", cov_kwds={"groups": g})
    except Exception:
        return m.fit(cov_type="cluster", cov_kwds={"groups": g[:, 1]})

def or_table(m, d, formula, names, label):
    se_jk, lo, hi = jackknife_ti(lambda d_: glm_fit(formula, d_, cluster=False).params.to_dict(), d, names)
    b = m.params[names]
    return pd.DataFrame({"modelo": label, "or_1dp": np.exp(b),
                         "ic95_cluster_inf": np.exp(m.conf_int().loc[names, 0]),
                         "ic95_cluster_sup": np.exp(m.conf_int().loc[names, 1]),
                         "ic95_jackknife_TI_inf": np.exp(b - 1.96 * se_jk),
                         "ic95_jackknife_TI_sup": np.exp(b + 1.96 * se_jk),
                         "or_min_deixando_1_TI": np.exp(lo), "or_max_deixando_1_TI": np.exp(hi),
                         "sinal_esperado": [EXPECT.get(n, 0) for n in names]}, index=names)

d = prep(segs)
seg_tabs = []

# (a) cicatriz x referencia
da = zscore(d.dropna(subset=SEG_REF), SEG_REF)
fa = "interrupcao ~ cicatriz + " + " + ".join(SEG_REF) + " + C(year)"
ma = glm_fit(fa, da)
seg_tabs.append(or_table(ma, da, fa, ["cicatriz"], "cicatriz x referencia"))
# (a2) o mesmo, ajustado pelo estado da faixa interna (a deteccao depende dele)
da2 = zscore(da.dropna(subset=["nbr_pre_int"]), ["nbr_pre_int"])
fa2 = fa + " + nbr_pre_int"
seg_tabs.append(or_table(glm_fit(fa2, da2), da2, fa2, ["cicatriz"], "cicatriz x referencia | + NBR antes"))

# (b) so bordas de cicatriz: referencia x + configuracao (+ deteccao)
sc = d[d["cicatriz"] == 1].dropna(subset=SEG_REF + SEG_CONF + SEG_DETEC).copy()
sc = zscore(sc, SEG_REF + SEG_CONF + SEG_DETEC)
fb_ref  = "interrupcao ~ " + " + ".join(SEG_REF) + " + C(year)"
fb_conf = "interrupcao ~ " + " + ".join(SEG_REF + SEG_CONF) + " + C(year)"
fb_det  = "interrupcao ~ " + " + ".join(SEG_REF + SEG_CONF + SEG_DETEC) + " + C(year)"
mb_ref, mb_conf, mb_det = glm_fit(fb_ref, sc), glm_fit(fb_conf, sc), glm_fit(fb_det, sc)
seg_tabs.append(or_table(mb_conf, sc, fb_conf, SEG_CONF, "cicatriz | configuracao"))
seg_tabs.append(or_table(mb_det, sc, fb_det, SEG_CONF + SEG_DETEC, "cicatriz | configuracao + deteccao"))
seg_tab = pd.concat(seg_tabs)

def cv(d_, f_list, splitter, groups):
    preds = {k: np.full(len(d_), np.nan) for k in f_list}
    for tr_, te_ in splitter.split(d_, d_["interrupcao"], groups):
        for k, f in f_list.items():
            try:
                preds[k][te_] = smf.glm(f, data=d_.iloc[tr_], family=sm.families.Binomial()).fit().predict(d_.iloc[te_])
            except Exception:
                pass
    ok = np.all([~np.isnan(p) for p in preds.values()], axis=0)
    y = d_["interrupcao"].to_numpy()[ok]
    return {k: {"logloss": log_loss(y, np.clip(p[ok], 1e-6, 1 - 1e-6)),
                "auc": roc_auc_score(y, p[ok]) if len(np.unique(y)) > 1 else np.nan} for k, p in preds.items()}

scr = sc.reset_index(drop=True)
fl = {"ref": fb_ref, "conf": fb_conf}
cv_f = cv(scr, fl, GroupKFold(5), scr["fire_id"].to_numpy())
cv_t = cv(scr, fl, LeaveOneGroupOut(), scr["ti_id"].to_numpy())
seg_summ = {"n_segmentos": len(sc), "n_incendios": sc["fire_id"].nunique(), "n_TIs": sc["ti_id"].nunique(),
            "aic_ref": mb_ref.aic, "aic_conf": mb_conf.aic,
            "logloss_incendios_fora_ref": cv_f["ref"]["logloss"], "logloss_incendios_fora_conf": cv_f["conf"]["logloss"],
            "logloss_TI_fora_ref": cv_t["ref"]["logloss"], "logloss_TI_fora_conf": cv_t["conf"]["logloss"],
            "auc_TI_fora_ref": cv_t["ref"]["auc"], "auc_TI_fora_conf": cv_t["conf"]["auc"]}
seg_tab.to_csv(stats_path("B09b_segmento_or_jackknife"))
pd.DataFrame([seg_summ]).to_csv(stats_path("B09b_segmento_resumo"), index=False)
console.print("[bold]SEGMENTO - razoes de chances com IC por cluster e por jackknife de TI[/bold]")
display(seg_tab.round(3)); display(pd.DataFrame([seg_summ]).round(4))

# ============================================================
# 4. CRITERIO DE EVIDENCIA (inferencia de poucos clusters)
# ============================================================

def _sig_dir_cell(v):
    r = cell_tab.loc[v]
    return (r["p_wild_webb"] < 0.05) and (np.sign(r["coef_pp_por_dp"]) == r["sinal_esperado"])

def _sig_dir_seg(v, label="cicatriz | configuracao"):
    r = seg_tab[(seg_tab["modelo"] == label)].loc[v]
    lo, hi = r["ic95_jackknife_TI_inf"], r["ic95_jackknife_TI_sup"]
    return ((lo > 1) or (hi < 1)) and (np.sign(np.log(r["or_1dp"])) == r["sinal_esperado"])

verd = pd.DataFrame([
    {"escala": "celula", "medida": v, "associada_no_sentido_esperado (wild p<0,05)": _sig_dir_cell(v),
     "melhora_ajuste": cell_summ["r2_within_conf"] > cell_summ["r2_within_ref"],
     "melhora_previsao_fora_TI": cell_summ["ganho_previsao_%"] > 0} for v in CELL_SES] + [
    {"escala": "segmento", "medida": v, "associada_no_sentido_esperado (jackknife TI)": _sig_dir_seg(v),
     "robusta_a_deteccao": _sig_dir_seg(v, "cicatriz | configuracao + deteccao"),
     "melhora_ajuste": seg_summ["aic_conf"] < seg_summ["aic_ref"],
     "melhora_previsao_fora_TI": seg_summ["logloss_TI_fora_conf"] < seg_summ["logloss_TI_fora_ref"]} for v in SEG_CONF])
verd.to_csv(stats_path("B09b_criterio_evidencia"), index=False)
console.print("[bold]Criterio de evidencia com inferencia valida para 8 TIs[/bold]")
display(verd)

# ============================================================
# 5. FIGURA (600 dpi): efeitos com os dois tipos de IC
# ============================================================

LAB = {"conf_split_ses_0": "Fragmentation SES", "conf_lpi_ses_0": "Largest-patch SES",
       "conf_npatch_ses_0": "N patches SES", "log_area_cicatriz": "Scar area (log)",
       "largura_local_100m": "Local scar width", "ses_cel_split": "Cell fragmentation SES",
       "nbr_pre_int": "Inner-strip NBR before", "gap_meses": "Months since scar"}
fig, axs = plt.subplots(1, 2, figsize=(17 / 2.54, 6.5 / 2.54), layout="constrained")
ax = axs[0]; y = np.arange(len(cell_tab))[::-1]
ax.axvline(0, color="black", lw=0.6, ls="--")
ax.errorbar(cell_tab["coef_pp_por_dp"], y, xerr=1.96 * cell_tab["ep_jackknife_TI_pp"], fmt="o", color="#1f5fc8", capsize=2)
for yi, (v, r) in zip(y, cell_tab.iterrows()):
    ax.text(r["coef_pp_por_dp"], yi + 0.25, f"wild p = {r['p_wild_webb']:.3f}", fontsize=6, ha="center")
ax.set_yticks(y); ax.set_yticklabels([LAB[v] for v in cell_tab.index], fontsize=7)
ax.set_xlabel("Later burned area (p.p.) per SD"); ax.set_title(f"(a) {CELL_M / 1000:.1f}-km cells", fontsize=8, loc="left")
ax = axs[1]
t = seg_tab[seg_tab["modelo"] == "cicatriz | configuracao + deteccao"]
y = np.arange(len(t))[::-1]
ax.axvline(1, color="black", lw=0.6, ls="--")
ax.errorbar(t["or_1dp"], y, xerr=[t["or_1dp"] - t["ic95_jackknife_TI_inf"], t["ic95_jackknife_TI_sup"] - t["or_1dp"]],
            fmt="o", color="#1f5fc8", capsize=2, label="jackknife by IL")
ax.errorbar(t["or_1dp"], y - 0.18, xerr=[t["or_1dp"] - t["ic95_cluster_inf"], t["ic95_cluster_sup"] - t["or_1dp"]],
            fmt="none", ecolor="#9a9992", capsize=2, label="clustered (fire + IL)")
ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels([LAB.get(v, v) for v in t.index], fontsize=7)
ax.set_xlabel("Odds of interruption per SD"); ax.legend(fontsize=6, frameon=False)
ax.set_title("(b) scar edges, adjusted for detectability", fontsize=8, loc="left")
for ext, kw in [("tif", {"pil_kwargs": {"compression": "tiff_lzw"}}), ("png", {}), ("pdf", {})]:
    fig.savefig(fig_path("B09b_efeitos", ext), dpi=600, facecolor="white", **kw)
plt.show(); plt.close(fig)

console.print("[bold green]OK BLOCO 13 completo.[/bold green]")


## 14 robustez

In [ ]:
# ======================================================================
# BLOCO 14 - Robustez: recorte sazonal, escala, criterios dos encontros, combustivel
# Depende das FUNCOES dos blocos 08, 10 e 12 (rode-os nesta sessao antes).
# ======================================================================
#
# Refaz as analises mudando UMA coisa de cada vez:
#   (1) recorte sazonal : artigo1 (principal) | local (60 mm) | local_100 (100 mm)
#   (2) escala (celula) : 2490 | 4980 (principal) | 9990 m
#   (3) encontros       : segmento 150/600 m; faixa 60/150 m; chegada 20%/50%
#   (4) combustivel     : todos os vegetados (principal) | natural + pastagem | so natural
# Para cada variante: n, teste do bloco de configuracao, ganho de ajuste/previsao
# e o coeficiente da configuracao (celula: n de manchas; segmento: conf_split_win).
# Cada variante de segmentos refaz todas as TIs-ano (dezenas de minutos).
#
# Output: 04_stats/B10_robustez_celulas.csv, B10_robustez_segmentos.csv
# ======================================================================

check_version()

RUN_SAZONAL, RUN_ESCALA, RUN_ENCONTROS, RUN_COMBUSTIVEL = True, True, True, True

SCHEMES = [s for s in ["artigo1", "local", "local_100"] if s in load_seasons()]
ENC_VARIANTS = {
    "padrao":           {},
    "segmento 150 m":   {"seg_len": 150},
    "segmento 600 m":   {"seg_len": 600},
    "faixa 60 m":       {"strip": 60},
    "faixa 150 m":      {"strip": 150},
    "chegada >= 20%":   {"arrival": 0.2},
    "chegada >= 50%":   {"arrival": 0.5},
}
for fn in ["build_cells", "build_segments", "fit_cells", "fit_segments", "fit_scar_vs_reference"]:
    if fn not in globals():
        raise RuntimeError(f"Funcao {fn} nao definida - rode os blocos 08, 10 e 12 nesta sessao antes do 14.")

CELL_TRACK = "conf_npatch"

def cell_row(cells, grupo, variante):
    s, c = fit_cells(cells, "y_resto", CELL_CONF, f"{grupo}: {variante}")
    est = c.loc[CELL_TRACK] if CELL_TRACK in c.index else None
    s.update({"grupo": grupo, "variante": variante,
              "coef_conf_celula": est["Estimate"] if est is not None else np.nan,
              "ic_inf": est["2.5%"] if est is not None else np.nan,
              "ic_sup": est["97.5%"] if est is not None else np.nan})
    return s

def seg_row(segs, grupo, variante):
    s, c = fit_segments(segs, SEG_CONF, f"{grupo}: {variante}")
    r = fit_scar_vs_reference(segs)
    s.update({"grupo": grupo, "variante": variante,
              "rc_conf_split": c.loc["conf_split_win", "razao_chances_1dp"],
              "ic_inf": c.loc["conf_split_win", "ic95_inf"], "ic_sup": c.loc["conf_split_win", "ic95_sup"],
              "rc_cicatriz_vs_ref": r["razao_chances_cicatriz"]})
    return s

rows_c, rows_s = [], []
if RUN_SAZONAL:
    for sch in SCHEMES:
        console.print(f"[cyan]Recorte sazonal: {sch}[/cyan]")
        rows_c.append(cell_row(build_cells(sch, CELL_M), "sazonal", sch))
        sg, _ = build_segments(sch)
        rows_s.append(seg_row(sg, "sazonal", sch))
if RUN_ESCALA:
    for cm in CELL_SIZES:
        console.print(f"[cyan]Escala: {cm} m[/cyan]")
        rows_c.append(cell_row(build_cells(SEASON_MAIN, cm), "escala", f"{cm} m"))
if RUN_ENCONTROS:
    for lab, P in ENC_VARIANTS.items():
        console.print(f"[cyan]Encontros: {lab}[/cyan]")
        sg, _ = build_segments(SEASON_MAIN, P)
        rows_s.append(seg_row(sg, "encontros", lab))
if RUN_COMBUSTIVEL:
    for fm in FUEL_MODES:
        if fm == FUEL_MAIN:
            continue
        console.print(f"[cyan]Combustivel: {fm}[/cyan]")
        rows_c.append(cell_row(build_cells(SEASON_MAIN, CELL_M, fuel_mode=fm), "combustivel", fm))
        sg, _ = build_segments(SEASON_MAIN, fuel_mode=fm)
        rows_s.append(seg_row(sg, "combustivel", fm))

rob_c, rob_s = pd.DataFrame(rows_c), pd.DataFrame(rows_s)
rob_c.to_csv(stats_path("B10_robustez_celulas"), index=False)
rob_s.to_csv(stats_path("B10_robustez_segmentos"), index=False)
display(rob_c[["grupo", "variante", "n", "coef_conf_celula", "ic_inf", "ic_sup", "wald_p",
               "ganho_r2_within", "ganho_previsao_fora_TI_%"]].round(4))
display(rob_s[["grupo", "variante", "n_segmentos", "prop_interrupcao", "rc_conf_split", "ic_inf", "ic_sup",
               "wald_p", "cv_TI_logloss_ref", "cv_TI_logloss_conf", "rc_cicatriz_vs_ref"]].round(4))

if len(rob_c):
    ref_sign = np.sign(rob_c["coef_conf_celula"].iloc[0])
    console.print(f"Celula: mesmo sinal em {(np.sign(rob_c['coef_conf_celula']) == ref_sign).sum()}/{len(rob_c)}; "
                  f"ganho de previsao fora da TI em {(rob_c['ganho_previsao_fora_TI_%'] > 0).sum()}/{len(rob_c)}.")
if len(rob_s):
    ref_sign = np.sign(np.log(rob_s["rc_conf_split"].iloc[0]))
    ganho = (rob_s["cv_TI_logloss_conf"] < rob_s["cv_TI_logloss_ref"]) & \
            (rob_s["cv_incendio_logloss_conf"] < rob_s["cv_incendio_logloss_ref"])
    console.print(f"Segmento: mesmo sinal em {(np.sign(np.log(rob_s['rc_conf_split'])) == ref_sign).sum()}/"
                  f"{len(rob_s)}; melhora a previsao em {ganho.sum()}/{len(rob_s)}.")
console.print("[bold green]OK BLOCO 14 completo.[/bold green]")


## 15 sintese_texto

In [ ]:
# ======================================================================
# BLOCO 15 - Sintese: Materiais e Metodos + Resultados (PT e EN)
# Depende do bloco 00 e das tabelas gravadas pelos blocos 02-14. Segundos.
# ======================================================================
#
# Le os resultados ja salvos e escreve:
#   08_textos/RSL_metodos_resultados_PT.docx (+ .md)
#   08_textos/RSL_methods_results_EN.docx   (+ .md)
# Os numeros vem SEMPRE das tabelas; a interpretacao segue regras fixas e e
# cautelosa: nada e atribuido as BRIFs; ausencia de fogo nao e contencao;
# associacao, nao causalidade. Bloco nao rodado -> "[falta: rode o bloco xx]".
# Rascunho fiel aos numeros: revise antes de usar.
# ======================================================================

check_version()

import docx
from docx.shared import Pt

OUT_DIR = PROJECT_ROOT / "08_textos"
OUT_DIR.mkdir(exist_ok=True)
REGRA_PRINCIPAL = "simetrica (Landsat + MODIS)"
REGRA_SENS      = "antiga (MapBiomas na referencia)"

def _csv(path, **kw):
    return pd.read_csv(path, **kw) if Path(path).exists() else None

def _json(path):
    return json.loads(Path(path).read_text(encoding="utf-8")) if Path(path).exists() else None

R = {
    "amostra":   _csv(table_path("amostra_final_tis")),
    "tipo":      _csv(stats_path("B06_tipologia_resumo"), index_col=0),
    "d1":        _csv(stats_path("D1_tempo")),
    "d2":        _csv(stats_path("D2_foco_noturno"), index_col=0),
    "d2_taxa":   _csv(stats_path("D2_foco_noturno_taxa"), index_col=0),
    "corr":      _csv(stats_path("B07_correlacao_quantidade_configuracao"), index_col=0),
    "ses_corr":  _csv(stats_path("B07b_correlacao_ses_quantidade"), index_col=0),
    "ses_res":   _csv(stats_path("B07b_ses_resumo")),
    "regra":     _csv(stats_path("B08a_regra_antiga_x_simetrica")),
    "ordem":     _csv(stats_path("B08_ordem_qualidade"), index_col=0),
    "conc":      _csv(stats_path("B08_concordancia_mapbiomas_landsat")),
    "modis_c":   _csv(stats_path("B08_contribuicao_modis"), index_col=0),
    "det_sens":  _csv(stats_path("B08b_sensibilidade_landsat"), index_col=0),
    "det_rr":    _csv(stats_path("B08b_modis_razao_taxas")),
    "det_curva": _csv(stats_path("B08b_curva_limiar")),
    "cell":      _csv(stats_path("B09_celula_resumo")),
    "ref":       _csv(stats_path("B09_cicatriz_vs_referencia")),
    "seg":       _csv(stats_path("B09_segmento_resumo")),
    "seg_coef":  _csv(stats_path("B09_segmento_coeficientes"), index_col=0),
    "b_cell":    _csv(stats_path("B09b_celula_ses_coeficientes"), index_col=0),
    "b_cell_s":  _csv(stats_path("B09b_celula_ses_resumo")),
    "b_seg":     _csv(stats_path("B09b_segmento_or_jackknife"), index_col=0),
    "b_seg_s":   _csv(stats_path("B09b_segmento_resumo")),
    "b_verd":    _csv(stats_path("B09b_criterio_evidencia")),
    "rob_s":     _csv(stats_path("B10_robustez_segmentos")),
    "rob_c":     _csv(stats_path("B10_robustez_celulas")),
}
faltam = [k for k, v in R.items() if v is None]
console.print(f"[cyan]Tabelas encontradas:[/cyan] {len(R) - len(faltam)}/{len(R)}"
              + (f" | [yellow]faltam: {faltam}[/yellow]" if faltam else ""))

def num(x, d=2, lang="pt"):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "NA"
    s = f"{x:,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".") if lang == "pt" else s

def pval(p, lang="pt"):
    if p is None or np.isnan(p):
        return "p = NA"
    if p < 0.001:
        return "p < 0,001" if lang == "pt" else "p < 0.001"
    return f"p = {num(p, 3, lang)}"

def falta(bloco, lang):
    return f"[falta: rode o bloco {bloco}]" if lang == "pt" else f"[missing: run block {bloco}]"

VAR = {"pt": {"conf_split_ses_0": "fragmentacao (SES)", "conf_lpi_ses_0": "maior mancha (SES)",
              "conf_npatch_ses_0": "numero de manchas (SES)", "log_area_cicatriz": "area da cicatriz (log)",
              "largura_local_100m": "largura local da cicatriz", "ses_cel_split": "fragmentacao da celula (SES)",
              "nbr_pre_int": "NBR da faixa interna antes", "gap_meses": "meses desde a cicatriz"},
       "en": {"conf_split_ses_0": "fragmentation (SES)", "conf_lpi_ses_0": "largest patch (SES)",
              "conf_npatch_ses_0": "number of patches (SES)", "log_area_cicatriz": "scar area (log)",
              "largura_local_100m": "local scar width", "ses_cel_split": "cell fragmentation (SES)",
              "nbr_pre_int": "inner-strip NBR before", "gap_meses": "months since scar"}}

# ============================================================
# MATERIAIS E METODOS
# ============================================================

def methods(lang):
    T = lambda pt, en: pt if lang == "pt" else en
    out = [("h1", T("Materiais e Metodos", "Materials and Methods"))]
    am = R["amostra"]
    if am is not None:
        eleg = am[am["elegivel"].astype(bool)]
        nomes = ", ".join(eleg["ti_name"].astype(str))
        out += [("h2", T("Area de estudo e amostra", "Study area and sample")),
                ("p", T(f"Partimos das 14 Terras Indigenas (TIs) do Cerrado com Brigadas Federais Indigenas (BRIF) "
                        f"do primeiro artigo, selecionadas pelo codigo FUNAI, recortadas no bioma e analisadas em "
                        f"EPSG:5880. O periodo e 2014-2024 (pos-BRIF); 2012-2013 entram so como historico de queima. "
                        f"Entraram as TIs com >= 6 anos de imagens Landsat 8/9 utilizaveis (cena <= 30% de nuvem, "
                        f"jun-out, <= 32 dias entre datas limpas) e >= 20 encontros entre cicatrizes do inicio da "
                        f"seca e queimadas posteriores: {len(eleg)} TIs ({nomes}).",
                        f"We started from the 14 Cerrado Indigenous Lands (ILs) with Indigenous Federal Brigades "
                        f"(BRIF) of our first article, selected by FUNAI code, clipped to the biome and analysed in "
                        f"EPSG:5880. The period is 2014-2024 (post-BRIF); 2012-2013 enter only as burn history. ILs "
                        f"were retained with >= 6 years of usable Landsat 8/9 imagery (scene cloud <= 30%, June-"
                        f"October, <= 32 days between clear dates) and >= 20 encounters between early dry-season "
                        f"scars and later fires: {len(eleg)} ILs ({nomes})."))]
    out += [
        ("h2", T("Dados", "Data")),
        ("p", T("Cicatrizes mensais (30 m) do MapBiomas Fogo Colecao 4 e cobertura do MapBiomas Colecao 10. A sequencia "
                "das queimadas foi verificada com o NBR mensal minimo do Landsat 8/9 Colecao 2 Nivel 2 e com focos "
                "ativos MODIS Colecao 6.1 (MOD14A1 Terra e MYD14A1 Aqua, 1 km, FireMask >= 8, como no primeiro artigo). "
                "Tempo diario do ERA5-Land, chuva mensal do CHIRPS, relevo do SRTM e rios do HydroSHEDS; agua e areas "
                "nao vegetadas foram barreiras permanentes.",
                "Monthly 30-m scars came from MapBiomas Fire Collection 4 and land cover from MapBiomas Collection 10. "
                "The burn sequence was verified with the monthly minimum NBR of Landsat 8/9 Collection 2 Level 2 and with "
                "MODIS Collection 6.1 daily active fires (MOD14A1 Terra and MYD14A1 Aqua, 1 km, FireMask >= 8, as in "
                "our first article). Daily weather came from ERA5-Land, monthly rainfall from CHIRPS, slope from SRTM "
                "and rivers from HydroSHEDS; water and non-vegetated areas were permanent barriers.")),
        ("h2", T("Encontros nas bordas", "Encounters at scar edges")),
        ("p", T("Bordas de cicatrizes iniciais (mai-jul) voltadas para combustivel nao queimado foram cortadas em "
                "segmentos de 300 m, com faixas externa e interna de 90 m. Houve CHEGADA quando >= 30% da faixa externa "
                "queimou em ago-out (MapBiomas) ou havia pixel MODIS com fogo a ate 1 km fora da borda; sem chegada, o "
                "segmento nao entra. Com chegada, CONTINUIDADE = nova queda de NBR >= 0,10 em >= 20% da faixa interna "
                "bem observada, ou centro de pixel MODIS com fogo >= 1 km dentro da cicatriz; INTERRUPCAO = faixa bem "
                "observada sem entrada. A ordem cicatriz -> chegada foi conferida com Landsat e MODIS. Como REFERENCIA, "
                "cada cicatriz com chegada recebeu uma pseudo-cicatriz de mesma forma, deslocada 2-10 km para vegetacao "
                "nao queimada da mesma TI, com a mesma regra.",
                "Edges of early scars (May-July) facing unburned fuel were cut into 300-m segments with 90-m outer and "
                "inner strips. A later fire ARRIVED when >= 30% of the outer strip burned in August-October (MapBiomas) "
                "or a MODIS fire pixel lay within 1 km outside the edge; segments without arrival were excluded. Given "
                "arrival, CONTINUITY = a new NBR drop >= 0.10 over >= 20% of a well-observed inner strip, or a MODIS "
                "fire-pixel centre >= 1 km inside the scar; INTERRUPTION = a well-observed strip with no entry. The scar "
                "-> arrival order was checked with Landsat and MODIS. As a REFERENCE, each scar with arrival received a "
                "same-shape pseudo-scar displaced 2-10 km into unburned vegetation of the same IL, with the same rule.")),
        ("h2", T("Detectabilidade", "Detectability")),
        ("p", T("Como uma nova queda de NBR e mais dificil de detectar sobre area ja queimada, o contraste cicatriz x "
                "referencia foi verificado (i) variando o limiar de queda (0,02-0,20, absoluto e relativo ao NBR "
                "anterior), (ii) medindo em que fracao o Landsat marca entrada quando o MODIS a confirma, e (iii) por um "
                "teste independente do NBR: a densidade de pixels-dia MODIS com fogo em ago-out nos nucleos das "
                "cicatrizes (> 1 km da borda) contra nucleos de combustivel nao queimado da mesma TI-ano (razao de "
                "taxas, Poisson com efeito fixo de TI-ano, IC por bootstrap de TIs).",
                "Because a new NBR drop is harder to detect over already burned ground, the scar-reference contrast was "
                "checked by (i) varying the drop threshold (0.02-0.20, absolute and relative to prior NBR), (ii) "
                "measuring how often Landsat flags entry when MODIS confirms it, and (iii) an NBR-independent test: the "
                "density of August-October MODIS fire pixel-days in scar cores (> 1 km from the edge) versus unburned "
                "fuel cores of the same IL-year (rate ratio, Poisson with IL-year fixed effects, IL bootstrap CI).")),
        ("h2", T("Configuracao alem da quantidade", "Configuration beyond quantity")),
        ("p", T(f"Em celulas de {num(CELL_M / 1000, 2, lang)} km, a configuracao do combustivel restante (fragmentacao "
                "pelo tamanho efetivo de malha, maior mancha e numero de manchas) foi comparada com um MODELO NULO: as "
                "cicatrizes de cada celula-ano foram reposicionadas ao acaso no combustivel disponivel, mantendo forma, "
                "area e numero (29 replicas), e a configuracao foi expressa como tamanho de efeito padronizado (SES). "
                "Assim, a configuracao independe da quantidade por construcao. Nas celulas, modelos lineares com efeitos "
                "fixos de celula e ano compararam a referencia (quantidade em t, vizinhas, t-1, t-2, cobertura, VPD e "
                "chuva) com o mesmo modelo + SES; nos segmentos de cicatriz, regressao logistica comparou a referencia "
                "(quantidade na janela de 1 km, cobertura, relevo, barreiras, tempo no dia da chegada, ano) com o mesmo "
                "modelo + area e largura local da cicatriz e SES da celula, e com o NBR da faixa interna antes do fogo "
                "como controle de detectabilidade. Com 8 TIs, a inferencia usou wild cluster bootstrap com pesos de "
                "Webb (9.999 replicas) nas celulas e jackknife por TI (CR3) nos segmentos; a previsao foi validada "
                "deixando incendios (5 grupos) ou TIs de fora. Configuracao so foi considerada relevante com "
                "associacao no sentido esperado, melhora do ajuste e da previsao fora da amostra.",
                f"In {num(CELL_M / 1000, 2, lang)}-km cells, the configuration of remaining fuel (fragmentation by "
                "effective mesh size, largest patch and number of patches) was compared with a NULL MODEL: the scars "
                "of each cell-year were randomly repositioned within available fuel, keeping shape, area and number "
                "(29 replicates), and configuration was expressed as a standardized effect size (SES), independent of "
                "quantity by construction. In cells, linear models with cell and year fixed effects compared a "
                "reference (quantity at t, neighbours, t-1, t-2, land cover, VPD and rainfall) with the same model + "
                "SES; at scar edges, logistic regression compared a reference (quantity in a 1-km window, land cover, "
                "terrain, barriers, arrival-day weather, year) with the same model + scar area, local width and cell "
                "SES, and with inner-strip NBR before the fire as a detectability control. With 8 ILs, inference used "
                "wild cluster bootstrap with Webb weights (9,999 replicates) for cells and IL jackknife (CR3) for "
                "edges; prediction was validated leaving fires (5 groups) or ILs out. Configuration was deemed relevant "
                "only with an association in the expected direction, better fit and better out-of-sample prediction.")),
        ("h2", T("Robustez", "Robustness")),
        ("p", T("Os modelos foram refeitos mudando uma escolha de cada vez: recorte sazonal (artigo 1; seca local "
                "CHIRPS < 60 e < 100 mm), celula (2,49; 4,98; 9,99 km), criterios dos encontros (segmento 150/600 m, "
                "faixa 60/150 m, chegada 20%/50%) e combustivel (toda vegetacao; natural + pastagem; so natural).",
                "Models were refitted changing one choice at a time: seasonal split (article 1; local CHIRPS < 60 and "
                "< 100 mm), cell size (2.49, 4.98, 9.99 km), encounter criteria (150/600-m segments, 60/150-m strips, "
                "20%/50% arrival) and fuel (all vegetation; natural + pasture; natural only).")),
    ]
    return out

# ============================================================
# RESULTADOS
# ============================================================

def results(lang):
    T = lambda pt, en: pt if lang == "pt" else en
    n = lambda x, d=2: num(x, d, lang)
    out = [("h1", T("Resultados", "Results"))]

    am = R["amostra"]
    if am is not None:
        excl = am[~am["elegivel"].astype(bool)]
        out += [("h2", T("Amostra", "Sample")),
                ("p", T(f"{int(am['elegivel'].sum())} das {len(am)} TIs entraram. Exclusoes: "
                        + "; ".join(f"{r.ti_name}: {r.motivo_exclusao}" for r in excl.itertuples()) + ".",
                        f"{int(am['elegivel'].sum())} of the {len(am)} ILs entered. Exclusions: "
                        + "; ".join(f"{r.ti_name}: {r.motivo_exclusao}" for r in excl.itertuples()) + "."))]

    out.append(("h2", T("Interrupcao nas bordas", "Interruption at scar edges")))
    rg = R["regra"]
    if rg is None:
        out.append(("p", falta("10", lang)))
    else:
        g = rg[rg["combustivel"] == FUEL_MAIN].set_index(["regra", "tipo_seg"])
        sim = g.xs(REGRA_PRINCIPAL, level="regra")
        tot = int(sim[["interrupcao", "continuidade", "indeterminado", "sem_chegada"]].sum().sum())
        sem = int(sim["sem_chegada"].sum())
        out.append(("p", T(
            f"Dos {n(tot, 0)} segmentos, {n(100 * sem / tot, 0)}% nao receberam fogo posterior e foram excluidos. O fogo "
            f"que chegou foi interrompido em {n(sim.loc['cicatriz', '% interrupcao'], 1)}% das bordas de cicatriz e em "
            f"{n(sim.loc['referencia', '% interrupcao'], 1)}% das bordas de referencia.",
            f"Of {n(tot, 0)} segments, {n(100 * sem / tot, 0)}% received no later fire and were excluded. Arriving fire was "
            f"interrupted at {n(sim.loc['cicatriz', '% interrupcao'], 1)}% of scar edges and at "
            f"{n(sim.loc['referencia', '% interrupcao'], 1)}% of reference edges.")))
    rf = R["ref"]
    if rf is not None:
        a = rf[(rf["combustivel"] == FUEL_MAIN) & (rf["regra"] == REGRA_PRINCIPAL)]
        if len(a):
            a = a.iloc[0]
            out.append(("p", T(f"Ajustada, a razao de chances de interrupcao cicatriz x referencia foi "
                               f"{n(a['razao_chances_cicatriz'], 1)} (IC {n(a['ic95_inf'], 1)}-{n(a['ic95_sup'], 1)}).",
                               f"Adjusted, the odds ratio of interruption at scar versus reference edges was "
                               f"{n(a['razao_chances_cicatriz'], 1)} (CI {n(a['ic95_inf'], 1)}-{n(a['ic95_sup'], 1)}).")))
    bs = R["b_seg"]
    if bs is not None and "cicatriz x referencia | + NBR antes" in set(bs["modelo"]):
        e = bs[bs["modelo"] == "cicatriz x referencia | + NBR antes"].iloc[0]
        out.append(("p", T(f"Controlando o NBR da faixa interna antes do fogo, a razao foi {n(e['or_1dp'], 1)} "
                           f"(IC jackknife por TI {n(e['ic95_jackknife_TI_inf'], 1)}-{n(e['ic95_jackknife_TI_sup'], 1)}).",
                           f"Controlling for inner-strip NBR before the fire, the ratio was {n(e['or_1dp'], 1)} "
                           f"(IL-jackknife CI {n(e['ic95_jackknife_TI_inf'], 1)}-{n(e['ic95_jackknife_TI_sup'], 1)}).")))

    out.append(("h2", T("Detectabilidade", "Detectability")))
    ds, rr = R["det_sens"], R["det_rr"]
    if ds is None or rr is None:
        out.append(("p", falta("11", lang)))
    else:
        txt_s = "; ".join(f"{T('cicatriz', 'scar') if i == 'cicatriz' else T('referencia', 'reference')} "
                          f"{n(100 * r['landsat_viu_entrada'], 0)}% (n = {int(r['n_segmentos'])})" for i, r in ds.iterrows())
        r0 = rr[(rr["core_m"] == rr["core_m"].min()) & (rr["referencia"] == "naoqueimado_local")]
        r0 = r0.iloc[0] if len(r0) else rr.iloc[0]
        barreira = r0["ic95_sup_boot_TI"] < 1
        out.append(("p", T(
            f"Quando o MODIS confirmou entrada do fogo, o Landsat tambem a marcou em: {txt_s}. Sem usar o NBR, a "
            f"densidade de pixels MODIS com fogo em ago-out nos nucleos das cicatrizes foi {n(r0['razao_taxas_agregada'], 2)} "
            f"vez a dos nucleos de combustivel nao queimado a ate 5 km (IC {n(r0['ic95_inf_boot_TI'], 2)}-"
            f"{n(r0['ic95_sup_boot_TI'], 2)}). "
            + ("O efeito de barreira, portanto, nao depende do detector." if barreira else
               "O teste independente nao confirma o efeito de barreira; o contraste das bordas deve ser lido com cautela."),
            f"When MODIS confirmed fire entry, Landsat also flagged it in: {txt_s}. Without NBR, the density of "
            f"August-October MODIS fire pixels in scar cores was {n(r0['razao_taxas_agregada'], 2)} times that in unburned "
            f"fuel cores within 5 km (CI {n(r0['ic95_inf_boot_TI'], 2)}-{n(r0['ic95_sup_boot_TI'], 2)}). "
            + ("The barrier effect therefore does not depend on the detector." if barreira else
               "The independent test does not confirm the barrier effect; the edge contrast must be read with caution."))))

    out.append(("h2", T("Configuracao alem da quantidade", "Configuration beyond quantity")))
    sc, sr = R["ses_corr"], R["ses_res"]
    if sc is not None and sr is not None:
        out.append(("p", T(
            f"A fragmentacao bruta acompanhou a quantidade (Spearman {n(sc.loc['q_inicio', 'conf_split'])}); o SES do "
            f"modelo nulo nao ({n(sc.loc['q_inicio', 'conf_split_ses'])}). Em "
            f"{n(100 * sr.iloc[0, 1], 0)}% das celulas-ano com cicatriz, as cicatrizes fragmentaram o combustivel mais "
            f"que o acaso (p <= 0,05) e em {n(100 * sr.iloc[0, 2], 0)}% menos.",
            f"Raw fragmentation tracked quantity (Spearman {n(sc.loc['q_inicio', 'conf_split'])}); the null-model SES did "
            f"not ({n(sc.loc['q_inicio', 'conf_split_ses'])}). In {n(100 * sr.iloc[0, 1], 0)}% of cell-years with scars, "
            f"scars split fuel more than chance (p <= 0.05) and in {n(100 * sr.iloc[0, 2], 0)}% less.")))
    bc, bcs = R["b_cell"], R["b_cell_s"]
    if bc is not None and bcs is not None:
        fr = "; ".join(f"{VAR[lang][v]}: {n(r['coef_pp_por_dp'], 2)} p.p./DP (wild p {n(r['p_wild_webb'], 3)})"
                       for v, r in bc.iterrows())
        s0 = bcs.iloc[0]
        out.append(("p", T(f"Celulas: {fr}. R2 within {n(s0['r2_within_ref'], 3)} -> {n(s0['r2_within_conf'], 3)}; "
                           f"erro de previsao em TIs novas mudou {n(s0['ganho_previsao_%'], 1)}% (positivo = melhora).",
                           f"Cells: {fr}. Within R2 {n(s0['r2_within_ref'], 3)} -> {n(s0['r2_within_conf'], 3)}; "
                           f"prediction error in new ILs changed by {n(s0['ganho_previsao_%'], 1)}% (positive = better).")))
    if bs is not None and R["b_seg_s"] is not None:
        t = bs[bs["modelo"] == "cicatriz | configuracao + deteccao"]
        fr = "; ".join(f"{VAR[lang].get(v, v)}: OR {n(r['or_1dp'])} (IC jk {n(r['ic95_jackknife_TI_inf'])}-"
                       f"{n(r['ic95_jackknife_TI_sup'])})" for v, r in t.iterrows())
        s0 = R["b_seg_s"].iloc[0]
        out.append(("p", T(f"Bordas de cicatriz ({n(s0['n_segmentos'], 0)} segmentos, {int(s0['n_TIs'])} TIs), com controle "
                           f"de detectabilidade: {fr}. Log-loss deixando TIs de fora {n(s0['logloss_TI_fora_ref'], 4)} -> "
                           f"{n(s0['logloss_TI_fora_conf'], 4)}; AUC {n(s0['auc_TI_fora_ref'], 3)} -> {n(s0['auc_TI_fora_conf'], 3)}.",
                           f"Scar edges ({n(s0['n_segmentos'], 0)} segments, {int(s0['n_TIs'])} ILs), with detectability "
                           f"control: {fr}. Leave-IL-out log-loss {n(s0['logloss_TI_fora_ref'], 4)} -> "
                           f"{n(s0['logloss_TI_fora_conf'], 4)}; AUC {n(s0['auc_TI_fora_ref'], 3)} -> {n(s0['auc_TI_fora_conf'], 3)}.")))
    if R["b_verd"] is not None:
        out += [("h2", T("Criterio de evidencia", "Evidence criterion")), ("table", R["b_verd"])]

    out.append(("h2", T("Robustez", "Robustness")))
    rs = R["rob_s"]
    if rs is None:
        out.append(("p", falta("14", lang)))
    else:
        out.append(("p", T(f"A razao de chances cicatriz x referencia variou de {n(rs['rc_cicatriz_vs_ref'].min(), 1)} a "
                           f"{n(rs['rc_cicatriz_vs_ref'].max(), 1)} entre {len(rs)} variantes de metodo.",
                           f"The scar versus reference odds ratio ranged from {n(rs['rc_cicatriz_vs_ref'].min(), 1)} to "
                           f"{n(rs['rc_cicatriz_vs_ref'].max(), 1)} across {len(rs)} method variants.")))

    out += [("h2", T("Limitacoes", "Limitations")),
            ("p", T("Associacoes observacionais em 8 TIs; nenhuma queimada foi atribuida as BRIFs nem chamada de prescrita. "
                    "O MODIS (1 km) so ve entrada em cicatrizes largas e chegada com incerteza de ~1 km; a data da chegada "
                    "usa o mes do MapBiomas quando nao ha foco. Os resultados valem para a mesma estacao seca.",
                    "Observational associations in 8 ILs; no burn was attributed to the brigades or called prescribed. "
                    "MODIS (1 km) sees entry only into wide scars and arrival with ~1-km uncertainty; arrival dates fall "
                    "back to the MapBiomas month without a detection. Results concern the same dry season only."))]
    return out

def write_doc(blocks, path_docx, title):
    d = docx.Document()
    st = d.styles["Normal"]; st.font.name = "Times New Roman"; st.font.size = Pt(11)
    d.add_heading(title, 0)
    md = [f"# {title}", ""]
    for kind, val in blocks:
        if kind == "h1":
            d.add_heading(val, 1); md += [f"## {val}", ""]
        elif kind == "h2":
            d.add_heading(val, 2); md += [f"### {val}", ""]
        elif kind == "p" and val:
            d.add_paragraph(val); md += [val, ""]
        elif kind == "table" and val is not None and len(val):
            t = d.add_table(rows=1, cols=len(val.columns)); t.style = "Light Grid Accent 1"
            for j, c in enumerate(val.columns):
                t.rows[0].cells[j].text = str(c)
            for row in val.itertuples(index=False):
                cells = t.add_row().cells
                for j, x in enumerate(row):
                    cells[j].text = f"{x:,.3f}" if isinstance(x, float) else str(x)
            md += ["```", val.to_string(index=False), "```", ""]
    d.save(path_docx)
    Path(str(path_docx).replace(".docx", ".md")).write_text("\n".join(md), encoding="utf-8")

for lang, (fname, title) in {
        "pt": ("RSL_metodos_resultados_PT.docx", "Configuracao das queimadas iniciais e continuidade do fogo"),
        "en": ("RSL_methods_results_EN.docx", "Early-burn configuration and fire continuity")}.items():
    write_doc(methods(lang) + results(lang), OUT_DIR / fname, title)
    console.print(f"[green]OK {OUT_DIR / fname}[/green]")
console.print("[bold green]OK BLOCO 15 completo (revise o texto antes de usar).[/bold green]")


## 16 figuras_letter

In [ ]:
# ======================================================================
# BLOCO 16 - Figuras e Tabela 1 da letter (Remote Sensing Letters, em ingles)
# Depende do bloco 00 e dos resultados ja salvos (01, 03, 07, 08, 10, 12, 14).
# Nao recalcula analises. 600 dpi: TIFF (LZW) + PDF vetorial + PNG.
# ======================================================================
#
# RSL: pagina de coluna unica; figuras na largura da mancha de texto
# (LARGURA_CM). Times New Roman (Liberation Serif no Colab), paleta azul/laranja
# segura para daltonismo, numeros escritos junto do dado, mapas com escala e norte.
#   Fig. 1  Premissa: (a) estacao dominante da queima por pixel nas TIs |
#           (b) area queimada anual, inicio x restante | (c) NBR mensal por classe
#   Fig. 2  O teste: (a) dNBR + cicatrizes + bordas atravessadas por TI |
#           (b) interrupcao cicatriz x referencia | (c) por TI
#   Fig. 3  Configuracao: (a) cicatrizes pela area | (b, c) % parado por area e
#           largura | (d) razoes de chances alem da quantidade
#   Fig. 4  Generalizacao: (a) % parado por celula de 5 km | (b) distribuicao por
#           TI | (c) robustez
#   Fig. S2 tipologia | Fig. S3 janela de 12 km a 30 m | Tabela 1
# JANELA = None escolhe a janela de 12 km sozinha; ou fixe JANELA = (ti_id, ano).
# ======================================================================

check_version()

JANELA = None
LARGURA_CM = 14.0
ESCALA_FONTE = 1.2

import sys, shutil
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, NullLocator, FormatStrFormatter, MaxNLocator
import rasterio
from shapely.geometry import shape, box as _box
import docx
from docx.shared import Pt, Cm as DocxCm

if "google.colab" in sys.modules:
    os.system("apt-get -qq install -y fonts-liberation > /dev/null 2>&1")
    font_manager.fontManager = font_manager._load_fontmanager(try_read_cache=False)

FIG_DIR = DIRS["figures"] / "letter"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TXT_DIR = PROJECT_ROOT / "08_textos"
TXT_DIR.mkdir(exist_ok=True)
FIRE_DIR = DIRS["exports"] / "fogo_mes"
SEQ_DIR = DIRS["exports"] / "sequencia_md"

# ============================================================
# ESTILO
# ============================================================
CM = 1 / 2.54
LARG = LARGURA_CM * CM
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRADE, EIXO, NEUTRO, FUNDO = "#e1e0d9", "#c3c2b7", "#f0efec", "#f4f3ef"
AZUL, LARANJA = "#2a78d6", "#eb6834"
AZUL_CLARO, ROXO, VINHO = "#a9c8ee", "#9b8ac4", "#7a1f3d"
CERRADO, CERRADO_BORDA = "#e2ead8", "#8aa36d"

def tam(pt):
    return round(pt * ESCALA_FONTE, 2)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": tam(7.5), "axes.labelsize": tam(7.5), "axes.titlesize": tam(7.5),
    "xtick.labelsize": tam(7), "ytick.labelsize": tam(7), "legend.fontsize": tam(6.8),
    "text.color": TINTA, "axes.labelcolor": TINTA, "axes.titlecolor": TINTA,
    "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.edgecolor": EIXO, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "legend.frameon": False, "legend.handlelength": 1.6,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.dpi": 600, "figure.dpi": 150,
})
HALO = [pe.withStroke(linewidth=1.8, foreground="white")]
_PAINEIS = []

def _raiz(ax):
    f = ax.figure
    while getattr(f, "figure", f) is not f and hasattr(f, "figure"):
        f = f.figure
    return f

def rotulo_painel(ax, letra, **_):
    """(a), (b)... posicionados no salvar(), depois do layout."""
    _PAINEIS.append((ax, letra))

def _colocar_rotulos(fig):
    meus = [p for p in _PAINEIS if _raiz(p[0]) is fig]
    if meus and fig.get_layout_engine() is not None:
        topo = 0.65 / (fig.get_figheight() * 2.54)
        fig.get_layout_engine().set(rect=(0, 0.25 * topo, 1, 1 - 1.25 * topo))
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    fixos = []
    for ax, letra in meus:
        bb = ax.get_tightbbox(r).transformed(inv)
        fixos.append((max(bb.x0, 0.0), bb.y1 + 0.004, letra))
    fig.set_layout_engine("none")
    for x, y, letra in fixos:
        fig.text(x, y, f"({letra})", fontweight="bold", fontsize=tam(8), ha="left", va="bottom")
    _PAINEIS[:] = [p for p in _PAINEIS if _raiz(p[0]) is not fig]

def salvar(fig, nome):
    _colocar_rotulos(fig)
    for ext, extra in [("pdf", {}), ("tif", {"pil_kwargs": {"compression": "tiff_lzw"}}), ("png", {})]:
        fig.savefig(FIG_DIR / f"{nome}.{ext}", dpi=600, facecolor="white", **extra)
    plt.show()
    plt.close(fig)
    console.print(f"[green]  {nome}: .pdf / .tif / .png (600 dpi)[/green]")

def moldura(ax, ext, barra_m=None, seta=True, fs=6, canto_barra="lower left", canto_seta="lower right"):
    """Mapa sem eixos, barra de escala preta/branca e seta de norte."""
    fs = tam(fs)
    ax.set_xlim(ext[0], ext[2]); ax.set_ylim(ext[1], ext[3])
    ax.set_aspect("equal"); ax.set_axis_off()
    w, h = ext[2] - ext[0], ext[3] - ext[1]
    if barra_m:
        x0 = ext[0] + 0.06 * w if "left" in canto_barra else ext[2] - 0.06 * w - barra_m
        y0 = ext[1] + 0.06 * h if "lower" in canto_barra else ext[3] - 0.08 * h
        th = 0.022 * h
        ax.add_patch(mpatches.Rectangle((x0, y0), barra_m / 2, th, fc=TINTA, ec=TINTA, lw=0.4, zorder=8))
        ax.add_patch(mpatches.Rectangle((x0 + barra_m / 2, y0), barra_m / 2, th, fc="white", ec=TINTA, lw=0.4, zorder=8))
        rot = f"{barra_m / 1000:g} km" if barra_m >= 1000 else f"{barra_m:g} m"
        ax.text(x0 + barra_m / 2, y0 - 0.012 * h, rot, fontsize=fs, ha="center", va="top", color=TINTA2,
                path_effects=HALO, zorder=8)
    if seta:
        xs = ext[0] + (0.90 if "right" in canto_seta else 0.10) * w
        ys = ext[1] + (0.10 if "lower" in canto_seta else 0.78) * h
        ax.annotate("", xy=(xs, ys + 0.09 * h), xytext=(xs, ys),
                    arrowprops=dict(arrowstyle="-|>", lw=0.7, color=TINTA, mutation_scale=7), zorder=8)
        ax.text(xs, ys - 0.01 * h, "N", ha="center", va="top", fontsize=fs + 1, fontweight="bold",
                path_effects=HALO, zorder=8)

def barra_redonda(largura_m, frac=0.25):
    alvo = frac * largura_m
    return max([k for k in [250, 500, 1000, 2000, 5000, 10000, 20000, 25000, 50000, 100000, 200000, 500000]
                if k <= alvo] or [250])

def dot_labels(ax, xs, ys, labels, dx, ha="left", fs=5.8, color=TINTA2):
    for x, y, s in zip(xs, ys, labels):
        ax.text(x + dx, y, s, fontsize=tam(fs), color=color, va="center", ha=ha)

TYPE_EN = {"Inicial grande": "Early, large", "Inicial pequena e fraca": "Early, small & weak",
           "Inicial pequena e intensa": "Early, small & intense", "Restante da seca": "Late season",
           "Fora da seca (nov-abr)": "Outside dry season", "Fragmento < 1 ha": "Fragments < 1 ha"}
TYPE_COL = {"Early, large": "#1d5aa6", "Early, small & weak": AZUL, "Early, small & intense": AZUL_CLARO,
            "Late season": LARANJA, "Outside dry season": "#c9c7bf", "Fragments < 1 ha": "#dcdad2"}
CMAP_NBR = LinearSegmentedColormap.from_list("nbr", ["#4a2a12", "#8c5a2b", "#c9a46b", "#efe3c6", "#a9cf8f", "#4f9a4a", "#1d5e2a"])
CMAP_DNBR = LinearSegmentedColormap.from_list("dnbr", ["#fbf6ea", "#fde0a3", "#f9a35a", "#e0532b", "#a8201a", "#5a0b10"])
SEM_DADO = "#e4e3de"
C_INI, C_TAR, C_PASSOU = "#1f5fc8", "#f28322", "#d1127a"
C_SEMQ, C_MISTO = "#e9e8e3", "#9a7fc6"
CMAP_PASSA = LinearSegmentedColormap.from_list("passa", ["#b2182b", "#ef8a62", "#fbe3c4", "#9ec9e2", "#2f78b4", "#163f73"])
CMAP_AREA2 = LinearSegmentedColormap.from_list("area2", ["#c6dbf7", "#6aa6e8", "#1f5fc8", "#0b2a66"])

# ============================================================
# DADOS
# ============================================================
def _csv(p, **kw):
    return pd.read_csv(p, **kw) if Path(p).exists() else None

STATS = ["B06_tipologia_resumo", "D1_tempo", "D2_foco_noturno_taxa", "D2_foco_noturno", "B09_celula_resumo",
         "B09_celula_coeficientes", "B08a_regra_antiga_x_simetrica", "B09_segmento_resumo",
         "B09_segmento_coeficientes", "B10_robustez_segmentos", "B09_cicatriz_vs_referencia"]
IDX0 = ("B06_tipologia_resumo", "D2_foco_noturno_taxa", "D2_foco_noturno", "B09_celula_coeficientes",
        "B09_segmento_coeficientes")
SEG_COLS = ["ti_id", "year", "tipo_seg", "resultado", "x", "y", "area_cicatriz_ha", "largura_local_m"]

AMOSTRA = load_sample_ids()
seg_file = cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet")
segs = pd.read_parquet(seg_file, columns=SEG_COLS)
art1 = _csv(DIRS["raw"] / "artigo1_tis.csv")
TI_NAME = dict(zip(tis["ti_id"].astype(int), tis["ti_name"]))
ORDER8 = tis[tis["ti_id"].astype(int).isin(AMOSTRA)].sort_values("ti_name")
NUM = {int(t): k for k, t in enumerate(ORDER8["ti_id"], start=1)}
R = {n: _csv(stats_path(n), index_col=0 if n in IDX0 else None) for n in STATS}
faltam = [n for n, v in R.items() if v is None]
if faltam:
    console.print(f"[yellow]WARNING faltam tabelas (paineis correspondentes saem vazios): {faltam}[/yellow]")
REGRA_OK = lambda s: s.astype(str).str.startswith("simetrica")

def annual_area():
    rows = []
    for ti in ORDER8["ti_id"].astype(int):
        p = FIRE_DIR / f"l9_{RUN_TAG}_firemonth_ti{ti}.tif"
        if not p.exists():
            continue
        with rasterio.open(p) as src:
            inside = src.read(1) == 1
            px_ha = abs(src.transform.a * src.transform.e) / 1e4
            for k, y in enumerate(SCAR_YEARS, start=2):
                if y in YEARS:
                    m = src.read(k)
                    rows.append({"ti_id": ti, "year": y, "early_ha": (inside & np.isin(m, EARLY)).sum() * px_ha,
                                 "late_ha": (inside & np.isin(m, LATE)).sum() * px_ha, "ti_ha": inside.sum() * px_ha})
    return pd.DataFrame(rows)

def nbr_profiles(step=3, max_px=4000, seed=7):
    """NBR mensal minimo (Landsat 8/9) por classe de pixel: cicatriz inicial, fogo posterior, nao queimado."""
    rng = np.random.default_rng(seed)
    acc = {}
    for ti in ORDER8["ti_id"].astype(int):
        for y in YEARS:
            try:
                r = read_raster(ti, y)
            except Exception:
                continue
            mon = r["month"][::step, ::step]
            cls = {"early": np.isin(mon, EARLY), "late": np.isin(mon, LATE), "unburned": mon == 0}
            for m in LS_MONTHS:
                v = r[f"nbr{m:02d}"][::step, ::step].astype(np.int16)
                ok = v > -101
                for c, mask in cls.items():
                    vals = v[ok & mask]
                    if len(vals) > max_px:
                        vals = rng.choice(vals, max_px, replace=False)
                    if len(vals):
                        acc.setdefault((c, m), []).append(vals / 100.0)
    rows = []
    for (c, m), lst in acc.items():
        a = np.concatenate(lst)
        rows.append({"classe": c, "mes": m, "p25": np.percentile(a, 25), "p50": np.median(a),
                     "p75": np.percentile(a, 75), "n": len(a)})
    return pd.DataFrame(rows).sort_values(["classe", "mes"])

area_anual = annual_area()
perfil_nbr = nbr_profiles()
perfil_nbr.to_csv(stats_path("F1_perfil_nbr_por_classe"), index=False)

_d_ok = segs[(segs["tipo_seg"] == "cicatriz") & segs["resultado"].isin(["interrupcao", "continuidade"])]
REP = {int(t): int(g.groupby("year").size().idxmax()) for t, g in _d_ok.groupby("ti_id")}

def pick_window():
    d = _d_ok
    if JANELA:
        cand = [tuple(JANELA)]
    else:
        g = d.groupby(["ti_id", "year"]).agg(n=("resultado", "size"),
                                             nc=("resultado", lambda r: (r == "continuidade").sum())).reset_index()
        g = g[g["nc"] >= 3] if (g["nc"] >= 3).any() else g
        cand = [tuple(int(v) for v in r) for r in g.sort_values("n", ascending=False)[["ti_id", "year"]].values[:15]]
    for ti, yr in cand:
        dd = d[(d["ti_id"] == ti) & (d["year"] == yr)]
        H, xe, ye = np.histogram2d(dd["x"], dd["y"], bins=[max(int(np.ptp(dd["x"]) // 3000), 1),
                                                           max(int(np.ptp(dd["y"]) // 3000), 1)])
        geom = tis[tis["ti_id"].astype(int) == ti].geometry.iloc[0].buffer(300)
        for flat in np.argsort(H, axis=None)[::-1][:300]:
            i, j = np.unravel_index(flat, H.shape)
            cx, cy = (xe[i] + xe[i + 1]) / 2, (ye[j] + ye[j + 1]) / 2
            win = (cx - 6000, cy - 6000, cx + 6000, cy + 6000)
            if geom.contains(_box(*win)):
                return ti, yr, win
    ti, yr = cand[0]
    dd = d[(d["ti_id"] == ti) & (d["year"] == yr)]
    cx, cy = dd["x"].median(), dd["y"].median()
    return ti, yr, (cx - 6000, cy - 6000, cx + 6000, cy + 6000)

W = pick_window()
console.print(f"[cyan]Janela 12 x 12 km:[/cyan] {TI_NAME.get(W[0])}, {W[1]}")

def window_nbr():
    r = read_raster(W[0], W[1])
    tr = r["_transform"]; px = abs(tr.a)
    c0, r0 = int(round((W[2][0] - tr.c) / px)), int(round((tr.f - W[2][3]) / px))
    n = int(round((W[2][2] - W[2][0]) / px))
    cut = lambda a: a[max(r0, 0):r0 + n, max(c0, 0):c0 + n]
    pre = cut(r["nbr07"]).astype(np.int16)
    post = np.stack([cut(r[f"nbr{m:02d}"]).astype(np.int16) for m in LATE])
    post = np.where(post > -101, post, 999).min(0); post[post == 999] = -128
    return {"pre": pre, "post": post, "month": cut(r["month"]).astype(np.int16),
            "ext": np.array([W[2][0], W[2][2], W[2][1], W[2][3]], dtype=float)}

JNBR = window_nbr()

def ti_geom(ti):
    return tis[tis["ti_id"].astype(int) == ti]

def context_outlines():
    out = {}
    try:
        g = ee.FeatureCollection(EE_ASSET_BIOME).geometry(ee.ErrorMargin(500)).simplify(2000)
        out["cerrado"] = gpd.GeoDataFrame(geometry=[shape(g.getInfo())], crs="EPSG:4326").to_crs(AREA_CRS)
    except Exception as e:
        console.print(f"[dim]contorno do Cerrado indisponivel: {str(e)[:80]}[/dim]")
    try:
        br = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq("country_na", "Brazil"))
        g = br.geometry(ee.ErrorMargin(1000)).simplify(5000)
        out["brasil"] = gpd.GeoDataFrame(geometry=[shape(g.getInfo())], crs="EPSG:4326").to_crs(AREA_CRS)
    except Exception as e:
        console.print(f"[dim]contorno do Brasil indisponivel: {str(e)[:80]}[/dim]")
    return out

CTX = context_outlines()

def mapa_localizacao(ax, cor_ti=None, cmap=None, norm=None, rotulos=True):
    """Cerrado + 14 TIs; cor_ti = {ti_id: valor} pinta as TIs analisadas."""
    if "brasil" in CTX:
        CTX["brasil"].plot(ax=ax, facecolor=FUNDO, edgecolor="#bdbcb6", lw=0.35, zorder=0)
    if "cerrado" in CTX:
        CTX["cerrado"].plot(ax=ax, facecolor=CERRADO, edgecolor=CERRADO_BORDA, lw=0.45, zorder=1)
    elig = tis["ti_id"].astype(int).isin(AMOSTRA)
    tis[~elig].plot(ax=ax, facecolor="white", edgecolor=TINTA2, lw=0.5, zorder=3)
    if cor_ti is None:
        tis[elig].plot(ax=ax, facecolor=LARANJA, edgecolor=TINTA, lw=0.35, zorder=4)
    for _, r in ORDER8.iterrows():
        p = r.geometry.representative_point(); ti = int(r["ti_id"])
        if cor_ti is not None and ti in cor_ti:
            ax.scatter(p.x, p.y, s=46, color=cmap(norm(cor_ti[ti])), edgecolor=TINTA, lw=0.6, zorder=5)
        if rotulos:
            ax.annotate(str(NUM[ti]), (p.x, p.y), xytext=(4.5, 2.5), textcoords="offset points",
                        fontsize=tam(6.2), fontweight="bold", color=TINTA, path_effects=HALO, zorder=6)
    b = CTX["cerrado"].total_bounds if "cerrado" in CTX else tis.total_bounds
    cx, cy, h = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2, max(b[2] - b[0], b[3] - b[1]) / 2 * 1.05
    return (cx - h, cy - h, cx + h, cy + h)

def colorbar_fina(fig, mappable, ax, label, ticks=None, extend="neither", orientation="horizontal", **kw):
    cb = fig.colorbar(mappable, ax=ax, orientation=orientation, fraction=0.05, pad=0.02, aspect=28,
                      shrink=kw.get("shrink", 0.9), extend=extend)
    cb.set_label(label, fontsize=tam(6.2))
    if ticks is not None:
        cb.set_ticks(ticks)
    cb.outline.set_linewidth(0.4)
    cb.ax.tick_params(labelsize=tam(5.8), width=0.5, length=2)
    return cb

def mapa_ti(ti):
    """Ano REP[ti]: sequencia (bloco 10), dNBR (Landsat) e area de cada cicatriz inicial (~700 px)."""
    from scipy import ndimage
    yr = REP.get(ti)
    p = SEQ_DIR / f"seq_ti{ti}_{yr}.tif"
    if yr is None or not p.exists():
        return None
    with rasterio.open(p) as src:
        seq = src.read(1); tr = src.transform; b = src.bounds
    early = np.isin(seq, [1, 3])
    lab, n = ndimage.label(early, structure=np.ones((3, 3), int))
    area = np.zeros(seq.shape, np.float32)
    if n:
        sizes = np.bincount(lab.ravel())[1:] * abs(tr.a * tr.e) / 1e4
        area[early] = sizes[lab[early] - 1]
    out = {"seq": seq, "area": area}
    try:
        r = read_raster(ti, yr)
        pre = r["nbr07"].astype(np.int16)
        post = np.stack([r[f"nbr{m:02d}"].astype(np.int16) for m in LATE])
        post = np.where(post > -101, post, 999).min(0)
        dn = np.where((pre > -101) & (post < 999), (pre - post) / 100.0, np.nan).astype(np.float32)
        if dn.shape == seq.shape:
            out["dnbr"] = dn
    except Exception as e:
        console.print(f"[dim]dNBR indisponivel para TI {ti}: {str(e)[:60]}[/dim]")
    f = int(np.ceil(max(seq.shape) / 700))
    out = {k: v[::f, ::f] for k, v in out.items()}
    out["ext"] = np.array([b.left, b.right, b.bottom, b.top])
    return out

MAPAS = {int(t): mapa_ti(int(t)) for t in ORDER8["ti_id"]}

from matplotlib.path import Path as _MPath
from matplotlib.patches import PathPatch as _PathPatch

def _geom_path(geom):
    verts, codes = [], []
    polys = list(geom.geoms) if hasattr(geom, "geoms") else [geom]
    for p in polys:
        for ring in [p.exterior] + list(p.interiors):
            xy = np.asarray(ring.coords)
            verts += xy.tolist(); codes += [_MPath.MOVETO] + [_MPath.LINETO] * (len(xy) - 2) + [_MPath.CLOSEPOLY]
    return _MPath(verts, codes)

def recortar_na_ti(ax, ti):
    pp = _PathPatch(_geom_path(ti_geom(ti).geometry.iloc[0]), transform=ax.transData, facecolor="none", edgecolor="none")
    for a in list(ax.images) + list(ax.collections):
        a.set_clip_path(pp)

def grade_tis(sf, desenhar, info=None, ncols=4):
    """As TIs analisadas, cada uma num painel quadrado na sua escala, recortado no limite da TI."""
    nrows = int(np.ceil(len(ORDER8) / ncols))
    axs = np.atleast_2d(sf.subplots(nrows, ncols))
    for ax, (_, r) in zip(axs.ravel(), ORDER8.iterrows()):
        ti = int(r["ti_id"])
        ti_geom(ti).plot(ax=ax, facecolor=C_SEMQ, edgecolor="none", zorder=0)
        desenhar(ax, ti)
        recortar_na_ti(ax, ti)
        ti_geom(ti).boundary.plot(ax=ax, color=TINTA, lw=0.45, zorder=6)
        bb = r.geometry.bounds
        h = max(bb[2] - bb[0], bb[3] - bb[1]) / 2 * 1.04; cx, cy = (bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2
        e = (cx - h, cy - h, cx + h, cy + h)
        canto = "lower left"
        for cand in ["lower left", "lower right", "upper left", "upper right"]:
            bw = barra_redonda(2 * h)
            xx = e[0] + 0.06 * 2 * h if "left" in cand else e[2] - 0.06 * 2 * h - bw
            yy = e[1] + 0.03 * 2 * h if "lower" in cand else e[3] - 0.12 * 2 * h
            if not _box(xx, yy, xx + bw, yy + 0.09 * 2 * h).intersects(r.geometry):
                canto = cand; break
        moldura(ax, e, barra_m=barra_redonda(2 * h), seta=(NUM[ti] == 1), fs=4.6, canto_barra=canto,
                canto_seta="upper right")
        ax.set_title(f"{NUM[ti]} {r['ti_name']}", fontsize=tam(5.8), pad=1.5, loc="left", fontweight="bold")
        if info:
            s = info(ti)
            if s:
                ax.text(0.0, -0.02, s, transform=ax.transAxes, ha="left", va="top", fontsize=tam(5.2), color=TINTA2)
    for ax in axs.ravel()[len(ORDER8):]:
        ax.set_axis_off()
    return axs

def _img(ax, arr, ext, **kw):
    return ax.imshow(arr, extent=(ext[0], ext[1], ext[2], ext[3]), interpolation="nearest", zorder=2,
                     rasterized=True, **kw)

def season_class(ti):
    p = FIRE_DIR / f"l9_{RUN_TAG}_firemonth_ti{ti}.tif"
    if not p.exists():
        return None, None
    with rasterio.open(p) as src:
        inside = src.read(1) == 1
        ne = np.zeros(inside.shape, np.uint8); nl = np.zeros(inside.shape, np.uint8)
        for k, y in enumerate(SCAR_YEARS, start=2):
            if y in YEARS:
                m = src.read(k)
                ne += np.isin(m, EARLY); nl += np.isin(m, LATE)
        b = src.bounds
    cl = np.zeros(inside.shape, np.uint8)
    cl[inside] = 1; cl[inside & (ne > nl)] = 2; cl[inside & (ne == nl) & (ne > 0)] = 3; cl[inside & (nl > ne)] = 4
    f = int(np.ceil(max(cl.shape) / 650))
    return np.ma.masked_equal(cl[::f, ::f], 0), (b.left, b.right, b.bottom, b.top)

def _regra_sim(rg):
    return rg[(rg["combustivel"] == FUEL_MAIN) & REGRA_OK(rg["regra"])].set_index("tipo_seg")

# ============================================================
# FIGURA 1 - PREMISSA
# ============================================================
def fig1():
    fig = plt.figure(figsize=(LARG, 13.6 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.45, 1], hspace=0.04)
    cmap = ListedColormap([C_SEMQ, C_INI, C_MISTO, C_TAR]); norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], 4)

    def desenhar(ax, ti):
        cl, e = season_class(ti)
        if cl is not None:
            ax.imshow(cl, cmap=cmap, norm=norm, extent=e, interpolation="nearest", zorder=2, rasterized=True)
    axs = grade_tis(sf_m, desenhar)
    sf_m.legend(handles=[mpatches.Patch(color=C_INI, label="Burned mostly in May–Jul"),
                         mpatches.Patch(color=C_TAR, label="Burned mostly in Aug–Oct"),
                         mpatches.Patch(color=C_MISTO, label="Equal"), mpatches.Patch(color=C_SEMQ, label="Not burned")],
                loc="outside lower center", ncol=4, fontsize=tam(5.8), handlelength=1.0, columnspacing=1.0)
    rotulo_painel(axs[0, 0], "a")

    axb, axc = sf_g.subplots(1, 2)
    a = area_anual.groupby("year")[["early_ha", "late_ha"]].sum() / 1000
    axb.fill_between(a.index, 0, a["early_ha"], color=C_INI, alpha=0.12, lw=0)
    axb.plot(a.index, a["late_ha"], lw=1.6, color=C_TAR, marker="s", ms=3.4, mec="white", mew=0.5, zorder=3)
    axb.plot(a.index, a["early_ha"], lw=1.8, color=C_INI, marker="o", ms=3.8, mec="white", mew=0.6, zorder=4)
    axb.text(a.index[-1] + 0.3, a["late_ha"].iloc[-1], "Aug–Oct", color=C_TAR, fontsize=tam(5.8), va="center")
    axb.text(a.index[-1] + 0.3, a["early_ha"].iloc[-1], "May–Jul", color=C_INI, fontsize=tam(5.8), va="center")
    axb.set_xlim(YEARS[0] - 0.5, YEARS[-1] + 2.4); axb.spines["bottom"].set_bounds(YEARS[0], YEARS[-1])
    axb.set_xticks(range(YEARS[0], YEARS[-1] + 1, 2)); axb.set_xticks(YEARS, minor=True)
    axb.grid(axis="x", visible=False)
    axb.set_ylim(0, a.values.max() * 1.25); axb.yaxis.set_major_locator(MaxNLocator(4))
    axb.set_ylabel("Burned area (kha)")
    sh = 100 * a["early_ha"].sum() / (a["early_ha"].sum() + a["late_ha"].sum())
    axb.set_title(f"early burning = {sh:.0f}% of dry-season area", fontsize=tam(5.8), color=MUDO, loc="left", pad=2)
    rotulo_painel(axb, "b")

    axc.axvspan(4.5, 7.5, color=C_INI, alpha=0.08, lw=0, zorder=0)
    axc.axvspan(7.5, 10.5, color=C_TAR, alpha=0.10, lw=0, zorder=0)
    for c, cor, lab in [("unburned", "#7d7c78", "Unburned"), ("late", C_TAR, "Burned\nAug–Oct"),
                        ("early", C_INI, "Burned\nMay–Jul")]:
        d = perfil_nbr[perfil_nbr["classe"] == c]
        if not len(d):
            continue
        axc.fill_between(d["mes"], d["p25"], d["p75"], color=cor, alpha=0.14, lw=0, zorder=1)
        axc.plot(d["mes"], d["p50"], color=cor, lw=1.8, marker="o", ms=3.2, mec="white", mew=0.5, zorder=3)
        ult = d.iloc[-1]
        axc.text(ult["mes"] + 0.2, ult["p50"], lab, color=cor, fontsize=tam(5.6), va="center", ha="left", linespacing=1.0)
    axc.set_xticks(LS_MONTHS); axc.set_xticklabels(["A", "M", "J", "J", "A", "S", "O", "N"][:len(LS_MONTHS)])
    axc.set_xlim(LS_MONTHS[0] - 0.4, LS_MONTHS[-1] + 2.4); axc.spines["bottom"].set_bounds(LS_MONTHS[0], LS_MONTHS[-1])
    axc.grid(axis="x", visible=False); axc.yaxis.set_major_locator(MaxNLocator(4))
    axc.set_ylabel("Landsat NBR")
    axc.set_title("NBR drops in the month each pixel burns", fontsize=tam(5.8), color=MUDO, loc="left", pad=2)
    rotulo_painel(axc, "c")
    salvar(fig, "Fig1_premise")

# ============================================================
# FIGURA 2 - O TESTE NAS BORDAS
# ============================================================
def fig2():
    fig = plt.figure(figsize=(LARG, 14.2 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.5, 1], hspace=0.04)
    norm_d = mpl.colors.Normalize(0, 0.6)

    def info(ti):
        dd = _d_ok[(_d_ok["ti_id"] == ti) & (_d_ok["year"] == REP.get(ti))]
        return f"{REP.get(ti)} · {100 * (dd['resultado'] == 'interrupcao').mean():.0f}% stopped" if len(dd) else ""

    def desenhar(ax, ti):
        m = MAPAS.get(ti)
        if m is None:
            return
        e = m["ext"]
        if "dnbr" in m:
            _img(ax, np.ma.masked_invalid(np.clip(m["dnbr"], 0, 0.8)), e, cmap=CMAP_DNBR, norm=norm_d)
        early = np.isin(m["seq"], [1, 3]).astype(float)
        if early.any():
            ax.contour(np.flipud(early), levels=[0.5], colors=[C_INI], linewidths=0.35, extent=(e[0], e[1], e[2], e[3]), zorder=4)
        dd = _d_ok[(_d_ok["ti_id"] == ti) & (_d_ok["year"] == REP.get(ti)) & (_d_ok["resultado"] == "continuidade")]
        ax.scatter(dd["x"], dd["y"], s=5, color=C_PASSOU, edgecolor="white", lw=0.25, zorder=5)
    axs = grade_tis(sf_m, desenhar, info=info)
    cb = sf_m.colorbar(mpl.cm.ScalarMappable(norm=norm_d, cmap=CMAP_DNBR), ax=axs, location="bottom",
                       shrink=0.32, aspect=30, pad=0.01, extend="max")
    cb.set_label("Landsat dNBR, July − min(Aug–Oct): later burning", fontsize=tam(5.8))
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=tam(5.4), width=0.4, length=1.5)
    sf_m.legend(handles=[Line2D([], [], color=C_INI, lw=1.0, label="Early-scar outline (May–Jul)"),
                         Line2D([], [], marker="o", ls="", ms=3.2, mfc=C_PASSOU, mec="white", mew=0.4,
                                label="Scar edge crossed by later fire")],
                loc="outside lower center", ncol=2, fontsize=tam(5.8), handlelength=1.2, columnspacing=1.4)
    rotulo_painel(axs[0, 0], "a")

    axb, axc = sf_g.subplots(1, 2, width_ratios=[1, 1.15])
    rg, rv = R["B08a_regra_antiga_x_simetrica"], R["B09_cicatriz_vs_referencia"]
    if rg is not None:
        g = _regra_sim(rg)
        for xi, (t, cor) in enumerate([("cicatriz", C_INI), ("referencia", "#8f8e88")]):
            v = g.loc[t, "% interrupcao"]; n = int(g.loc[t, "interrupcao"] + g.loc[t, "continuidade"])
            axb.bar(xi, v, width=0.62, color=cor, zorder=2)
            axb.text(xi, v + 2, f"{v:.1f}%", ha="center", va="bottom", fontsize=tam(6.4), fontweight="bold", color=cor)
            axb.text(xi, -4, f"n = {n:,}", ha="center", va="top", fontsize=tam(5.2), color=MUDO)
        axb.set_xticks([0, 1]); axb.set_xticklabels(["Early-scar\nedges", "Reference\nedges"], fontsize=tam(6.0))
        axb.tick_params(axis="x", length=0, pad=12)
        axb.set_ylim(0, 112); axb.set_yticks([0, 50, 100]); axb.set_ylabel("Arriving fires stopped (%)")
        axb.grid(axis="x", visible=False)
    if rv is not None:
        r_ = rv[(rv["combustivel"] == FUEL_MAIN) & REGRA_OK(rv["regra"])]
        if len(r_):
            r_ = r_.iloc[0]
            axb.set_title(f"adjusted OR {r_['razao_chances_cicatriz']:.0f} ({r_['ic95_inf']:.0f}–{r_['ic95_sup']:.0f})",
                          fontsize=tam(5.8), color=MUDO, loc="left", pad=2)
    rotulo_painel(axb, "b")
    per = (segs[segs["resultado"].isin(["interrupcao", "continuidade"])]
           .assign(i=lambda x: x["resultado"] == "interrupcao").groupby(["ti_id", "tipo_seg"])["i"].mean().unstack())
    tl = ORDER8["ti_id"].astype(int).tolist()[::-1]
    for yi, t in enumerate(tl):
        if t not in per.index:
            continue
        s, r_ = 100 * per.loc[t].get("cicatriz", np.nan), 100 * per.loc[t].get("referencia", np.nan)
        axc.plot([r_, s], [yi, yi], color="#cfcec7", lw=2.2, zorder=1, solid_capstyle="round")
        axc.scatter(r_, yi, s=24, color="#8f8e88", edgecolor="white", lw=0.5, zorder=3)
        axc.scatter(s, yi, s=28, color=C_INI, edgecolor="white", lw=0.5, zorder=3)
    axc.set_yticks(range(len(tl))); axc.set_yticklabels([f"{NUM[t]} {TI_NAME[t]}" for t in tl], fontsize=tam(5.6))
    axc.set_xlim(-3, 103); axc.set_xticks([0, 50, 100]); axc.set_xlabel("Arriving fires stopped (%)", fontsize=tam(6.0))
    axc.grid(axis="y", visible=False); axc.tick_params(axis="y", length=0)
    axc.set_title("each IL: reference (grey) vs early-scar edges (blue)", fontsize=tam(5.6), color=MUDO, loc="left", pad=2)
    rotulo_painel(axc, "c")
    salvar(fig, "Fig2_edge_test")

# ============================================================
# FIGURA 3 - CONFIGURACAO ALEM DA QUANTIDADE
# ============================================================
BINS_AREA, LAB_AREA = [1, 10, 100, 1000, np.inf], ["1–10", "10–100", "100–1000", "> 1000"]
BINS_LARG, LAB_LARG = [0, 150, 300, 600, np.inf], ["< 150", "150–300", "300–600", "> 600"]

def taxa_por_faixa(col, bins, labs):
    d = _d_ok.dropna(subset=[col]).copy()
    d["faixa"] = pd.cut(d[col], bins, labels=labs, right=False)
    g = d.groupby("faixa", observed=False)["resultado"].agg(n="size", p=lambda r: (r == "interrupcao").mean())
    g["ep"] = np.sqrt(g["p"] * (1 - g["p"]) / g["n"].clip(lower=1))
    return g

def fig3():
    fig = plt.figure(figsize=(LARG, 14.0 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.45, 1], hspace=0.04)
    norm_a = mpl.colors.LogNorm(1, 1000)

    def desenhar(ax, ti):
        m = MAPAS.get(ti)
        if m is None:
            return
        e = m["ext"]
        _img(ax, np.ma.masked_where(m["seq"] != 2, np.ones(m["seq"].shape)), e, cmap=ListedColormap(["#f9c9a2"]), vmin=0, vmax=1)
        _img(ax, np.ma.masked_less_equal(m["area"], 0), e, cmap=CMAP_AREA2, norm=norm_a)
        dd = _d_ok[(_d_ok["ti_id"] == ti) & (_d_ok["year"] == REP.get(ti)) & (_d_ok["resultado"] == "continuidade")]
        ax.scatter(dd["x"], dd["y"], s=6, color=C_PASSOU, edgecolor="white", lw=0.3, zorder=5)
    axs = grade_tis(sf_m, desenhar, info=lambda ti: f"{REP.get(ti)}")
    cb = sf_m.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=CMAP_AREA2), ax=axs, location="bottom",
                       shrink=0.36, aspect=30, pad=0.03)
    cb.set_ticks([1, 10, 100, 1000]); cb.set_ticklabels(["1", "10", "100", "1000"])
    cb.set_label("Size of each early scar (ha)", fontsize=tam(5.8))
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=tam(5.4), width=0.4, length=1.5)
    sf_m.legend(handles=[mpatches.Patch(color="#f9c9a2", label="Burned Aug–Oct"),
                         Line2D([], [], marker="o", ls="", ms=3.4, mfc=C_PASSOU, mec="white", mew=0.4,
                                label="Scar edge crossed by later fire")],
                loc="outside lower center", ncol=2, fontsize=tam(5.8), handlelength=1.2, columnspacing=1.4)
    rotulo_painel(axs[0, 0], "a")

    axb, axc, axd = sf_g.subplots(1, 3, width_ratios=[1, 1, 1.15])
    for ax, (col, bins, labs, xl, letra) in zip([axb, axc], [("area_cicatriz_ha", BINS_AREA, LAB_AREA, "Early-scar size (ha)", "b"),
                                                             ("largura_local_m", BINS_LARG, LAB_LARG, "Local scar width (m)", "c")]):
        g = taxa_por_faixa(col, bins, labs)
        x = np.arange(len(g)); cores = [CMAP_AREA2(v) for v in np.linspace(0.3, 1.0, len(g))]
        ax.bar(x, 100 * g["p"], width=0.66, color=cores, zorder=2)
        ax.errorbar(x, 100 * g["p"], yerr=196 * g["ep"], fmt="none", ecolor=TINTA, elinewidth=0.7, capsize=1.5)
        for xi, (_, r) in zip(x, g.iterrows()):
            if r["n"]:
                ax.text(xi, 100 * r["p"] + 196 * r["ep"] + 0.25, f"{100 * r['p']:.1f}", ha="center", va="bottom",
                        fontsize=tam(5.6), color=TINTA)
        lo = max(0, np.floor(100 * (g["p"] - 2.2 * g["ep"]).min() / 2) * 2 - 2)
        ax.set_ylim(lo, 100.8)
        ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=tam(5.6))
        ax.tick_params(axis="x", length=0); ax.grid(axis="x", visible=False)
        ax.yaxis.set_major_locator(MaxNLocator(4)); ax.set_xlabel(xl, fontsize=tam(6.0))
        rotulo_painel(ax, letra)
    axb.set_ylabel("Arriving fires stopped (%)", fontsize=tam(6.0))
    sc = R["B09_segmento_coeficientes"]
    lab = {"log_area_cicatriz": "Scar size", "largura_local_100m": "Scar width",
           "conf_split_win": "Fragmentation", "conf_lpi_win": "Largest patch"}
    if sc is not None:
        k = sc[(sc["combustivel"] == FUEL_MAIN) & (sc["modelo"] == "segmento | config")]
        vars_ = [v for v in lab if v in k.index]; y = np.arange(len(vars_))[::-1].astype(float)
        axd.axvline(1.0, color=TINTA2, lw=0.7, ls=(0, (3, 2)), zorder=1)
        his = []
        for yi, v in zip(y, vars_):
            e = k.loc[v]; sig = e["ic95_inf"] > 1 or e["ic95_sup"] < 1
            cor = C_INI if sig else "#9a9992"
            axd.plot([e["ic95_inf"], e["ic95_sup"]], [yi, yi], color=cor, lw=2.0, solid_capstyle="round", zorder=2)
            axd.scatter(e["razao_chances_1dp"], yi, s=34, color=cor if sig else "white", edgecolor="white" if sig else cor,
                        lw=0.6 if sig else 1.0, zorder=3)
            axd.text(e["ic95_sup"] * 1.05, yi, f"{e['razao_chances_1dp']:.2f}", fontsize=tam(5.8), color=cor,
                     va="center", fontweight="bold" if sig else "normal")
            his.append(e["ic95_sup"])
        axd.set_xscale("log"); axd.set_xlim(0.65, max(his) * 1.55)
        axd.xaxis.set_major_locator(FixedLocator([0.75, 1, 1.5, 2, 3])); axd.xaxis.set_minor_locator(NullLocator())
        axd.xaxis.set_major_formatter(FormatStrFormatter("%g"))
        axd.set_yticks(y); axd.set_yticklabels([lab[v] for v in vars_], fontsize=tam(5.8))
        axd.set_ylim(-0.6, len(vars_) - 0.4)
        axd.grid(axis="y", visible=False); axd.tick_params(axis="y", length=0)
        axd.set_xlabel("Odds of stopping per SD,\nbeyond quantity", fontsize=tam(6.0))
    rotulo_painel(axd, "d")
    salvar(fig, "Fig3_configuration")

# ============================================================
# FIGURA 4 - GENERALIZACAO
# ============================================================
def fig4():
    fig = plt.figure(figsize=(LARG, 13.8 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.45, 1], hspace=0.04)
    d = _d_ok.copy()
    d["x"] = np.floor(d["x"] / CELL_M) * CELL_M + CELL_M / 2
    d["y"] = np.floor(d["y"] / CELL_M) * CELL_M + CELL_M / 2
    gcell = (d.groupby(["ti_id", "x", "y"]).agg(n=("resultado", "size"),
             rate=("resultado", lambda r: 100 * (r == "interrupcao").mean())).reset_index())
    gcell = gcell[gcell["n"] >= 10]
    lo = float(np.floor(gcell["rate"].quantile(0.02) / 5) * 5) if len(gcell) else 80
    norm = mpl.colors.Normalize(lo, 100)

    def desenhar(ax, ti):
        g = gcell[gcell["ti_id"] == ti]
        if not len(g):
            return
        h = CELL_M / 2
        gg = gpd.GeoDataFrame(g[["rate"]], geometry=[_box(x - h, y - h, x + h, y + h) for x, y in zip(g["x"], g["y"])],
                              crs=AREA_CRS)
        gg.plot(ax=ax, column="rate", cmap=CMAP_PASSA, norm=norm, edgecolor="white", linewidth=0.15, zorder=2)
    axs = grade_tis(sf_m, desenhar)
    cb = sf_m.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=CMAP_PASSA), ax=axs, location="bottom",
                       shrink=0.42, aspect=34, pad=0.03, extend="min" if gcell["rate"].min() < lo else "neither")
    cb.set_label(f"Arriving fires stopped at early-scar edges (%), {CELL_M / 1000:.0f}-km cells, {YEARS[0]}–{YEARS[-1]}",
                 fontsize=tam(5.8))
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=tam(5.4), width=0.4, length=1.5)
    rotulo_painel(axs[0, 0], "a")

    axb, axc = sf_g.subplots(1, 2, width_ratios=[1, 1.25])
    tl = ORDER8["ti_id"].astype(int).tolist()
    for xi, t in enumerate(tl):
        v = gcell.loc[gcell["ti_id"] == t, "rate"].to_numpy()
        if not len(v):
            continue
        jit = (np.random.default_rng(t).random(len(v)) - 0.5) * 0.5
        axb.scatter(xi + jit, v, s=5, c=CMAP_PASSA(norm(v)), lw=0, alpha=0.85, zorder=2, rasterized=True)
        axb.plot([xi - 0.3, xi + 0.3], [np.median(v)] * 2, color=TINTA, lw=1.2, zorder=3)
    axb.set_xticks(range(len(tl))); axb.set_xticklabels([str(NUM[t]) for t in tl], fontsize=tam(5.8))
    axb.tick_params(axis="x", length=0); axb.grid(axis="x", visible=False)
    axb.set_ylim(max(lo - 3, 0), 101); axb.yaxis.set_major_locator(MaxNLocator(4))
    axb.set_xlabel("IL (numbers as in a)", fontsize=tam(6.0)); axb.set_ylabel("Stopped (%), per cell", fontsize=tam(6.0))
    rotulo_painel(axb, "b")
    rob = R["B10_robustez_segmentos"]
    grupos = [("Seasonal split", ["artigo1", "local", "local_100"], ["Article 1", "Local, 60 mm", "Local, 100 mm"]),
              ("Encounter criteria", ["segmento 150 m", "segmento 600 m", "faixa 60 m", "faixa 150 m",
                                      "chegada >= 20%", "chegada >= 50%"],
               ["Segment 150 m", "Segment 600 m", "Strip 60 m", "Strip 150 m", "Arrival ≥ 20%", "Arrival ≥ 50%"]),
              ("Fuel", ["nativa+pastagem", "nativa"], ["Natural + pasture", "Natural only"])]
    if rob is not None and len(rob):
        rb = rob.drop_duplicates("variante").set_index("variante")
        y = 0.0; ticks, labs = [], []
        axc.axvline(1.0, color=TINTA2, lw=0.7, ls=(0, (3, 2)), zorder=1)
        if "padrao" in rb.index:
            axc.axvspan(rb.loc["padrao", "ic_inf"], rb.loc["padrao", "ic_sup"], color=C_INI, alpha=0.10, lw=0, zorder=0)
        for gname, keys, names in grupos[::-1]:
            ks = [(k, n) for k, n in zip(keys, names) if k in rb.index]
            for k, n in ks[::-1]:
                r = rb.loc[k]; sig = r["ic_inf"] > 1 or r["ic_sup"] < 1; cor = C_INI if sig else "#9a9992"
                axc.plot([r["ic_inf"], r["ic_sup"]], [y, y], color=cor, lw=1.6, solid_capstyle="round", zorder=2)
                axc.scatter(r["rc_conf_split"], y, s=20, color=cor, edgecolor="white", lw=0.5, zorder=3)
                ticks.append(y); labs.append(n); y += 1
            axc.text(1.0, y - 0.35, gname, transform=mpl.transforms.blended_transform_factory(axc.transAxes, axc.transData),
                     ha="right", va="bottom", fontsize=tam(5.4), fontstyle="italic", color=MUDO)
            y += 0.9
        axc.set_yticks(ticks); axc.set_yticklabels(labs, fontsize=tam(5.4))
        axc.set_ylim(-0.7, y - 0.4)
        axc.grid(axis="y", visible=False); axc.tick_params(axis="y", length=0)
        axc.set_xlabel("Odds ratio, window fragmentation (band = default)", fontsize=tam(6.0))
    rotulo_painel(axc, "c")
    salvar(fig, "Fig4_generalisation")

# ============================================================
# FIGURA S2 - TIPOLOGIA DESCRITIVA
# ============================================================
def figS2():
    fig = plt.figure(figsize=(LARG, 6.4 * CM), layout="constrained")
    gs = fig.add_gridspec(1, 3)
    rows = ["Early, large", "Early, small & weak", "Early, small & intense", "Late season"]
    ypos = np.array([3.4, 2.4, 1.4, 0.0])
    axes = [fig.add_subplot(gs[0])]
    axes += [fig.add_subplot(gs[k], sharey=axes[0]) for k in (1, 2)]
    tp, d1, d2t = R["B06_tipologia_resumo"], R["D1_tempo"], R["D2_foco_noturno_taxa"]
    if tp is not None:
        t = tp.copy(); t.index = [TYPE_EN.get(i, i) for i in t.index]
        v = [t.loc[k, "area_ha"] / 1e6 for k in rows]
        axes[0].scatter(v, ypos, s=26, color=[TYPE_COL[k] for k in rows], edgecolor="white", lw=0.6, zorder=3)
        dot_labels(axes[0], v, ypos + 0.42, [f"{x:.2f}" if x >= 0.01 else f"{x:.3f}" for x in v], 0, ha="center", fs=5.4)
        axes[0].set_xscale("log"); axes[0].set_xlim(5e-4, 40)
        axes[0].xaxis.set_major_locator(FixedLocator([0.001, 0.1, 10])); axes[0].xaxis.set_minor_locator(NullLocator())
        axes[0].xaxis.set_major_formatter(FormatStrFormatter("%g"))
    axes[0].set_xlabel("Burned area (Mha)", fontsize=tam(6.2))
    if d1 is not None:
        v = d1[d1["variavel"] == "P_vpd_max"].copy(); v["lab"] = v["tipo"].map(TYPE_EN); v = v.set_index("lab")
        rr = [k for k in rows if k in v.index]
        m = [v.loc[k, "media_percentil"] for k in rr]; yy = [ypos[rows.index(k)] for k in rr]
        axes[1].axvline(0.5, color=MUDO, lw=0.6, ls=(0, (3, 2)), zorder=1)
        for k, yi in zip(rr, yy):
            axes[1].plot([v.loc[k, "ic95_inf"], v.loc[k, "ic95_sup"]], [yi, yi], color=TYPE_COL[k], lw=1.3, alpha=0.85,
                         solid_capstyle="round", zorder=2)
        axes[1].scatter(m, yy, s=26, color=[TYPE_COL[k] for k in rr], edgecolor="white", lw=0.6, zorder=3)
        dot_labels(axes[1], m, np.array(yy) + 0.42, [f"{x:.2f}" for x in m], 0, ha="center", fs=5.4)
        axes[1].set_xlim(0.40, 0.80)
    axes[1].set_xlabel("VPD percentile, 1st MODIS detection", fontsize=tam(6.2))
    if d2t is not None:
        t = d2t.copy(); t.index = [TYPE_EN.get(i, i) for i in t.index]
        rr = [k for k in rows if k in t.index]
        v = [100 * t.loc[k, "mean"] for k in rr]; yy = [ypos[rows.index(k)] for k in rr]
        axes[2].scatter(v, yy, s=26, color=[TYPE_COL[k] for k in rr], edgecolor="white", lw=0.6, zorder=3)
        dot_labels(axes[2], v, np.array(yy) + 0.42, [f"{x:.0f}%" for x in v], 0, ha="center", fs=5.4)
        axes[2].set_xlim(-3, max(v) * 1.3 if v else 10); axes[2].xaxis.set_major_locator(MaxNLocator(3, integer=True))
    axes[2].set_xlabel("Night-time MODIS detections (%)", fontsize=tam(6.2))
    for k_, ax in enumerate(axes):
        ax.axhline(0.7, color=EIXO, lw=0.5, ls=(0, (2, 2)))
        ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0, labelleft=(k_ == 0))
        rotulo_painel(ax, "abc"[k_])
    axes[0].set_yticks(ypos)
    axes[0].set_yticklabels(["Early, large", "Early, small\n& weak", "Early, small\n& intense", "Late season"],
                            fontsize=tam(6.0), linespacing=1.0)
    axes[0].set_ylim(-0.5, 4.0)
    salvar(fig, "FigS2_scar_types")

# ============================================================
# FIGURA S3 - JANELA DE 12 km A 30 m
# ============================================================
def figS3():
    fig = plt.figure(figsize=(LARG, 9.4 * CM), layout="constrained")
    sf_m, sf_d = fig.subfigures(2, 1, height_ratios=[1.62, 0.62], hspace=0.0)
    ti, yr, win = W
    e = JNBR["ext"]; ext_img = (e[0], e[1], e[2], e[3])
    early = np.isin(JNBR["month"], EARLY)
    dd = segs[(segs["ti_id"] == ti) & (segs["year"] == yr) & (segs["tipo_seg"] == "cicatriz") &
              segs["x"].between(win[0], win[2]) & segs["y"].between(win[1], win[3]) &
              segs["resultado"].isin(["interrupcao", "continuidade"])]
    norm_nbr = mpl.colors.TwoSlopeNorm(vmin=-0.3, vcenter=0.15, vmax=0.6)
    norm_d = mpl.colors.Normalize(0, 0.6)
    axs = list(sf_m.subplots(1, 3))
    titulos = [f"NBR, July {yr}", f"NBR, min. Aug–Oct {yr}", "dNBR, July − (Aug–Oct)"]
    pre = np.ma.masked_less(JNBR["pre"], -100) / 100.0
    post = np.ma.masked_less(JNBR["post"], -100) / 100.0
    for k, ax in enumerate(axs):
        ax.add_patch(mpatches.Rectangle(win[:2], win[2] - win[0], win[3] - win[1], color=SEM_DADO, zorder=0))
        if k == 0:
            im_n = ax.imshow(pre, extent=ext_img, cmap=CMAP_NBR, norm=norm_nbr, interpolation="nearest", zorder=1, rasterized=True)
        elif k == 1:
            ax.imshow(post, extent=ext_img, cmap=CMAP_NBR, norm=norm_nbr, interpolation="nearest", zorder=1, rasterized=True)
        else:
            im_d = ax.imshow(np.ma.clip(pre - post, -0.1, 0.8), extent=ext_img, cmap=CMAP_DNBR, norm=norm_d,
                             interpolation="nearest", zorder=1, rasterized=True)
        if early.any():
            ax.contour(np.flipud(early.astype(float)), levels=[0.5], colors=[AZUL], linewidths=0.45, extent=ext_img, zorder=3)
        if k == 2 and len(dd):
            q = dd[dd["resultado"] == "continuidade"]
            ax.scatter(q["x"], q["y"], s=11, color="#d63aa8", edgecolor="white", lw=0.5, zorder=5)
        ti_geom(ti).boundary.plot(ax=ax, color=TINTA, lw=0.5, zorder=4)
        moldura(ax, win, barra_m=2000 if k == 0 else None, seta=(k == 2), fs=5)
        ax.set_title(titulos[k], fontsize=tam(6.4), pad=2, loc="center")
        rotulo_painel(ax, "abc"[k])
    colorbar_fina(sf_m, im_n, axs[:2], "Landsat 8/9 NBR (grey: > 500 m from any burn)", ticks=[-0.3, 0, 0.15, 0.3, 0.6], shrink=0.7)
    colorbar_fina(sf_m, im_d, axs[2], "dNBR", ticks=[0, 0.2, 0.4, 0.6], extend="max", shrink=0.9)
    if len(dd):
        pi = 100 * (dd["resultado"] == "interrupcao").mean()
        axs[2].text(0.02, 0.98, f"{len(dd):,} edges reached\n{pi:.0f}% interrupted", transform=axs[2].transAxes,
                    ha="left", va="top", fontsize=tam(5.6), color=TINTA, linespacing=1.15, path_effects=HALO, zorder=9)
    sf_m.legend(handles=[Line2D([], [], color=AZUL, lw=0.9, label="Early-scar outline (May–Jul)"),
                         Line2D([], [], marker="o", ls="", ms=3.4, mfc="#d63aa8", mec="white", mew=0.5,
                                label="Scar edge crossed by later fire")],
                loc="outside lower center", ncol=2, fontsize=tam(5.8), handlelength=1.2, columnspacing=1.2)
    ax = sf_d.subplots()
    rg = R["B08a_regra_antiga_x_simetrica"]
    if rg is not None:
        g = _regra_sim(rg)
        nomes = [("cicatriz", "Early-scar edges", AZUL), ("referencia", "Reference edges (no earlier burn)", MUDO)]
        for yi, (t, lab, cor) in zip([1, 0], nomes):
            v = g.loc[t, "% interrupcao"]; n = int(g.loc[t, "interrupcao"] + g.loc[t, "continuidade"])
            ax.barh(yi, v, height=0.5, color=cor, alpha=0.9, zorder=2)
            ax.text(v + 1.2 if v < 85 else v - 1.2, yi, f"{v:.1f}%  (n = {n:,})", fontsize=tam(5.8),
                    color=TINTA if v < 85 else "white", ha="left" if v < 85 else "right", va="center", zorder=3)
        ax.set_yticks([1, 0]); ax.set_yticklabels([n[1] for n in nomes], fontsize=tam(6.2))
        ax.set_ylim(-0.5, 1.5); ax.set_xlim(0, 100); ax.set_xticks([0, 25, 50, 75, 100])
        ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0)
        ax.set_xlabel(f"Arriving fires interrupted (%), {len(ORDER8)} ILs, {YEARS[0]}–{YEARS[-1]}", fontsize=tam(6.2))
    rotulo_painel(ax, "d")
    salvar(fig, "FigS3_window_landsat")

# ============================================================
# TABELA 1
# ============================================================
def table1():
    rows = []
    for ti in ORDER8["ti_id"].astype(int):
        r = {"#": NUM.get(ti), "IL": TI_NAME.get(ti, ti)}
        a = tis.loc[tis["ti_id"].astype(int) == ti, "area_ha"]
        r["Area (km²)"] = int(round(float(a.iloc[0]) / 100)) if len(a) else np.nan
        aa = area_anual[area_anual["ti_id"] == ti]
        if len(aa):
            r["Early burned (% yr⁻¹)"] = round(100 * (aa["early_ha"] / aa["ti_ha"]).mean(), 1)
            r["Late burned (% yr⁻¹)"] = round(100 * (aa["late_ha"] / aa["ti_ha"]).mean(), 1)
        g = segs[segs["ti_id"] == ti]
        for t, nome in [("cicatriz", "scar"), ("referencia", "reference")]:
            q = g[(g["tipo_seg"] == t) & g["resultado"].isin(["interrupcao", "continuidade"])]
            if t == "cicatriz":
                r["Scar edges with arrival"] = len(q)
            r[f"Interrupted, {nome} edges (%)"] = round(100 * (q["resultado"] == "interrupcao").mean(), 1) if len(q) else np.nan
        if art1 is not None:
            cod = tis.loc[tis["ti_id"].astype(int) == ti, "terrai_cod"]
            a1 = art1[art1["terrai_cod"] == int(cod.iloc[0])] if len(cod) else art1.iloc[0:0]
            r["Article-1 class"] = a1["classe"].iloc[0] if len(a1) else ""
            r["FRRI"] = round(float(a1["frri"].iloc[0]), 2) if len(a1) and pd.notna(a1["frri"].iloc[0]) else np.nan
        rows.append(r)
    t = pd.DataFrame(rows)
    t.to_csv(FIG_DIR / "Table1_analysed_ILs.csv", index=False)
    display(t)
    return t

CAP = {
    "Fig1_premise": "Figure 1. Early dry-season scars and their timing. (a) Dominant burn season of each 30-m pixel in "
                    f"the analysed Indigenous Lands (ILs), {YEARS[0]}–{YEARS[-1]} (MapBiomas Fire). (b) Annual burned area "
                    "in May–July and August–October. (c) Monthly minimum Landsat 8/9 NBR of pixels burned early, burned "
                    "later and unburned (median and interquartile range; NBR within 500 m of burns).",
    "Fig2_edge_test": "Figure 2. Do early scars stop arriving fire? (a) Each IL in the year with most scar edges reached by "
                      "later fire: Landsat dNBR between July and the August–October minimum, early-scar outlines and 300-m "
                      "scar edges crossed by later fire (magenta). (b) Arriving fires stopped at early-scar and reference "
                      "edges with the same Landsat + MODIS entry rule, and the adjusted odds ratio. (c) The same in each IL.",
    "Fig3_configuration": "Figure 3. Configuration beyond quantity. (a) Early scars coloured by size, later fire and crossed "
                          "edges. (b, c) Share of arriving fires stopped by early-scar size and local width (95% binomial "
                          "interval). (d) Odds ratios of stopping per SD beyond the reference model with quantity (SE "
                          "clustered by fire and IL; open symbol, CI includes 1).",
    "Fig4_generalisation": f"Figure 4. Generalisation. (a) Share of arriving fires stopped at early-scar edges in "
                           f"{CELL_M / 1000:.0f}-km cells (>= 10 edges). (b) Distribution of the cells in each IL (bar, "
                           "median). (c) Odds ratio for window fragmentation under alternative seasonal splits, encounter "
                           "criteria and fuel definitions, one change at a time (95% CI; band, default model).",
    "FigS2_scar_types": "Figure S2. Descriptive types of early dry-season scars (>= 1 ha): (a) burned area; (b) mean VPD "
                        "percentile on the day of the first MODIS detection (ERA5-Land; IL-bootstrap 95% CI); (c) share of "
                        "scars with night-time MODIS detections. Types are not attributed to management or brigades.",
    "FigS3_window_landsat": f"Figure S3. Detail at 30 m: a 12 × 12 km window of {TI_NAME.get(W[0])}, {W[1]}. (a–c) Landsat "
                            "8/9 NBR in July, minimum NBR of August–October and dNBR, with early-scar outlines and crossed "
                            "edges. (d) Share of arriving fires stopped at early-scar and reference edges, all ILs.",
}

console.print("[bold cyan]-- Figuras da letter --[/bold cyan]")
for fn in [fig1, fig2, fig3, fig4, figS2, figS3]:
    try:
        fn()
    except Exception as e:
        console.print(f"[yellow]{fn.__name__} pulada: {str(e)[:160]}[/yellow]")
t1 = table1()

d = docx.Document()
d.styles["Normal"].font.name = "Times New Roman"; d.styles["Normal"].font.size = Pt(10)
d.add_heading("Figures and Table – early-burn configuration and fire continuity", 1)
for name, cap in CAP.items():
    png = FIG_DIR / f"{name}.png"
    if png.exists():
        d.add_picture(str(png), width=DocxCm(LARGURA_CM))
        d.add_paragraph(cap)
d.add_paragraph("Table 1. Analysed Indigenous Lands (numbers as in Fig. 1): area in the Cerrado, mean annual share burned "
                f"in May–July and August–October ({YEARS[0]}–{YEARS[-1]}), early-scar edges reached by later fire with a "
                "defined outcome, share interrupted at scar and reference edges (same Landsat + MODIS rule), and the class "
                "and FRRI of our first article.")
tb = d.add_table(rows=1, cols=len(t1.columns)); tb.style = "Light Grid Accent 1"
for j, c in enumerate(t1.columns):
    tb.rows[0].cells[j].text = str(c)
for row in t1.itertuples(index=False):
    cl_ = tb.add_row().cells
    for j, x in enumerate(row):
        cl_[j].text = "" if (isinstance(x, float) and np.isnan(x)) else (f"{x:,}" if isinstance(x, (int, np.integer)) else str(x))
d.save(TXT_DIR / "RSL_figuras_tabela.docx")
console.print(f"[bold green]OK BLOCO 16 completo: figuras em {FIG_DIR}.[/bold green]")


## 17 figuras_sensoriamento

In [ ]:
# ======================================================================
# BLOCO 17 - Figuras de sensoriamento remoto 600 dpi (TIFF LZW + PDF vetorial + PNG), em ingles
# Rode DEPOIS do bloco 16 e do bloco 11 na MESMA sessao (usa o estilo e as funcoes deles:
# salvar, moldura, grade_tis, mapa_localizacao, W, JNBR, CTX, core_counts).
# Depende tambem dos blocos 02, 05 (MODIS), 09 (SES) e 10 (segmentos).
# ======================================================================

check_version()
for _f in ["salvar", "moldura", "grade_tis", "mapa_localizacao", "rotulo_painel", "W", "CTX", "ORDER8"]:
    if _f not in globals():
        raise RuntimeError(f"'{_f}' nao definido - rode o bloco 16 nesta sessao antes do bloco 17.")

# ============================================================
# Figuras (todas com mapa; numeros escritos no proprio dado):
#   FigR1  Area e amostra: (a) localizacao no Brasil/Cerrado com grade lat/long,
#          14 TIs (8 analisadas, 6 excluidas com o motivo) | (b) triagem TI x ano:
#          maior intervalo entre imagens Landsat limpas e encontros candidatos
#   FigR2  O encontro visto do satelite (30 m, janela de 12 km do bloco 16):
#          (a) Landsat 8/9 falsa-cor SWIR2-NIR-red, fim do inicio da seca |
#          (b) idem, fim da seca | (c) dNBR + cicatrizes iniciais (MapBiomas),
#          pegadas dos pixels MODIS do restante e bordas interrompidas/atravessadas
#   FigR3  Teste independente do NBR: (a) mapa dos nucleos de cicatriz e do
#          combustivel nao queimado com os focos MODIS do restante | (b) razao
#          de taxas por TI (mapa) | (c) razao agregada com IC por bootstrap de TIs
#   FigR4  Configuracao pelo modelo nulo: SES medio da fragmentacao por celula
#          de 5 km em cada TI (azul = cicatrizes dividem o combustivel mais que o acaso)
# ============================================================

from pyproj import Transformer
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as mpatches

TAG_SEG    = globals().get("TAG_SEG", f"{TAG_YEARS}_md")
FOCOS_NAME = globals().get("FOCOS_NAME", f"modis_focos_v1_{TAG_YEARS}")
_to_ll = Transformer.from_crs(AREA_CRS, "EPSG:4326", always_xy=True)
_to_xy = Transformer.from_crs("EPSG:4326", AREA_CRS, always_xy=True)
GRAT = "#b9b8b0"

# ------------------------------------------------------------
# Grade de coordenadas (lat/long) desenhada sobre mapas em EPSG:5880
# ------------------------------------------------------------
def graticule(ax, ext, step=None, labels=True, fs=5.4):
    x0, y0, x1, y1 = ext
    lons, lats = _to_ll.transform([x0, x1, x0, x1], [y0, y0, y1, y1])
    lo0, lo1, la0, la1 = min(lons), max(lons), min(lats), max(lats)
    if step is None:
        span = max(lo1 - lo0, la1 - la0)
        step = next(s for s in [0.05, 0.1, 0.25, 0.5, 1, 2, 5, 10] if span / s <= 5)
    fmt = (lambda v, ew: f"{abs(v):.{0 if step >= 1 else (1 if step >= 0.1 else 2)}f}°{ew}")
    for lo in np.arange(np.ceil(lo0 / step) * step, lo1, step):
        la = np.linspace(la0 - 1, la1 + 1, 200)
        xs, ys = _to_xy.transform(np.full_like(la, lo), la)
        ax.plot(xs, ys, color=GRAT, lw=0.3, zorder=1.5, clip_on=True)
        if labels:
            xb, _ = _to_xy.transform(lo, la0)
            if x0 < xb < x1:
                ax.text(xb, y0 + 0.01 * (y1 - y0), fmt(lo, "W" if lo < 0 else "E"), fontsize=tam(fs), color=TINTA2,
                        ha="center", va="bottom", path_effects=HALO, zorder=9)
    for la in np.arange(np.ceil(la0 / step) * step, la1, step):
        lo = np.linspace(lo0 - 1, lo1 + 1, 200)
        xs, ys = _to_xy.transform(lo, np.full_like(lo, la))
        ax.plot(xs, ys, color=GRAT, lw=0.3, zorder=1.5, clip_on=True)
        if labels:
            _, yb = _to_xy.transform(lo0, la)
            if y0 < yb < y1:
                ax.text(x0 + 0.01 * (x1 - x0), yb, fmt(la, "S" if la < 0 else "N"), fontsize=tam(fs), color=TINTA2,
                        ha="left", va="center", path_effects=HALO, zorder=9)

# ============================================================
# FIG R1 - AREA DE ESTUDO E TRIAGEM DA AMOSTRA
# ============================================================
def figR1():
    am_p = table_path("amostra_final_tis")
    am = pd.read_csv(am_p)
    ty_p = stats_path("elegibilidade_encontros_ti_ano")
    ty = pd.read_csv(ty_p)
    fig = plt.figure(figsize=(LARG, 15.5 * CM), layout="constrained")
    sf_a, sf_b = fig.subfigures(2, 1, height_ratios=[1.35, 1], hspace=0.03)
    ax = sf_a.subplots()
    ext = mapa_localizacao(ax, rotulos=True)
    excl = am[~am["elegivel"].astype(bool)]
    for _, r in excl.iterrows():
        g = tis[tis["ti_id"].astype(int) == int(r["ti_id"])]
        if len(g):
            p = g.geometry.iloc[0].representative_point()
            cod = "I" if "imagens" in str(r["motivo_exclusao"]) else "E"
            ax.annotate(cod, (p.x, p.y), xytext=(4, -6), textcoords="offset points", fontsize=tam(5.6),
                        color=TINTA2, fontstyle="italic", path_effects=HALO, zorder=6)
    graticule(ax, ext, step=5)
    moldura(ax, ext, barra_m=500000, seta=True, fs=5.6)
    # inset: America do Sul / Brasil
    axi = ax.inset_axes([0.70, 0.66, 0.30, 0.34])
    if "brasil" in CTX:
        CTX["brasil"].plot(ax=axi, facecolor=FUNDO, edgecolor="#9a9992", lw=0.3)
    if "cerrado" in CTX:
        CTX["cerrado"].plot(ax=axi, facecolor=CERRADO, edgecolor=CERRADO_BORDA, lw=0.3)
    axi.add_patch(mpatches.Rectangle(ext[:2], ext[2] - ext[0], ext[3] - ext[1], fill=False, ec=TINTA, lw=0.5))
    axi.set_aspect("equal"); axi.set_axis_off()
    sf_a.legend(handles=[mpatches.Patch(fc=LARANJA, ec=TINTA, lw=0.35, label="Analysed IL (numbers as in Table 1)"),
                         mpatches.Patch(fc="white", ec=TINTA2, lw=0.5, label="Excluded IL: I = imagery, E = encounters"),
                         mpatches.Patch(fc=CERRADO, ec=CERRADO_BORDA, lw=0.45, label="Cerrado biome")],
                loc="outside lower center", ncol=3, fontsize=tam(5.6), handlelength=1.1, columnspacing=1.0)
    rotulo_painel(ax, "a")

    axb = sf_b.subplots()
    order = am.sort_values(["elegivel", "ti_name"], ascending=[False, True])["ti_id"].astype(int).tolist()
    gap = ty.pivot_table(index="ti_id", columns="year", values="max_gap_dias").reindex(order)
    enc = ty.pivot_table(index="ti_id", columns="year", values="n_encontros", fill_value=0).reindex(order)
    cm_gap = LinearSegmentedColormap.from_list("gap", ["#1f5fc8", "#a9c8ee", "#fbe3c4", "#e0532b", "#7a1f3d"])
    im = axb.imshow(gap.values, cmap=cm_gap, norm=mpl.colors.Normalize(8, 96), aspect="auto", interpolation="nearest")
    for i in range(gap.shape[0]):
        for j in range(gap.shape[1]):
            v = int(enc.values[i, j])
            if v:
                axb.text(j, i, f"{v}" if v < 1000 else f"{v / 1000:.1f}k", ha="center", va="center",
                         fontsize=tam(4.6), color="white" if gap.values[i, j] < 20 or gap.values[i, j] > 70 else TINTA)
    axb.set_xticks(range(gap.shape[1])); axb.set_xticklabels(gap.columns, fontsize=tam(5.4))
    axb.set_yticks(range(gap.shape[0]))
    nm = tis.set_index("ti_id")["ti_name"]
    axb.set_yticklabels([f"{NUM.get(t, '–')} {nm[t]}" for t in order], fontsize=tam(5.4))
    n_ok = int(am["elegivel"].sum())
    axb.axhline(n_ok - 0.5, color=TINTA, lw=0.8)
    axb.grid(False); axb.tick_params(length=0)
    cb = sf_b.colorbar(im, ax=axb, fraction=0.03, pad=0.01, extend="max")
    cb.set_label("Longest gap between clear\nLandsat 8/9 dates, Jun–Oct (days)", fontsize=tam(5.6))
    cb.ax.tick_params(labelsize=tam(5.2)); cb.outline.set_linewidth(0.4)
    axb.set_title("numbers: candidate encounters (late fire touching early scar); line: analysed above, excluded below",
                  fontsize=tam(5.6), color=MUDO, loc="left", pad=2)
    rotulo_painel(axb, "b")
    salvar(fig, "FigR1_study_area_sample")

# ============================================================
# FIG R2 - O ENCONTRO VISTO PELO LANDSAT E PELO MODIS (30 m)
# ============================================================
def landsat_rgb(win, d0, d1, px=30):
    """Mediana Landsat 8/9 SR (SWIR2, NIR, red) sem nuvem na janela, grade EPSG:5880 de 30 m alinhada."""
    x0, y0, x1, y1 = win
    n_c, n_r = int(round((x1 - x0) / px)), int(round((y1 - y0) / px))
    rect = ee.Geometry.Rectangle([x0, y0, x1, y1], PROJ, False)
    def prep(i):
        i = ee.Image(i)
        clear = i.select("QA_PIXEL").bitwiseAnd(0b11111).eq(0)
        return i.select(["SR_B7", "SR_B5", "SR_B4"]).multiply(0.0000275).add(-0.2).updateMask(clear)
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(rect).filterDate(d0, d1))
    n_img = col.size().getInfo()
    img = col.map(prep).median().unmask(-1).toFloat()
    arr = ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY",
                                 "grid": {"dimensions": {"width": n_c, "height": n_r},
                                          "affineTransform": {"scaleX": px, "shearX": 0, "translateX": x0,
                                                              "shearY": 0, "scaleY": -px, "translateY": y1},
                                          "crsCode": AREA_CRS}})
    rgb = np.dstack([arr[b] for b in ["SR_B7", "SR_B5", "SR_B4"]]).astype(float)
    rgb[rgb < 0] = np.nan
    return rgb, n_img

def stretch(rgb, lo=2, hi=98, ref=None):
    ref = rgb if ref is None else ref
    out = np.empty_like(rgb)
    for k in range(3):
        a, b = np.nanpercentile(ref[..., k], [lo, hi])
        out[..., k] = np.clip((rgb[..., k] - a) / (b - a), 0, 1)
    return np.nan_to_num(out ** 0.9, nan=0.9)

def figR2():
    ti, yr, win = W
    pre, n_pre = landsat_rgb(win, f"{yr}-06-15", f"{yr}-08-01")
    post, n_post = landsat_rgb(win, f"{yr}-09-15", f"{yr}-11-01")
    both = np.concatenate([pre, post], axis=0)                     # mesmo estiramento nas duas datas
    pre_s, post_s = stretch(pre, ref=both), stretch(post, ref=both)
    ext_i = (win[0], win[2], win[1], win[3])
    early = np.isin(JNBR["month"], EARLY).astype(float) if JNBR is not None else None
    fo = read_table(cache_path(FOCOS_NAME, "parquet"))
    fo = fo[(fo["confidence"].astype(str).str[0] != "l")]
    fo["acq_date"] = pd.to_datetime(fo["acq_date"])
    fo = fo[(fo["acq_date"].dt.year == yr) & fo["acq_date"].dt.month.isin(LATE)]
    fx, fy = _to_xy.transform(fo["longitude"].to_numpy(), fo["latitude"].to_numpy())
    inw = (fx > win[0]) & (fx < win[2]) & (fy > win[1]) & (fy < win[3])
    fx, fy = fx[inw], fy[inw]
    sg = pd.read_parquet(cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet"),
                         columns=["ti_id", "year", "tipo_seg", "resultado", "x", "y"])
    sg = sg[(sg["ti_id"] == ti) & (sg["year"] == yr) & (sg["tipo_seg"] == "cicatriz") &
            sg["x"].between(win[0], win[2]) & sg["y"].between(win[1], win[3])]

    fig = plt.figure(figsize=(LARG, 7.6 * CM), layout="constrained")
    axs = fig.subplots(1, 3)
    titles = [f"Landsat 8/9, 15 Jun–31 Jul {yr} (n = {n_pre})", f"Landsat 8/9, 15 Sep–31 Oct {yr} (n = {n_post})",
              "dNBR, scars, MODIS and edges"]
    axs[0].imshow(pre_s, extent=ext_i, interpolation="nearest", rasterized=True, zorder=1)
    axs[1].imshow(post_s, extent=ext_i, interpolation="nearest", rasterized=True, zorder=1)
    if JNBR is not None:
        dn = np.ma.masked_less(JNBR["pre"], -100) / 100.0 - np.ma.masked_less(JNBR["post"], -100) / 100.0
        e = JNBR["ext"]
        axs[2].add_patch(mpatches.Rectangle(win[:2], win[2] - win[0], win[3] - win[1], color=SEM_DADO, zorder=0))
        im = axs[2].imshow(np.ma.clip(dn, -0.1, 0.8), extent=(e[0], e[1], e[2], e[3]), cmap=CMAP_DNBR,
                           norm=mpl.colors.Normalize(0, 0.6), interpolation="nearest", rasterized=True, zorder=1)
    for ax in axs:
        if early is not None and early.any():
            ax.contour(np.flipud(early), levels=[0.5], colors=["#5fd3ff" if ax is not axs[2] else C_INI],
                       linewidths=0.45, extent=ext_i, zorder=3)
    for x, y in zip(fx, fy):                                         # pegada nominal do pixel MODIS (~926 m)
        axs[2].add_patch(mpatches.Rectangle((x - 463, y - 463), 926, 926, fill=False, ec="#7a1f3d", lw=0.5, zorder=4))
    ok = sg[sg["resultado"] == "interrupcao"]; ct = sg[sg["resultado"] == "continuidade"]
    axs[2].scatter(ok["x"], ok["y"], s=2.5, color="#163f73", lw=0, zorder=5)
    axs[2].scatter(ct["x"], ct["y"], s=9, color=C_PASSOU, edgecolor="white", lw=0.4, zorder=6)
    for k, ax in enumerate(axs):
        ti_geom(ti).boundary.plot(ax=ax, color="white" if k < 2 else TINTA, lw=0.5, zorder=4)
        graticule(ax, win, labels=(k == 0))
        moldura(ax, win, barra_m=2000 if k == 0 else None, seta=(k == 2), fs=5)
        ax.set_title(titles[k], fontsize=tam(5.8), pad=2)
        rotulo_painel(ax, "abc"[k])
    if JNBR is not None:
        colorbar_fina(fig, im, axs[2], "dNBR (July − min Aug–Oct)", ticks=[0, 0.2, 0.4, 0.6], extend="max")
    fig.legend(handles=[Line2D([], [], color=C_INI, lw=0.9, label="Early scar, May–Jul (MapBiomas)"),
                        mpatches.Patch(fill=False, ec="#7a1f3d", lw=0.6, label="MODIS fire pixel, Aug–Oct"),
                        Line2D([], [], marker="o", ls="", ms=2, color="#163f73", label="Edge: fire interrupted"),
                        Line2D([], [], marker="o", ls="", ms=3.4, mfc=C_PASSOU, mec="white", label="Edge: fire crossed")],
               loc="outside lower center", ncol=4, fontsize=tam(5.4), handlelength=1.0, columnspacing=0.9)
    salvar(fig, "FigR2_encounter_landsat_modis")

# ============================================================
# FIG R3 - TESTE INDEPENDENTE DO NBR (FOCOS MODIS NOS NUCLEOS)
# ============================================================
def figR3():
    rr = pd.read_csv(stats_path("B08b_modis_razao_taxas"))
    pt = pd.read_csv(stats_path("B08b_modis_razao_taxas_por_TI"), index_col=0)
    nuc = pd.read_csv(stats_path("B08b_modis_nucleos_ti_ano"))
    fig = plt.figure(figsize=(LARG, 13.2 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.25, 1], hspace=0.03)
    # (a) mapa do TI-ano com maior area de nucleo de cicatriz
    ex = nuc[nuc["core_m"] == nuc["core_m"].min()].sort_values("km2_cicatriz", ascending=False).iloc[0]
    _, M = core_counts(int(ex["ti_id"]), int(ex["year"]), int(ex["core_m"]), keep_masks=True)
    tr = M["tr"]; Hh, Ww = M["E"].shape
    e = (tr.c, tr.f - Hh * PIX_M, tr.c + Ww * PIX_M, tr.f)
    cls = np.zeros(M["E"].shape, np.uint8)
    cls[M["inside"]] = 1; cls[M["L"]] = 2; cls[M["E"]] = 3; cls[M["coreE"]] = 4; cls[M["coreU"]] = 5
    cmap = ListedColormap(["#ffffff", NEUTRO, "#f9c9a2", AZUL_CLARO, C_INI, "#cfe3c3"])
    ax = sf_m.subplots()
    f = int(np.ceil(max(cls.shape) / 1800))
    ax.imshow(np.ma.masked_equal(cls[::f, ::f], 0), cmap=cmap, vmin=0, vmax=5, extent=(e[0], e[2], e[1], e[3]),
              interpolation="nearest", rasterized=True, zorder=1)
    ax.scatter(*M["pts"], s=2.2, color=VINHO, lw=0, zorder=4)
    ti_geom(int(ex["ti_id"])).boundary.plot(ax=ax, color=TINTA, lw=0.5, zorder=5)
    bb = ti_geom(int(ex["ti_id"])).total_bounds
    h = max(bb[2] - bb[0], bb[3] - bb[1]) / 2 * 1.03; cx, cy = (bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2
    ext = (cx - h, cy - h, cx + h, cy + h)
    graticule(ax, ext)
    moldura(ax, ext, barra_m=barra_redonda(2 * h), seta=True, fs=5.4)
    ax.set_title(f"{TI_NAME.get(int(ex['ti_id']))}, {int(ex['year'])}", fontsize=tam(6.2), loc="left")
    sf_m.legend(handles=[mpatches.Patch(color=AZUL_CLARO, label="Early scar (May–Jul)"),
                         mpatches.Patch(color=C_INI, label="Scar core, > 1 km from edge"),
                         mpatches.Patch(color="#cfe3c3", label="Unburned-fuel core, > 1 km"),
                         mpatches.Patch(color="#f9c9a2", label="Burned Aug–Oct"),
                         Line2D([], [], marker="o", ls="", ms=2.6, color=VINHO, label="MODIS fire pixel, Aug–Oct")],
                loc="outside right center", ncol=1, fontsize=tam(5.6), handlelength=1.0)
    rotulo_painel(ax, "a")
    # (b) razao de taxas por TI no mapa de localizacao
    axb, axc = sf_g.subplots(1, 2, width_ratios=[1.25, 1])
    vals = {int(k): float(v) for k, v in pt["razao_taxas"].items() if np.isfinite(v) and v > 0}
    norm = mpl.colors.LogNorm(max(min(vals.values()), 1e-3), 1) if vals else mpl.colors.LogNorm(1e-2, 1)
    cm_rr = LinearSegmentedColormap.from_list("rr", ["#163f73", "#2f78b4", "#9ec9e2", "#fbe3c4"])
    ext_l = mapa_localizacao(axb, cor_ti=vals, cmap=cm_rr, norm=norm)
    graticule(axb, ext_l, step=5, fs=5)
    moldura(axb, ext_l, barra_m=500000, seta=False, fs=5)
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cm_rr), ax=axb, fraction=0.04, pad=0.01)
    cb.set_label("MODIS rate ratio,\nscar / unburned core", fontsize=tam(5.6)); cb.ax.tick_params(labelsize=tam(5))
    rotulo_painel(axb, "b")
    # (c) razao agregada (log)
    lab = {"naoqueimado": "All unburned fuel", "naoqueimado_local": "Unburned ≤ 5 km of a scar"}
    y = 0
    for cm_ in sorted(rr["core_m"].unique()):
        for ref in ["naoqueimado", "naoqueimado_local"]:
            r = rr[(rr["core_m"] == cm_) & (rr["referencia"] == ref)]
            if not len(r):
                continue
            r = r.iloc[0]
            axc.plot([r["ic95_inf_boot_TI"], r["ic95_sup_boot_TI"]], [y, y], color=C_INI, lw=1.8, solid_capstyle="round")
            axc.scatter(r["razao_taxas_agregada"], y, s=26, color=C_INI, edgecolor="white", zorder=3)
            axc.text(r["ic95_sup_boot_TI"] * 1.15, y, f"{r['razao_taxas_agregada']:.2f}", va="center", fontsize=tam(5.6))
            axc.text(-0.02, y, f"{lab[ref]}\ncore > {cm_ / 1000:g} km", transform=mpl.transforms.blended_transform_factory(
                axc.transAxes, axc.transData), ha="right", va="center", fontsize=tam(5.4), linespacing=1.0)
            y += 1
    axc.axvline(1, color=TINTA2, lw=0.7, ls=(0, (3, 2)))
    axc.set_xscale("log"); axc.set_yticks([]); axc.set_ylim(-0.6, y - 0.4)
    axc.set_xlabel("Later-fire MODIS density, scar core / unburned core\n(95% CI, bootstrap over ILs)", fontsize=tam(5.8))
    axc.grid(axis="y", visible=False)
    rotulo_painel(axc, "c")
    salvar(fig, "FigR3_modis_independent_test")

# ============================================================
# FIG R4 - CONFIGURACAO PELO MODELO NULO, NO MAPA
# ============================================================
def figR4():
    cn = pd.read_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[FUEL_MAIN]}_{TAG_YEARS}_nulo", "parquet"))
    cn = cn[cn["q_inicio"] > 0].dropna(subset=["conf_split_ses"])
    g = cn.groupby(["ti_id", "x", "y"]).agg(ses=("conf_split_ses", "mean"), n=("year", "size")).reset_index()
    g = g[g["n"] >= 3]
    norm = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
    cm_ses = LinearSegmentedColormap.from_list("ses", ["#163f73", "#2f78b4", "#9ec9e2", "#f4f3ef", "#f9c9a2", "#e0532b", "#7a1f3d"])
    fig = plt.figure(figsize=(LARG, 9.8 * CM), layout="constrained")

    def desenhar(ax, ti):
        d_ = g[g["ti_id"] == ti]
        hh = CELL_M / 2
        if len(d_):
            gg = gpd.GeoDataFrame(d_[["ses"]], geometry=[_box(x - hh, y - hh, x + hh, y + hh) for x, y in zip(d_["x"], d_["y"])],
                                  crs=AREA_CRS)
            gg.plot(ax=ax, column="ses", cmap=cm_ses, norm=norm, edgecolor="white", linewidth=0.12, zorder=2)
    axs = grade_tis(fig, desenhar, info=lambda ti: (f"median SES {g.loc[g['ti_id'] == ti, 'ses'].median():.2f}"
                                                   if (g["ti_id"] == ti).any() else ""))
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cm_ses), ax=axs, location="bottom", shrink=0.45,
                      aspect=34, pad=0.02, extend="both")
    cb.set_label("Fuel-fragmentation SES vs null model (mean 2014–2024; blue: early scars split fuel more than chance)",
                 fontsize=tam(5.6))
    cb.ax.tick_params(labelsize=tam(5.2)); cb.outline.set_linewidth(0.4)
    rotulo_painel(axs[0, 0], "a")
    salvar(fig, "FigR4_null_model_configuration")

# ============================================================
# RODAR (cada figura isolada: se faltar um insumo, so ela e pulada)
# ============================================================
for fn in [figR1, figR2, figR3, figR4]:
    try:
        fn()
    except Exception as e:
        console.print(f"[yellow]{fn.__name__} pulada: {str(e)[:160]}[/yellow]")
console.print(f"[bold green]OK BLOCO 17 completo: figuras em {FIG_DIR} (600 dpi .tif/.pdf/.png).[/bold green]")


## 18 todas_figuras

In [ ]:
# ======================================================================
# BLOCO 18 - TODAS AS FIGURAS DO PROJETO, num bloco so (600 dpi TIFF LZW + PDF vetorial + PNG)
# Depende so do bloco 00 e dos resultados ja gravados pelos blocos 01-14. Nao recalcula analises.
# Cada figura roda isolada: se faltar um insumo, so ela e pulada (aviso amarelo).
#
#   Letter     Fig1 premissa | Fig2 o teste | Fig3 configuracao | Fig4 generalizacao   (eram do bloco 16)
#   Sensoriam. FigR1 area e triagem | FigR2 encontro Landsat+MODIS | FigR3 teste MODIS | FigR4 SES  (bloco 17)
#   Suplem.    FigS2 tipologia (07) | FigS3 janela 30 m (16) | FigS4 estacao seca (06) | FigS5 MODIS x
#              cicatrizes (05) | FigS6 detectabilidade (11) | FigS7 inferencia 8 TIs (13) | FigS8 modelo
#              nulo x quantidade (09) | FigS9 resultado dos segmentos (10)
#   + Tabela 1 e 08_textos/RSL_figuras_tabela.docx (figuras + legendas)
# Saida: 05_figures/letter/
# ======================================================================

# Confere se o 00_configuracao.py DESTA pasta (RSL - LETTER II) rodou nesta sessao
if globals().get("BLOCKS_VERSION") != "2026-10-17-final" or "TAG_SEG" not in globals():
    raise RuntimeError(
        "O 00_configuracao.py da pasta RSL - LETTER II nao foi rodado nesta sessao "
        f"(versao carregada: {globals().get('BLOCKS_VERSION')}). "
        "Faca: Ambiente de execucao > Reiniciar sessao; rode o 00_configuracao.py; depois rode este bloco 18.")

JANELA = None
LARGURA_CM = 14.0
ESCALA_FONTE = 1.2

import sys, shutil
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, NullLocator, FormatStrFormatter, MaxNLocator
import rasterio
from shapely.geometry import shape, box as _box
import docx
from docx.shared import Pt, Cm as DocxCm

if "google.colab" in sys.modules:
    os.system("apt-get -qq install -y fonts-liberation > /dev/null 2>&1")
    font_manager.fontManager = font_manager._load_fontmanager(try_read_cache=False)

FIG_DIR = DIRS["figures"] / "letter"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TXT_DIR = PROJECT_ROOT / "08_textos"
TXT_DIR.mkdir(exist_ok=True)
FIRE_DIR = DIRS["exports"] / "fogo_mes"
SEQ_DIR = DIRS["exports"] / "sequencia_md"

# ============================================================
# ESTILO
# ============================================================
CM = 1 / 2.54
LARG = LARGURA_CM * CM
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRADE, EIXO, NEUTRO, FUNDO = "#e1e0d9", "#c3c2b7", "#f0efec", "#f4f3ef"
AZUL, LARANJA = "#2a78d6", "#eb6834"
AZUL_CLARO, ROXO, VINHO = "#a9c8ee", "#9b8ac4", "#7a1f3d"
CERRADO, CERRADO_BORDA = "#e2ead8", "#8aa36d"

def tam(pt):
    return round(pt * ESCALA_FONTE, 2)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": tam(7.5), "axes.labelsize": tam(7.5), "axes.titlesize": tam(7.5),
    "xtick.labelsize": tam(7), "ytick.labelsize": tam(7), "legend.fontsize": tam(6.8),
    "text.color": TINTA, "axes.labelcolor": TINTA, "axes.titlecolor": TINTA,
    "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.edgecolor": EIXO, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "legend.frameon": False, "legend.handlelength": 1.6,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.dpi": 600, "figure.dpi": 150,
})
HALO = [pe.withStroke(linewidth=1.8, foreground="white")]
_PAINEIS = []

def _raiz(ax):
    f = ax.figure
    while getattr(f, "figure", f) is not f and hasattr(f, "figure"):
        f = f.figure
    return f

def rotulo_painel(ax, letra, **_):
    """(a), (b)... posicionados no salvar(), depois do layout."""
    _PAINEIS.append((ax, letra))

def _colocar_rotulos(fig):
    meus = [p for p in _PAINEIS if _raiz(p[0]) is fig]
    if meus and fig.get_layout_engine() is not None:
        topo = 0.65 / (fig.get_figheight() * 2.54)
        fig.get_layout_engine().set(rect=(0, 0.25 * topo, 1, 1 - 1.25 * topo))
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    fixos = []
    for ax, letra in meus:
        bb = ax.get_tightbbox(r).transformed(inv)
        fixos.append((max(bb.x0, 0.0), bb.y1 + 0.004, letra))
    fig.set_layout_engine("none")
    for i, (x, y, l_) in enumerate(fixos):            # letras da mesma linha na mesma altura
        mesma = [yy for _, yy, _ in fixos if abs(yy - y) < 0.12]
        fixos[i] = (x, max(mesma), l_)
    for x, y, letra in fixos:
        fig.text(x, y, f"({letra})", fontweight="bold", fontsize=tam(8), ha="left", va="bottom")
    _PAINEIS[:] = [p for p in _PAINEIS if _raiz(p[0]) is not fig]

def salvar(fig, nome):
    _colocar_rotulos(fig)
    for ext, extra in [("pdf", {}), ("tif", {"pil_kwargs": {"compression": "tiff_lzw"}}), ("png", {})]:
        fig.savefig(FIG_DIR / f"{nome}.{ext}", dpi=600, facecolor="white", **extra)
    plt.show()
    plt.close(fig)
    console.print(f"[green]  {nome}: .pdf / .tif / .png (600 dpi)[/green]")

from matplotlib.offsetbox import AnchoredOffsetbox, AuxTransformBox, DrawingArea, HPacker, TextArea, VPacker

FS_MAPA = 5.2          # mesmo tamanho de texto de escala e norte em todos os mapas

def _caixa(ax, filho, canto):
    ab = AnchoredOffsetbox(loc=canto, child=filho, pad=0.25, borderpad=0.35, frameon=True)
    ab.patch.set_boxstyle("round,pad=0.12")
    ab.patch.set_facecolor("white"); ab.patch.set_edgecolor("none"); ab.patch.set_alpha(0.88)
    ab.set_zorder(9)
    ax.add_artist(ab)

def moldura(ax, ext, barra_m=None, seta=True, canto_barra="lower left", canto_seta="lower right", **_):
    """Padrao de todos os mapas: barra de escala no canto inferior esquerdo, com o rotulo AO LADO da barra,
    e seta de norte no canto inferior direito, ambas numa caixa branca (nunca sobre texto ou outra marca)."""
    fs = tam(FS_MAPA)
    ax.set_xlim(ext[0], ext[2]); ax.set_ylim(ext[1], ext[3])
    ax.set_aspect("equal")
    if not getattr(ax, "_grat", False):
        ax.set_axis_off()
    h = ext[3] - ext[1]
    if barra_m:
        th = 0.018 * h
        bar = AuxTransformBox(ax.transData)
        bar.add_artist(mpatches.Rectangle((0, 0), barra_m / 2, th, fc=TINTA, ec=TINTA, lw=0.4))
        bar.add_artist(mpatches.Rectangle((barra_m / 2, 0), barra_m / 2, th, fc="white", ec=TINTA, lw=0.4))
        rot = f"{barra_m / 1000:g} km" if barra_m >= 1000 else f"{barra_m:g} m"
        txt = TextArea(rot, textprops=dict(fontsize=fs, color=TINTA))
        _caixa(ax, HPacker(children=[bar, txt], align="center", pad=0, sep=2.5), canto_barra)
    if seta:
        w_, h_ = fs * 0.75, fs * 1.25
        da = DrawingArea(w_, h_, 0, 0)
        da.add_artist(mpatches.Polygon([(w_ / 2, h_), (0, 0), (w_ / 2, 0.3 * h_)], closed=True, fc=TINTA, ec=TINTA, lw=0.4))
        da.add_artist(mpatches.Polygon([(w_ / 2, h_), (w_, 0), (w_ / 2, 0.3 * h_)], closed=True, fc="white", ec=TINTA, lw=0.4))
        n_ = TextArea("N", textprops=dict(fontsize=fs, fontweight="bold", color=TINTA))
        _caixa(ax, VPacker(children=[n_, da], align="center", pad=0, sep=0.6), canto_seta)

def titulo_mapa(ax, titulo, sub=None, negrito=False):
    """Padrao de titulo: sempre a esquerda; a informacao extra vai numa segunda linha, abaixo, cinza."""
    fs_t, fs_s = tam(5.8), tam(5.2)
    ax.set_title(titulo, fontsize=fs_t, loc="left", fontweight="bold" if negrito else "normal",
                 pad=(fs_s * 1.2 + 2.0) if sub else 2.0)
    if sub:
        ax.annotate(sub, xy=(0, 1), xycoords="axes fraction", xytext=(0, 1.2), textcoords="offset points",
                    ha="left", va="bottom", fontsize=fs_s, color=TINTA2, annotation_clip=False)

def barra_redonda(largura_m, frac=0.25):
    alvo = frac * largura_m
    return max([k for k in [250, 500, 1000, 2000, 5000, 10000, 20000, 25000, 50000, 100000, 200000, 500000]
                if k <= alvo] or [250])

def dot_labels(ax, xs, ys, labels, dx, ha="left", fs=5.8, color=TINTA2):
    for x, y, s in zip(xs, ys, labels):
        ax.text(x + dx, y, s, fontsize=tam(fs), color=color, va="center", ha=ha)

TYPE_EN = {"Inicial grande": "Early, large", "Inicial pequena e fraca": "Early, small & weak",
           "Inicial pequena e intensa": "Early, small & intense", "Restante da seca": "Late season",
           "Fora da seca (nov-abr)": "Outside dry season", "Fragmento < 1 ha": "Fragments < 1 ha"}
TYPE_COL = {"Early, large": "#1d5aa6", "Early, small & weak": AZUL, "Early, small & intense": AZUL_CLARO,
            "Late season": LARANJA, "Outside dry season": "#c9c7bf", "Fragments < 1 ha": "#dcdad2"}
CMAP_NBR = LinearSegmentedColormap.from_list("nbr", ["#4a2a12", "#8c5a2b", "#c9a46b", "#efe3c6", "#a9cf8f", "#4f9a4a", "#1d5e2a"])
CMAP_DNBR = LinearSegmentedColormap.from_list("dnbr", ["#fbf6ea", "#fde0a3", "#f9a35a", "#e0532b", "#a8201a", "#5a0b10"])
SEM_DADO = "#e4e3de"
C_INI, C_TAR, C_PASSOU = "#1f5fc8", "#f28322", "#d1127a"
C_SEMQ, C_MISTO = "#e9e8e3", "#9a7fc6"
CMAP_PASSA = LinearSegmentedColormap.from_list("passa", ["#b2182b", "#ef8a62", "#fbe3c4", "#9ec9e2", "#2f78b4", "#163f73"])
CMAP_AREA2 = LinearSegmentedColormap.from_list("area2", ["#c6dbf7", "#6aa6e8", "#1f5fc8", "#0b2a66"])

# ============================================================
# DADOS
# ============================================================
def _csv(p, **kw):
    return pd.read_csv(p, **kw) if Path(p).exists() else None

STATS = ["B06_tipologia_resumo", "D1_tempo", "D2_foco_noturno_taxa", "D2_foco_noturno", "B09_celula_resumo",
         "B09_celula_coeficientes", "B08a_regra_antiga_x_simetrica", "B09_segmento_resumo",
         "B09_segmento_coeficientes", "B10_robustez_segmentos", "B09_cicatriz_vs_referencia"]
IDX0 = ("B06_tipologia_resumo", "D2_foco_noturno_taxa", "D2_foco_noturno", "B09_celula_coeficientes",
        "B09_segmento_coeficientes")
SEG_COLS = ["ti_id", "year", "tipo_seg", "resultado", "x", "y", "area_cicatriz_ha", "largura_local_m"]

AMOSTRA = load_sample_ids()
seg_file = cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet")
segs = pd.read_parquet(seg_file, columns=SEG_COLS)
art1 = _csv(DIRS["raw"] / "artigo1_tis.csv")
TI_NAME = dict(zip(tis["ti_id"].astype(int), tis["ti_name"]))
ORDER8 = tis[tis["ti_id"].astype(int).isin(AMOSTRA)].sort_values("ti_name")
NUM = {int(t): k for k, t in enumerate(ORDER8["ti_id"], start=1)}
R = {n: _csv(stats_path(n), index_col=0 if n in IDX0 else None) for n in STATS}
faltam = [n for n, v in R.items() if v is None]
if faltam:
    console.print(f"[yellow]WARNING faltam tabelas (paineis correspondentes saem vazios): {faltam}[/yellow]")
REGRA_OK = lambda s: s.astype(str).str.startswith("simetrica")

def annual_area():
    rows = []
    for ti in ORDER8["ti_id"].astype(int):
        p = FIRE_DIR / f"l9_{RUN_TAG}_firemonth_ti{ti}.tif"
        if not p.exists():
            continue
        with rasterio.open(p) as src:
            inside = src.read(1) == 1
            px_ha = abs(src.transform.a * src.transform.e) / 1e4
            for k, y in enumerate(SCAR_YEARS, start=2):
                if y in YEARS:
                    m = src.read(k)
                    rows.append({"ti_id": ti, "year": y, "early_ha": (inside & np.isin(m, EARLY)).sum() * px_ha,
                                 "late_ha": (inside & np.isin(m, LATE)).sum() * px_ha, "ti_ha": inside.sum() * px_ha})
    return pd.DataFrame(rows)

def nbr_profiles(step=3, max_px=4000, seed=7):
    """NBR mensal minimo (Landsat 8/9) por classe de pixel: cicatriz inicial, fogo posterior, nao queimado."""
    rng = np.random.default_rng(seed)
    acc = {}
    for ti in ORDER8["ti_id"].astype(int):
        for y in YEARS:
            try:
                r = read_raster(ti, y)
            except Exception:
                continue
            mon = r["month"][::step, ::step]
            cls = {"early": np.isin(mon, EARLY), "late": np.isin(mon, LATE), "unburned": mon == 0}
            for m in LS_MONTHS:
                v = r[f"nbr{m:02d}"][::step, ::step].astype(np.int16)
                ok = v > -101
                for c, mask in cls.items():
                    vals = v[ok & mask]
                    if len(vals) > max_px:
                        vals = rng.choice(vals, max_px, replace=False)
                    if len(vals):
                        acc.setdefault((c, m), []).append(vals / 100.0)
    rows = []
    for (c, m), lst in acc.items():
        a = np.concatenate(lst)
        rows.append({"classe": c, "mes": m, "p25": np.percentile(a, 25), "p50": np.median(a),
                     "p75": np.percentile(a, 75), "n": len(a)})
    return pd.DataFrame(rows).sort_values(["classe", "mes"])

area_anual = annual_area()
perfil_nbr = nbr_profiles()
perfil_nbr.to_csv(stats_path("F1_perfil_nbr_por_classe"), index=False)

_d_ok = segs[(segs["tipo_seg"] == "cicatriz") & segs["resultado"].isin(["interrupcao", "continuidade"])]
REP = {int(t): int(g.groupby("year").size().idxmax()) for t, g in _d_ok.groupby("ti_id")}

def pick_window():
    d = _d_ok
    if JANELA:
        cand = [tuple(JANELA)]
    else:
        g = d.groupby(["ti_id", "year"]).agg(n=("resultado", "size"),
                                             nc=("resultado", lambda r: (r == "continuidade").sum())).reset_index()
        g = g[g["nc"] >= 3] if (g["nc"] >= 3).any() else g
        cand = [tuple(int(v) for v in r) for r in g.sort_values("n", ascending=False)[["ti_id", "year"]].values[:15]]
    for ti, yr in cand:
        dd = d[(d["ti_id"] == ti) & (d["year"] == yr)]
        H, xe, ye = np.histogram2d(dd["x"], dd["y"], bins=[max(int(np.ptp(dd["x"]) // 3000), 1),
                                                           max(int(np.ptp(dd["y"]) // 3000), 1)])
        geom = tis[tis["ti_id"].astype(int) == ti].geometry.iloc[0].buffer(300)
        for flat in np.argsort(H, axis=None)[::-1][:300]:
            i, j = np.unravel_index(flat, H.shape)
            cx, cy = (xe[i] + xe[i + 1]) / 2, (ye[j] + ye[j + 1]) / 2
            win = (cx - 6000, cy - 6000, cx + 6000, cy + 6000)
            if geom.contains(_box(*win)):
                return ti, yr, win
    ti, yr = cand[0]
    dd = d[(d["ti_id"] == ti) & (d["year"] == yr)]
    cx, cy = dd["x"].median(), dd["y"].median()
    return ti, yr, (cx - 6000, cy - 6000, cx + 6000, cy + 6000)

W = pick_window()
console.print(f"[cyan]Janela 12 x 12 km:[/cyan] {TI_NAME.get(W[0])}, {W[1]}")

def window_nbr():
    r = read_raster(W[0], W[1])
    tr = r["_transform"]; px = abs(tr.a)
    c0, r0 = int(round((W[2][0] - tr.c) / px)), int(round((tr.f - W[2][3]) / px))
    n = int(round((W[2][2] - W[2][0]) / px))
    cut = lambda a: a[max(r0, 0):r0 + n, max(c0, 0):c0 + n]
    pre = cut(r["nbr07"]).astype(np.int16)
    post = np.stack([cut(r[f"nbr{m:02d}"]).astype(np.int16) for m in LATE])
    post = np.where(post > -101, post, 999).min(0); post[post == 999] = -128
    return {"pre": pre, "post": post, "month": cut(r["month"]).astype(np.int16),
            "ext": np.array([W[2][0], W[2][2], W[2][1], W[2][3]], dtype=float)}

JNBR = window_nbr()

def ti_geom(ti):
    return tis[tis["ti_id"].astype(int) == ti]

def context_outlines():
    out = {}
    try:
        g = ee.FeatureCollection(EE_ASSET_BIOME).geometry(ee.ErrorMargin(500)).simplify(2000)
        out["cerrado"] = gpd.GeoDataFrame(geometry=[shape(g.getInfo())], crs="EPSG:4326").to_crs(AREA_CRS)
    except Exception as e:
        console.print(f"[dim]contorno do Cerrado indisponivel: {str(e)[:80]}[/dim]")
    try:
        br = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq("country_na", "Brazil"))
        g = br.geometry(ee.ErrorMargin(1000)).simplify(5000)
        out["brasil"] = gpd.GeoDataFrame(geometry=[shape(g.getInfo())], crs="EPSG:4326").to_crs(AREA_CRS)
    except Exception as e:
        console.print(f"[dim]contorno do Brasil indisponivel: {str(e)[:80]}[/dim]")
    return out

CTX = context_outlines()

def mapa_localizacao(ax, cor_ti=None, cmap=None, norm=None, rotulos=True):
    """Cerrado + 14 TIs; cor_ti = {ti_id: valor} pinta as TIs analisadas."""
    if "brasil" in CTX:
        CTX["brasil"].plot(ax=ax, facecolor=FUNDO, edgecolor="#bdbcb6", lw=0.35, zorder=0)
    if "cerrado" in CTX:
        CTX["cerrado"].plot(ax=ax, facecolor=CERRADO, edgecolor=CERRADO_BORDA, lw=0.45, zorder=1)
    elig = tis["ti_id"].astype(int).isin(AMOSTRA)
    tis[~elig].plot(ax=ax, facecolor="white", edgecolor=TINTA2, lw=0.5, zorder=3)
    if cor_ti is None:
        tis[elig].plot(ax=ax, facecolor=LARANJA, edgecolor=TINTA, lw=0.35, zorder=4)
    for _, r in ORDER8.iterrows():
        p = r.geometry.representative_point(); ti = int(r["ti_id"])
        if cor_ti is not None and ti in cor_ti:
            ax.scatter(p.x, p.y, s=46, color=cmap(norm(cor_ti[ti])), edgecolor=TINTA, lw=0.6, zorder=5)
        if rotulos:
            ax.annotate(str(NUM[ti]), (p.x, p.y), xytext=(4.5, 2.5), textcoords="offset points",
                        fontsize=tam(6.2), fontweight="bold", color=TINTA, path_effects=HALO, zorder=6)
    b = CTX["cerrado"].total_bounds if "cerrado" in CTX else tis.total_bounds
    cx, cy, h = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2, max(b[2] - b[0], b[3] - b[1]) / 2 * 1.05
    return (cx - h, cy - h, cx + h, cy + h)

def colorbar_fina(fig, mappable, ax, label, ticks=None, extend="neither", orientation="horizontal", **kw):
    cb = fig.colorbar(mappable, ax=ax, orientation=orientation, fraction=0.05, pad=0.02, aspect=28,
                      shrink=kw.get("shrink", 0.9), extend=extend)
    cb.set_label(label, fontsize=tam(6.2))
    if ticks is not None:
        cb.set_ticks(ticks)
    cb.outline.set_linewidth(0.4)
    cb.ax.tick_params(labelsize=tam(5.8), width=0.5, length=2)
    return cb

def mapa_ti(ti):
    """Ano REP[ti]: sequencia (bloco 10), dNBR (Landsat) e area de cada cicatriz inicial (~700 px)."""
    from scipy import ndimage
    yr = REP.get(ti)
    p = SEQ_DIR / f"seq_ti{ti}_{yr}.tif"
    if yr is None or not p.exists():
        return None
    with rasterio.open(p) as src:
        seq = src.read(1); tr = src.transform; b = src.bounds
    early = np.isin(seq, [1, 3])
    lab, n = ndimage.label(early, structure=np.ones((3, 3), int))
    area = np.zeros(seq.shape, np.float32)
    if n:
        sizes = np.bincount(lab.ravel())[1:] * abs(tr.a * tr.e) / 1e4
        area[early] = sizes[lab[early] - 1]
    out = {"seq": seq, "area": area}
    try:
        r = read_raster(ti, yr)
        pre = r["nbr07"].astype(np.int16)
        post = np.stack([r[f"nbr{m:02d}"].astype(np.int16) for m in LATE])
        post = np.where(post > -101, post, 999).min(0)
        dn = np.where((pre > -101) & (post < 999), (pre - post) / 100.0, np.nan).astype(np.float32)
        if dn.shape == seq.shape:
            out["dnbr"] = dn
    except Exception as e:
        console.print(f"[dim]dNBR indisponivel para TI {ti}: {str(e)[:60]}[/dim]")
    f = int(np.ceil(max(seq.shape) / 700))
    out = {k: v[::f, ::f] for k, v in out.items()}
    out["ext"] = np.array([b.left, b.right, b.bottom, b.top])
    return out

MAPAS = {int(t): mapa_ti(int(t)) for t in ORDER8["ti_id"]}

from matplotlib.path import Path as _MPath
from matplotlib.patches import PathPatch as _PathPatch

def _geom_path(geom):
    verts, codes = [], []
    polys = list(geom.geoms) if hasattr(geom, "geoms") else [geom]
    for p in polys:
        for ring in [p.exterior] + list(p.interiors):
            xy = np.asarray(ring.coords)
            verts += xy.tolist(); codes += [_MPath.MOVETO] + [_MPath.LINETO] * (len(xy) - 2) + [_MPath.CLOSEPOLY]
    return _MPath(verts, codes)

def recortar_na_ti(ax, ti):
    pp = _PathPatch(_geom_path(ti_geom(ti).geometry.iloc[0]), transform=ax.transData, facecolor="none", edgecolor="none")
    for a in list(ax.images) + list(ax.collections):
        a.set_clip_path(pp)

def grade_tis(sf, desenhar, info=None, ncols=4):
    """As TIs analisadas, cada uma num painel quadrado na sua escala, recortado no limite da TI."""
    nrows = int(np.ceil(len(ORDER8) / ncols))
    axs = np.atleast_2d(sf.subplots(nrows, ncols))
    for ax, (_, r) in zip(axs.ravel(), ORDER8.iterrows()):
        ti = int(r["ti_id"])
        ti_geom(ti).plot(ax=ax, facecolor=C_SEMQ, edgecolor="none", zorder=0)
        desenhar(ax, ti)
        recortar_na_ti(ax, ti)
        ti_geom(ti).boundary.plot(ax=ax, color=TINTA, lw=0.45, zorder=6)
        bb = r.geometry.bounds
        H = max(bb[2] - bb[0], bb[3] - bb[1]) * 1.04
        S = H / 0.80                                   # 20% de baixo ficam livres para escala e norte
        cx, cy = (bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2
        e = (cx - S / 2, cy - 0.6 * S, cx + S / 2, cy + 0.4 * S)
        moldura(ax, e, barra_m=barra_redonda(S), seta=True)
        titulo_mapa(ax, f"{NUM[ti]} {r['ti_name']}", info(ti) if info else None, negrito=True)
    for ax in axs.ravel()[len(ORDER8):]:
        ax.set_axis_off()
    return axs

def _img(ax, arr, ext, **kw):
    return ax.imshow(arr, extent=(ext[0], ext[1], ext[2], ext[3]), interpolation="nearest", zorder=2,
                     rasterized=True, **kw)

def season_class(ti):
    p = FIRE_DIR / f"l9_{RUN_TAG}_firemonth_ti{ti}.tif"
    if not p.exists():
        return None, None
    with rasterio.open(p) as src:
        inside = src.read(1) == 1
        ne = np.zeros(inside.shape, np.uint8); nl = np.zeros(inside.shape, np.uint8)
        for k, y in enumerate(SCAR_YEARS, start=2):
            if y in YEARS:
                m = src.read(k)
                ne += np.isin(m, EARLY); nl += np.isin(m, LATE)
        b = src.bounds
    cl = np.zeros(inside.shape, np.uint8)
    cl[inside] = 1; cl[inside & (ne > nl)] = 2; cl[inside & (ne == nl) & (ne > 0)] = 3; cl[inside & (nl > ne)] = 4
    f = int(np.ceil(max(cl.shape) / 650))
    return np.ma.masked_equal(cl[::f, ::f], 0), (b.left, b.right, b.bottom, b.top)

def _regra_sim(rg):
    return rg[(rg["combustivel"] == FUEL_MAIN) & REGRA_OK(rg["regra"])].set_index("tipo_seg")

# ============================================================
# FIGURA 1 - PREMISSA
# ============================================================
def fig1():
    fig = plt.figure(figsize=(LARG, 13.6 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.45, 1], hspace=0.04)
    cmap = ListedColormap([C_SEMQ, C_INI, C_MISTO, C_TAR]); norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], 4)

    def desenhar(ax, ti):
        cl, e = season_class(ti)
        if cl is not None:
            ax.imshow(cl, cmap=cmap, norm=norm, extent=e, interpolation="nearest", zorder=2, rasterized=True)
    axs = grade_tis(sf_m, desenhar)
    sf_m.legend(handles=[mpatches.Patch(color=C_INI, label="Burned mostly in May–Jul"),
                         mpatches.Patch(color=C_TAR, label="Burned mostly in Aug–Oct"),
                         mpatches.Patch(color=C_MISTO, label="Equal"), mpatches.Patch(color=C_SEMQ, label="Not burned")],
                loc="outside lower center", ncol=4, fontsize=tam(5.8), handlelength=1.0, columnspacing=1.0)
    rotulo_painel(axs[0, 0], "a")

    axb, axc = sf_g.subplots(1, 2)
    a = area_anual.groupby("year")[["early_ha", "late_ha"]].sum() / 1000
    axb.fill_between(a.index, 0, a["early_ha"], color=C_INI, alpha=0.12, lw=0)
    axb.plot(a.index, a["late_ha"], lw=1.6, color=C_TAR, marker="s", ms=3.4, mec="white", mew=0.5, zorder=3)
    axb.plot(a.index, a["early_ha"], lw=1.8, color=C_INI, marker="o", ms=3.8, mec="white", mew=0.6, zorder=4)
    axb.text(a.index[-1] + 0.3, a["late_ha"].iloc[-1], "Aug–Oct", color=C_TAR, fontsize=tam(5.8), va="center")
    axb.text(a.index[-1] + 0.3, a["early_ha"].iloc[-1], "May–Jul", color=C_INI, fontsize=tam(5.8), va="center")
    axb.set_xlim(YEARS[0] - 0.5, YEARS[-1] + 2.4); axb.spines["bottom"].set_bounds(YEARS[0], YEARS[-1])
    axb.set_xticks(range(YEARS[0], YEARS[-1] + 1, 2)); axb.set_xticks(YEARS, minor=True)
    axb.grid(axis="x", visible=False)
    axb.set_ylim(0, a.values.max() * 1.25); axb.yaxis.set_major_locator(MaxNLocator(4))
    axb.set_ylabel("Burned area (kha)")
    sh = 100 * a["early_ha"].sum() / (a["early_ha"].sum() + a["late_ha"].sum())
    axb.set_title(f"early burning = {sh:.0f}% of dry-season area", fontsize=tam(5.8), color=MUDO, loc="left", pad=2)
    rotulo_painel(axb, "b")

    _fim = []
    axc.axvspan(4.5, 7.5, color=C_INI, alpha=0.08, lw=0, zorder=0)
    axc.axvspan(7.5, 10.5, color=C_TAR, alpha=0.10, lw=0, zorder=0)
    for c, cor, lab in [("unburned", "#7d7c78", "Unburned"), ("late", C_TAR, "Burned\nAug–Oct"),
                        ("early", C_INI, "Burned\nMay–Jul")]:
        d = perfil_nbr[perfil_nbr["classe"] == c]
        if not len(d):
            continue
        axc.fill_between(d["mes"], d["p25"], d["p75"], color=cor, alpha=0.14, lw=0, zorder=1)
        axc.plot(d["mes"], d["p50"], color=cor, lw=1.8, marker="o", ms=3.2, mec="white", mew=0.5, zorder=3)
        _fim.append([d.iloc[-1]["p50"], lab, cor, d.iloc[-1]["mes"]])
    if _fim:
        _fim.sort(key=lambda z: z[0])
        ys = [z[0] for z in _fim]
        gap = 0.075 * (axc.get_ylim()[1] - axc.get_ylim()[0])
        for i in range(1, len(ys)):
            ys[i] = max(ys[i], ys[i - 1] + gap)
        for (y0, lab, cor, mx), yl in zip(_fim, ys):
            axc.annotate(lab.replace("\n", " "), (mx, y0), xytext=(mx + 0.25, yl), color=cor, fontsize=tam(5.6),
                         va="center", ha="left", arrowprops=dict(arrowstyle="-", color=cor, lw=0.4, shrinkA=0, shrinkB=2))
    axc.set_xticks(LS_MONTHS); axc.set_xticklabels(["A", "M", "J", "J", "A", "S", "O", "N"][:len(LS_MONTHS)])
    axc.set_xlim(LS_MONTHS[0] - 0.4, LS_MONTHS[-1] + 3.2); axc.spines["bottom"].set_bounds(LS_MONTHS[0], LS_MONTHS[-1])
    axc.grid(axis="x", visible=False); axc.yaxis.set_major_locator(MaxNLocator(4))
    axc.set_ylabel("Landsat NBR")
    axc.set_title("NBR drops in the month each pixel burns", fontsize=tam(5.8), color=MUDO, loc="left", pad=2)
    rotulo_painel(axc, "c")
    salvar(fig, "Fig1_premise")

# ============================================================
# FIGURA 2 - O TESTE NAS BORDAS
# ============================================================
def fig2():
    fig = plt.figure(figsize=(LARG, 14.2 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.5, 1], hspace=0.04)
    norm_d = mpl.colors.Normalize(0, 0.6)

    def info(ti):
        dd = _d_ok[(_d_ok["ti_id"] == ti) & (_d_ok["year"] == REP.get(ti))]
        return f"{REP.get(ti)} · {100 * (dd['resultado'] == 'interrupcao').mean():.0f}% stopped" if len(dd) else ""

    def desenhar(ax, ti):
        m = MAPAS.get(ti)
        if m is None:
            return
        e = m["ext"]
        if "dnbr" in m:
            _img(ax, np.ma.masked_invalid(np.clip(m["dnbr"], 0, 0.8)), e, cmap=CMAP_DNBR, norm=norm_d)
        early = np.isin(m["seq"], [1, 3]).astype(float)
        if early.any():
            ax.contour(np.flipud(early), levels=[0.5], colors=[C_INI], linewidths=0.22, alpha=0.9, extent=(e[0], e[1], e[2], e[3]), zorder=4)
        dd = _d_ok[(_d_ok["ti_id"] == ti) & (_d_ok["year"] == REP.get(ti)) & (_d_ok["resultado"] == "continuidade")]
        ax.scatter(dd["x"], dd["y"], s=2.2, color=C_PASSOU, edgecolor="none", alpha=0.85, zorder=5)
    axs = grade_tis(sf_m, desenhar, info=info)
    cb = sf_m.colorbar(mpl.cm.ScalarMappable(norm=norm_d, cmap=CMAP_DNBR), ax=axs, location="bottom",
                       shrink=0.32, aspect=30, pad=0.01, extend="max")
    cb.set_label("Landsat dNBR, July − min(Aug–Oct): later burning", fontsize=tam(5.8))
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=tam(5.4), width=0.4, length=1.5)
    sf_m.legend(handles=[Line2D([], [], color=C_INI, lw=1.0, label="Early-scar outline (May–Jul)"),
                         Line2D([], [], marker="o", ls="", ms=3.2, mfc=C_PASSOU, mec="white", mew=0.4,
                                label="Scar edge crossed by later fire")],
                loc="outside lower center", ncol=2, fontsize=tam(5.8), handlelength=1.2, columnspacing=1.4)
    rotulo_painel(axs[0, 0], "a")

    axb, axc = sf_g.subplots(1, 2, width_ratios=[1, 1.15])
    rg, rv = R["B08a_regra_antiga_x_simetrica"], R["B09_cicatriz_vs_referencia"]
    if rg is not None:
        g = _regra_sim(rg)
        for xi, (t, cor) in enumerate([("cicatriz", C_INI), ("referencia", "#8f8e88")]):
            v = g.loc[t, "% interrupcao"]; n = int(g.loc[t, "interrupcao"] + g.loc[t, "continuidade"])
            axb.bar(xi, v, width=0.62, color=cor, zorder=2)
            axb.text(xi, v + 2, f"{v:.1f}%", ha="center", va="bottom", fontsize=tam(6.4), fontweight="bold", color=cor)
            axb.text(xi, -4, f"n = {n:,}", ha="center", va="top", fontsize=tam(5.2), color=MUDO)
        axb.set_xticks([0, 1]); axb.set_xticklabels(["Early-scar\nedges", "Reference\nedges"], fontsize=tam(6.0))
        axb.tick_params(axis="x", length=0, pad=12)
        axb.set_ylim(0, 112); axb.set_yticks([0, 50, 100]); axb.set_ylabel("Arriving fires stopped (%)")
        axb.grid(axis="x", visible=False)
    if rv is not None:
        r_ = rv[(rv["combustivel"] == FUEL_MAIN) & REGRA_OK(rv["regra"])]
        if len(r_):
            r_ = r_.iloc[0]
            axb.set_title(f"adjusted OR {r_['razao_chances_cicatriz']:.0f} ({r_['ic95_inf']:.0f}–{r_['ic95_sup']:.0f})",
                          fontsize=tam(5.8), color=MUDO, loc="left", pad=2)
    rotulo_painel(axb, "b")
    per = (segs[segs["resultado"].isin(["interrupcao", "continuidade"])]
           .assign(i=lambda x: x["resultado"] == "interrupcao").groupby(["ti_id", "tipo_seg"])["i"].mean().unstack())
    tl = ORDER8["ti_id"].astype(int).tolist()[::-1]
    for yi, t in enumerate(tl):
        if t not in per.index:
            continue
        s, r_ = 100 * per.loc[t].get("cicatriz", np.nan), 100 * per.loc[t].get("referencia", np.nan)
        axc.plot([r_, s], [yi, yi], color="#cfcec7", lw=2.2, zorder=1, solid_capstyle="round")
        axc.scatter(r_, yi, s=24, color="#8f8e88", edgecolor="white", lw=0.5, zorder=3)
        axc.scatter(s, yi, s=28, color=C_INI, edgecolor="white", lw=0.5, zorder=3)
    axc.set_yticks(range(len(tl))); axc.set_yticklabels([f"{NUM[t]} {TI_NAME[t]}" for t in tl], fontsize=tam(5.6))
    axc.set_xlim(-3, 103); axc.set_xticks([0, 50, 100]); axc.set_xlabel("Arriving fires stopped (%)", fontsize=tam(6.0))
    axc.grid(axis="y", visible=False); axc.tick_params(axis="y", length=0)
    axc.set_title("each IL: reference (grey) vs early-scar edges (blue)", fontsize=tam(5.6), color=MUDO, loc="left", pad=2)
    rotulo_painel(axc, "c")
    salvar(fig, "Fig2_edge_test")

# ============================================================
# FIGURA 3 - CONFIGURACAO ALEM DA QUANTIDADE
# ============================================================
BINS_AREA, LAB_AREA = [1, 10, 100, 1000, np.inf], ["1–10", "10–\n100", "100–\n1000", "> 1000"]
BINS_LARG, LAB_LARG = [0, 150, 300, 600, np.inf], ["< 150", "150–\n300", "300–\n600", "> 600"]

def taxa_por_faixa(col, bins, labs):
    d = _d_ok.dropna(subset=[col]).copy()
    d["faixa"] = pd.cut(d[col], bins, labels=labs, right=False)
    g = d.groupby("faixa", observed=False)["resultado"].agg(n="size", p=lambda r: (r == "interrupcao").mean())
    g["ep"] = np.sqrt(g["p"] * (1 - g["p"]) / g["n"].clip(lower=1))
    return g

def fig3():
    fig = plt.figure(figsize=(LARG, 14.0 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.45, 1], hspace=0.04)
    norm_a = mpl.colors.LogNorm(1, 1000)

    def desenhar(ax, ti):
        m = MAPAS.get(ti)
        if m is None:
            return
        e = m["ext"]
        _img(ax, np.ma.masked_where(m["seq"] != 2, np.ones(m["seq"].shape)), e, cmap=ListedColormap(["#f9c9a2"]), vmin=0, vmax=1)
        _img(ax, np.ma.masked_less_equal(m["area"], 0), e, cmap=CMAP_AREA2, norm=norm_a)
        dd = _d_ok[(_d_ok["ti_id"] == ti) & (_d_ok["year"] == REP.get(ti)) & (_d_ok["resultado"] == "continuidade")]
        ax.scatter(dd["x"], dd["y"], s=2.2, color=C_PASSOU, edgecolor="none", alpha=0.85, zorder=5)
    axs = grade_tis(sf_m, desenhar, info=lambda ti: f"{REP.get(ti)}")
    cb = sf_m.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=CMAP_AREA2), ax=axs, location="bottom",
                       shrink=0.36, aspect=30, pad=0.03)
    cb.set_ticks([1, 10, 100, 1000]); cb.set_ticklabels(["1", "10", "100", "1000"])
    cb.set_label("Size of each early scar (ha)", fontsize=tam(5.8))
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=tam(5.4), width=0.4, length=1.5)
    sf_m.legend(handles=[mpatches.Patch(color="#f9c9a2", label="Burned Aug–Oct"),
                         Line2D([], [], marker="o", ls="", ms=3.4, mfc=C_PASSOU, mec="white", mew=0.4,
                                label="Scar edge crossed by later fire")],
                loc="outside lower center", ncol=2, fontsize=tam(5.8), handlelength=1.2, columnspacing=1.4)
    rotulo_painel(axs[0, 0], "a")

    axb, axc, axd = sf_g.subplots(1, 3, width_ratios=[1, 1, 1.15])
    for ax, (col, bins, labs, xl, letra) in zip([axb, axc], [("area_cicatriz_ha", BINS_AREA, LAB_AREA, "Early-scar size (ha)", "b"),
                                                             ("largura_local_m", BINS_LARG, LAB_LARG, "Local scar width (m)", "c")]):
        g = taxa_por_faixa(col, bins, labs)
        x = np.arange(len(g)); cores = [CMAP_AREA2(v) for v in np.linspace(0.3, 1.0, len(g))]
        ax.bar(x, 100 * g["p"], width=0.66, color=cores, zorder=2)
        ax.errorbar(x, 100 * g["p"], yerr=196 * g["ep"], fmt="none", ecolor=TINTA, elinewidth=0.7, capsize=1.5)
        for xi, (_, r) in zip(x, g.iterrows()):
            if r["n"]:
                ax.text(xi, 100 * r["p"] + 196 * r["ep"] + 0.25, f"{100 * r['p']:.1f}", ha="center", va="bottom",
                        fontsize=tam(5.6), color=TINTA)
        lo = max(0, np.floor(100 * (g["p"] - 2.2 * g["ep"]).min() / 2) * 2 - 2)
        ax.set_ylim(lo, 100.8)
        ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=tam(5.4), linespacing=0.95)
        ax.tick_params(axis="x", length=0); ax.grid(axis="x", visible=False)
        ax.yaxis.set_major_locator(MaxNLocator(4)); ax.set_xlabel(xl, fontsize=tam(6.0))
        rotulo_painel(ax, letra)
    axb.set_ylabel("Arriving fires stopped (%)", fontsize=tam(6.0))
    sc = R["B09_segmento_coeficientes"]
    lab = {"log_area_cicatriz": "Scar size", "largura_local_100m": "Scar width",
           "conf_split_win": "Fragmentation", "conf_lpi_win": "Largest patch"}
    if sc is not None:
        k = sc[(sc["combustivel"] == FUEL_MAIN) & (sc["modelo"] == "segmento | config")]
        vars_ = [v for v in lab if v in k.index]; y = np.arange(len(vars_))[::-1].astype(float)
        axd.axvline(1.0, color=TINTA2, lw=0.7, ls=(0, (3, 2)), zorder=1)
        his = []
        for yi, v in zip(y, vars_):
            e = k.loc[v]; sig = e["ic95_inf"] > 1 or e["ic95_sup"] < 1
            cor = C_INI if sig else "#9a9992"
            axd.plot([e["ic95_inf"], e["ic95_sup"]], [yi, yi], color=cor, lw=2.0, solid_capstyle="round", zorder=2)
            axd.scatter(e["razao_chances_1dp"], yi, s=34, color=cor if sig else "white", edgecolor="white" if sig else cor,
                        lw=0.6 if sig else 1.0, zorder=3)
            axd.text(e["ic95_sup"] * 1.05, yi, f"{e['razao_chances_1dp']:.2f}", fontsize=tam(5.8), color=cor,
                     va="center", fontweight="bold" if sig else "normal")
            his.append(e["ic95_sup"])
        axd.set_xscale("log"); axd.set_xlim(0.65, max(his) * 1.55)
        axd.xaxis.set_major_locator(FixedLocator([0.75, 1, 1.5, 2, 3])); axd.xaxis.set_minor_locator(NullLocator())
        axd.xaxis.set_major_formatter(FormatStrFormatter("%g"))
        axd.set_yticks(y); axd.set_yticklabels([lab[v] for v in vars_], fontsize=tam(5.8))
        axd.set_ylim(-0.6, len(vars_) - 0.4)
        axd.grid(axis="y", visible=False); axd.tick_params(axis="y", length=0)
        axd.set_xlabel("Odds of stopping per SD,\nbeyond quantity", fontsize=tam(6.0))
    rotulo_painel(axd, "d")
    salvar(fig, "Fig3_configuration")

# ============================================================
# FIGURA 4 - GENERALIZACAO
# ============================================================
def fig4():
    fig = plt.figure(figsize=(LARG, 13.8 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.45, 1], hspace=0.04)
    d = _d_ok.copy()
    d["x"] = np.floor(d["x"] / CELL_M) * CELL_M + CELL_M / 2
    d["y"] = np.floor(d["y"] / CELL_M) * CELL_M + CELL_M / 2
    gcell = (d.groupby(["ti_id", "x", "y"]).agg(n=("resultado", "size"),
             rate=("resultado", lambda r: 100 * (r == "interrupcao").mean())).reset_index())
    gcell = gcell[gcell["n"] >= 10]
    lo = float(np.floor(gcell["rate"].quantile(0.02) / 5) * 5) if len(gcell) else 80
    norm = mpl.colors.Normalize(lo, 100)

    def desenhar(ax, ti):
        g = gcell[gcell["ti_id"] == ti]
        if not len(g):
            return
        h = CELL_M / 2
        gg = gpd.GeoDataFrame(g[["rate"]], geometry=[_box(x - h, y - h, x + h, y + h) for x, y in zip(g["x"], g["y"])],
                              crs=AREA_CRS)
        gg.plot(ax=ax, column="rate", cmap=CMAP_PASSA, norm=norm, edgecolor="white", linewidth=0.15, zorder=2)
    axs = grade_tis(sf_m, desenhar)
    cb = sf_m.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=CMAP_PASSA), ax=axs, location="bottom",
                       shrink=0.42, aspect=34, pad=0.03, extend="min" if gcell["rate"].min() < lo else "neither")
    cb.set_label(f"Arriving fires stopped at early-scar edges (%), {CELL_M / 1000:.0f}-km cells, {YEARS[0]}–{YEARS[-1]}",
                 fontsize=tam(5.8))
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=tam(5.4), width=0.4, length=1.5)
    rotulo_painel(axs[0, 0], "a")

    axb, axc = sf_g.subplots(1, 2, width_ratios=[1, 1.25])
    tl = ORDER8["ti_id"].astype(int).tolist()
    for xi, t in enumerate(tl):
        v = gcell.loc[gcell["ti_id"] == t, "rate"].to_numpy()
        if not len(v):
            continue
        jit = (np.random.default_rng(t).random(len(v)) - 0.5) * 0.5
        axb.scatter(xi + jit, v, s=5, c=CMAP_PASSA(norm(v)), lw=0, alpha=0.85, zorder=2, rasterized=True)
        axb.plot([xi - 0.3, xi + 0.3], [np.median(v)] * 2, color=TINTA, lw=1.2, zorder=3)
    axb.set_xticks(range(len(tl))); axb.set_xticklabels([str(NUM[t]) for t in tl], fontsize=tam(5.8))
    axb.tick_params(axis="x", length=0); axb.grid(axis="x", visible=False)
    axb.set_ylim(max(lo - 3, 0), 101); axb.yaxis.set_major_locator(MaxNLocator(4))
    axb.set_xlabel("IL (numbers as in a)", fontsize=tam(6.0)); axb.set_ylabel("Stopped (%), per cell", fontsize=tam(6.0))
    rotulo_painel(axb, "b")
    rob = R["B10_robustez_segmentos"]
    grupos = [("Seasonal split", ["artigo1", "local", "local_100"], ["Dias et al. (2026)", "Local, 60 mm", "Local, 100 mm"]),
              ("Encounter criteria", ["segmento 150 m", "segmento 600 m", "faixa 60 m", "faixa 150 m",
                                      "chegada >= 20%", "chegada >= 50%"],
               ["Segment 150 m", "Segment 600 m", "Strip 60 m", "Strip 150 m", "Arrival ≥ 20%", "Arrival ≥ 50%"]),
              ("Fuel", ["nativa+pastagem", "nativa"], ["Natural + pasture", "Natural only"])]
    if rob is not None and len(rob):
        rb = rob.drop_duplicates("variante").set_index("variante")
        y = 0.0; ticks, labs = [], []
        axc.axvline(1.0, color=TINTA2, lw=0.7, ls=(0, (3, 2)), zorder=1)
        if "padrao" in rb.index:
            axc.axvspan(rb.loc["padrao", "ic_inf"], rb.loc["padrao", "ic_sup"], color=C_INI, alpha=0.10, lw=0, zorder=0)
        for gname, keys, names in grupos[::-1]:
            ks = [(k, n) for k, n in zip(keys, names) if k in rb.index]
            for k, n in ks[::-1]:
                r = rb.loc[k]; sig = r["ic_inf"] > 1 or r["ic_sup"] < 1; cor = C_INI if sig else "#9a9992"
                axc.plot([r["ic_inf"], r["ic_sup"]], [y, y], color=cor, lw=1.6, solid_capstyle="round", zorder=2)
                axc.scatter(r["rc_conf_split"], y, s=20, color=cor, edgecolor="white", lw=0.5, zorder=3)
                ticks.append(y); labs.append(n); y += 1
            ticks.append(y - 0.15); labs.append(gname.upper()); y += 1.1
        axc.set_yticks(ticks); axc.set_yticklabels(labs, fontsize=tam(5.4))
        for tl_ in axc.get_yticklabels():
            if tl_.get_text().isupper():
                tl_.set_fontsize(tam(4.8)); tl_.set_color(MUDO); tl_.set_fontweight("bold")
        axc.set_ylim(-0.7, y - 0.4)
        axc.grid(axis="y", visible=False); axc.tick_params(axis="y", length=0)
        axc.set_xlabel("Odds ratio, window fragmentation (band = default)", fontsize=tam(6.0))
    rotulo_painel(axc, "c")
    salvar(fig, "Fig4_generalisation")

# ============================================================
# FIGURA S2 - TIPOLOGIA DESCRITIVA
# ============================================================
def figS2():
    fig = plt.figure(figsize=(LARG, 6.4 * CM), layout="constrained")
    gs = fig.add_gridspec(1, 3)
    rows = ["Early, large", "Early, small & weak", "Early, small & intense", "Late season"]
    ypos = np.array([3.4, 2.4, 1.4, 0.0])
    axes = [fig.add_subplot(gs[0])]
    axes += [fig.add_subplot(gs[k], sharey=axes[0]) for k in (1, 2)]
    tp, d1, d2t = R["B06_tipologia_resumo"], R["D1_tempo"], R["D2_foco_noturno_taxa"]
    if tp is not None:
        t = tp.copy(); t.index = [TYPE_EN.get(i, i) for i in t.index]
        v = [t.loc[k, "area_ha"] / 1e6 for k in rows]
        axes[0].scatter(v, ypos, s=26, color=[TYPE_COL[k] for k in rows], edgecolor="white", lw=0.6, zorder=3)
        dot_labels(axes[0], v, ypos + 0.42, [f"{x:.2f}" if x >= 0.01 else f"{x:.3f}" for x in v], 0, ha="center", fs=5.4)
        axes[0].set_xscale("log"); axes[0].set_xlim(5e-4, 40)
        axes[0].xaxis.set_major_locator(FixedLocator([0.001, 0.1, 10])); axes[0].xaxis.set_minor_locator(NullLocator())
        axes[0].xaxis.set_major_formatter(FormatStrFormatter("%g"))
    axes[0].set_xlabel("Burned area (Mha)", fontsize=tam(6.2))
    if d1 is not None:
        v = d1[d1["variavel"] == "P_vpd_max"].copy(); v["lab"] = v["tipo"].map(TYPE_EN); v = v.set_index("lab")
        rr = [k for k in rows if k in v.index]
        m = [v.loc[k, "media_percentil"] for k in rr]; yy = [ypos[rows.index(k)] for k in rr]
        axes[1].axvline(0.5, color=MUDO, lw=0.6, ls=(0, (3, 2)), zorder=1)
        for k, yi in zip(rr, yy):
            axes[1].plot([v.loc[k, "ic95_inf"], v.loc[k, "ic95_sup"]], [yi, yi], color=TYPE_COL[k], lw=1.3, alpha=0.85,
                         solid_capstyle="round", zorder=2)
        axes[1].scatter(m, yy, s=26, color=[TYPE_COL[k] for k in rr], edgecolor="white", lw=0.6, zorder=3)
        dot_labels(axes[1], m, np.array(yy) + 0.42, [f"{x:.2f}" for x in m], 0, ha="center", fs=5.4)
        axes[1].set_xlim(0.40, 0.80)
    axes[1].set_xlabel("VPD percentile, 1st MODIS detection", fontsize=tam(6.2))
    if d2t is not None:
        t = d2t.copy(); t.index = [TYPE_EN.get(i, i) for i in t.index]
        rr = [k for k in rows if k in t.index]
        v = [100 * t.loc[k, "mean"] for k in rr]; yy = [ypos[rows.index(k)] for k in rr]
        axes[2].scatter(v, yy, s=26, color=[TYPE_COL[k] for k in rr], edgecolor="white", lw=0.6, zorder=3)
        dot_labels(axes[2], v, np.array(yy) + 0.42, [f"{x:.0f}%" for x in v], 0, ha="center", fs=5.4)
        axes[2].set_xlim(-3, max(v) * 1.3 if v else 10); axes[2].xaxis.set_major_locator(MaxNLocator(3, integer=True))
    axes[2].set_xlabel("Night-time MODIS detections (%)", fontsize=tam(6.2))
    for k_, ax in enumerate(axes):
        ax.axhline(0.7, color=EIXO, lw=0.5, ls=(0, (2, 2)))
        ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0, labelleft=(k_ == 0))
        rotulo_painel(ax, "abc"[k_])
    axes[0].set_yticks(ypos)
    axes[0].set_yticklabels(["Early, large", "Early, small\n& weak", "Early, small\n& intense", "Late season"],
                            fontsize=tam(6.0), linespacing=1.0)
    axes[0].set_ylim(-0.5, 4.0)
    salvar(fig, "FigS2_scar_types")

# ============================================================
# FIGURA S3 - JANELA DE 12 km A 30 m
# ============================================================
def figS3():
    fig = plt.figure(figsize=(LARG, 9.4 * CM), layout="constrained")
    sf_m, sf_d = fig.subfigures(2, 1, height_ratios=[1.62, 0.62], hspace=0.0)
    ti, yr, win = W
    e = JNBR["ext"]; ext_img = (e[0], e[1], e[2], e[3])
    early = np.isin(JNBR["month"], EARLY)
    dd = segs[(segs["ti_id"] == ti) & (segs["year"] == yr) & (segs["tipo_seg"] == "cicatriz") &
              segs["x"].between(win[0], win[2]) & segs["y"].between(win[1], win[3]) &
              segs["resultado"].isin(["interrupcao", "continuidade"])]
    norm_nbr = mpl.colors.TwoSlopeNorm(vmin=-0.3, vcenter=0.15, vmax=0.6)
    norm_d = mpl.colors.Normalize(0, 0.6)
    axs = list(sf_m.subplots(1, 3))
    titulos = [f"NBR, July {yr}", f"NBR, min. Aug–Oct {yr}", "dNBR, July − (Aug–Oct)"]
    pre = np.ma.masked_less(JNBR["pre"], -100) / 100.0
    post = np.ma.masked_less(JNBR["post"], -100) / 100.0
    for k, ax in enumerate(axs):
        ax.add_patch(mpatches.Rectangle(win[:2], win[2] - win[0], win[3] - win[1], color=SEM_DADO, zorder=0))
        if k == 0:
            im_n = ax.imshow(pre, extent=ext_img, cmap=CMAP_NBR, norm=norm_nbr, interpolation="nearest", zorder=1, rasterized=True)
        elif k == 1:
            ax.imshow(post, extent=ext_img, cmap=CMAP_NBR, norm=norm_nbr, interpolation="nearest", zorder=1, rasterized=True)
        else:
            im_d = ax.imshow(np.ma.clip(pre - post, -0.1, 0.8), extent=ext_img, cmap=CMAP_DNBR, norm=norm_d,
                             interpolation="nearest", zorder=1, rasterized=True)
        if early.any():
            ax.contour(np.flipud(early.astype(float)), levels=[0.5], colors=[AZUL], linewidths=0.45, extent=ext_img, zorder=3)
        if k == 2 and len(dd):
            q = dd[dd["resultado"] == "continuidade"]
            ax.scatter(q["x"], q["y"], s=11, color="#d63aa8", edgecolor="white", lw=0.5, zorder=5)
        ti_geom(ti).boundary.plot(ax=ax, color=TINTA, lw=0.5, zorder=4)
        moldura(ax, win, barra_m=2000, seta=True)
        titulo_mapa(ax, titulos[k])
        rotulo_painel(ax, "abc"[k])
    colorbar_fina(sf_m, im_n, axs[:2], "Landsat 8/9 NBR (grey: > 500 m from any burn)", ticks=[-0.3, 0, 0.15, 0.3, 0.6], shrink=0.7)
    colorbar_fina(sf_m, im_d, axs[2], "dNBR", ticks=[0, 0.2, 0.4, 0.6], extend="max", shrink=0.9)
    if len(dd):
        pi = 100 * (dd["resultado"] == "interrupcao").mean()
        axs[2].text(0.02, 0.98, f"{len(dd):,} edges reached\n{pi:.0f}% interrupted", transform=axs[2].transAxes,
                    ha="left", va="top", fontsize=tam(5.6), color=TINTA, linespacing=1.15, path_effects=HALO, zorder=9)
    sf_m.legend(handles=[Line2D([], [], color=AZUL, lw=0.9, label="Early-scar outline (May–Jul)"),
                         Line2D([], [], marker="o", ls="", ms=3.4, mfc="#d63aa8", mec="white", mew=0.5,
                                label="Scar edge crossed by later fire")],
                loc="outside lower center", ncol=2, fontsize=tam(5.8), handlelength=1.2, columnspacing=1.2)
    ax = sf_d.subplots()
    rg = R["B08a_regra_antiga_x_simetrica"]
    if rg is not None:
        g = _regra_sim(rg)
        nomes = [("cicatriz", "Early-scar edges", AZUL), ("referencia", "Reference edges (no earlier burn)", MUDO)]
        for yi, (t, lab, cor) in zip([1, 0], nomes):
            v = g.loc[t, "% interrupcao"]; n = int(g.loc[t, "interrupcao"] + g.loc[t, "continuidade"])
            ax.barh(yi, v, height=0.5, color=cor, alpha=0.9, zorder=2)
            ax.text(v + 1.2 if v < 85 else v - 1.2, yi, f"{v:.1f}%  (n = {n:,})", fontsize=tam(5.8),
                    color=TINTA if v < 85 else "white", ha="left" if v < 85 else "right", va="center", zorder=3)
        ax.set_yticks([1, 0]); ax.set_yticklabels([n[1] for n in nomes], fontsize=tam(6.2))
        ax.set_ylim(-0.5, 1.5); ax.set_xlim(0, 100); ax.set_xticks([0, 25, 50, 75, 100])
        ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0)
        ax.set_xlabel(f"Arriving fires interrupted (%), {len(ORDER8)} ILs, {YEARS[0]}–{YEARS[-1]}", fontsize=tam(6.2))
    rotulo_painel(ax, "d")
    salvar(fig, "FigS3_window_landsat")

# ============================================================
# TABELA 1
# ============================================================
def table1():
    rows = []
    for ti in ORDER8["ti_id"].astype(int):
        r = {"#": NUM.get(ti), "IL": TI_NAME.get(ti, ti)}
        a = tis.loc[tis["ti_id"].astype(int) == ti, "area_ha"]
        r["Area (km²)"] = int(round(float(a.iloc[0]) / 100)) if len(a) else np.nan
        aa = area_anual[area_anual["ti_id"] == ti]
        if len(aa):
            r["Early burned (% yr⁻¹)"] = round(100 * (aa["early_ha"] / aa["ti_ha"]).mean(), 1)
            r["Late burned (% yr⁻¹)"] = round(100 * (aa["late_ha"] / aa["ti_ha"]).mean(), 1)
        g = segs[segs["ti_id"] == ti]
        for t, nome in [("cicatriz", "scar"), ("referencia", "reference")]:
            q = g[(g["tipo_seg"] == t) & g["resultado"].isin(["interrupcao", "continuidade"])]
            if t == "cicatriz":
                r["Scar edges with arrival"] = len(q)
            r[f"Interrupted, {nome} edges (%)"] = round(100 * (q["resultado"] == "interrupcao").mean(), 1) if len(q) else np.nan
        if art1 is not None:
            cod = tis.loc[tis["ti_id"].astype(int) == ti, "terrai_cod"]
            a1 = art1[art1["terrai_cod"] == int(cod.iloc[0])] if len(cod) else art1.iloc[0:0]
            r["Dias et al. (2026) class"] = a1["classe"].iloc[0] if len(a1) else ""
            r["FRRI"] = round(float(a1["frri"].iloc[0]), 2) if len(a1) and pd.notna(a1["frri"].iloc[0]) else np.nan
        rows.append(r)
    t = pd.DataFrame(rows)
    t.to_csv(FIG_DIR / "Table1_analysed_ILs.csv", index=False)
    display(t)
    return t

CAP = {
    "Fig1_premise": "Figure 1. Early dry-season scars and their timing. (a) Dominant burn season of each 30-m pixel in "
                    f"the analysed Indigenous Lands (ILs), {YEARS[0]}–{YEARS[-1]} (MapBiomas Fire). (b) Annual burned area "
                    "in May–July and August–October. (c) Monthly minimum Landsat 8/9 NBR of pixels burned early, burned "
                    "later and unburned (median and interquartile range; NBR within 500 m of burns).",
    "Fig2_edge_test": "Figure 2. Do early scars stop arriving fire? (a) Each IL in the year with most scar edges reached by "
                      "later fire: Landsat dNBR between July and the August–October minimum, early-scar outlines and 300-m "
                      "scar edges crossed by later fire (magenta). (b) Arriving fires stopped at early-scar and reference "
                      "edges with the same Landsat + MODIS entry rule, and the adjusted odds ratio. (c) The same in each IL.",
    "Fig3_configuration": "Figure 3. Configuration beyond quantity. (a) Early scars coloured by size, later fire and crossed "
                          "edges. (b, c) Share of arriving fires stopped by early-scar size and local width (95% binomial "
                          "interval). (d) Odds ratios of stopping per SD beyond the reference model with quantity (SE "
                          "clustered by fire and IL; open symbol, CI includes 1).",
    "Fig4_generalisation": f"Figure 4. Generalisation. (a) Share of arriving fires stopped at early-scar edges in "
                           f"{CELL_M / 1000:.0f}-km cells (>= 10 edges). (b) Distribution of the cells in each IL (bar, "
                           "median). (c) Odds ratio for window fragmentation under alternative seasonal splits, encounter "
                           "criteria and fuel definitions, one change at a time (95% CI; band, default model).",
    "FigS2_scar_types": "Figure S2. Descriptive types of early dry-season scars (>= 1 ha): (a) burned area; (b) mean VPD "
                        "percentile on the day of the first MODIS detection (ERA5-Land; IL-bootstrap 95% CI); (c) share of "
                        "scars with night-time MODIS detections. Types are not attributed to management or brigades.",
    "FigS3_window_landsat": f"Figure S3. Detail at 30 m: a 12 × 12 km window of {TI_NAME.get(W[0])}, {W[1]}. (a–c) Landsat "
                            "8/9 NBR in July, minimum NBR of August–October and dNBR, with early-scar outlines and crossed "
                            "edges. (d) Share of arriving fires stopped at early-scar and reference edges, all ILs.",
}


# ============================================================
# FIGURAS DE SENSORIAMENTO (eram do bloco 17)
# ============================================================
from pyproj import Transformer
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as mpatches

TAG_SEG    = globals().get("TAG_SEG", f"{TAG_YEARS}_md")
FOCOS_NAME = globals().get("FOCOS_NAME", f"modis_focos_v1_{TAG_YEARS}")
_to_ll = Transformer.from_crs(AREA_CRS, "EPSG:4326", always_xy=True)
_to_xy = Transformer.from_crs("EPSG:4326", AREA_CRS, always_xy=True)
GRAT = "#b9b8b0"

# ------------------------------------------------------------
# Grade de coordenadas (lat/long) desenhada sobre mapas em EPSG:5880
# ------------------------------------------------------------
def graticule(ax, ext, step=None, labels=True, fs=5.2):
    """Linhas de lat/long (EPSG:5880) e rotulos FORA do quadro: longitude embaixo, latitude a esquerda."""
    x0, y0, x1, y1 = ext
    lons, lats = _to_ll.transform([x0, x1, x0, x1], [y0, y0, y1, y1])
    lo0, lo1, la0, la1 = min(lons), max(lons), min(lats), max(lats)
    if step is None:
        span = max(lo1 - lo0, la1 - la0)
        step = next(s for s in [0.05, 0.1, 0.25, 0.5, 1, 2, 5, 10] if span / s <= 4)
    fmt = (lambda v, ew: f"{abs(v):.{0 if step >= 1 else (1 if step >= 0.1 else 2)}f}°{ew}")
    xt, xl, yt, yl = [], [], [], []
    for lo in np.arange(np.ceil(lo0 / step) * step, lo1, step):
        la = np.linspace(la0 - 1, la1 + 1, 200)
        xs, ys = _to_xy.transform(np.full_like(la, lo), la)
        ax.plot(xs, ys, color=GRAT, lw=0.3, zorder=1.5, clip_on=True)
        xb, _ = _to_xy.transform(lo, la0)
        if x0 < xb < x1:
            xt.append(xb); xl.append(fmt(lo, "W" if lo < 0 else "E"))
    for la in np.arange(np.ceil(la0 / step) * step, la1, step):
        lo = np.linspace(lo0 - 1, lo1 + 1, 200)
        xs, ys = _to_xy.transform(lo, np.full_like(lo, la))
        ax.plot(xs, ys, color=GRAT, lw=0.3, zorder=1.5, clip_on=True)
        _, yb = _to_xy.transform(lo0, la)
        if y0 < yb < y1:
            yt.append(yb); yl.append(fmt(la, "S" if la < 0 else "N"))
    ax._grat = True
    ax.grid(False)
    ax.set_xticks(xt); ax.set_xticklabels(xl if labels else [])
    ax.set_yticks(yt); ax.set_yticklabels(yl if labels else [], rotation=90, va="center")
    ax.tick_params(labelsize=tam(fs), length=1.8, width=0.4, pad=1.2, colors=TINTA2)
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.5); s.set_color(TINTA2)

# ============================================================
# FIG R1 - AREA DE ESTUDO E TRIAGEM DA AMOSTRA
# ============================================================
def figR1():
    am_p = table_path("amostra_final_tis")
    am = pd.read_csv(am_p)
    ty_p = stats_path("elegibilidade_encontros_ti_ano")
    ty = pd.read_csv(ty_p)
    fig = plt.figure(figsize=(LARG, 15.5 * CM), layout="constrained")
    sf_a, sf_b = fig.subfigures(2, 1, height_ratios=[1.35, 1], hspace=0.03)
    ax = sf_a.subplots()
    ext = mapa_localizacao(ax, rotulos=True)
    excl = am[~am["elegivel"].astype(bool)]
    for _, r in excl.iterrows():
        g = tis[tis["ti_id"].astype(int) == int(r["ti_id"])]
        if len(g):
            p = g.geometry.iloc[0].representative_point()
            cod = "I" if "imagens" in str(r["motivo_exclusao"]) else "E"
            ax.annotate(cod, (p.x, p.y), xytext=(4, -6), textcoords="offset points", fontsize=tam(5.6),
                        color=TINTA2, fontstyle="italic", path_effects=HALO, zorder=6)
    graticule(ax, ext, step=5)
    moldura(ax, ext, barra_m=500000, seta=True)
    # inset: America do Sul / Brasil
    axi = ax.inset_axes([0.70, 0.66, 0.30, 0.34])
    if "brasil" in CTX:
        CTX["brasil"].plot(ax=axi, facecolor=FUNDO, edgecolor="#9a9992", lw=0.3)
    if "cerrado" in CTX:
        CTX["cerrado"].plot(ax=axi, facecolor=CERRADO, edgecolor=CERRADO_BORDA, lw=0.3)
    axi.add_patch(mpatches.Rectangle(ext[:2], ext[2] - ext[0], ext[3] - ext[1], fill=False, ec=TINTA, lw=0.5))
    axi.set_aspect("equal"); axi.set_axis_off()
    sf_a.legend(handles=[mpatches.Patch(fc=LARANJA, ec=TINTA, lw=0.35, label="Analysed IL (numbers as in Table 1)"),
                         mpatches.Patch(fc="white", ec=TINTA2, lw=0.5, label="Excluded IL: I = imagery, E = encounters"),
                         mpatches.Patch(fc=CERRADO, ec=CERRADO_BORDA, lw=0.45, label="Cerrado biome")],
                loc="outside lower center", ncol=3, fontsize=tam(5.6), handlelength=1.1, columnspacing=1.0)
    rotulo_painel(ax, "a")

    axb = sf_b.subplots()
    order = am.sort_values(["elegivel", "ti_name"], ascending=[False, True])["ti_id"].astype(int).tolist()
    gap = ty.pivot_table(index="ti_id", columns="year", values="max_gap_dias").reindex(order)
    enc = ty.pivot_table(index="ti_id", columns="year", values="n_encontros", fill_value=0).reindex(order)
    cm_gap = LinearSegmentedColormap.from_list("gap", ["#1f5fc8", "#a9c8ee", "#fbe3c4", "#e0532b", "#7a1f3d"])
    im = axb.imshow(gap.values, cmap=cm_gap, norm=mpl.colors.Normalize(8, 96), aspect="auto", interpolation="nearest")
    for i in range(gap.shape[0]):
        for j in range(gap.shape[1]):
            v = int(enc.values[i, j])
            if v:
                axb.text(j, i, f"{v}" if v < 1000 else f"{v / 1000:.1f}k", ha="center", va="center",
                         fontsize=tam(4.6), color="white" if gap.values[i, j] < 20 or gap.values[i, j] > 70 else TINTA)
    axb.set_xticks(range(gap.shape[1])); axb.set_xticklabels(gap.columns, fontsize=tam(5.4))
    axb.set_yticks(range(gap.shape[0]))
    nm = tis.set_index("ti_id")["ti_name"]
    axb.set_yticklabels([f"{NUM.get(t, '–')} {nm[t]}" for t in order], fontsize=tam(5.4))
    n_ok = int(am["elegivel"].sum())
    axb.axhline(n_ok - 0.5, color=TINTA, lw=0.8)
    axb.grid(False); axb.tick_params(length=0)
    cb = sf_b.colorbar(im, ax=axb, fraction=0.03, pad=0.01, extend="max")
    cb.set_label("Longest gap between clear\nLandsat 8/9 dates, Jun–Oct (days)", fontsize=tam(5.6))
    cb.ax.tick_params(labelsize=tam(5.2)); cb.outline.set_linewidth(0.4)
    axb.set_title("numbers: candidate encounters (late fire touching early scar); line: analysed above, excluded below",
                  fontsize=tam(5.6), color=MUDO, loc="left", pad=2)
    rotulo_painel(axb, "b")
    salvar(fig, "FigR1_study_area_sample")

# ============================================================
# FIG R2 - O ENCONTRO VISTO PELO LANDSAT E PELO MODIS (30 m)
# ============================================================
def landsat_rgb(win, d0, d1, px=30):
    """Mediana Landsat 8/9 SR (SWIR2, NIR, red) sem nuvem na janela, grade EPSG:5880 de 30 m alinhada."""
    x0, y0, x1, y1 = win
    n_c, n_r = int(round((x1 - x0) / px)), int(round((y1 - y0) / px))
    rect = ee.Geometry.Rectangle([x0, y0, x1, y1], PROJ, False)
    def prep(i):
        i = ee.Image(i)
        clear = i.select("QA_PIXEL").bitwiseAnd(0b11111).eq(0)
        return i.select(["SR_B7", "SR_B5", "SR_B4"]).multiply(0.0000275).add(-0.2).updateMask(clear)
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(rect).filterDate(d0, d1))
    n_img = col.size().getInfo()
    img = col.map(prep).median().unmask(-1).toFloat()
    arr = ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY",
                                 "grid": {"dimensions": {"width": n_c, "height": n_r},
                                          "affineTransform": {"scaleX": px, "shearX": 0, "translateX": x0,
                                                              "shearY": 0, "scaleY": -px, "translateY": y1},
                                          "crsCode": AREA_CRS}})
    rgb = np.dstack([arr[b] for b in ["SR_B7", "SR_B5", "SR_B4"]]).astype(float)
    rgb[rgb < 0] = np.nan
    return rgb, n_img

def stretch(rgb, lo=2, hi=98, ref=None):
    ref = rgb if ref is None else ref
    out = np.empty_like(rgb)
    for k in range(3):
        a, b = np.nanpercentile(ref[..., k], [lo, hi])
        out[..., k] = np.clip((rgb[..., k] - a) / (b - a), 0, 1)
    return np.nan_to_num(out ** 0.9, nan=0.9)

def figR2():
    ti, yr, win = W
    pre, n_pre = landsat_rgb(win, f"{yr}-06-15", f"{yr}-08-01")
    post, n_post = landsat_rgb(win, f"{yr}-09-15", f"{yr}-11-01")
    both = np.concatenate([pre, post], axis=0)                     # mesmo estiramento nas duas datas
    pre_s, post_s = stretch(pre, ref=both), stretch(post, ref=both)
    ext_i = (win[0], win[2], win[1], win[3])
    early = np.isin(JNBR["month"], EARLY).astype(float) if JNBR is not None else None
    fo = read_table(cache_path(FOCOS_NAME, "parquet"))
    fo = fo[(fo["confidence"].astype(str).str[0] != "l")]
    fo["acq_date"] = pd.to_datetime(fo["acq_date"])
    fo = fo[(fo["acq_date"].dt.year == yr) & fo["acq_date"].dt.month.isin(LATE)]
    fx, fy = _to_xy.transform(fo["longitude"].to_numpy(), fo["latitude"].to_numpy())
    inw = (fx > win[0]) & (fx < win[2]) & (fy > win[1]) & (fy < win[3])
    fx, fy = fx[inw], fy[inw]
    sg = pd.read_parquet(cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet"),
                         columns=["ti_id", "year", "tipo_seg", "resultado", "x", "y"])
    sg = sg[(sg["ti_id"] == ti) & (sg["year"] == yr) & (sg["tipo_seg"] == "cicatriz") &
            sg["x"].between(win[0], win[2]) & sg["y"].between(win[1], win[3])]

    fig = plt.figure(figsize=(LARG, 7.6 * CM), layout="constrained")
    axs = fig.subplots(1, 3)
    titles = [f"Landsat 8/9, 15 Jun–31 Jul {yr} (n = {n_pre})", f"Landsat 8/9, 15 Sep–31 Oct {yr} (n = {n_post})",
              "dNBR, scars, MODIS and edges"]
    axs[0].imshow(pre_s, extent=ext_i, interpolation="nearest", rasterized=True, zorder=1)
    axs[1].imshow(post_s, extent=ext_i, interpolation="nearest", rasterized=True, zorder=1)
    if JNBR is not None:
        dn = np.ma.masked_less(JNBR["pre"], -100) / 100.0 - np.ma.masked_less(JNBR["post"], -100) / 100.0
        e = JNBR["ext"]
        axs[2].add_patch(mpatches.Rectangle(win[:2], win[2] - win[0], win[3] - win[1], color=SEM_DADO, zorder=0))
        im = axs[2].imshow(np.ma.clip(dn, -0.1, 0.8), extent=(e[0], e[1], e[2], e[3]), cmap=CMAP_DNBR,
                           norm=mpl.colors.Normalize(0, 0.6), interpolation="nearest", rasterized=True, zorder=1)
    for ax in axs:
        if early is not None and early.any():
            ax.contour(np.flipud(early), levels=[0.5], colors=["#5fd3ff" if ax is not axs[2] else C_INI],
                       linewidths=0.45, extent=ext_i, zorder=3)
    for x, y in zip(fx, fy):                                         # pegada nominal do pixel MODIS (~926 m)
        axs[2].add_patch(mpatches.Rectangle((x - 463, y - 463), 926, 926, fill=False, ec="#7a1f3d", lw=0.5, zorder=4))
    ok = sg[sg["resultado"] == "interrupcao"]; ct = sg[sg["resultado"] == "continuidade"]
    axs[2].scatter(ok["x"], ok["y"], s=2.5, color="#163f73", lw=0, zorder=5)
    axs[2].scatter(ct["x"], ct["y"], s=9, color=C_PASSOU, edgecolor="white", lw=0.4, zorder=6)
    for k, ax in enumerate(axs):
        ti_geom(ti).boundary.plot(ax=ax, color="white" if k < 2 else TINTA, lw=0.5, zorder=4)
        graticule(ax, win)
        moldura(ax, win, barra_m=2000, seta=True)
        titulo_mapa(ax, titles[k])
        rotulo_painel(ax, "abc"[k])
    if JNBR is not None:
        colorbar_fina(fig, im, axs[2], "dNBR (July − min Aug–Oct)", ticks=[0, 0.2, 0.4, 0.6], extend="max")
    fig.legend(handles=[Line2D([], [], color=C_INI, lw=0.9, label="Early scar, May–Jul (MapBiomas)"),
                        mpatches.Patch(fill=False, ec="#7a1f3d", lw=0.6, label="MODIS fire pixel, Aug–Oct"),
                        Line2D([], [], marker="o", ls="", ms=2, color="#163f73", label="Edge: fire interrupted"),
                        Line2D([], [], marker="o", ls="", ms=3.4, mfc=C_PASSOU, mec="white", label="Edge: fire crossed")],
               loc="outside lower center", ncol=4, fontsize=tam(5.4), handlelength=1.0, columnspacing=0.9)
    salvar(fig, "FigR2_encounter_landsat_modis")

# ============================================================
# FIG R3 - TESTE INDEPENDENTE DO NBR (FOCOS MODIS NOS NUCLEOS)
# ============================================================
def figR3():
    rr = pd.read_csv(stats_path("B08b_modis_razao_taxas"))
    pt = pd.read_csv(stats_path("B08b_modis_razao_taxas_por_TI"), index_col=0)
    nuc = pd.read_csv(stats_path("B08b_modis_nucleos_ti_ano"))
    fig = plt.figure(figsize=(LARG, 13.2 * CM), layout="constrained")
    sf_m, sf_g = fig.subfigures(2, 1, height_ratios=[1.25, 1], hspace=0.03)
    # (a) mapa do TI-ano com maior area de nucleo de cicatriz
    ex = nuc[nuc["core_m"] == nuc["core_m"].min()].sort_values("km2_cicatriz", ascending=False).iloc[0]
    _, M = core_counts(int(ex["ti_id"]), int(ex["year"]), int(ex["core_m"]), keep_masks=True)
    tr = M["tr"]; Hh, Ww = M["E"].shape
    e = (tr.c, tr.f - Hh * PIX_M, tr.c + Ww * PIX_M, tr.f)
    cls = np.zeros(M["E"].shape, np.uint8)
    cls[M["inside"]] = 1; cls[M["L"]] = 2; cls[M["E"]] = 3; cls[M["coreE"]] = 4; cls[M["coreU"]] = 5
    cmap = ListedColormap(["#ffffff", NEUTRO, "#f9c9a2", AZUL_CLARO, C_INI, "#cfe3c3"])
    ax = sf_m.subplots()
    f = int(np.ceil(max(cls.shape) / 1800))
    ax.imshow(np.ma.masked_equal(cls[::f, ::f], 0), cmap=cmap, vmin=0, vmax=5, extent=(e[0], e[2], e[1], e[3]),
              interpolation="nearest", rasterized=True, zorder=1)
    ax.scatter(*M["pts"], s=2.2, color=VINHO, lw=0, zorder=4)
    ti_geom(int(ex["ti_id"])).boundary.plot(ax=ax, color=TINTA, lw=0.5, zorder=5)
    bb = ti_geom(int(ex["ti_id"])).total_bounds
    h = max(bb[2] - bb[0], bb[3] - bb[1]) / 2 * 1.03; cx, cy = (bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2
    ext = (cx - h, cy - h, cx + h, cy + h)
    graticule(ax, ext)
    moldura(ax, ext, barra_m=barra_redonda(2 * h), seta=True)
    titulo_mapa(ax, TI_NAME.get(int(ex['ti_id'])), f"{int(ex['year'])}", negrito=True)
    sf_m.legend(handles=[mpatches.Patch(color=AZUL_CLARO, label="Early scar (May–Jul)"),
                         mpatches.Patch(color=C_INI, label="Scar core, > 1 km from edge"),
                         mpatches.Patch(color="#cfe3c3", label="Unburned-fuel core, > 1 km"),
                         mpatches.Patch(color="#f9c9a2", label="Burned Aug–Oct"),
                         Line2D([], [], marker="o", ls="", ms=2.6, color=VINHO, label="MODIS fire pixel, Aug–Oct")],
                loc="outside right center", ncol=1, fontsize=tam(5.6), handlelength=1.0)
    rotulo_painel(ax, "a")
    # (b) razao de taxas por TI no mapa de localizacao
    axb, axc = sf_g.subplots(1, 2, width_ratios=[1.25, 1])
    vals = {int(k): float(v) for k, v in pt["razao_taxas"].items() if np.isfinite(v) and v > 0}
    norm = mpl.colors.LogNorm(max(min(vals.values()), 1e-3), 1) if vals else mpl.colors.LogNorm(1e-2, 1)
    cm_rr = LinearSegmentedColormap.from_list("rr", ["#163f73", "#2f78b4", "#9ec9e2", "#fbe3c4"])
    ext_l = mapa_localizacao(axb, cor_ti=vals, cmap=cm_rr, norm=norm)
    graticule(axb, ext_l, step=5)
    moldura(axb, ext_l, barra_m=500000, seta=True)
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cm_rr), ax=axb, fraction=0.04, pad=0.01)
    cb.set_label("MODIS rate ratio,\nscar / unburned core", fontsize=tam(5.6)); cb.ax.tick_params(labelsize=tam(5))
    rotulo_painel(axb, "b")
    # (c) razao agregada (log)
    lab = {"naoqueimado": "All unburned fuel", "naoqueimado_local": "Unburned ≤ 5 km of a scar"}
    y = 0
    for cm_ in sorted(rr["core_m"].unique()):
        for ref in ["naoqueimado", "naoqueimado_local"]:
            r = rr[(rr["core_m"] == cm_) & (rr["referencia"] == ref)]
            if not len(r):
                continue
            r = r.iloc[0]
            axc.plot([r["ic95_inf_boot_TI"], r["ic95_sup_boot_TI"]], [y, y], color=C_INI, lw=1.8, solid_capstyle="round")
            axc.scatter(r["razao_taxas_agregada"], y, s=26, color=C_INI, edgecolor="white", zorder=3)
            axc.text(r["ic95_sup_boot_TI"] * 1.15, y, f"{r['razao_taxas_agregada']:.2f}", va="center", fontsize=tam(5.6))
            axc.text(-0.02, y, f"{lab[ref]}\ncore > {cm_ / 1000:g} km", transform=mpl.transforms.blended_transform_factory(
                axc.transAxes, axc.transData), ha="right", va="center", fontsize=tam(5.4), linespacing=1.0)
            y += 1
    axc.axvline(1, color=TINTA2, lw=0.7, ls=(0, (3, 2)))
    axc.set_xscale("log"); axc.set_yticks([]); axc.set_ylim(-0.6, y - 0.4)
    axc.set_xlabel("Later-fire MODIS density, scar core / unburned core\n(95% CI, bootstrap over ILs)", fontsize=tam(5.8))
    axc.grid(axis="y", visible=False)
    rotulo_painel(axc, "c")
    salvar(fig, "FigR3_modis_independent_test")

# ============================================================
# FIG R4 - CONFIGURACAO PELO MODELO NULO, NO MAPA
# ============================================================
def figR4():
    cn = pd.read_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[FUEL_MAIN]}_{TAG_YEARS}_nulo", "parquet"))
    cn = cn[cn["q_inicio"] > 0].dropna(subset=["conf_split_ses"])
    g = cn.groupby(["ti_id", "x", "y"]).agg(ses=("conf_split_ses", "mean"), n=("year", "size")).reset_index()
    g = g[g["n"] >= 3]
    norm = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
    cm_ses = LinearSegmentedColormap.from_list("ses", ["#163f73", "#2f78b4", "#9ec9e2", "#f4f3ef", "#f9c9a2", "#e0532b", "#7a1f3d"])
    fig = plt.figure(figsize=(LARG, 9.8 * CM), layout="constrained")

    def desenhar(ax, ti):
        d_ = g[g["ti_id"] == ti]
        hh = CELL_M / 2
        if len(d_):
            gg = gpd.GeoDataFrame(d_[["ses"]], geometry=[_box(x - hh, y - hh, x + hh, y + hh) for x, y in zip(d_["x"], d_["y"])],
                                  crs=AREA_CRS)
            gg.plot(ax=ax, column="ses", cmap=cm_ses, norm=norm, edgecolor="white", linewidth=0.12, zorder=2)
    axs = grade_tis(fig, desenhar, info=lambda ti: (f"median SES {g.loc[g['ti_id'] == ti, 'ses'].median():.2f}"
                                                   if (g["ti_id"] == ti).any() else ""))
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cm_ses), ax=axs, location="bottom", shrink=0.45,
                      aspect=34, pad=0.02, extend="both")
    cb.set_label("Fuel-fragmentation SES vs null model (mean 2014–2024; blue: early scars split fuel more than chance)",
                 fontsize=tam(5.6))
    cb.ax.tick_params(labelsize=tam(5.2)); cb.outline.set_linewidth(0.4)
    rotulo_painel(axs[0, 0], "a")
    salvar(fig, "FigR4_null_model_configuration")


# ============================================================
# FIGURAS DOS OUTROS BLOCOS (refeitas no mesmo estilo, a partir do que ja foi gravado)
# ============================================================
from scipy import ndimage as _nd
SEASONS = load_seasons()
CORE_M, LOCAL_M = [1000, 1500], 5000          # mesmos valores do bloco 11

_fo_cache = {}
def _focos():
    if "fo" not in _fo_cache:
        fo = read_table(cache_path(FOCOS_NAME, "parquet"))
        fo = fo[fo["confidence"].astype(str).str[0] != "l"].copy()
        fo["acq_date"] = pd.to_datetime(fo["acq_date"])
        fo = fo.drop_duplicates(["latitude", "longitude", "acq_date"])
        x, y = _to_xy.transform(fo["longitude"].to_numpy(), fo["latitude"].to_numpy())
        fo["x"], fo["y"] = x, y
        fo["year"], fo["mes"] = fo["acq_date"].dt.year, fo["acq_date"].dt.month
        _fo_cache["fo"] = fo
    return _fo_cache["fo"]

def core_counts(ti, y, core_m, scheme=SEASON_MAIN, fuel_mode=FUEL_MAIN, keep_masks=False):
    """Mesma conta do bloco 11: focos MODIS do restante da seca nos nucleos de cicatriz x nao queimado."""
    fo = _focos()
    st, yr = static_of(ti), read_raster(ti, y)
    ini, res = SEASONS[scheme][int(ti)]
    inside = st["ti"] == ti
    queim = inside & fuel_from_cover(yr["cover"], fuel_mode) & (st["barrier"] == 0)
    sm_ = season_masks(yr["month"], ti, scheme, SEASONS)
    E = sm_["inicio"] & queim
    U = queim & ~sm_["antes"] & ~E
    k = core_m / PIX_M
    coreE = _nd.distance_transform_edt(E) > k
    coreU = _nd.distance_transform_edt(U) > k
    tr = st["_transform"]
    p = fo[(fo["ti_id"] == ti) & (fo["year"] == y) & fo["mes"].isin(res)]
    c_ = np.floor((p["x"].to_numpy() - tr.c) / PIX_M).astype(int)
    r_ = np.floor((tr.f - p["y"].to_numpy()) / PIX_M).astype(int)
    ok = (r_ >= 0) & (r_ < E.shape[0]) & (c_ >= 0) & (c_ < E.shape[1])
    out = {"ti_id": ti, "year": y, "core_m": core_m}
    if keep_masks:
        return out, dict(coreE=coreE, coreU=coreU, E=E, L=sm_["resto"] & inside, inside=inside, tr=tr,
                         pts=(p["x"].to_numpy()[ok], p["y"].to_numpy()[ok]))
    return out

# ---------- S4: estacao seca de cada TI (bloco 06) ----------
def figS4():
    pr = pd.read_csv(stats_path("chuva_mensal_por_ti"))
    fig = plt.figure(figsize=(LARG, 8.6 * CM), layout="constrained")
    axs = fig.subplots(2, 4, sharex=True, sharey=True)
    mes_lab = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
    for ax, (_, r) in zip(axs.ravel(), ORDER8.iterrows()):
        ti = int(r["ti_id"])
        g = pr[pr["ti_id"] == ti].sort_values("mes")
        ini, res = SEASONS.get("local", SEASONS["artigo1"]).get(ti, (EARLY, LATE))
        cores = [C_INI if m in ini else C_TAR if m in res else "#cfcec7" for m in g["mes"]]
        ax.axvspan(4.5, 7.5, color=C_INI, alpha=0.07, lw=0); ax.axvspan(7.5, 10.5, color=C_TAR, alpha=0.08, lw=0)
        ax.bar(g["mes"], g["chuva_mm"], color=cores, width=0.75, zorder=2)
        ax.axhline(60, color=TINTA2, lw=0.6, ls=(0, (3, 2)), zorder=3)
        ax.set_title(f"{NUM[ti]} {r['ti_name']}", fontsize=tam(5.8), pad=1.5, loc="left", fontweight="bold")
        ax.set_xticks(range(1, 13)); ax.set_xticklabels(mes_lab, fontsize=tam(5))
        ax.grid(axis="x", visible=False); ax.tick_params(axis="x", length=0)
    for ax in axs[:, 0]:
        ax.set_ylabel("Rain (mm)", fontsize=tam(6))
    fig.legend(handles=[mpatches.Patch(color=C_INI, label="Local early dry season"),
                        mpatches.Patch(color=C_TAR, label="Local rest of dry season"),
                        Line2D([], [], color=TINTA2, lw=0.6, ls=(0, (3, 2)), label="60 mm"),
                        mpatches.Patch(color=C_INI, alpha=0.15, label="Dias et al. (2026): May–Jul | Aug–Oct")],
               loc="outside lower center", ncol=4, fontsize=tam(5.6), handlelength=1.2, columnspacing=1.0)
    rotulo_painel(axs[0, 0], "a")
    salvar(fig, "FigS4_dry_season_per_IL")

# ---------- S5: focos MODIS sobre as cicatrizes (bloco 05) ----------
def figS5():
    fo = _focos()
    scars = read_table(cache_path(SCARS_NAME, "gpkg"))
    fig = plt.figure(figsize=(LARG, 9.6 * CM), layout="constrained")

    def desenhar(ax, ti):
        yr = REP.get(ti)
        s = scars[(scars["ti_id"] == ti) & (scars["year"] == yr)]
        s[s["month"].isin(EARLY)].plot(ax=ax, color=C_INI, lw=0, zorder=2, rasterized=True)
        s[s["month"].isin(LATE)].plot(ax=ax, color=C_TAR, lw=0, zorder=2, rasterized=True)
        f = fo[(fo["ti_id"] == ti) & (fo["year"] == yr)]
        ax.scatter(f["x"], f["y"], s=1.6, color=np.where(f["mes"].isin(EARLY), "#0b2a66", VINHO), lw=0, zorder=4)
    axs = grade_tis(fig, desenhar, info=lambda ti: str(REP.get(ti, "")))
    fig.legend(handles=[mpatches.Patch(color=C_INI, label="Scar May–Jul (MapBiomas)"),
                        mpatches.Patch(color=C_TAR, label="Scar Aug–Oct (MapBiomas)"),
                        Line2D([], [], marker="o", ls="", ms=2.4, color="#0b2a66", label="MODIS fire pixel, May–Jul"),
                        Line2D([], [], marker="o", ls="", ms=2.4, color=VINHO, label="MODIS fire pixel, Aug–Oct")],
               loc="outside lower center", ncol=4, fontsize=tam(5.6), handlelength=1.0, columnspacing=1.0)
    rotulo_painel(axs[0, 0], "a")
    salvar(fig, "FigS5_modis_over_scars")

# ---------- S6: detectabilidade (bloco 11) ----------
def figS6():
    curva = pd.read_csv(stats_path("B08b_curva_limiar"))
    sens = pd.read_csv(stats_path("B08b_sensibilidade_landsat"), index_col=0)
    rr = pd.read_csv(stats_path("B08b_modis_razao_taxas"))
    fig = plt.figure(figsize=(LARG, 6.6 * CM), layout="constrained")
    ax1, ax2, ax3 = fig.subplots(1, 3, width_ratios=[1.2, 0.8, 1.1])
    for tipo, cor, lab in [("cicatriz", C_INI, "Early-scar edges"), ("referencia", "#8f8e88", "Reference edges")]:
        g = curva[curva["tipo_seg"] == tipo]
        ax1.plot(g["limiar"], g["interrupcao_abs_%"], color=cor, lw=1.6, label=lab)
        ax1.plot(g["limiar"], g["interrupcao_rel_%"], color=cor, lw=1.0, ls=(0, (3, 2)))
    ax1.axvline(0.10, color=TINTA2, lw=0.6, ls=":")
    ax1.text(0.102, 2, "used", fontsize=tam(5.2), color=TINTA2)
    ax1.set_xlabel("NBR-drop threshold", fontsize=tam(6)); ax1.set_ylabel("Interrupted (%)", fontsize=tam(6))
    ax1.set_ylim(0, 102); ax1.legend(fontsize=tam(5.4), loc="center right")
    rotulo_painel(ax1, "a")
    x = np.arange(len(sens))
    ax2.bar(x, 100 * sens["landsat_viu_entrada"], color=[C_INI, "#8f8e88"][:len(sens)], width=0.6, zorder=2)
    for k_, (i_, r) in enumerate(sens.iterrows()):
        ax2.text(k_, 100 * r["landsat_viu_entrada"] + 2, f"{100 * r['landsat_viu_entrada']:.0f}%\nn = {int(r['n_segmentos']):,}",
                 ha="center", va="bottom", fontsize=tam(5.2), linespacing=1.0)
    ax2.set_xticks(x); ax2.set_xticklabels([{"cicatriz": "Scar", "referencia": "Ref."}.get(i, i) for i in sens.index])
    ax2.set_ylim(0, 118); ax2.set_ylabel("Landsat saw the entry (%)", fontsize=tam(6))
    ax2.grid(axis="x", visible=False); ax2.tick_params(axis="x", length=0)
    rotulo_painel(ax2, "b")
    y = 0; labs = []
    lab = {"naoqueimado": "all unburned", "naoqueimado_local": "unburned ≤ 5 km"}
    for cm_ in sorted(rr["core_m"].unique()):
        for ref in ["naoqueimado", "naoqueimado_local"]:
            r = rr[(rr["core_m"] == cm_) & (rr["referencia"] == ref)]
            if not len(r):
                continue
            r = r.iloc[0]
            ax3.plot([r["ic95_inf_boot_TI"], r["ic95_sup_boot_TI"]], [y, y], color=C_INI, lw=1.8, solid_capstyle="round")
            ax3.scatter(r["razao_taxas_agregada"], y, s=26, color=C_INI, edgecolor="white", zorder=3)
            ax3.text(r["ic95_sup_boot_TI"] * 1.12, y, f"{r['razao_taxas_agregada']:.2f}", va="center", fontsize=tam(5.4))
            labs.append(f"core > {cm_ / 1000:g} km, {lab[ref]}"); y += 1
    ax3.axvline(1, color=TINTA2, lw=0.7, ls=(0, (3, 2)))
    ax3.set_xscale("log"); ax3.set_yticks(range(len(labs))); ax3.set_yticklabels(labs, fontsize=tam(5.2))
    ax3.set_ylim(-0.6, len(labs) - 0.4); ax3.grid(axis="y", visible=False); ax3.tick_params(axis="y", length=0)
    ax3.set_xlabel("MODIS density, scar / unburned", fontsize=tam(6))
    rotulo_painel(ax3, "c")
    salvar(fig, "FigS6_detectability")

# ---------- S7: inferencia para poucas TIs (bloco 13) ----------
def figS7():
    ct = pd.read_csv(stats_path("B09b_celula_ses_coeficientes"), index_col=0)
    st_ = pd.read_csv(stats_path("B09b_segmento_or_jackknife"), index_col=0)
    LAB = {"conf_split_ses_0": "Fragmentation SES", "conf_lpi_ses_0": "Largest-patch SES",
           "conf_npatch_ses_0": "N patches SES", "log_area_cicatriz": "Scar size (log)",
           "largura_local_100m": "Local scar width", "ses_cel_split": "Cell fragmentation SES",
           "nbr_pre_int": "Inner-strip NBR before", "gap_meses": "Months since scar"}
    fig = plt.figure(figsize=(LARG, 6.8 * CM), layout="constrained")
    axa, axb = fig.subplots(1, 2)
    y = np.arange(len(ct))[::-1]
    axa.axvline(0, color=TINTA2, lw=0.7, ls=(0, (3, 2)))
    for yi, (v, r) in zip(y, ct.iterrows()):
        lo, hi = r["coef_pp_por_dp"] - 1.96 * r["ep_jackknife_TI_pp"], r["coef_pp_por_dp"] + 1.96 * r["ep_jackknife_TI_pp"]
        sig = r["p_wild_webb"] < 0.05; cor = C_INI if sig else "#9a9992"
        axa.plot([lo, hi], [yi, yi], color=cor, lw=1.8, solid_capstyle="round")
        axa.scatter(r["coef_pp_por_dp"], yi, s=28, color=cor if sig else "white", edgecolor="white" if sig else cor, lw=0.9, zorder=3)
        axa.text(hi, yi + 0.22, f"wild p = {r['p_wild_webb']:.3f}", fontsize=tam(5), color=TINTA2, ha="right", va="bottom")
    axa.set_yticks(y); axa.set_yticklabels([LAB.get(v, v) for v in ct.index], fontsize=tam(5.8))
    axa.grid(axis="y", visible=False); axa.tick_params(axis="y", length=0)
    axa.set_xlabel("Later burned area (p.p. per SD)", fontsize=tam(6))
    axa.set_title(f"{CELL_M / 1000:.0f}-km cells", fontsize=tam(5.8), color=MUDO, loc="right", pad=2)
    rotulo_painel(axa, "a")
    t = st_[st_["modelo"] == "cicatriz | configuracao + deteccao"] if "modelo" in st_ else st_
    y = np.arange(len(t))[::-1]
    axb.axvline(1, color=TINTA2, lw=0.7, ls=(0, (3, 2)))
    for yi, (v, r) in zip(y, t.iterrows()):
        sig = r["ic95_jackknife_TI_inf"] > 1 or r["ic95_jackknife_TI_sup"] < 1; cor = C_INI if sig else "#9a9992"
        axb.plot([r["ic95_jackknife_TI_inf"], r["ic95_jackknife_TI_sup"]], [yi + 0.12] * 2, color=cor, lw=1.8,
                 solid_capstyle="round")
        axb.scatter(r["or_1dp"], yi + 0.12, s=28, color=cor if sig else "white", edgecolor="white" if sig else cor,
                    lw=0.9, zorder=3)
        axb.plot([r["ic95_cluster_inf"], r["ic95_cluster_sup"]], [yi - 0.18] * 2, color="#bdbcb4", lw=1.2,
                 solid_capstyle="round")
    axb.set_xscale("log")
    axb.xaxis.set_major_locator(FixedLocator([0.5, 0.75, 1, 1.5, 2, 3])); axb.xaxis.set_minor_locator(NullLocator())
    axb.xaxis.set_major_formatter(FormatStrFormatter("%g"))
    axb.set_yticks(y); axb.set_yticklabels([LAB.get(v, v) for v in t.index], fontsize=tam(5.8))
    axb.grid(axis="y", visible=False); axb.tick_params(axis="y", length=0)
    axb.set_xlabel("Odds of stopping per SD", fontsize=tam(6))
    axb.legend(handles=[Line2D([], [], color=C_INI, lw=1.8, label="Jackknife by IL"),
                        Line2D([], [], color="#bdbcb4", lw=1.2, label="Clustered (fire + IL)")],
               loc="lower right", fontsize=tam(5.2))
    axb.set_title("scar edges, adjusted for detectability", fontsize=tam(5.8), color=MUDO, loc="right", pad=2)
    rotulo_painel(axb, "b")
    salvar(fig, "FigS7_few_IL_inference")

# ---------- S8: configuracao pelo modelo nulo x quantidade (bloco 09) ----------
def figS8():
    cn = pd.read_parquet(cache_path(f"celulas_{SEASON_MAIN}_{CELL_M}_{FUEL_TAG[FUEL_MAIN]}_{TAG_YEARS}_nulo", "parquet"))
    sub = cn[cn["q_inicio"] > 0].dropna(subset=["conf_split_ses"])
    from scipy.stats import spearmanr
    fig = plt.figure(figsize=(LARG, 6.4 * CM), layout="constrained")
    axa, axb = fig.subplots(1, 2)
    for ax, col, cor, yl, let in [(axa, "conf_split", "#8f8e88", "Raw fragmentation\nlog(m$_{eff}$ after/before)", "a"),
                                  (axb, "conf_split_ses", C_INI, "Fragmentation SES\nvs null model", "b")]:
        hb = ax.hexbin(sub["q_inicio"], sub[col], gridsize=40, bins="log", mincnt=1, linewidths=0,
                       cmap=LinearSegmentedColormap.from_list("h", ["#f1f0eb", cor]))
        rho = spearmanr(sub["q_inicio"], sub[col]).correlation
        ax.set_title(f"Spearman ρ = {rho:.2f}", fontsize=tam(5.8), color=MUDO, loc="right", pad=2)
        ax.set_xlabel("Early burned fraction of the cell", fontsize=tam(6)); ax.set_ylabel(yl, fontsize=tam(6))
        if col.endswith("ses"):
            ax.axhline(0, color=TINTA2, lw=0.7, ls=(0, (3, 2)))
        rotulo_painel(ax, let)
    salvar(fig, "FigS8_null_model_vs_quantity")

# ---------- S9: resultado dos segmentos (bloco 10) ----------
def figS9():
    sg = pd.read_parquet(cache_path(f"segmentos_{SEASON_MAIN}_s300_f90_a30_{FUEL_TAG[FUEL_MAIN]}_{TAG_SEG}", "parquet"),
                         columns=["tipo_seg", "resultado"])
    ordem = ["interrupcao", "continuidade", "indeterminado", "sem_chegada"]
    nomes = {"interrupcao": "Stopped", "continuidade": "Crossed", "indeterminado": "Undetermined",
             "sem_chegada": "No later fire arrived"}
    cores = {"interrupcao": C_INI, "continuidade": C_PASSOU, "indeterminado": "#bdbcb4", "sem_chegada": "#ecebe6"}
    tab = sg.groupby(["tipo_seg", "resultado"]).size().unstack(fill_value=0).reindex(columns=ordem, fill_value=0)
    pct = 100 * tab.div(tab.sum(axis=1), axis=0)
    fig = plt.figure(figsize=(LARG, 4.6 * CM), layout="constrained")
    ax = fig.subplots()
    rows = [r for r in ["cicatriz", "referencia"] if r in pct.index]
    for yi, t in enumerate(rows[::-1]):
        left = 0
        for c in ordem:
            v = pct.loc[t, c]
            ax.barh(yi, v, left=left, color=cores[c], height=0.6, edgecolor="white", lw=0.6)
            if v >= 6:
                ax.text(left + v / 2, yi, f"{v:.0f}%", ha="center", va="center", fontsize=tam(5.4),
                        color="white" if c in ("interrupcao", "continuidade") else TINTA)
            left += v
        ax.text(101, yi, f"n = {int(tab.loc[t].sum()):,}", va="center", fontsize=tam(5.4), color=TINTA2)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([{"cicatriz": "Early-scar edges", "referencia": "Reference edges"}[t]
                                                         for t in rows[::-1]])
    ax.set_xlim(0, 118); ax.set_xticks([0, 25, 50, 75, 100]); ax.set_xlabel("Share of 300-m edge segments (%)", fontsize=tam(6))
    ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0)
    fig.legend(handles=[mpatches.Patch(color=cores[c], label=nomes[c]) for c in ordem], loc="outside lower center",
               ncol=4, fontsize=tam(5.6), handlelength=1.0)
    rotulo_painel(ax, "a")
    salvar(fig, "FigS9_edge_outcomes")

CAP.update({
    "FigS4_dry_season_per_IL": "Figure S4. Mean monthly rainfall (CHIRPS) in each IL; bars coloured by the local early and "
                               "rest of the dry season (months < 60 mm); shading, the Dias et al. (2026) split.",
    "FigS5_modis_over_scars": "Figure S5. MapBiomas scars (May–July, August–October) and MODIS MOD14A1/MYD14A1 fire pixels "
                              "(FireMask >= 8) in each IL, in the same year as Fig. 2.",
    "FigS6_detectability": "Figure S6. Detectability. (a) Share of interrupted edges by NBR-drop threshold (dashed, "
                           "relative drop). (b) Share of MODIS-confirmed entries also detected by Landsat. (c) Density of "
                           "later MODIS fire pixels in early-scar cores relative to unburned-fuel cores (95% CI, bootstrap "
                           "over ILs).",
    "FigS7_few_IL_inference": "Figure S7. Inference with eight ILs. (a) Cell-scale effects of configuration (null-model SES) "
                              "with jackknife-by-IL 95% CI and wild-bootstrap p. (b) Odds ratios at scar edges adjusted for "
                              "detectability: jackknife-by-IL and fire-and-IL clustered 95% CI.",
    "FigS8_null_model_vs_quantity": "Figure S8. Raw fuel fragmentation is almost a function of the early burned fraction "
                                    "(a); the null-model SES removes that dependence (b). Hexagons, cell-years.",
    "FigS9_edge_outcomes": "Figure S9. Outcome of all 300-m edge segments: stopped, crossed, undetermined and without later "
                           "fire arrival (excluded: absence of fire is not containment).",
    "FigR1_study_area_sample": "Figure R1. (a) The 14 ILs with BRIF and the 8 analysed. (b) Screening: longest gap between "
                               "clear Landsat 8/9 dates (colour) and candidate encounters (numbers) per IL and year.",
    "FigR2_encounter_landsat_modis": "Figure R2. The encounter seen by satellite: Landsat 8/9 SWIR2–NIR–red before and "
                                     "after the later fire, and dNBR with early scars, MODIS fire pixels and edge outcomes.",
    "FigR3_modis_independent_test": "Figure R3. NBR-independent test: MODIS later-fire pixels in early-scar cores and in "
                                    "unburned-fuel cores; rate ratio per IL and pooled (95% CI, bootstrap over ILs).",
    "FigR4_null_model_configuration": "Figure R4. Fuel-fragmentation SES against the null model per 5-km cell (mean "
                                      "2014–2024); blue, early scars split fuel more than chance.",
})

# ============================================================
# RODAR TUDO
# ============================================================
console.print("[bold cyan]-- Bloco 18: todas as figuras (versao 2026-10-25b, mapas padronizados) --[/bold cyan]")
for fn in [fig1, fig2, fig3, fig4, figR1, figR2, figR3, figR4, figS2, figS3, figS4, figS5, figS6, figS7, figS8, figS9]:
    try:
        fn()
    except Exception as e:
        console.print(f"[yellow]{fn.__name__} pulada: {str(e)[:160]}[/yellow]")
t1 = table1()

d = docx.Document()
d.styles["Normal"].font.name = "Times New Roman"; d.styles["Normal"].font.size = Pt(10)
d.add_heading("Figures and Table – early-burn configuration and fire continuity", 1)
for name, cap in CAP.items():
    png = FIG_DIR / f"{name}.png"
    if png.exists():
        d.add_picture(str(png), width=DocxCm(LARGURA_CM))
        d.add_paragraph(cap)
d.add_paragraph("Table 1. Analysed Indigenous Lands (numbers as in Fig. 1): area in the Cerrado, mean annual share burned "
                f"in May–July and August–October ({YEARS[0]}–{YEARS[-1]}), early-scar edges reached by later fire with a "
                "defined outcome, share interrupted at scar and reference edges (same Landsat + MODIS rule), and the class "
                "and FRRI of Dias et al. (2026).")
tb = d.add_table(rows=1, cols=len(t1.columns)); tb.style = "Light Grid Accent 1"
for j, c in enumerate(t1.columns):
    tb.rows[0].cells[j].text = str(c)
for row in t1.itertuples(index=False):
    cl_ = tb.add_row().cells
    for j, x in enumerate(row):
        cl_[j].text = "" if (isinstance(x, float) and np.isnan(x)) else (f"{x:,}" if isinstance(x, (int, np.integer)) else str(x))
d.save(TXT_DIR / "RSL_figuras_tabela.docx")
console.print(f"[bold green]OK BLOCO 18 completo: figuras em {FIG_DIR} e legendas em "
              f"{TXT_DIR / 'RSL_figuras_tabela.docx'}.[/bold green]")
